# **Automatic Classification of Fresh and Rotten Fruits Using Deep Learning: A Benchmark for Visual Food Quality Assessment**

# **Cell 1 — General configuration**

In [ ]:
from pathlib import Path
import os
import random
import warnings

import numpy as np
import pandas as pd

# General configuration
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)

# Reproducible PyTorch configuration
try:
    import torch

    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    TORCH_AVAILABLE = True

except ImportError:
    TORCH_AVAILABLE = False

# Original dataset

DATASET_ROOT = Path(
    "/kaggle/input/datasets/andreamencotovar/fruit-roller2025/mixted_images_fruit/mixted"
)

# Main directory for the new process

WORK_ROOT = Path(
    "/kaggle/working/fruit_dataset_clean_rebuild"
)

MANIFEST_DIR = WORK_ROOT / "manifests"
REPORT_DIR = WORK_ROOT / "reports"
HASH_DIR = WORK_ROOT / "hashes"
VISUAL_REVIEW_DIR = WORK_ROOT / "visual_review"
CLEAN_DATASET_DIR = WORK_ROOT / "clean_dataset"
SPLIT_DIR = WORK_ROOT / "splits"

OUTPUT_DIRECTORIES = [
    WORK_ROOT,
    MANIFEST_DIR,
    REPORT_DIR,
    HASH_DIR,
    VISUAL_REVIEW_DIR,
    CLEAN_DATASET_DIR,
    SPLIT_DIR
]

for directory in OUTPUT_DIRECTORIES:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

#  Initial validations

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"The original dataset was not found:\n{DATASET_ROOT}"
    )

class_directories = sorted(
    [
        directory
        for directory in DATASET_ROOT.iterdir()
        if directory.is_dir()
    ],
    key=lambda path: path.name
)

if len(class_directories) != 24:
    raise ValueError(
        "Expected 24 class directories, "
        f"but found {len(class_directories)}."
    )

# Configuration summary

print("=" * 80)
print("NOTEBOOK CONFIGURATION")
print("=" * 80)

print(f"Global seed: {SEED}")
print(f"Original dataset: {DATASET_ROOT}")
print(f"Working directory: {WORK_ROOT}")
print(f"Number of class directories: {len(class_directories)}")
print(f"PyTorch available: {TORCH_AVAILABLE}")
print(f"GPU available: {torch.cuda.is_available() if TORCH_AVAILABLE else False}")

print("\nClass directories:")

for index, class_directory in enumerate(
    class_directories,
    start=1
):
    print(
        f"{index:02d}. {class_directory.name}"
    )

print("\nIMPORTANT:")
print(
    "The original dataset will not be modified. "
    "All results will be saved in /kaggle/working."
)

# **Cell 2 — Inventory from original images and read validation**
This cell:

Iterates through the 24 original folders. Records the 40,219 file paths. Extracts the class, product, and condition (Healthy/Rotten). Attempts to open and decode each file. Identifies unreadable files. Does not yet calculate duplicates. Does not delete or copy any images.

In [ ]:
from PIL import Image, ImageFile
from tqdm.auto import tqdm


ImageFile.LOAD_TRUNCATED_IMAGES = False

#  Supported extensions

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}

# Validation function

def inspect_image_file(image_path):
    """
    Verifies the file structure and then attempts
    to fully decode its pixels.

    Returns metadata and any error message.
    """

    try:
        # First opening: structural verification
        with Image.open(image_path) as image:
            detected_format = image.format
            image.verify()

        # Second opening: full decoding
        with Image.open(image_path) as image:
            image.load()

            width, height = image.size
            mode = image.mode

            n_frames = getattr(
                image,
                "n_frames",
                1
            )

        if width <= 0 or height <= 0:
            raise ValueError(
                f"Invalid dimensions: {width} x {height}"
            )

        return {
            "is_readable": True,
            "detected_format": detected_format,
            "image_mode": mode,
            "width": int(width),
            "height": int(height),
            "n_frames": int(n_frames),
            "read_error": None
        }

    except Exception as error:

        return {
            "is_readable": False,
            "detected_format": None,
            "image_mode": None,
            "width": None,
            "height": None,
            "n_frames": None,
            "read_error": (
                f"{type(error).__name__}: {error}"
            )
        }

# Locate all files

image_paths = []
ignored_files = []

for class_directory in class_directories:

    for file_path in sorted(
        class_directory.rglob("*"),
        key=lambda path: str(path).lower()
    ):

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() in IMAGE_EXTENSIONS:
            image_paths.append(file_path)
        else:
            ignored_files.append(file_path)

image_paths = sorted(
    image_paths,
    key=lambda path: (
        path.parent.name,
        path.name.lower(),
        str(path)
    )
)

print("=" * 80)
print("ORIGINAL DATASET INVENTORY")
print("=" * 80)

print(
    f"Image files found: "
    f"{len(image_paths):,}"
)

print(
    f"Files with other extensions ignored: "
    f"{len(ignored_files):,}"
)

# Build inventory and validate each image

inventory_records = []

for dataset_id, image_path in enumerate(
    tqdm(
        image_paths,
        desc="Validating images"
    )
):

    class_name = image_path.parent.name

    if "__" not in class_name:
        raise ValueError(
            f"Unrecognized class name: {class_name}"
        )

    product_name, quality_label = (
        class_name.rsplit("__", 1)
    )

    inspection = inspect_image_file(
        image_path
    )

    inventory_records.append({
        "dataset_id": dataset_id,
        "image_path": str(image_path),
        "relative_path": str(
            image_path.relative_to(DATASET_ROOT)
        ),
        "file_name": image_path.name,
        "extension": image_path.suffix.lower(),
        "class_name": class_name,
        "product_name": product_name,
        "quality_label": quality_label,
        "file_size_bytes": image_path.stat().st_size,
        **inspection
    })

inventory_df = pd.DataFrame(
    inventory_records
)

#  Structural validations

if len(inventory_df) != 40219:
    raise ValueError(
        "Expected 40,219 images, but found "
        f"{len(inventory_df):,}."
    )

if inventory_df["image_path"].duplicated().any():
    duplicated_paths = inventory_df[
        inventory_df["image_path"].duplicated(
            keep=False
        )
    ]

    raise ValueError(
        "Duplicate paths were found:\n"
        f"{duplicated_paths[['image_path']].head()}"
    )

if inventory_df["dataset_id"].duplicated().any():
    raise ValueError(
        "Duplicate dataset_id identifiers were generated."
    )

expected_quality_labels = {
    "Healthy",
    "Rotten"
}

found_quality_labels = set(
    inventory_df["quality_label"].unique()
)

if found_quality_labels != expected_quality_labels:
    raise ValueError(
        "The quality labels do not match "
        f"{expected_quality_labels}. Found: "
        f"{found_quality_labels}"
    )

if inventory_df["class_name"].nunique() != 24:
    raise ValueError(
        "The inventory does not contain the 24 expected classes."
    )

#  Summaries

readability_summary_df = pd.DataFrame({
    "metric": [
        "total_image_files",
        "unique_image_paths",
        "number_of_classes",
        "number_of_products",
        "readable_images",
        "unreadable_images",
        "ignored_non_image_files"
    ],
    "value": [
        len(inventory_df),
        inventory_df["image_path"].nunique(),
        inventory_df["class_name"].nunique(),
        inventory_df["product_name"].nunique(),
        int(inventory_df["is_readable"].sum()),
        int((~inventory_df["is_readable"]).sum()),
        len(ignored_files)
    ]
})

class_distribution_df = (
    inventory_df
    .groupby(
        [
            "class_name",
            "product_name",
            "quality_label"
        ],
        as_index=False
    )
    .agg(
        total_files=("image_path", "size"),
        readable_files=("is_readable", "sum")
    )
)

class_distribution_df[
    "unreadable_files"
] = (
    class_distribution_df["total_files"]
    - class_distribution_df["readable_files"]
)

class_distribution_df = (
    class_distribution_df
    .sort_values("class_name")
    .reset_index(drop=True)
)

unreadable_df = (
    inventory_df[
        ~inventory_df["is_readable"]
    ]
    .copy()
    .reset_index(drop=True)
)

# Save results

INVENTORY_PATH = (
    MANIFEST_DIR
    / "01_original_dataset_inventory.csv"
)

READABILITY_SUMMARY_PATH = (
    REPORT_DIR
    / "01_readability_summary.csv"
)

CLASS_DISTRIBUTION_PATH = (
    REPORT_DIR
    / "01_original_class_distribution.csv"
)

UNREADABLE_PATH = (
    REPORT_DIR
    / "01_unreadable_images.csv"
)

inventory_df.to_csv(
    INVENTORY_PATH,
    index=False
)

readability_summary_df.to_csv(
    READABILITY_SUMMARY_PATH,
    index=False
)

class_distribution_df.to_csv(
    CLASS_DISTRIBUTION_PATH,
    index=False
)

unreadable_df.to_csv(
    UNREADABLE_PATH,
    index=False
)

if ignored_files:

    ignored_files_df = pd.DataFrame({
        "file_path": [
            str(path)
            for path in ignored_files
        ],
        "extension": [
            path.suffix.lower()
            for path in ignored_files
        ]
    })

    ignored_files_df.to_csv(
        REPORT_DIR
        / "01_ignored_non_image_files.csv",
        index=False
    )

# Display results

print("\n" + "=" * 80)
print("READABILITY SUMMARY")
print("=" * 80)

display(readability_summary_df)

print("\nOriginal class distribution:")
display(class_distribution_df)

print("\nUnreadable files:")

if unreadable_df.empty:

    print(
        "No unreadable files were detected."
    )

else:

    display(
        unreadable_df[
            [
                "dataset_id",
                "class_name",
                "file_name",
                "file_size_bytes",
                "read_error",
                "image_path"
            ]
        ]
    )

print("\nInventory saved to:")
print(INVENTORY_PATH)

print(
    "\nNo files have been deleted, moved, or copied "
    "at this stage."
)

# **Cell 3 — Exact hashes and label conflicts**
This cell recalculates SHA-256 hashes from scratch for the entire original corpus. It will identify:

Groups of exactly identical files. Redundant copies. Duplicates within the same class. Identical images assigned to different classes. "Healthy–Rotten" conflicts.

In [ ]:
import hashlib
from tqdm.auto import tqdm

#  Recover the inventory if the kernel was restarted

INVENTORY_PATH = (
    MANIFEST_DIR
    / "01_original_dataset_inventory.csv"
)

if "inventory_df" not in globals():

    if not INVENTORY_PATH.exists():
        raise FileNotFoundError(
            f"The inventory was not found:\n{INVENTORY_PATH}"
        )

    inventory_df = pd.read_csv(
        INVENTORY_PATH,
        low_memory=False
    )

inventory_sha_df = inventory_df.copy()

# SHA-256 function

def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024
):
    """
    Calculates SHA-256 by reading the file in chunks.
    """

    sha256 = hashlib.sha256()

    try:
        with open(file_path, "rb") as file:

            while True:

                chunk = file.read(chunk_size)

                if not chunk:
                    break

                sha256.update(chunk)

        return {
            "sha256": sha256.hexdigest(),
            "sha256_error": None
        }

    except Exception as error:

        return {
            "sha256": None,
            "sha256_error": (
                f"{type(error).__name__}: {error}"
            )
        }

# Calculate hashes for the 40,219 images

print("=" * 80)
print("SHA-256 EXACT HASH CALCULATION")
print("=" * 80)

hash_records = []

for image_path in tqdm(
    inventory_sha_df["image_path"],
    total=len(inventory_sha_df),
    desc="Calculating SHA-256"
):

    hash_records.append(
        calculate_sha256(image_path)
    )

hash_results_df = pd.DataFrame(
    hash_records
)

inventory_sha_df = pd.concat(
    [
        inventory_sha_df.reset_index(drop=True),
        hash_results_df.reset_index(drop=True)
    ],
    axis=1
)

# Validate errors

sha_error_df = (
    inventory_sha_df[
        inventory_sha_df["sha256"].isna()
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    f"\nHashing errors: {len(sha_error_df):,}"
)

# Helper functions

def join_unique(values):
    """
    Joins sorted unique values.
    """

    return " | ".join(
        sorted(
            {
                str(value)
                for value in values
                if pd.notna(value)
            }
        )
    )

# Statistics by hash

valid_sha_df = (
    inventory_sha_df[
        inventory_sha_df["sha256"].notna()
    ]
    .copy()
)

sha_group_df = (
    valid_sha_df
    .groupby("sha256")
    .agg(
        n_files=(
            "image_path",
            "size"
        ),
        n_classes=(
            "class_name",
            "nunique"
        ),
        classes=(
            "class_name",
            join_unique
        ),
        n_products=(
            "product_name",
            "nunique"
        ),
        products=(
            "product_name",
            join_unique
        ),
        n_quality_labels=(
            "quality_label",
            "nunique"
        ),
        quality_labels=(
            "quality_label",
            join_unique
        ),
        extensions=(
            "extension",
            join_unique
        ),
        file_sizes=(
            "file_size_bytes",
            lambda values: " | ".join(
                map(
                    str,
                    sorted(set(values))
                )
            )
        )
    )
    .reset_index()
)

sha_group_df["is_exact_duplicate_group"] = (
    sha_group_df["n_files"] > 1
)

sha_group_df["redundant_copies"] = (
    sha_group_df["n_files"] - 1
)

sha_group_df["class_conflict"] = (
    sha_group_df["n_classes"] > 1
)

sha_group_df["product_conflict"] = (
    sha_group_df["n_products"] > 1
)

sha_group_df["quality_conflict"] = (
    sha_group_df["n_quality_labels"] > 1
)

sha_group_df["any_label_conflict"] = (
    sha_group_df["class_conflict"]
    | sha_group_df["product_conflict"]
    | sha_group_df["quality_conflict"]
)

# Exact duplicate groups

exact_duplicate_groups_df = (
    sha_group_df[
        sha_group_df[
            "is_exact_duplicate_group"
        ]
    ]
    .copy()
    .sort_values(
        [
            "any_label_conflict",
            "n_files"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

exact_duplicate_groups_df[
    "exact_group_id"
] = [
    f"EXACT_{index:05d}"
    for index in range(
        1,
        len(exact_duplicate_groups_df) + 1
    )
]

# Members of each duplicate group

exact_duplicate_members_df = (
    valid_sha_df
    .merge(
        exact_duplicate_groups_df[
            [
                "sha256",
                "exact_group_id",
                "n_files",
                "n_classes",
                "classes",
                "n_products",
                "products",
                "n_quality_labels",
                "quality_labels",
                "class_conflict",
                "product_conflict",
                "quality_conflict",
                "any_label_conflict"
            ]
        ],
        on="sha256",
        how="inner",
        validate="many_to_one"
    )
    .sort_values(
        [
            "exact_group_id",
            "class_name",
            "file_name",
            "image_path"
        ]
    )
    .reset_index(drop=True)
)

#  Groups with conflicts

exact_conflict_groups_df = (
    exact_duplicate_groups_df[
        exact_duplicate_groups_df[
            "any_label_conflict"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

conflict_hashes = set(
    exact_conflict_groups_df["sha256"]
)

exact_conflict_members_df = (
    exact_duplicate_members_df[
        exact_duplicate_members_df[
            "sha256"
        ].isin(conflict_hashes)
    ]
    .copy()
    .reset_index(drop=True)
)

# Summary by class

exact_duplicates_by_class_df = (
    exact_duplicate_members_df
    .groupby(
        "class_name",
        as_index=False
    )
    .agg(
        files_in_exact_duplicate_groups=(
            "image_path",
            "size"
        ),
        exact_duplicate_groups=(
            "exact_group_id",
            "nunique"
        ),
        files_in_conflicting_groups=(
            "any_label_conflict",
            "sum"
        )
    )
    .sort_values(
        "files_in_exact_duplicate_groups",
        ascending=False
    )
    .reset_index(drop=True)
)

# Overall summary

n_duplicate_groups = len(
    exact_duplicate_groups_df
)

n_files_in_duplicate_groups = len(
    exact_duplicate_members_df
)

n_redundant_copies = int(
    exact_duplicate_groups_df[
        "redundant_copies"
    ].sum()
)

n_conflict_groups = len(
    exact_conflict_groups_df
)

n_files_in_conflict_groups = len(
    exact_conflict_members_df
)

exact_summary_df = pd.DataFrame({
    "metric": [
        "total_files",
        "valid_sha256_files",
        "sha256_errors",
        "unique_sha256_hashes",
        "exact_duplicate_groups",
        "files_in_exact_duplicate_groups",
        "redundant_exact_copies",
        "same_class_exact_duplicate_groups",
        "cross_class_exact_duplicate_groups",
        "quality_conflict_groups",
        "any_label_conflict_groups",
        "files_in_any_label_conflict_groups"
    ],
    "value": [
        len(inventory_sha_df),
        len(valid_sha_df),
        len(sha_error_df),
        valid_sha_df["sha256"].nunique(),
        n_duplicate_groups,
        n_files_in_duplicate_groups,
        n_redundant_copies,
        int(
            (
                exact_duplicate_groups_df[
                    "n_classes"
                ] == 1
            ).sum()
        ),
        int(
            (
                exact_duplicate_groups_df[
                    "n_classes"
                ] > 1
            ).sum()
        ),
        int(
            exact_duplicate_groups_df[
                "quality_conflict"
            ].sum()
        ),
        n_conflict_groups,
        n_files_in_conflict_groups
    ]
})

#  Save results

INVENTORY_SHA_PATH = (
    MANIFEST_DIR
    / "02_inventory_with_sha256.csv"
)

SHA_SUMMARY_PATH = (
    REPORT_DIR
    / "02_sha256_summary.csv"
)

SHA_GROUPS_PATH = (
    HASH_DIR
    / "02_sha256_all_groups.csv"
)

EXACT_GROUPS_PATH = (
    HASH_DIR
    / "02_exact_duplicate_groups.csv"
)

EXACT_MEMBERS_PATH = (
    HASH_DIR
    / "02_exact_duplicate_members.csv"
)

EXACT_CONFLICT_GROUPS_PATH = (
    REPORT_DIR
    / "02_exact_conflict_groups.csv"
)

EXACT_CONFLICT_MEMBERS_PATH = (
    REPORT_DIR
    / "02_exact_conflict_members.csv"
)

EXACT_BY_CLASS_PATH = (
    REPORT_DIR
    / "02_exact_duplicates_by_class.csv"
)

SHA_ERRORS_PATH = (
    REPORT_DIR
    / "02_sha256_errors.csv"
)

inventory_sha_df.to_csv(
    INVENTORY_SHA_PATH,
    index=False
)

exact_summary_df.to_csv(
    SHA_SUMMARY_PATH,
    index=False
)

sha_group_df.to_csv(
    SHA_GROUPS_PATH,
    index=False
)

exact_duplicate_groups_df.to_csv(
    EXACT_GROUPS_PATH,
    index=False
)

exact_duplicate_members_df.to_csv(
    EXACT_MEMBERS_PATH,
    index=False
)

exact_conflict_groups_df.to_csv(
    EXACT_CONFLICT_GROUPS_PATH,
    index=False
)

exact_conflict_members_df.to_csv(
    EXACT_CONFLICT_MEMBERS_PATH,
    index=False
)

exact_duplicates_by_class_df.to_csv(
    EXACT_BY_CLASS_PATH,
    index=False
)

sha_error_df.to_csv(
    SHA_ERRORS_PATH,
    index=False
)

# Display results

print("\n" + "=" * 80)
print("EXACT DUPLICATES SUMMARY")
print("=" * 80)

display(exact_summary_df)

print("\nExact duplicates by class:")
display(exact_duplicates_by_class_df)

print("\nExact duplicate groups with label conflicts:")

if exact_conflict_groups_df.empty:

    print(
        "No label conflicts were detected."
    )

else:

    display(
        exact_conflict_groups_df[
            [
                "exact_group_id",
                "n_files",
                "classes",
                "products",
                "quality_labels",
                "class_conflict",
                "quality_conflict"
            ]
        ]
    )

    print("\nFiles belonging to conflicting groups:")

    display(
        exact_conflict_members_df[
            [
                "exact_group_id",
                "class_name",
                "quality_label",
                "file_name",
                "image_path",
                "sha256"
            ]
        ]
    )

print("\nManifest with SHA-256 saved to:")
print(INVENTORY_SHA_PATH)

print(
    "\nNo files have been deleted, moved, or copied "
    "at this stage."
)

#  **Cell 4 — Calculating perceptual hashes from scratch**
This cell:

Calculates pHash, dHash, and aHash. Corrects EXIF orientation whenever possible. Logs metadata errors without halting the process. Handles transparency by placing it against a white background. Detects groups with repeated perceptual hashes.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from PIL import Image, ImageOps, ImageFile
from tqdm.auto import tqdm


ImageFile.LOAD_TRUNCATED_IMAGES = True


# Load manifest with SHA-256

INVENTORY_SHA_PATH = (
    MANIFEST_DIR
    / "02_inventory_with_sha256.csv"
)

if not INVENTORY_SHA_PATH.exists():
    raise FileNotFoundError(
        f"The manifest with SHA-256 was not found:\n"
        f"{INVENTORY_SHA_PATH}"
    )

perceptual_source_df = pd.read_csv(
    INVENTORY_SHA_PATH,
    low_memory=False
)

required_columns = {
    "dataset_id",
    "image_path",
    "relative_path",
    "file_name",
    "class_name",
    "product_name",
    "quality_label",
    "sha256"
}

missing_columns = (
    required_columns
    - set(perceptual_source_df.columns)
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )

if len(perceptual_source_df) != 40219:
    raise ValueError(
        "The manifest does not contain the "
        "40,219 expected images."
    )

# Previously determined manual exclusions

MANUAL_EXCLUSIONS = {
    (
        "Strawberry__Rotten/"
        "rottenStrawberry (3).jpeg"
    ): (
        "Previously failed during standardized perceptual "
        "processing; excluded by conservative audit policy."
    )
}

perceptual_source_df[
    "manual_exclusion_requested"
] = perceptual_source_df[
    "relative_path"
].isin(MANUAL_EXCLUSIONS)

perceptual_source_df[
    "manual_exclusion_reason"
] = perceptual_source_df[
    "relative_path"
].map(MANUAL_EXCLUSIONS)

# Compatibility across Pillow versions

try:
    RESAMPLING_METHOD = Image.Resampling.LANCZOS
except AttributeError:
    RESAMPLING_METHOD = Image.LANCZOS

# Convert 64 bits to hexadecimal

def bits_to_hex64(bits):
    """
    Converts exactly 64 Boolean values
    into a 16-character hexadecimal string.
    """

    bits = np.asarray(
        bits,
        dtype=np.uint8
    ).reshape(-1)

    if bits.size != 64:
        raise ValueError(
            f"Expected 64 bits, but obtained "
            f"{bits.size}."
        )

    value = 0

    for bit in bits:
        value = (
            value << 1
        ) | int(bit)

    return f"{value:016x}"

#  Robust image opening and normalization

def open_standardized_rgb(image_path):
    """
    Opens and decodes the image.

    Attempts to correct EXIF orientation. If the metadata
    are corrupted, preserves the original pixels and
    records a warning.
    """

    with Image.open(image_path) as original_image:

        original_image.load()

        original_width, original_height = (
            original_image.size
        )

        original_mode = original_image.mode
        detected_format = original_image.format

        processing_warning = None

        try:
            image = ImageOps.exif_transpose(
                original_image
            )

        except Exception as error:

            image = original_image.copy()

            processing_warning = (
                "EXIF transpose failed: "
                f"{type(error).__name__}: {error}"
            )

        # Handle alpha channels or palette transparency
        if (
            image.mode in {"RGBA", "LA"}
            or "transparency" in image.info
        ):

            image_rgba = image.convert("RGBA")

            white_background = Image.new(
                "RGBA",
                image_rgba.size,
                (255, 255, 255, 255)
            )

            image_rgb = Image.alpha_composite(
                white_background,
                image_rgba
            ).convert("RGB")

        else:

            image_rgb = image.convert("RGB")

        return {
            "image": image_rgb.copy(),
            "original_width": int(original_width),
            "original_height": int(original_height),
            "original_mode": original_mode,
            "detected_format": detected_format,
            "processing_warning": processing_warning
        }

# Calculate pHash, dHash, and aHash

def calculate_perceptual_hashes(image_path):
    """
    Calculates three 64-bit perceptual hashes:

    pHash:
        Based on DCT and spatial frequencies.

    dHash:
        Based on horizontal differences.

    aHash:
        Based on mean intensity.
    """

    try:
        opened = open_standardized_rgb(
            image_path
        )

        image_rgb = opened["image"]
        gray_image = image_rgb.convert("L")

        #  64-bit pHash
        
        phash_image = gray_image.resize(
            (32, 32),
            RESAMPLING_METHOD
        )

        phash_array = np.asarray(
            phash_image,
            dtype=np.float32
        )

        dct_matrix = cv2.dct(
            phash_array
        )

        low_frequency_dct = (
            dct_matrix[:8, :8]
        )

        # The DC coefficient is excluded when computing
        # the median, as in standard pHash.
        median_value = np.median(
            low_frequency_dct
            .reshape(-1)[1:]
        )

        phash_bits = (
            low_frequency_dct
            > median_value
        )

        phash64 = bits_to_hex64(
            phash_bits
        )

        # 64-bit horizontal dHash
       
        dhash_image = gray_image.resize(
            (9, 8),
            RESAMPLING_METHOD
        )

        dhash_array = np.asarray(
            dhash_image,
            dtype=np.int16
        )

        dhash_bits = (
            dhash_array[:, 1:]
            > dhash_array[:, :-1]
        )

        dhash64 = bits_to_hex64(
            dhash_bits
        )

        # 64-bit aHash
        
        ahash_image = gray_image.resize(
            (8, 8),
            RESAMPLING_METHOD
        )

        ahash_array = np.asarray(
            ahash_image,
            dtype=np.float32
        )

        ahash_bits = (
            ahash_array
            > ahash_array.mean()
        )

        ahash64 = bits_to_hex64(
            ahash_bits
        )

        width = opened["original_width"]
        height = opened["original_height"]

        aspect_ratio = (
            width / height
            if height > 0
            else np.nan
        )

        return {
            "phash64": phash64,
            "dhash64": dhash64,
            "ahash64": ahash64,
            "perceptual_width": width,
            "perceptual_height": height,
            "aspect_ratio": aspect_ratio,
            "perceptual_original_mode": (
                opened["original_mode"]
            ),
            "perceptual_detected_format": (
                opened["detected_format"]
            ),
            "perceptual_processing_warning": (
                opened["processing_warning"]
            ),
            "perceptual_hash_ok": True,
            "perceptual_hash_error": None
        }

    except Exception as error:

        return {
            "phash64": None,
            "dhash64": None,
            "ahash64": None,
            "perceptual_width": None,
            "perceptual_height": None,
            "aspect_ratio": None,
            "perceptual_original_mode": None,
            "perceptual_detected_format": None,
            "perceptual_processing_warning": None,
            "perceptual_hash_ok": False,
            "perceptual_hash_error": (
                f"{type(error).__name__}: {error}"
            )
        }

# Calculate perceptual hashes

print("=" * 80)
print("PERCEPTUAL HASH CALCULATION")
print("=" * 80)

print(
    f"Images to process: "
    f"{len(perceptual_source_df):,}"
)

perceptual_records = []

for image_path in tqdm(
    perceptual_source_df["image_path"],
    total=len(perceptual_source_df),
    desc="Calculating pHash, dHash, and aHash"
):

    perceptual_records.append(
        calculate_perceptual_hashes(
            image_path
        )
    )

perceptual_results_df = pd.DataFrame(
    perceptual_records
)

inventory_perceptual_df = pd.concat(
    [
        perceptual_source_df.reset_index(
            drop=True
        ),
        perceptual_results_df.reset_index(
            drop=True
        )
    ],
    axis=1
)

# Validate results

perceptual_error_df = (
    inventory_perceptual_df[
        ~inventory_perceptual_df[
            "perceptual_hash_ok"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

processing_warning_df = (
    inventory_perceptual_df[
        inventory_perceptual_df[
            "perceptual_processing_warning"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

valid_perceptual_df = (
    inventory_perceptual_df[
        inventory_perceptual_df[
            "perceptual_hash_ok"
        ]
    ]
    .copy()
)

# Helper functions for grouping

def join_unique(values):
    return " | ".join(
        sorted(
            {
                str(value)
                for value in values
                if pd.notna(value)
            }
        )
    )

# Statistics for identical pHash values

phash_group_df = (
    valid_perceptual_df
    .groupby("phash64")
    .agg(
        n_files=(
            "image_path",
            "size"
        ),
        n_sha256=(
            "sha256",
            "nunique"
        ),
        n_classes=(
            "class_name",
            "nunique"
        ),
        classes=(
            "class_name",
            join_unique
        ),
        n_products=(
            "product_name",
            "nunique"
        ),
        products=(
            "product_name",
            join_unique
        ),
        n_quality_labels=(
            "quality_label",
            "nunique"
        ),
        quality_labels=(
            "quality_label",
            join_unique
        )
    )
    .reset_index()
)

phash_group_df[
    "repeated_phash"
] = (
    phash_group_df["n_files"] > 1
)

phash_group_df[
    "different_binary_files"
] = (
    phash_group_df["n_sha256"] > 1
)

phash_group_df[
    "class_conflict_candidate"
] = (
    phash_group_df["n_classes"] > 1
)

phash_group_df[
    "quality_conflict_candidate"
] = (
    phash_group_df[
        "n_quality_labels"
    ] > 1
)

repeated_phash_groups_df = (
    phash_group_df[
        phash_group_df["repeated_phash"]
    ]
    .copy()
    .sort_values(
        [
            "quality_conflict_candidate",
            "n_files"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

nonexact_repeated_phash_df = (
    repeated_phash_groups_df[
        repeated_phash_groups_df[
            "different_binary_files"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# Members of non-exact repeated pHash groups

nonexact_phash_members_df = (
    valid_perceptual_df
    .merge(
        nonexact_repeated_phash_df[
            [
                "phash64",
                "n_files",
                "n_sha256",
                "n_classes",
                "classes",
                "n_products",
                "products",
                "n_quality_labels",
                "quality_labels",
                "class_conflict_candidate",
                "quality_conflict_candidate"
            ]
        ],
        on="phash64",
        how="inner",
        validate="many_to_one"
    )
    .sort_values(
        [
            "phash64",
            "class_name",
            "file_name",
            "image_path"
        ]
    )
    .reset_index(drop=True)
)

#  Overall summary

perceptual_summary_df = pd.DataFrame({
    "metric": [
        "total_files",
        "valid_perceptual_hashes",
        "perceptual_hash_errors",
        "processing_warnings",
        "manual_exclusions_requested",
        "unique_phash64",
        "unique_dhash64",
        "unique_ahash64",
        "repeated_phash_groups",
        "repeated_phash_groups_with_different_sha256",
        "files_in_nonexact_repeated_phash_groups",
        "cross_class_phash_candidate_groups",
        "cross_quality_phash_candidate_groups"
    ],
    "value": [
        len(inventory_perceptual_df),
        len(valid_perceptual_df),
        len(perceptual_error_df),
        len(processing_warning_df),
        int(
            inventory_perceptual_df[
                "manual_exclusion_requested"
            ].sum()
        ),
        valid_perceptual_df[
            "phash64"
        ].nunique(),
        valid_perceptual_df[
            "dhash64"
        ].nunique(),
        valid_perceptual_df[
            "ahash64"
        ].nunique(),
        len(repeated_phash_groups_df),
        len(nonexact_repeated_phash_df),
        len(nonexact_phash_members_df),
        int(
            nonexact_repeated_phash_df[
                "class_conflict_candidate"
            ].sum()
        ),
        int(
            nonexact_repeated_phash_df[
                "quality_conflict_candidate"
            ].sum()
        )
    ]
})

# Save results

INVENTORY_PERCEPTUAL_PATH = (
    MANIFEST_DIR
    / "03_inventory_with_perceptual_hashes.csv"
)

PERCEPTUAL_SUMMARY_PATH = (
    REPORT_DIR
    / "03_perceptual_hash_summary.csv"
)

PERCEPTUAL_ERRORS_PATH = (
    REPORT_DIR
    / "03_perceptual_hash_errors.csv"
)

PROCESSING_WARNINGS_PATH = (
    REPORT_DIR
    / "03_perceptual_processing_warnings.csv"
)

PHASH_GROUPS_PATH = (
    HASH_DIR
    / "03_phash_all_groups.csv"
)

REPEATED_PHASH_GROUPS_PATH = (
    HASH_DIR
    / "03_phash_repeated_groups.csv"
)

NONEXACT_PHASH_GROUPS_PATH = (
    HASH_DIR
    / "03_phash_repeated_nonexact_groups.csv"
)

NONEXACT_PHASH_MEMBERS_PATH = (
    HASH_DIR
    / "03_phash_repeated_nonexact_members.csv"
)

inventory_perceptual_df.to_csv(
    INVENTORY_PERCEPTUAL_PATH,
    index=False
)

perceptual_summary_df.to_csv(
    PERCEPTUAL_SUMMARY_PATH,
    index=False
)

perceptual_error_df.to_csv(
    PERCEPTUAL_ERRORS_PATH,
    index=False
)

processing_warning_df.to_csv(
    PROCESSING_WARNINGS_PATH,
    index=False
)

phash_group_df.to_csv(
    PHASH_GROUPS_PATH,
    index=False
)

repeated_phash_groups_df.to_csv(
    REPEATED_PHASH_GROUPS_PATH,
    index=False
)

nonexact_repeated_phash_df.to_csv(
    NONEXACT_PHASH_GROUPS_PATH,
    index=False
)

nonexact_phash_members_df.to_csv(
    NONEXACT_PHASH_MEMBERS_PATH,
    index=False
)

# Display results

print("\n" + "=" * 80)
print("PERCEPTUAL HASH SUMMARY")
print("=" * 80)

display(perceptual_summary_df)

print("\nPerceptual processing errors:")

if perceptual_error_df.empty:

    print(
        "No errors occurred."
    )

else:

    display(
        perceptual_error_df[
            [
                "dataset_id",
                "class_name",
                "file_name",
                "relative_path",
                "perceptual_hash_error"
            ]
        ]
    )

print("\nWarnings during processing:")

if processing_warning_df.empty:

    print(
        "No warnings occurred."
    )

else:

    display(
        processing_warning_df[
            [
                "dataset_id",
                "class_name",
                "file_name",
                "relative_path",
                "perceptual_processing_warning"
            ]
        ]
    )

print("\nFile flagged for manual exclusion:")

manual_exclusion_df = (
    inventory_perceptual_df[
        inventory_perceptual_df[
            "manual_exclusion_requested"
        ]
    ]
    .copy()
)

display(
    manual_exclusion_df[
        [
            "dataset_id",
            "class_name",
            "file_name",
            "relative_path",
            "perceptual_hash_ok",
            "perceptual_processing_warning",
            "perceptual_hash_error",
            "manual_exclusion_reason"
        ]
    ]
)

print("\npHash groups with potential quality-label conflicts:")

phash_conflict_candidates_df = (
    nonexact_repeated_phash_df[
        nonexact_repeated_phash_df[
            "quality_conflict_candidate"
        ]
    ]
    .copy()
)

if phash_conflict_candidates_df.empty:

    print(
        "No perceptual candidates were detected "
        "with different Healthy/Rotten labels."
    )

else:

    display(
        phash_conflict_candidates_df[
            [
                "phash64",
                "n_files",
                "n_sha256",
                "classes",
                "quality_labels"
            ]
        ]
    )

print("\nPerceptual manifest saved to:")
print(INVENTORY_PERCEPTUAL_PATH)

print(
    "\nNo files have been deleted, moved, or copied "
    "at this stage."
)

# **Cell 5 — Record the 265 confirmed duplicates**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Paths

REVIEW_TEMPLATE_PATH = (
    REPORT_DIR
    / "07_near_phash_manual_review_template.csv"
)

if not REVIEW_TEMPLATE_PATH.exists():
    raise FileNotFoundError(
        f"The review template was not found:\n"
        f"{REVIEW_TEMPLATE_PATH}"
    )

# Load the complete review queue

review_completed_df = pd.read_csv(
    REVIEW_TEMPLATE_PATH,
    low_memory=False
)

required_columns = {
    "candidate_id",
    "near_duplicate_status",
    "sha256_a",
    "sha256_b",
    "image_path_a",
    "image_path_b",
    "relative_path_a",
    "relative_path_b",
    "class_name_a",
    "class_name_b",
    "product_name_a",
    "product_name_b",
    "quality_label_a",
    "quality_label_b",
    "same_class",
    "same_product",
    "same_quality_label"
}

missing_columns = (
    required_columns
    - set(review_completed_df.columns)
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )

# Integrity checks

if len(review_completed_df) != 265:
    raise ValueError(
        "Expected 265 unique pairs in the review, "
        f"but found {len(review_completed_df):,}."
    )

if review_completed_df[
    "candidate_id"
].duplicated().any():

    duplicated_ids = review_completed_df[
        review_completed_df[
            "candidate_id"
        ].duplicated(keep=False)
    ]

    raise ValueError(
        "The template contains duplicate candidate_id values:\n"
        f"{duplicated_ids[['candidate_id']].head()}"
    )

if (
    review_completed_df["sha256_a"]
    == review_completed_df["sha256_b"]
).any():
    raise ValueError(
        "The review contains pairs with identical SHA-256 values. "
        "These should have been handled as exact duplicates."
    )

# Record the decision reported by the researcher

review_completed_df[
    "manual_decision"
] = "confirmed_duplicate"

review_completed_df[
    "review_notes"
] = (
    "Both files were visually reviewed and confirmed "
    "to represent the same underlying image."
)

review_completed_df[
    "decision_source"
] = "full_visual_review_by_researcher"

review_completed_df[
    "include_as_duplicate_edge"
] = True

# Create a deterministic identifier for each pair

def build_pair_key(row):
    """
    Generates a key independent of A/B ordering.
    """

    hashes = sorted(
        [
            str(row["sha256_a"]),
            str(row["sha256_b"])
        ]
    )

    return (
        hashes[0]
        + "__"
        + hashes[1]
    )

review_completed_df[
    "duplicate_pair_key"
] = review_completed_df.apply(
    build_pair_key,
    axis=1
)

if review_completed_df[
    "duplicate_pair_key"
].duplicated().any():

    raise ValueError(
        "Duplicate SHA-256 pair relationships exist "
        "within the visual review."
    )

# Identify confirmed conflicts

review_completed_df[
    "confirmed_class_conflict"
] = (
    ~review_completed_df["same_class"]
)

review_completed_df[
    "confirmed_product_conflict"
] = (
    ~review_completed_df["same_product"]
)

review_completed_df[
    "confirmed_quality_conflict"
] = (
    ~review_completed_df[
        "same_quality_label"
    ]
)

review_completed_df[
    "confirmed_any_label_conflict"
] = (
    review_completed_df[
        "confirmed_class_conflict"
    ]
    | review_completed_df[
        "confirmed_product_conflict"
    ]
    | review_completed_df[
        "confirmed_quality_conflict"
    ]
)


# Summary by original automated status

decision_by_status_df = (
    review_completed_df
    .groupby(
        "near_duplicate_status",
        as_index=False
    )
    .agg(
        reviewed_pairs=(
            "candidate_id",
            "size"
        ),
        confirmed_duplicate_pairs=(
            "include_as_duplicate_edge",
            "sum"
        ),
        cross_class_pairs=(
            "confirmed_class_conflict",
            "sum"
        ),
        cross_product_pairs=(
            "confirmed_product_conflict",
            "sum"
        ),
        cross_quality_pairs=(
            "confirmed_quality_conflict",
            "sum"
        )
    )
    .sort_values(
        "reviewed_pairs",
        ascending=False
    )
    .reset_index(drop=True)
)

# Overall summary

unique_sha256_in_review = set(
    review_completed_df["sha256_a"]
).union(
    set(
        review_completed_df["sha256_b"]
    )
)

manual_decision_summary_df = pd.DataFrame({
    "metric": [
        "unique_reviewed_pairs",
        "confirmed_duplicate_pairs",
        "different_image_pairs",
        "uncertain_pairs",
        "unique_sha256_in_confirmed_pairs",
        "confirmed_same_class_pairs",
        "confirmed_cross_class_pairs",
        "confirmed_cross_product_pairs",
        "confirmed_cross_quality_pairs",
        "confirmed_any_label_conflict_pairs"
    ],
    "value": [
        len(review_completed_df),
        int(
            review_completed_df[
                "manual_decision"
            ].eq(
                "confirmed_duplicate"
            ).sum()
        ),
        int(
            review_completed_df[
                "manual_decision"
            ].eq(
                "different_image"
            ).sum()
        ),
        int(
            review_completed_df[
                "manual_decision"
            ].eq(
                "uncertain"
            ).sum()
        ),
        len(unique_sha256_in_review),
        int(
            review_completed_df[
                "same_class"
            ].sum()
        ),
        int(
            review_completed_df[
                "confirmed_class_conflict"
            ].sum()
        ),
        int(
            review_completed_df[
                "confirmed_product_conflict"
            ].sum()
        ),
        int(
            review_completed_df[
                "confirmed_quality_conflict"
            ].sum()
        ),
        int(
            review_completed_df[
                "confirmed_any_label_conflict"
            ].sum()
        )
    ]
})

# Separate pairs with label conflicts

confirmed_conflict_pairs_df = (
    review_completed_df[
        review_completed_df[
            "confirmed_any_label_conflict"
        ]
    ]
    .copy()
    .sort_values(
        [
            "confirmed_quality_conflict",
            "class_name_a",
            "class_name_b"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

# Save results

COMPLETED_REVIEW_PATH = (
    REPORT_DIR
    / "07_near_phash_manual_review_completed.csv"
)

CONFIRMED_NEAR_PAIRS_PATH = (
    HASH_DIR
    / "07_confirmed_near_duplicate_pairs.csv"
)

MANUAL_DECISION_SUMMARY_PATH = (
    REPORT_DIR
    / "07_manual_decision_summary.csv"
)

DECISION_BY_STATUS_PATH = (
    REPORT_DIR
    / "07_manual_decision_by_status.csv"
)

CONFIRMED_CONFLICT_PAIRS_PATH = (
    REPORT_DIR
    / "07_confirmed_near_duplicate_conflicts.csv"
)

review_completed_df.to_csv(
    COMPLETED_REVIEW_PATH,
    index=False
)

review_completed_df.to_csv(
    CONFIRMED_NEAR_PAIRS_PATH,
    index=False
)

manual_decision_summary_df.to_csv(
    MANUAL_DECISION_SUMMARY_PATH,
    index=False
)

decision_by_status_df.to_csv(
    DECISION_BY_STATUS_PATH,
    index=False
)

confirmed_conflict_pairs_df.to_csv(
    CONFIRMED_CONFLICT_PAIRS_PATH,
    index=False
)

# Display results

print("=" * 80)
print("MANUAL DECISION RECORD")
print("=" * 80)

display(manual_decision_summary_df)

print("\nDecisions by original automated status:")
display(decision_by_status_df)

print("\nConfirmed duplicates with label conflicts:")

if confirmed_conflict_pairs_df.empty:

    print(
        "There are no confirmed pairs with "
        "label conflicts."
    )

else:

    display(
        confirmed_conflict_pairs_df[
            [
                "candidate_id",
                "near_duplicate_status",
                "class_name_a",
                "class_name_b",
                "quality_label_a",
                "quality_label_b",
                "relative_path_a",
                "relative_path_b"
            ]
        ]
    )

print("\nCompleted review saved to:")
print(COMPLETED_REVIEW_PATH)

print("\nConfirmed pairs saved to:")
print(CONFIRMED_NEAR_PAIRS_PATH)

print(
    "\nNo files have been deleted, moved, or copied "
    "at this stage."
)

# **Cell 6 — Auditing augmentation variants via names**
Identify families of images whose names share a common base after removing explicit transformation prefixes.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

# Load the complete manifest with hashes
INVENTORY_PERCEPTUAL_PATH = (
    MANIFEST_DIR
    / "03_inventory_with_perceptual_hashes.csv"
)

if not INVENTORY_PERCEPTUAL_PATH.exists():
    raise FileNotFoundError(
        f"Not found:\n{INVENTORY_PERCEPTUAL_PATH}"
    )

filename_df = pd.read_csv(
    INVENTORY_PERCEPTUAL_PATH,
    low_memory=False
)

required_columns = {
    "dataset_id",
    "image_path",
    "relative_path",
    "file_name",
    "class_name",
    "product_name",
    "quality_label",
    "sha256",
    "phash64"
}

missing_columns = (
    required_columns
    - set(filename_df.columns)
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )

# Explicit transformation patterns
# Only prefixes clearly associated with augmentation are removed.
# Common numeric filenames are not modified.

AUGMENTATION_PATTERNS = [
    (
        "rotation",
        re.compile(
            r"^(?:rotated[_\-\s]*by[_\-\s]*"
            r"[-+]?\d+(?:\.\d+)?[_\-\s]+)",
            flags=re.IGNORECASE
        )
    ),
    (
        "rotation",
        re.compile(
            r"^(?:rotated|rotation)[_\-\s]*"
            r"[-+]?\d+(?:\.\d+)?[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "horizontal_flip",
        re.compile(
            r"^(?:horizontal[_\-\s]*flip|"
            r"flip[_\-\s]*horizontal|"
            r"flipped[_\-\s]*horizontal)"
            r"[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "vertical_flip",
        re.compile(
            r"^(?:vertical[_\-\s]*flip|"
            r"flip[_\-\s]*vertical|"
            r"flipped[_\-\s]*vertical)"
            r"[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "generic_flip",
        re.compile(
            r"^(?:flipped|flip)[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "brightness",
        re.compile(
            r"^(?:brightness|brightened|darkened)"
            r"(?:[_\-\s]*[-+]?\d+(?:\.\d+)?)?"
            r"[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "contrast",
        re.compile(
            r"^(?:contrast)"
            r"(?:[_\-\s]*[-+]?\d+(?:\.\d+)?)?"
            r"[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "zoom",
        re.compile(
            r"^(?:zoomed|zoom)"
            r"(?:[_\-\s]*[-+]?\d+(?:\.\d+)?)?"
            r"[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "crop",
        re.compile(
            r"^(?:cropped|crop)[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "shear",
        re.compile(
            r"^(?:sheared|shear)"
            r"(?:[_\-\s]*[-+]?\d+(?:\.\d+)?)?"
            r"[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "translation",
        re.compile(
            r"^(?:translated|translation|shifted|shift)"
            r"(?:[_\-\s]*[-+]?\d+(?:\.\d+)?)?"
            r"[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "blur",
        re.compile(
            r"^(?:blurred|blur)"
            r"(?:[_\-\s]*[-+]?\d+(?:\.\d+)?)?"
            r"[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "noise",
        re.compile(
            r"^(?:noisy|noise|gaussian[_\-\s]*noise)"
            r"(?:[_\-\s]*[-+]?\d+(?:\.\d+)?)?"
            r"[_\-\s]+",
            flags=re.IGNORECASE
        )
    ),
    (
        "augmentation",
        re.compile(
            r"^(?:augmented|augmentation|aug)"
            r"[_\-\s]+",
            flags=re.IGNORECASE
        )
    )
]

# Conservative filename normalization
def normalize_text(value):
    """
    Normalizes spaces and separators without changing
    the semantic content of the filename.
    """

    value = str(value).strip()

    value = re.sub(
        r"[\s_\-]+",
        " ",
        value
    )

    return value.strip().lower()


def extract_augmentation_family(file_name):
    """
    Removes one or more explicit transformation prefixes.

    Returns:
    - candidate base name;
    - detected transformations;
    - number of prefixes removed.
    """

    path = Path(str(file_name))

    original_stem = path.stem
    working_stem = original_stem

    detected_operations = []

    # Some filenames may contain several consecutive prefixes.
    # Repeat until no further changes occur.
    for _ in range(10):

        matched = False

        for operation_name, pattern in AUGMENTATION_PATTERNS:

            match = pattern.match(
                working_stem
            )

            if match:

                detected_operations.append(
                    operation_name
                )

                working_stem = (
                    working_stem[
                        match.end():
                    ]
                    .strip(" _-")
                )

                matched = True
                break

        if not matched:
            break

    normalized_original = normalize_text(
        original_stem
    )

    normalized_base = normalize_text(
        working_stem
    )

    has_explicit_prefix = (
        len(detected_operations) > 0
        and normalized_base != normalized_original
        and len(normalized_base) > 0
    )

    return pd.Series({
        "original_stem": original_stem,
        "augmentation_base_name": normalized_base,
        "augmentation_operations": (
            " | ".join(detected_operations)
        ),
        "n_augmentation_prefixes": len(
            detected_operations
        ),
        "has_explicit_augmentation_prefix": (
            has_explicit_prefix
        )
    })

# Apply extraction
augmentation_columns_df = (
    filename_df["file_name"]
    .apply(extract_augmentation_family)
)

filename_audit_df = pd.concat(
    [
        filename_df.reset_index(drop=True),
        augmentation_columns_df.reset_index(
            drop=True
        )
    ],
    axis=1
)

# Define a family key
# The same base name is initially evaluated within each product.
# A global key is also retained to detect conflicts
# across classes or products.

filename_audit_df[
    "augmentation_family_key_product"
] = (
    filename_audit_df[
        "product_name"
    ].str.lower()
    + "||"
    + filename_audit_df[
        "augmentation_base_name"
    ]
)

filename_audit_df[
    "augmentation_family_key_global"
] = filename_audit_df[
    "augmentation_base_name"
]

# The audit includes only images with an explicit prefix
# or another image that could serve as the base file
# of the same family.
explicit_augmented_df = (
    filename_audit_df[
        filename_audit_df[
            "has_explicit_augmentation_prefix"
        ]
    ]
    .copy()
)

explicit_base_names = set(
    explicit_augmented_df[
        "augmentation_base_name"
    ]
)

family_candidate_df = (
    filename_audit_df[
        filename_audit_df[
            "augmentation_base_name"
        ].isin(explicit_base_names)
    ]
    .copy()
)

# Summary of detected operations
operation_records = []

for operations in explicit_augmented_df[
    "augmentation_operations"
].dropna():

    for operation in str(operations).split(" | "):

        operation = operation.strip()

        if operation:
            operation_records.append(
                operation
            )

operation_summary_df = (
    pd.Series(
        operation_records,
        name="augmentation_operation"
    )
    .value_counts()
    .rename_axis(
        "augmentation_operation"
    )
    .reset_index(name="n_files")
)

# Family statistics by product
def join_unique(values):

    return " | ".join(
        sorted(
            {
                str(value)
                for value in values
                if pd.notna(value)
            }
        )
    )

product_family_stats_df = (
    family_candidate_df
    .groupby(
        "augmentation_family_key_product"
    )
    .agg(
        product_name=(
            "product_name",
            "first"
        ),
        augmentation_base_name=(
            "augmentation_base_name",
            "first"
        ),
        n_files=(
            "image_path",
            "size"
        ),
        n_sha256=(
            "sha256",
            "nunique"
        ),
        n_classes=(
            "class_name",
            "nunique"
        ),
        classes=(
            "class_name",
            join_unique
        ),
        n_quality_labels=(
            "quality_label",
            "nunique"
        ),
        quality_labels=(
            "quality_label",
            join_unique
        ),
        n_explicit_augmented_files=(
            "has_explicit_augmentation_prefix",
            "sum"
        ),
        operations=(
            "augmentation_operations",
            join_unique
        )
    )
    .reset_index()
)

product_family_stats_df = (
    product_family_stats_df[
        product_family_stats_df[
            "n_files"
        ] > 1
    ]
    .copy()
)

product_family_stats_df[
    "class_conflict_candidate"
] = (
    product_family_stats_df[
        "n_classes"
    ] > 1
)

product_family_stats_df[
    "quality_conflict_candidate"
] = (
    product_family_stats_df[
        "n_quality_labels"
    ] > 1
)

product_family_stats_df = (
    product_family_stats_df
    .sort_values(
        [
            "quality_conflict_candidate",
            "class_conflict_candidate",
            "n_files"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

product_family_stats_df[
    "filename_family_id"
] = [
    f"NAME_FAMILY_{index:06d}"
    for index in range(
        1,
        len(product_family_stats_df) + 1
    )
]

# Members of candidate families
filename_family_members_df = (
    family_candidate_df
    .merge(
        product_family_stats_df[
            [
                "augmentation_family_key_product",
                "filename_family_id",
                "n_files",
                "n_sha256",
                "classes",
                "quality_labels",
                "operations",
                "class_conflict_candidate",
                "quality_conflict_candidate"
            ]
        ],
        on="augmentation_family_key_product",
        how="inner",
        validate="many_to_one"
    )
    .sort_values(
        [
            "filename_family_id",
            "class_name",
            "has_explicit_augmentation_prefix",
            "file_name"
        ],
        ascending=[
            True,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

# Families with potential conflicts
filename_conflict_families_df = (
    product_family_stats_df[
        product_family_stats_df[
            "class_conflict_candidate"
        ]
        | product_family_stats_df[
            "quality_conflict_candidate"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

filename_conflict_members_df = (
    filename_family_members_df[
        filename_family_members_df[
            "filename_family_id"
        ].isin(
            set(
                filename_conflict_families_df[
                    "filename_family_id"
                ]
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# Overall summary
filename_audit_summary_df = pd.DataFrame({
    "metric": [
        "total_files",
        "files_with_explicit_augmentation_prefix",
        "unique_explicit_augmentation_base_names",
        "candidate_family_files",
        "multi_file_candidate_families",
        "unique_sha256_in_candidate_families",
        "candidate_families_with_class_conflict",
        "candidate_families_with_quality_conflict",
        "files_in_conflicting_filename_families"
    ],
    "value": [
        len(filename_audit_df),
        len(explicit_augmented_df),
        explicit_augmented_df[
            "augmentation_base_name"
        ].nunique(),
        len(filename_family_members_df),
        len(product_family_stats_df),
        filename_family_members_df[
            "sha256"
        ].nunique(),
        int(
            product_family_stats_df[
                "class_conflict_candidate"
            ].sum()
        ),
        int(
            product_family_stats_df[
                "quality_conflict_candidate"
            ].sum()
        ),
        len(filename_conflict_members_df)
    ]
})

# Save results
FILENAME_AUDIT_PATH = (
    MANIFEST_DIR
    / "08_inventory_with_filename_audit.csv"
)

FILENAME_SUMMARY_PATH = (
    REPORT_DIR
    / "08_filename_augmentation_summary.csv"
)

OPERATION_SUMMARY_PATH = (
    REPORT_DIR
    / "08_augmentation_operation_counts.csv"
)

FILENAME_FAMILIES_PATH = (
    REPORT_DIR
    / "08_filename_candidate_families.csv"
)

FILENAME_FAMILY_MEMBERS_PATH = (
    REPORT_DIR
    / "08_filename_candidate_family_members.csv"
)

FILENAME_CONFLICT_FAMILIES_PATH = (
    REPORT_DIR
    / "08_filename_conflict_families.csv"
)

FILENAME_CONFLICT_MEMBERS_PATH = (
    REPORT_DIR
    / "08_filename_conflict_family_members.csv"
)

filename_audit_df.to_csv(
    FILENAME_AUDIT_PATH,
    index=False
)

filename_audit_summary_df.to_csv(
    FILENAME_SUMMARY_PATH,
    index=False
)

operation_summary_df.to_csv(
    OPERATION_SUMMARY_PATH,
    index=False
)

product_family_stats_df.to_csv(
    FILENAME_FAMILIES_PATH,
    index=False
)

filename_family_members_df.to_csv(
    FILENAME_FAMILY_MEMBERS_PATH,
    index=False
)

filename_conflict_families_df.to_csv(
    FILENAME_CONFLICT_FAMILIES_PATH,
    index=False
)

filename_conflict_members_df.to_csv(
    FILENAME_CONFLICT_MEMBERS_PATH,
    index=False
)

# Display results

print("=" * 80)
print("PRE-AUGMENTED VARIANT AUDIT")
print("=" * 80)

display(filename_audit_summary_df)

print("\nDetected operations:")
display(operation_summary_df)

print("\nFamilies with potential conflicts:")

if filename_conflict_families_df.empty:

    print(
        "No filename-based families with class or "
        "quality-label conflicts were detected."
    )

else:

    display(
        filename_conflict_families_df[
            [
                "filename_family_id",
                "product_name",
                "augmentation_base_name",
                "n_files",
                "n_sha256",
                "classes",
                "quality_labels",
                "operations"
            ]
        ]
    )

    print("\nMembers of conflicting families:")

    display(
        filename_conflict_members_df[
            [
                "filename_family_id",
                "class_name",
                "quality_label",
                "has_explicit_augmentation_prefix",
                "augmentation_operations",
                "file_name",
                "relative_path",
                "sha256"
            ]
        ]
    )

print("\nLargest families:")

display(
    product_family_stats_df[
        [
            "filename_family_id",
            "product_name",
            "augmentation_base_name",
            "n_files",
            "n_sha256",
            "classes",
            "quality_labels",
            "operations"
        ]
    ].head(30)
)

print("\nResults saved to:")
print(FILENAME_AUDIT_PATH)

print(
    "\nNo files have been deleted, moved, or copied "
    "at this stage."
)


# **Cell 7 — Audit the composition of pre-augmented families**
This cell will check how many families contain:

exactly one original image; no original images; more than one potential original image.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Load filename-based family members
FILENAME_FAMILY_MEMBERS_PATH = (
    REPORT_DIR
    / "08_filename_candidate_family_members.csv"
)

if not FILENAME_FAMILY_MEMBERS_PATH.exists():
    raise FileNotFoundError(
        f"Not found:\n{FILENAME_FAMILY_MEMBERS_PATH}"
    )

family_members_df = pd.read_csv(
    FILENAME_FAMILY_MEMBERS_PATH,
    low_memory=False
)

required_columns = {
    "filename_family_id",
    "image_path",
    "relative_path",
    "file_name",
    "class_name",
    "product_name",
    "quality_label",
    "sha256",
    "has_explicit_augmentation_prefix",
    "augmentation_operations",
    "file_size_bytes",
    "width",
    "height"
}

missing_columns = (
    required_columns
    - set(family_members_df.columns)
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )

# Ensure Boolean values
if family_members_df[
    "has_explicit_augmentation_prefix"
].dtype != bool:

    family_members_df[
        "has_explicit_augmentation_prefix"
    ] = (
        family_members_df[
            "has_explicit_augmentation_prefix"
        ]
        .astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False
        })
    )

if family_members_df[
    "has_explicit_augmentation_prefix"
].isna().any():
    raise ValueError(
        "Unable to interpret all explicit "
        "augmentation indicators."
    )

# Identify original image candidates
family_members_df[
    "is_unaugmented_base_candidate"
] = (
    ~family_members_df[
        "has_explicit_augmentation_prefix"
    ]
)

family_members_df[
    "pixel_area"
] = (
    family_members_df["width"]
    * family_members_df["height"]
)

# Helper functions
def join_unique(values):

    return " | ".join(
        sorted(
            {
                str(value)
                for value in values
                if pd.notna(value)
                and str(value).strip() != ""
            }
        )
    )


def join_base_paths(group_df):

    paths = group_df.loc[
        group_df[
            "is_unaugmented_base_candidate"
        ],
        "relative_path"
    ].tolist()

    return " | ".join(
        sorted(map(str, paths))
    )

# Per-family statistics
family_composition_records = []

for family_id, group_df in family_members_df.groupby(
    "filename_family_id",
    sort=True
):

    n_files = len(group_df)

    n_augmented = int(
        group_df[
            "has_explicit_augmentation_prefix"
        ].sum()
    )

    n_base_candidates = int(
        group_df[
            "is_unaugmented_base_candidate"
        ].sum()
    )

    if n_base_candidates == 1:

        family_status = (
            "one_unaugmented_base_available"
        )

    elif n_base_candidates == 0:

        family_status = (
            "no_unaugmented_base_available"
        )

    else:

        family_status = (
            "multiple_unaugmented_base_candidates"
        )

    family_composition_records.append({
        "filename_family_id": family_id,
        "family_status": family_status,
        "n_files": n_files,
        "n_augmented_files": n_augmented,
        "n_unaugmented_base_candidates": (
            n_base_candidates
        ),
        "redundant_files_if_one_retained": (
            n_files - 1
        ),
        "n_sha256": group_df[
            "sha256"
        ].nunique(),
        "n_classes": group_df[
            "class_name"
        ].nunique(),
        "classes": join_unique(
            group_df["class_name"]
        ),
        "n_products": group_df[
            "product_name"
        ].nunique(),
        "products": join_unique(
            group_df["product_name"]
        ),
        "n_quality_labels": group_df[
            "quality_label"
        ].nunique(),
        "quality_labels": join_unique(
            group_df["quality_label"]
        ),
        "operations": join_unique(
            group_df[
                "augmentation_operations"
            ]
        ),
        "base_candidate_paths": (
            join_base_paths(group_df)
        ),
        "maximum_pixel_area": (
            group_df["pixel_area"].max()
        ),
        "maximum_file_size_bytes": (
            group_df[
                "file_size_bytes"
            ].max()
        )
    })

family_composition_df = pd.DataFrame(
    family_composition_records
)

# Safety validations
if len(family_composition_df) != 1511:
    raise ValueError(
        "Expected 1,511 families, but "
        f"found {len(family_composition_df):,}."
    )

if len(family_members_df) != 9180:
    raise ValueError(
        "Expected 9,180 members, but "
        f"found {len(family_members_df):,}."
    )

if (
    family_composition_df["n_classes"] > 1
).any():
    raise ValueError(
        "Some filename-based families contain more than one class."
    )

if (
    family_composition_df[
        "n_quality_labels"
    ] > 1
).any():
    raise ValueError(
        "Some filename-based families contain different "
        "Healthy/Rotten labels."
    )

# Separate family types
one_base_families_df = (
    family_composition_df[
        family_composition_df[
            "family_status"
        ] == "one_unaugmented_base_available"
    ]
    .copy()
    .reset_index(drop=True)
)

no_base_families_df = (
    family_composition_df[
        family_composition_df[
            "family_status"
        ] == "no_unaugmented_base_available"
    ]
    .copy()
    .reset_index(drop=True)
)

multiple_base_families_df = (
    family_composition_df[
        family_composition_df[
            "family_status"
        ]
        == "multiple_unaugmented_base_candidates"
    ]
    .copy()
    .reset_index(drop=True)
)

# Family size distribution
family_size_distribution_df = (
    family_composition_df[
        "n_files"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("n_files_per_family")
    .reset_index(name="n_families")
)

# Per-class summary
family_class_summary_df = (
    family_members_df
    .groupby(
        "class_name",
        as_index=False
    )
    .agg(
        files_in_filename_families=(
            "image_path",
            "size"
        ),
        filename_families=(
            "filename_family_id",
            "nunique"
        ),
        explicit_augmented_files=(
            "has_explicit_augmentation_prefix",
            "sum"
        ),
        unaugmented_base_candidates=(
            "is_unaugmented_base_candidate",
            "sum"
        )
    )
)

family_class_summary_df[
    "files_removed_if_one_per_family"
] = (
    family_class_summary_df[
        "files_in_filename_families"
    ]
    - family_class_summary_df[
        "filename_families"
    ]
)

family_class_summary_df = (
    family_class_summary_df
    .sort_values(
        "files_removed_if_one_per_family",
        ascending=False
    )
    .reset_index(drop=True)
)

# Overall summary
family_composition_summary_df = pd.DataFrame({
    "metric": [
        "filename_families",
        "files_in_filename_families",
        "families_with_one_unaugmented_base",
        "families_without_unaugmented_base",
        "families_with_multiple_unaugmented_bases",
        "total_unaugmented_base_candidates",
        "total_explicit_augmented_files",
        "files_removed_if_one_retained_per_family",
        "families_with_class_conflict",
        "families_with_quality_conflict"
    ],
    "value": [
        len(family_composition_df),
        len(family_members_df),
        len(one_base_families_df),
        len(no_base_families_df),
        len(multiple_base_families_df),
        int(
            family_members_df[
                "is_unaugmented_base_candidate"
            ].sum()
        ),
        int(
            family_members_df[
                "has_explicit_augmentation_prefix"
            ].sum()
        ),
        int(
            family_composition_df[
                "redundant_files_if_one_retained"
            ].sum()
        ),
        int(
            (
                family_composition_df[
                    "n_classes"
                ] > 1
            ).sum()
        ),
        int(
            (
                family_composition_df[
                    "n_quality_labels"
                ] > 1
            ).sum()
        )
    ]
})

# Save results
FAMILY_COMPOSITION_PATH = (
    REPORT_DIR
    / "09_filename_family_composition.csv"
)

FAMILY_COMPOSITION_SUMMARY_PATH = (
    REPORT_DIR
    / "09_filename_family_composition_summary.csv"
)

FAMILY_SIZE_DISTRIBUTION_PATH = (
    REPORT_DIR
    / "09_filename_family_size_distribution.csv"
)

NO_BASE_FAMILIES_PATH = (
    REPORT_DIR
    / "09_filename_families_without_original_base.csv"
)

MULTIPLE_BASE_FAMILIES_PATH = (
    REPORT_DIR
    / "09_filename_families_with_multiple_bases.csv"
)

FAMILY_CLASS_SUMMARY_PATH = (
    REPORT_DIR
    / "09_filename_family_summary_by_class.csv"
)

family_composition_df.to_csv(
    FAMILY_COMPOSITION_PATH,
    index=False
)

family_composition_summary_df.to_csv(
    FAMILY_COMPOSITION_SUMMARY_PATH,
    index=False
)

family_size_distribution_df.to_csv(
    FAMILY_SIZE_DISTRIBUTION_PATH,
    index=False
)

no_base_families_df.to_csv(
    NO_BASE_FAMILIES_PATH,
    index=False
)

multiple_base_families_df.to_csv(
    MULTIPLE_BASE_FAMILIES_PATH,
    index=False
)

family_class_summary_df.to_csv(
    FAMILY_CLASS_SUMMARY_PATH,
    index=False
)

# Display results
print("=" * 80)
print("COMPOSITION OF PRE-AUGMENTED FAMILIES")
print("=" * 80)

display(
    family_composition_summary_df
)

print("\nFamily size distribution:")
display(
    family_size_distribution_df
)

print("\nPer-class summary:")
display(
    family_class_summary_df
)

print("\nFamilies without an identifiable original image:")

if no_base_families_df.empty:

    print(
        "All families contain an original "
        "image without a prefix."
    )

else:

    display(
        no_base_families_df[
            [
                "filename_family_id",
                "classes",
                "n_files",
                "n_augmented_files",
                "operations",
                "maximum_pixel_area",
                "maximum_file_size_bytes"
            ]
        ].head(30)
    )

print("\nFamilies with multiple possible originals:")

if multiple_base_families_df.empty:

    print(
        "No families contain more than one possible "
        "original image."
    )

else:

    display(
        multiple_base_families_df[
            [
                "filename_family_id",
                "classes",
                "n_files",
                "n_unaugmented_base_candidates",
                "base_candidate_paths"
            ]
        ]
    )

print(
    "\nNo files have been selected, deleted, "
    "moved, or copied yet."
)

# **Cell 8 — Build provisional global components**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Input paths
INVENTORY_PATH = (
    MANIFEST_DIR
    / "03_inventory_with_perceptual_hashes.csv"
)

EQUAL_PHASH_CONFIRMED_PATH = (
    HASH_DIR
    / "04_equal_phash_confirmed_duplicate_pairs.csv"
)

EQUAL_PHASH_METRICS_PATH = (
    HASH_DIR
    / "04_equal_phash_pair_metrics.csv"
)

EQUAL_PHASH_MANUAL_PATH = (
    HASH_DIR
    / "04_equal_phash_manual_review_pairs.csv"
)

NEAR_CONFIRMED_PATH = (
    HASH_DIR
    / "07_confirmed_near_duplicate_pairs.csv"
)

NEAR_MANUAL_PATH = (
    HASH_DIR
    / "06_near_phash_manual_review_pairs.csv"
)

FILENAME_FAMILY_MEMBERS_PATH = (
    REPORT_DIR
    / "08_filename_candidate_family_members.csv"
)

required_paths = [
    INVENTORY_PATH,
    EQUAL_PHASH_CONFIRMED_PATH,
    EQUAL_PHASH_METRICS_PATH,
    EQUAL_PHASH_MANUAL_PATH,
    NEAR_CONFIRMED_PATH,
    NEAR_MANUAL_PATH,
    FILENAME_FAMILY_MEMBERS_PATH
]

for required_path in required_paths:

    if not required_path.exists():
        raise FileNotFoundError(
            f"Not found:\n{required_path}"
        )

# Load files
inventory_df = pd.read_csv(
    INVENTORY_PATH,
    low_memory=False
)

equal_confirmed_df = pd.read_csv(
    EQUAL_PHASH_CONFIRMED_PATH,
    low_memory=False
)

equal_metrics_df = pd.read_csv(
    EQUAL_PHASH_METRICS_PATH,
    low_memory=False
)

equal_manual_df = pd.read_csv(
    EQUAL_PHASH_MANUAL_PATH,
    low_memory=False
)

near_confirmed_df = pd.read_csv(
    NEAR_CONFIRMED_PATH,
    low_memory=False
)

near_manual_df = pd.read_csv(
    NEAR_MANUAL_PATH,
    low_memory=False
)

filename_members_df = pd.read_csv(
    FILENAME_FAMILY_MEMBERS_PATH,
    low_memory=False
)

# Validate inventory
required_inventory_columns = {
    "dataset_id",
    "image_path",
    "relative_path",
    "class_name",
    "product_name",
    "quality_label",
    "sha256",
    "manual_exclusion_requested"
}

missing_columns = (
    required_inventory_columns
    - set(inventory_df.columns)
)

if missing_columns:
    raise ValueError(
        "Missing columns in the inventory: "
        f"{sorted(missing_columns)}"
    )

if len(inventory_df) != 40219:
    raise ValueError(
        "The inventory does not contain the expected "
        "40,219 images."
    )

if inventory_df["dataset_id"].duplicated().any():
    raise ValueError(
        "Duplicate dataset_id values exist."
    )

inventory_df["dataset_id"] = (
    inventory_df["dataset_id"].astype(int)
)

inventory_df = (
    inventory_df
    .sort_values("dataset_id")
    .reset_index(drop=True)
)

expected_ids = np.arange(len(inventory_df))

if not np.array_equal(
    inventory_df["dataset_id"].to_numpy(),
    expected_ids
):
    raise ValueError(
        "The dataset_id values do not form the continuous sequence "
        "0...40218."
    )

# Interpret Boolean columns
def normalize_boolean_series(series):
    """
    Converts Boolean, text, or numeric columns to Boolean.
    """

    if series.dtype == bool:
        return series

    normalized = (
        series.astype(str)
        .str.strip()
        .str.lower()
    )

    mapped = normalized.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })

    if mapped.isna().any():
        invalid_values = sorted(
            normalized[
                mapped.isna()
            ].unique()
        )

        raise ValueError(
            "Unrecognized Boolean values: "
            f"{invalid_values}"
        )

    return mapped.astype(bool)


inventory_df[
    "manual_exclusion_requested"
] = normalize_boolean_series(
    inventory_df[
        "manual_exclusion_requested"
    ]
)

for dataframe in [
    equal_metrics_df,
    equal_manual_df,
    near_confirmed_df,
    near_manual_df
]:

    for column in [
        "same_class",
        "same_product",
        "same_quality_label"
    ]:

        if column in dataframe.columns:

            dataframe[column] = (
                normalize_boolean_series(
                    dataframe[column]
                )
            )

# Union-Find structure
class UnionFind:

    def __init__(self, size):

        self.parent = list(
            range(size)
        )

        self.rank = [0] * size

    def find(self, value):

        while self.parent[value] != value:

            self.parent[value] = self.parent[
                self.parent[value]
            ]

            value = self.parent[value]

        return value

    def union(self, value_a, value_b):

        root_a = self.find(value_a)
        root_b = self.find(value_b)

        if root_a == root_b:
            return False

        if self.rank[root_a] < self.rank[root_b]:

            self.parent[root_a] = root_b

        elif self.rank[root_a] > self.rank[root_b]:

            self.parent[root_b] = root_a

        else:

            self.parent[root_b] = root_a
            self.rank[root_a] += 1

        return True


union_find = UnionFind(
    len(inventory_df)
)

# SHA-256 to dataset_id mappings
sha_to_dataset_ids = (
    inventory_df
    .groupby("sha256")["dataset_id"]
    .apply(list)
    .to_dict()
)

sha_representative_id = {
    sha256: min(dataset_ids)
    for sha256, dataset_ids
    in sha_to_dataset_ids.items()
}

# Merge exact duplicates
exact_union_operations = 0
exact_duplicate_group_count = 0

for sha256, dataset_ids in sha_to_dataset_ids.items():

    if len(dataset_ids) < 2:
        continue

    exact_duplicate_group_count += 1

    first_id = dataset_ids[0]

    for other_id in dataset_ids[1:]:

        if union_find.union(
            first_id,
            other_id
        ):
            exact_union_operations += 1

# Helper functions for pairwise relationships
def build_pair_key_from_values(
    sha256_a,
    sha256_b
):

    hashes = sorted(
        [
            str(sha256_a),
            str(sha256_b)
        ]
    )

    return (
        hashes[0]
        + "__"
        + hashes[1]
    )


def union_pair_dataframe(
    dataframe,
    source_name
):
    """
    Merges the two SHA-256 hashes in each row and returns
    operation statistics.
    """

    required_pair_columns = {
        "sha256_a",
        "sha256_b"
    }

    missing_pair_columns = (
        required_pair_columns
        - set(dataframe.columns)
    )

    if missing_pair_columns:
        raise ValueError(
            f"Missing columns in {source_name}: "
            f"{sorted(missing_pair_columns)}"
        )

    processed_pairs = 0
    new_union_operations = 0

    for row in dataframe.itertuples(
        index=False
    ):

        sha_a = str(row.sha256_a)
        sha_b = str(row.sha256_b)

        if sha_a not in sha_representative_id:
            raise KeyError(
                f"SHA-256 A not found: {sha_a}"
            )

        if sha_b not in sha_representative_id:
            raise KeyError(
                f"SHA-256 B not found: {sha_b}"
            )

        processed_pairs += 1

        if union_find.union(
            sha_representative_id[sha_a],
            sha_representative_id[sha_b]
        ):
            new_union_operations += 1

    return {
        "relation_source": source_name,
        "input_relations": processed_pairs,
        "new_union_operations": (
            new_union_operations
        )
    }

# Identical pHash confirmed by a validated rule
relation_summary_records = []

relation_summary_records.append(
    union_pair_dataframe(
        equal_confirmed_df,
        "equal_phash_validated_rule"
    )
)

# Previously confirmed identical-pHash conflicts
# These are the carrot Healthy-Rotten pairs that remained
# marked as manual_review in this notebook, but were already
# visually inspected and confirmed by the researcher.

equal_manual_cross_quality_df = (
    equal_metrics_df[
        (
            equal_metrics_df[
                "duplicate_status"
            ] == "manual_review"
        )
        & (
            ~equal_metrics_df[
                "same_quality_label"
            ]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

if len(equal_manual_cross_quality_df) != 2:
    raise ValueError(
        "Expected 2 previously confirmed identical-pHash "
        "Healthy-Rotten pairs, "
        f"but found "
        f"{len(equal_manual_cross_quality_df)}."
    )

equal_manual_cross_quality_df[
    "manual_decision"
] = "confirmed_duplicate"

equal_manual_cross_quality_df[
    "decision_source"
] = (
    "visual_confirmation_in_previous_audit"
)

relation_summary_records.append(
    union_pair_dataframe(
        equal_manual_cross_quality_df,
        "equal_phash_manual_cross_quality"
    )
)

# Visually confirmed near-pHash pairs
if len(near_confirmed_df) != 265:
    raise ValueError(
        "Expected 265 confirmed near-pHash pairs, "
        f"but found "
        f"{len(near_confirmed_df)}."
    )

relation_summary_records.append(
    union_pair_dataframe(
        near_confirmed_df,
        "near_phash_full_visual_review"
    )
)

# Merge pre-augmented families
required_filename_columns = {
    "filename_family_id",
    "dataset_id"
}

missing_filename_columns = (
    required_filename_columns
    - set(filename_members_df.columns)
)

if missing_filename_columns:
    raise ValueError(
        "Missing columns in las familias nominales: "
        f"{sorted(missing_filename_columns)}"
    )

filename_family_union_operations = 0

for family_id, group_df in filename_members_df.groupby(
    "filename_family_id",
    sort=True
):

    dataset_ids = (
        group_df["dataset_id"]
        .astype(int)
        .tolist()
    )

    first_id = dataset_ids[0]

    for other_id in dataset_ids[1:]:

        if union_find.union(
            first_id,
            other_id
        ):
            filename_family_union_operations += 1

relation_summary_records.append({
    "relation_source": (
        "explicit_filename_augmentation_family"
    ),
    "input_relations": int(
        len(filename_members_df)
        - filename_members_df[
            "filename_family_id"
        ].nunique()
    ),
    "new_union_operations": (
        filename_family_union_operations
    )
})

relation_summary_records.insert(
    0,
    {
        "relation_source": "exact_sha256",
        "input_relations": int(
            sum(
                max(
                    len(dataset_ids) - 1,
                    0
                )
                for dataset_ids
                in sha_to_dataset_ids.values()
            )
        ),
        "new_union_operations": (
            exact_union_operations
        )
    }
)

relation_summary_df = pd.DataFrame(
    relation_summary_records
)

# Create deterministic components
inventory_components_df = (
    inventory_df.copy()
)

inventory_components_df[
    "component_root"
] = inventory_components_df[
    "dataset_id"
].apply(
    union_find.find
)

component_minimum_ids = (
    inventory_components_df
    .groupby("component_root")[
        "dataset_id"
    ]
    .min()
    .sort_values()
)

root_to_component_id = {
    root: f"DUPCOMP_{index:06d}"
    for index, root in enumerate(
        component_minimum_ids.index,
        start=1
    )
}

inventory_components_df[
    "duplicate_component_id"
] = inventory_components_df[
    "component_root"
].map(
    root_to_component_id
)

# Component-level statistics
def join_unique(values):

    return " | ".join(
        sorted(
            {
                str(value)
                for value in values
                if pd.notna(value)
            }
        )
    )


component_stats_df = (
    inventory_components_df
    .groupby(
        "duplicate_component_id",
        as_index=False
    )
    .agg(
        n_files=(
            "image_path",
            "size"
        ),
        n_sha256=(
            "sha256",
            "nunique"
        ),
        n_classes=(
            "class_name",
            "nunique"
        ),
        classes=(
            "class_name",
            join_unique
        ),
        n_products=(
            "product_name",
            "nunique"
        ),
        products=(
            "product_name",
            join_unique
        ),
        n_quality_labels=(
            "quality_label",
            "nunique"
        ),
        quality_labels=(
            "quality_label",
            join_unique
        ),
        contains_manual_exclusion=(
            "manual_exclusion_requested",
            "max"
        ),
        minimum_dataset_id=(
            "dataset_id",
            "min"
        )
    )
)

component_stats_df[
    "is_duplicate_component"
] = (
    component_stats_df["n_files"] > 1
)

component_stats_df[
    "class_conflict"
] = (
    component_stats_df["n_classes"] > 1
)

component_stats_df[
    "product_conflict"
] = (
    component_stats_df["n_products"] > 1
)

component_stats_df[
    "quality_conflict"
] = (
    component_stats_df[
        "n_quality_labels"
    ] > 1
)

component_stats_df[
    "any_label_conflict"
] = (
    component_stats_df["class_conflict"]
    | component_stats_df["product_conflict"]
    | component_stats_df["quality_conflict"]
)

component_stats_df[
    "mandatory_full_component_exclusion"
] = (
    component_stats_df[
        "any_label_conflict"
    ]
)

inventory_components_df = (
    inventory_components_df
    .merge(
        component_stats_df,
        on="duplicate_component_id",
        how="left",
        validate="many_to_one",
        suffixes=(
            "",
            "_component"
        )
    )
)

# SHA-to-component mapping
sha_to_component = (
    inventory_components_df
    .drop_duplicates("sha256")
    .set_index("sha256")[
        "duplicate_component_id"
    ]
    .to_dict()
)

# Prepare manual candidates not yet confirmed
confirmed_equal_pair_keys = set(
    equal_confirmed_df.apply(
        lambda row: build_pair_key_from_values(
            row["sha256_a"],
            row["sha256_b"]
        ),
        axis=1
    )
)

confirmed_equal_pair_keys.update(
    equal_manual_cross_quality_df.apply(
        lambda row: build_pair_key_from_values(
            row["sha256_a"],
            row["sha256_b"]
        ),
        axis=1
    )
)

confirmed_near_pair_keys = set(
    near_confirmed_df.apply(
        lambda row: build_pair_key_from_values(
            row["sha256_a"],
            row["sha256_b"]
        ),
        axis=1
    )
)

equal_manual_df[
    "candidate_source"
] = "equal_phash_manual_review"

near_manual_df[
    "candidate_source"
] = "near_phash_manual_review"

equal_manual_df[
    "duplicate_pair_key"
] = equal_manual_df.apply(
    lambda row: build_pair_key_from_values(
        row["sha256_a"],
        row["sha256_b"]
    ),
    axis=1
)

near_manual_df[
    "duplicate_pair_key"
] = near_manual_df.apply(
    lambda row: build_pair_key_from_values(
        row["sha256_a"],
        row["sha256_b"]
    ),
    axis=1
)

equal_unresolved_df = (
    equal_manual_df[
        ~equal_manual_df[
            "duplicate_pair_key"
        ].isin(
            confirmed_equal_pair_keys
        )
    ]
    .copy()
)

near_unresolved_df = (
    near_manual_df[
        ~near_manual_df[
            "duplicate_pair_key"
        ].isin(
            confirmed_near_pair_keys
        )
    ]
    .copy()
)

common_columns = [
    "duplicate_pair_key",
    "candidate_source",
    "sha256_a",
    "sha256_b",
    "image_path_a",
    "image_path_b",
    "relative_path_a",
    "relative_path_b",
    "class_name_a",
    "class_name_b",
    "quality_label_a",
    "quality_label_b",
    "ssim",
    "pixel_correlation",
    "histogram_correlation",
    "normalized_mad"
]

for column in common_columns:

    if column not in equal_unresolved_df.columns:
        equal_unresolved_df[column] = np.nan

    if column not in near_unresolved_df.columns:
        near_unresolved_df[column] = np.nan

unresolved_pairs_df = pd.concat(
    [
        equal_unresolved_df[
            common_columns
        ],
        near_unresolved_df[
            common_columns
        ]
    ],
    ignore_index=True
)

# Keep one row per pair and record all sources
unresolved_pair_records = []

for pair_key, group_df in unresolved_pairs_df.groupby(
    "duplicate_pair_key",
    sort=True
):

    best_row = (
        group_df
        .sort_values(
            [
                "ssim",
                "pixel_correlation"
            ],
            ascending=[
                False,
                False
            ]
        )
        .iloc[0]
    )

    record = best_row.to_dict()

    record["candidate_sources"] = " | ".join(
        sorted(
            set(
                group_df[
                    "candidate_source"
                ].astype(str)
            )
        )
    )

    unresolved_pair_records.append(
        record
    )

unresolved_unique_df = pd.DataFrame(
    unresolved_pair_records
)

unresolved_unique_df[
    "component_id_a"
] = unresolved_unique_df[
    "sha256_a"
].map(
    sha_to_component
)

unresolved_unique_df[
    "component_id_b"
] = unresolved_unique_df[
    "sha256_b"
].map(
    sha_to_component
)

if unresolved_unique_df[
    [
        "component_id_a",
        "component_id_b"
    ]
].isna().any().any():

    raise ValueError(
        "Could not assign components to "
        "all manual candidates."
    )

unresolved_unique_df[
    "already_same_component"
] = (
    unresolved_unique_df[
        "component_id_a"
    ]
    == unresolved_unique_df[
        "component_id_b"
    ]
)

unresolved_unique_df[
    "cross_class"
] = (
    unresolved_unique_df[
        "class_name_a"
    ]
    != unresolved_unique_df[
        "class_name_b"
    ]
)

unresolved_unique_df[
    "cross_quality"
] = (
    unresolved_unique_df[
        "quality_label_a"
    ]
    != unresolved_unique_df[
        "quality_label_b"
    ]
)

residual_cross_component_df = (
    unresolved_unique_df[
        ~unresolved_unique_df[
            "already_same_component"
        ]
    ]
    .copy()
    .sort_values(
        [
            "cross_quality",
            "cross_class",
            "ssim",
            "pixel_correlation"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

residual_cross_component_df[
    "residual_candidate_id"
] = [
    f"RESIDUAL_{index:06d}"
    for index in range(
        1,
        len(
            residual_cross_component_df
        ) + 1
    )
]

# Overall component summary
duplicate_components_df = (
    component_stats_df[
        component_stats_df[
            "is_duplicate_component"
        ]
    ]
    .copy()
)

conflict_components_df = (
    component_stats_df[
        component_stats_df[
            "any_label_conflict"
        ]
    ]
    .copy()
)

component_summary_df = pd.DataFrame({
    "metric": [
        "total_files",
        "total_components",
        "singleton_components",
        "duplicate_components",
        "files_in_duplicate_components",
        "redundant_files_if_one_per_component",
        "label_conflict_components",
        "files_in_label_conflict_components",
        "components_containing_manual_exclusion",
        "files_in_components_containing_manual_exclusion",
        "largest_component_size",
        "unresolved_manual_pair_candidates",
        "unresolved_pairs_already_within_same_component",
        "residual_pairs_connecting_different_components",
        "residual_cross_class_pairs",
        "residual_cross_quality_pairs"
    ],
    "value": [
        len(inventory_components_df),
        len(component_stats_df),
        int(
            (
                component_stats_df[
                    "n_files"
                ] == 1
            ).sum()
        ),
        len(duplicate_components_df),
        int(
            duplicate_components_df[
                "n_files"
            ].sum()
        ),
        int(
            (
                duplicate_components_df[
                    "n_files"
                ] - 1
            ).sum()
        ),
        len(conflict_components_df),
        int(
            conflict_components_df[
                "n_files"
            ].sum()
        ),
        int(
            component_stats_df[
                "contains_manual_exclusion"
            ].sum()
        ),
        int(
            component_stats_df.loc[
                component_stats_df[
                    "contains_manual_exclusion"
                ],
                "n_files"
            ].sum()
        ),
        int(
            component_stats_df[
                "n_files"
            ].max()
        ),
        len(unresolved_unique_df),
        int(
            unresolved_unique_df[
                "already_same_component"
            ].sum()
        ),
        len(residual_cross_component_df),
        int(
            residual_cross_component_df[
                "cross_class"
            ].sum()
        ),
        int(
            residual_cross_component_df[
                "cross_quality"
            ].sum()
        )
    ]
})

# Save results
COMPONENT_MANIFEST_PATH = (
    MANIFEST_DIR
    / "10_inventory_with_provisional_components.csv"
)

COMPONENT_STATS_PATH = (
    REPORT_DIR
    / "10_provisional_component_statistics.csv"
)

COMPONENT_SUMMARY_PATH = (
    REPORT_DIR
    / "10_provisional_component_summary.csv"
)

RELATION_SUMMARY_PATH = (
    REPORT_DIR
    / "10_component_relation_summary.csv"
)

CONFLICT_COMPONENTS_PATH = (
    REPORT_DIR
    / "10_provisional_conflict_components.csv"
)

RESIDUAL_CANDIDATES_PATH = (
    REPORT_DIR
    / "10_residual_cross_component_candidates.csv"
)

EQUAL_MANUAL_CONFIRMED_PATH = (
    REPORT_DIR
    / "10_equal_phash_manual_conflicts_confirmed.csv"
)

inventory_components_df.to_csv(
    COMPONENT_MANIFEST_PATH,
    index=False
)

component_stats_df.to_csv(
    COMPONENT_STATS_PATH,
    index=False
)

component_summary_df.to_csv(
    COMPONENT_SUMMARY_PATH,
    index=False
)

relation_summary_df.to_csv(
    RELATION_SUMMARY_PATH,
    index=False
)

conflict_components_df.to_csv(
    CONFLICT_COMPONENTS_PATH,
    index=False
)

residual_cross_component_df.to_csv(
    RESIDUAL_CANDIDATES_PATH,
    index=False
)

equal_manual_cross_quality_df.to_csv(
    EQUAL_MANUAL_CONFIRMED_PATH,
    index=False
)

# Display results
print("=" * 80)
print("PROVISIONAL GLOBAL COMPONENTS")
print("=" * 80)

display(
    component_summary_df
)

print("\nRelationships incorporated:")
display(
    relation_summary_df
)

print("\nComponents with label conflicts:")

if conflict_components_df.empty:

    print(
        "No conflicting components exist."
    )

else:

    display(
        conflict_components_df[
            [
                "duplicate_component_id",
                "n_files",
                "n_sha256",
                "classes",
                "products",
                "quality_labels",
                "contains_manual_exclusion"
            ]
        ]
    )

print(
    "\nResidual manual candidates connecting "
    "different components:"
)

print(
    f"{len(residual_cross_component_df):,} pairs"
)

if not residual_cross_component_df.empty:

    display(
        residual_cross_component_df[
            [
                "residual_candidate_id",
                "candidate_sources",
                "component_id_a",
                "component_id_b",
                "class_name_a",
                "class_name_b",
                "quality_label_a",
                "quality_label_b",
                "ssim",
                "pixel_correlation",
                "histogram_correlation",
                "normalized_mad",
                "relative_path_a",
                "relative_path_b"
            ]
        ].head(30)
    )

print("\nProvisional manifest saved to:")
print(COMPONENT_MANIFEST_PATH)

print(
    "\nNo files have been selected, deleted, "
    "moved, or copied yet."
)

# **Cell 9 — Final components and selection of the clean corpus**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# Input paths
PROVISIONAL_MANIFEST_PATH = (
    MANIFEST_DIR
    / "10_inventory_with_provisional_components.csv"
)

PROVISIONAL_COMPONENT_STATS_PATH = (
    REPORT_DIR
    / "10_provisional_component_statistics.csv"
)

RESIDUAL_PAIRS_PATH = (
    REPORT_DIR
    / "10_residual_cross_component_candidates.csv"
)

FILENAME_AUDIT_PATH = (
    MANIFEST_DIR
    / "08_inventory_with_filename_audit.csv"
)

required_paths = [
    PROVISIONAL_MANIFEST_PATH,
    PROVISIONAL_COMPONENT_STATS_PATH,
    RESIDUAL_PAIRS_PATH,
    FILENAME_AUDIT_PATH
]

for required_path in required_paths:

    if not required_path.exists():

        raise FileNotFoundError(
            f"Not found:\n{required_path}"
        )


# Load files
provisional_df = pd.read_csv(
    PROVISIONAL_MANIFEST_PATH,
    low_memory=False
)

provisional_stats_df = pd.read_csv(
    PROVISIONAL_COMPONENT_STATS_PATH,
    low_memory=False
)

residual_df = pd.read_csv(
    RESIDUAL_PAIRS_PATH,
    low_memory=False
)

filename_audit_df = pd.read_csv(
    FILENAME_AUDIT_PATH,
    low_memory=False
)


# Basic validations
if len(provisional_df) != 40219:

    raise ValueError(
        "The provisional manifest does not contain "
        "the expected 40,219 images."
    )

if provisional_df[
    "dataset_id"
].duplicated().any():

    raise ValueError(
        "Duplicate dataset_id values exist."
    )

if provisional_df[
    "duplicate_component_id"
].nunique() != 28358:

    raise ValueError(
        "Expected 28,358 provisional components, "
        "but found "
        f"{provisional_df['duplicate_component_id'].nunique():,}."
    )

required_manifest_columns = {
    "dataset_id",
    "image_path",
    "relative_path",
    "file_name",
    "class_name",
    "product_name",
    "quality_label",
    "sha256",
    "duplicate_component_id",
    "manual_exclusion_requested",
    "file_size_bytes"
}

missing_manifest_columns = (
    required_manifest_columns
    - set(provisional_df.columns)
)

if missing_manifest_columns:

    raise ValueError(
        "Missing columns in the provisional manifest: "
        f"{sorted(missing_manifest_columns)}"
    )

required_residual_columns = {
    "component_id_a",
    "component_id_b",
    "class_name_a",
    "class_name_b",
    "quality_label_a",
    "quality_label_b"
}

missing_residual_columns = (
    required_residual_columns
    - set(residual_df.columns)
)

if missing_residual_columns:

    raise ValueError(
        "Missing columns in the residual candidates: "
        f"{sorted(missing_residual_columns)}"
    )


# Normalize Boolean columns
def normalize_boolean_series(series):

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    normalized = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapped = normalized.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })

    if mapped.isna().any():

        invalid_values = sorted(
            normalized[
                mapped.isna()
            ].unique()
        )

        raise ValueError(
            "Unrecognized Boolean values: "
            f"{invalid_values}"
        )

    return mapped.astype(bool)


provisional_df[
    "manual_exclusion_requested"
] = normalize_boolean_series(
    provisional_df[
        "manual_exclusion_requested"
    ]
)


# Include explicit augmentation information
required_filename_columns = {
    "dataset_id",
    "has_explicit_augmentation_prefix",
    "augmentation_operations"
}

missing_filename_columns = (
    required_filename_columns
    - set(filename_audit_df.columns)
)

if missing_filename_columns:

    raise ValueError(
        "Missing columns in the filename audit: "
        f"{sorted(missing_filename_columns)}"
    )

augmentation_df = (
    filename_audit_df[
        [
            "dataset_id",
            "has_explicit_augmentation_prefix",
            "augmentation_operations"
        ]
    ]
    .drop_duplicates(
        subset="dataset_id",
        keep="first"
    )
    .copy()
)

augmentation_df[
    "has_explicit_augmentation_prefix"
] = normalize_boolean_series(
    augmentation_df[
        "has_explicit_augmentation_prefix"
    ]
)

selection_df = provisional_df.merge(
    augmentation_df,
    on="dataset_id",
    how="left",
    validate="one_to_one"
)

selection_df[
    "has_explicit_augmentation_prefix"
] = (
    selection_df[
        "has_explicit_augmentation_prefix"
    ]
    .fillna(False)
    .astype(bool)
)


# Identify image dimension columns
dimension_candidates = [
    ("width", "height"),
    ("image_width", "image_height"),
    ("perceptual_width", "perceptual_height")
]

width_column = None
height_column = None

for candidate_width, candidate_height in dimension_candidates:

    if {
        candidate_width,
        candidate_height
    }.issubset(selection_df.columns):

        width_column = candidate_width
        height_column = candidate_height
        break

if width_column is None or height_column is None:

    raise ValueError(
        "Image width and height columns were not found."
    )

selection_df["pixel_area"] = (
    pd.to_numeric(
        selection_df[width_column],
        errors="coerce"
    )
    *
    pd.to_numeric(
        selection_df[height_column],
        errors="coerce"
    )
)

selection_df["pixel_area"] = (
    selection_df["pixel_area"]
    .fillna(0)
)

selection_df["file_size_bytes"] = (
    pd.to_numeric(
        selection_df["file_size_bytes"],
        errors="coerce"
    )
    .fillna(0)
)


# Recalculate statistics by provisional component
def join_unique(values):

    return " | ".join(
        sorted(
            {
                str(value)
                for value in values
                if pd.notna(value)
            }
        )
    )


component_stats_df = (
    selection_df
    .groupby(
        "duplicate_component_id",
        as_index=False
    )
    .agg(
        n_files=(
            "dataset_id",
            "size"
        ),
        n_sha256=(
            "sha256",
            "nunique"
        ),
        n_classes=(
            "class_name",
            "nunique"
        ),
        classes=(
            "class_name",
            join_unique
        ),
        n_products=(
            "product_name",
            "nunique"
        ),
        products=(
            "product_name",
            join_unique
        ),
        n_quality_labels=(
            "quality_label",
            "nunique"
        ),
        quality_labels=(
            "quality_label",
            join_unique
        ),
        contains_manual_exclusion=(
            "manual_exclusion_requested",
            "max"
        ),
        minimum_dataset_id=(
            "dataset_id",
            "min"
        )
    )
)

component_stats_df[
    "class_conflict"
] = (
    component_stats_df["n_classes"] > 1
)

component_stats_df[
    "product_conflict"
] = (
    component_stats_df["n_products"] > 1
)

component_stats_df[
    "quality_conflict"
] = (
    component_stats_df[
        "n_quality_labels"
    ] > 1
)

component_stats_df[
    "any_label_conflict"
] = (
    component_stats_df["class_conflict"]
    | component_stats_df["product_conflict"]
    | component_stats_df["quality_conflict"]
)


# Components requiring complete exclusion
conflict_component_ids = set(
    component_stats_df.loc[
        component_stats_df[
            "any_label_conflict"
        ],
        "duplicate_component_id"
    ]
)

manual_exclusion_component_ids = set(
    component_stats_df.loc[
        component_stats_df[
            "contains_manual_exclusion"
        ],
        "duplicate_component_id"
    ]
)

if len(conflict_component_ids) != 9:

    raise ValueError(
        "Expected 9 conflicting components, "
        f"but found {len(conflict_component_ids)}."
    )

if len(manual_exclusion_component_ids) != 1:

    raise ValueError(
        "Expected 1 component with a manual exclusion, "
        f"but found "
        f"{len(manual_exclusion_component_ids)}."
    )

overlapping_excluded_components = (
    conflict_component_ids
    & manual_exclusion_component_ids
)

if overlapping_excluded_components:

    raise ValueError(
        "The manually excluded component is also identified "
        "as conflicting. It must be reviewed before proceeding."
    )

excluded_component_ids = (
    conflict_component_ids
    | manual_exclusion_component_ids
)

eligible_component_ids = set(
    component_stats_df.loc[
        ~component_stats_df[
            "duplicate_component_id"
        ].isin(excluded_component_ids),
        "duplicate_component_id"
    ]
)

if len(eligible_component_ids) != 28348:

    raise ValueError(
        "Expected 28,348 eligible components, "
        f"but obtained {len(eligible_component_ids):,}."
    )


# Select one canonical image per component
# Priority:
# Image without explicit augmentation.
# Higher resolution.
# Larger file size.
# Lower dataset_id as a reproducible tie-breaker.

eligible_rows_df = (
    selection_df[
        selection_df[
            "duplicate_component_id"
        ].isin(eligible_component_ids)
    ]
    .copy()
)

eligible_rows_df = (
    eligible_rows_df
    .sort_values(
        [
            "duplicate_component_id",
            "has_explicit_augmentation_prefix",
            "pixel_area",
            "file_size_bytes",
            "dataset_id"
        ],
        ascending=[
            True,
            True,
            False,
            False,
            True
        ]
    )
)

selected_indices = (
    eligible_rows_df
    .drop_duplicates(
        subset="duplicate_component_id",
        keep="first"
    )
    .index
)

selection_df[
    "selected_for_clean_dataset"
] = False

selection_df.loc[
    selected_indices,
    "selected_for_clean_dataset"
] = True


# Validate canonical selection
clean_selected_df = (
    selection_df[
        selection_df[
            "selected_for_clean_dataset"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

if len(clean_selected_df) != 28348:

    raise ValueError(
        "Expected 28,348 selected images, "
        f"but obtained {len(clean_selected_df):,}."
    )

if clean_selected_df[
    "duplicate_component_id"
].duplicated().any():

    raise ValueError(
        "More than one image was selected in a "
        "provisional component."
    )

if clean_selected_df[
    "sha256"
].duplicated().any():

    raise ValueError(
        "The selected dataset still contains "
        "duplicate SHA-256 hashes."
    )

if clean_selected_df[
    "manual_exclusion_requested"
].any():

    raise ValueError(
        "The image marked for exclusion was selected."
    )

selected_component_ids = set(
    clean_selected_df[
        "duplicate_component_id"
    ]
)

if selected_component_ids != eligible_component_ids:

    raise ValueError(
        "The selection does not exactly match the "
        "eligible components."
    )


# Union-Find for split-protection groups
# Residual relationships do NOT remove images.
# They only require their components to be assigned
# together to train, val, or test.

class UnionFind:

    def __init__(self, size):

        self.parent = list(
            range(size)
        )

        self.rank = [0] * size

    def find(self, value):

        if self.parent[value] != value:

            self.parent[value] = self.find(
                self.parent[value]
            )

        return self.parent[value]

    def union(self, value_a, value_b):

        root_a = self.find(value_a)
        root_b = self.find(value_b)

        if root_a == root_b:
            return False

        if self.rank[root_a] < self.rank[root_b]:

            self.parent[root_a] = root_b

        elif self.rank[root_a] > self.rank[root_b]:

            self.parent[root_b] = root_a

        else:

            self.parent[root_b] = root_a
            self.rank[root_a] += 1

        return True


eligible_component_list = sorted(
    eligible_component_ids
)

component_to_index = {
    component_id: index
    for index, component_id in enumerate(
        eligible_component_list
    )
}

guard_union_find = UnionFind(
    len(eligible_component_list)
)


# Validate residual relationships
residual_df["cross_class"] = (
    residual_df["class_name_a"]
    != residual_df["class_name_b"]
)

residual_df["cross_quality"] = (
    residual_df["quality_label_a"]
    != residual_df["quality_label_b"]
)

if residual_df["cross_class"].any():

    raise ValueError(
        "Residual relationships exist between "
        "different classes."
    )

if residual_df["cross_quality"].any():

    raise ValueError(
        "Residual relationships exist between "
        "Healthy and Rotten."
    )


# Create protection groups using residual relationships
residual_relations_processed = 0
residual_relations_used = 0
residual_relations_skipped_excluded = 0
effective_guard_unions = 0

for row in residual_df.itertuples(
    index=False
):

    residual_relations_processed += 1

    component_a = row.component_id_a
    component_b = row.component_id_b

    # If either belongs to an excluded component,
    # the relationship is not needed for the split.
    if (
        component_a not in eligible_component_ids
        or component_b not in eligible_component_ids
    ):

        residual_relations_skipped_excluded += 1
        continue

    residual_relations_used += 1

    if guard_union_find.union(
        component_to_index[component_a],
        component_to_index[component_b]
    ):

        effective_guard_unions += 1


# Deterministic protection-group identifiers
component_to_guard_root = {
    component_id: guard_union_find.find(
        component_to_index[component_id]
    )
    for component_id in eligible_component_list
}

component_minimum_dataset_id = (
    component_stats_df
    .set_index(
        "duplicate_component_id"
    )[
        "minimum_dataset_id"
    ]
    .to_dict()
)

guard_root_minimum_ids = {}

for component_id, guard_root in (
    component_to_guard_root.items()
):

    minimum_id = component_minimum_dataset_id[
        component_id
    ]

    if guard_root not in guard_root_minimum_ids:

        guard_root_minimum_ids[
            guard_root
        ] = minimum_id

    else:

        guard_root_minimum_ids[
            guard_root
        ] = min(
            guard_root_minimum_ids[
                guard_root
            ],
            minimum_id
        )

ordered_guard_roots = [
    root
    for root, minimum_id
    in sorted(
        guard_root_minimum_ids.items(),
        key=lambda item: item[1]
    )
]

guard_root_to_id = {
    root: f"SPLITGUARD_{index:06d}"
    for index, root in enumerate(
        ordered_guard_roots,
        start=1
    )
}

component_to_guard_id = {
    component_id: guard_root_to_id[
        guard_root
    ]
    for component_id, guard_root
    in component_to_guard_root.items()
}

selection_df[
    "split_guard_group_id"
] = selection_df[
    "duplicate_component_id"
].map(
    component_to_guard_id
)

clean_selected_df[
    "split_guard_group_id"
] = clean_selected_df[
    "duplicate_component_id"
].map(
    component_to_guard_id
)

if clean_selected_df[
    "split_guard_group_id"
].isna().any():

    raise ValueError(
        "Some selected images were not assigned "
        "a split_guard_group_id."
    )


# Verify that each protection group belongs to one class
guard_group_stats_df = (
    clean_selected_df
    .groupby(
        "split_guard_group_id",
        as_index=False
    )
    .agg(
        n_selected_images=(
            "dataset_id",
            "size"
        ),
        n_confirmed_components=(
            "duplicate_component_id",
            "nunique"
        ),
        n_classes=(
            "class_name",
            "nunique"
        ),
        classes=(
            "class_name",
            join_unique
        ),
        n_products=(
            "product_name",
            "nunique"
        ),
        products=(
            "product_name",
            join_unique
        ),
        n_quality_labels=(
            "quality_label",
            "nunique"
        ),
        quality_labels=(
            "quality_label",
            join_unique
        ),
        minimum_dataset_id=(
            "dataset_id",
            "min"
        )
    )
)

if (
    guard_group_stats_df[
        "n_classes"
    ] > 1
).any():

    invalid_guard_groups = (
        guard_group_stats_df[
            guard_group_stats_df[
                "n_classes"
            ] > 1
        ]
    )

    raise ValueError(
        "Some protection groups contain "
        "more than one class:\n"
        f"{invalid_guard_groups.head(20)}"
    )

if (
    guard_group_stats_df[
        "n_quality_labels"
    ] > 1
).any():

    raise ValueError(
        "Some protection groups contain "
        "Healthy and Rotten."
    )


# Assign cleaning action and reason
selection_df["cleaning_action"] = (
    "remove_duplicate_or_augmented_variant"
)

selection_df["cleaning_reason"] = (
    "Another canonical representative was selected "
    "from the same confirmed duplicate component."
)

conflict_mask = selection_df[
    "duplicate_component_id"
].isin(
    conflict_component_ids
)

manual_exclusion_mask = selection_df[
    "duplicate_component_id"
].isin(
    manual_exclusion_component_ids
)

selected_mask = selection_df[
    "selected_for_clean_dataset"
]

selection_df.loc[
    conflict_mask,
    "cleaning_action"
] = (
    "exclude_full_label_conflict_component"
)

selection_df.loc[
    conflict_mask,
    "cleaning_reason"
] = (
    "The confirmed duplicate component contains more "
    "than one class, product, or quality label."
)

selection_df.loc[
    manual_exclusion_mask,
    "cleaning_action"
] = (
    "exclude_manual_problematic_component"
)

selection_df.loc[
    manual_exclusion_mask,
    "cleaning_reason"
] = (
    "The component contains the image manually excluded "
    "under the conservative cleaning policy."
)

selection_df.loc[
    selected_mask,
    "cleaning_action"
] = "retain_canonical_image"

selection_df.loc[
    selected_mask,
    "cleaning_reason"
] = (
    "Canonical representative retained from the "
    "confirmed duplicate component."
)


# Update final tables
clean_selected_df = (
    selection_df[
        selection_df[
            "selected_for_clean_dataset"
        ]
    ]
    .copy()
)

clean_selected_df[
    "split_guard_group_id"
] = clean_selected_df[
    "duplicate_component_id"
].map(
    component_to_guard_id
)

clean_selected_df = (
    clean_selected_df
    .sort_values(
        [
            "class_name",
            "split_guard_group_id",
            "duplicate_component_id"
        ]
    )
    .reset_index(drop=True)
)

removed_df = (
    selection_df[
        ~selection_df[
            "selected_for_clean_dataset"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# Removal summary
removed_duplicate_variants = int(
    (
        removed_df[
            "cleaning_action"
        ]
        == "remove_duplicate_or_augmented_variant"
    ).sum()
)

removed_label_conflicts = int(
    (
        removed_df[
            "cleaning_action"
        ]
        == "exclude_full_label_conflict_component"
    ).sum()
)

removed_manual_exclusion = int(
    (
        removed_df[
            "cleaning_action"
        ]
        == "exclude_manual_problematic_component"
    ).sum()
)

if (
    removed_duplicate_variants
    + removed_label_conflicts
    + removed_manual_exclusion
    != len(removed_df)
):

    raise ValueError(
        "The removal categories do not add up to "
        "the total number of removed images."
    )


# Overall summary
selection_summary_df = pd.DataFrame({
    "metric": [
        "original_files",
        "provisional_confirmed_components",
        "confirmed_label_conflict_components",
        "files_in_label_conflict_components",
        "manual_exclusion_components",
        "files_in_manual_exclusion_components",
        "eligible_confirmed_components",
        "clean_selected_images",
        "removed_images_total",
        "removed_duplicate_or_augmented_variants",
        "removed_due_to_label_conflicts",
        "removed_due_to_manual_exclusion",
        "selected_unique_sha256",
        "selected_number_of_classes",
        "residual_relations_processed",
        "residual_relations_used_as_split_guards",
        "residual_relations_skipped_excluded",
        "effective_split_guard_unions",
        "split_guard_groups",
        "split_guard_groups_with_multiple_images",
        "selected_images_in_multi_image_guard_groups",
        "largest_split_guard_group"
    ],
    "value": [
        len(selection_df),
        component_stats_df[
            "duplicate_component_id"
        ].nunique(),
        len(conflict_component_ids),
        int(
            selection_df[
                conflict_mask
            ].shape[0]
        ),
        len(manual_exclusion_component_ids),
        int(
            selection_df[
                manual_exclusion_mask
            ].shape[0]
        ),
        len(eligible_component_ids),
        len(clean_selected_df),
        len(removed_df),
        removed_duplicate_variants,
        removed_label_conflicts,
        removed_manual_exclusion,
        clean_selected_df[
            "sha256"
        ].nunique(),
        clean_selected_df[
            "class_name"
        ].nunique(),
        residual_relations_processed,
        residual_relations_used,
        residual_relations_skipped_excluded,
        effective_guard_unions,
        len(guard_group_stats_df),
        int(
            (
                guard_group_stats_df[
                    "n_selected_images"
                ] > 1
            ).sum()
        ),
        int(
            guard_group_stats_df.loc[
                guard_group_stats_df[
                    "n_selected_images"
                ] > 1,
                "n_selected_images"
            ].sum()
        ),
        int(
            guard_group_stats_df[
                "n_selected_images"
            ].max()
        )
    ]
})


# Clean dataset distribution by class
original_distribution_df = (
    selection_df[
        "class_name"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("class_name")
    .reset_index(name="original_images")
)

clean_distribution_df = (
    clean_selected_df[
        "class_name"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("class_name")
    .reset_index(name="clean_images")
)

class_distribution_df = (
    original_distribution_df
    .merge(
        clean_distribution_df,
        on="class_name",
        how="left"
    )
)

class_distribution_df[
    "clean_images"
] = (
    class_distribution_df[
        "clean_images"
    ]
    .fillna(0)
    .astype(int)
)

class_distribution_df[
    "removed_images"
] = (
    class_distribution_df[
        "original_images"
    ]
    - class_distribution_df[
        "clean_images"
    ]
)

class_distribution_df[
    "retained_percentage"
] = (
    100
    * class_distribution_df[
        "clean_images"
    ]
    / class_distribution_df[
        "original_images"
    ]
)


# Protection-group size distribution
guard_size_distribution_df = (
    guard_group_stats_df[
        "n_selected_images"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "selected_images_per_guard_group"
    )
    .reset_index(
        name="n_guard_groups"
    )
)


# Save results
CORRECTED_COMPONENT_MANIFEST_PATH = (
    MANIFEST_DIR
    / "11_corrected_component_manifest.csv"
)

CLEAN_SELECTED_PATH = (
    MANIFEST_DIR
    / "11_clean_selected_images.csv"
)

REMOVED_IMAGES_PATH = (
    REPORT_DIR
    / "11_removed_images.csv"
)

CORRECTED_COMPONENT_STATS_PATH = (
    REPORT_DIR
    / "11_corrected_component_statistics.csv"
)

SELECTION_SUMMARY_PATH = (
    REPORT_DIR
    / "11_corrected_selection_summary.csv"
)

CLASS_DISTRIBUTION_PATH = (
    REPORT_DIR
    / "11_clean_class_distribution.csv"
)

SPLIT_GUARD_GROUPS_PATH = (
    REPORT_DIR
    / "11_split_guard_groups.csv"
)

SPLIT_GUARD_SIZE_DISTRIBUTION_PATH = (
    REPORT_DIR
    / "11_split_guard_size_distribution.csv"
)

selection_df.to_csv(
    CORRECTED_COMPONENT_MANIFEST_PATH,
    index=False
)

clean_selected_df.to_csv(
    CLEAN_SELECTED_PATH,
    index=False
)

removed_df.to_csv(
    REMOVED_IMAGES_PATH,
    index=False
)

component_stats_df.to_csv(
    CORRECTED_COMPONENT_STATS_PATH,
    index=False
)

selection_summary_df.to_csv(
    SELECTION_SUMMARY_PATH,
    index=False
)

class_distribution_df.to_csv(
    CLASS_DISTRIBUTION_PATH,
    index=False
)

guard_group_stats_df.to_csv(
    SPLIT_GUARD_GROUPS_PATH,
    index=False
)

guard_size_distribution_df.to_csv(
    SPLIT_GUARD_SIZE_DISTRIBUTION_PATH,
    index=False
)


# Display results
print("=" * 80)
print("CONSERVATIVE SELECTION OF THE CLEAN DATASET")
print("=" * 80)

display(selection_summary_df)

print("\nClean distribution by class:")
display(class_distribution_df)

print("\nProtection-group size distribution:")
display(guard_size_distribution_df)

print("\nSelected images manifest:")
print(CLEAN_SELECTED_PATH)

print("\nSplit-protection groups:")
print(SPLIT_GUARD_GROUPS_PATH)

print(
    "\nNo images have been copied yet "
    "and the 80/10/10 split has not been created."
)

# **Cell 10 — Create the 80/10/10 split and copy the cleaned corpus**

In [ ]:
from itertools import permutations
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold


# Configuration
if "SEED" not in globals():
    SEED = 42

if "SPLIT_DIR" not in globals():
    SPLIT_DIR = (
        WORK_ROOT
        / "splits"
    )

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEAN_SELECTED_PATH = (
    MANIFEST_DIR
    / "11_clean_selected_images.csv"
)

if not CLEAN_SELECTED_PATH.exists():

    raise FileNotFoundError(
        f"Not found:\n{CLEAN_SELECTED_PATH}"
    )


# Load the clean dataset
clean_df = pd.read_csv(
    CLEAN_SELECTED_PATH,
    low_memory=False
)

required_columns = {
    "dataset_id",
    "image_path",
    "relative_path",
    "file_name",
    "class_name",
    "product_name",
    "quality_label",
    "sha256",
    "duplicate_component_id",
    "split_guard_group_id"
}

missing_columns = (
    required_columns
    - set(clean_df.columns)
)

if missing_columns:

    raise ValueError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )


# Initial validations
if len(clean_df) != 28348:

    raise ValueError(
        "Expected 28,348 clean images, "
        f"but found {len(clean_df):,}."
    )

if clean_df["class_name"].nunique() != 24:

    raise ValueError(
        "The dataset does not contain all 24 classes."
    )

if clean_df["dataset_id"].duplicated().any():

    raise ValueError(
        "Duplicate dataset_id values exist."
    )

if clean_df["sha256"].duplicated().any():

    raise ValueError(
        "The dataset contains duplicate SHA-256 hashes."
    )

if clean_df[
    "duplicate_component_id"
].duplicated().any():

    raise ValueError(
        "More than one image was selected "
        "per confirmed component."
    )

if clean_df[
    "split_guard_group_id"
].isna().any():

    raise ValueError(
        "Some images have no split protection group."
    )

if (
    clean_df[
        "split_guard_group_id"
    ].nunique()
    != 25041
):

    raise ValueError(
        "Expected 25,041 split protection groups, "
        "but found "
        f"{clean_df['split_guard_group_id'].nunique():,}."
    )


# Verify that each group belongs to a single class
guard_integrity_df = (
    clean_df
    .groupby(
        "split_guard_group_id",
        as_index=False
    )
    .agg(
        n_images=(
            "dataset_id",
            "size"
        ),
        n_classes=(
            "class_name",
            "nunique"
        ),
        n_products=(
            "product_name",
            "nunique"
        ),
        n_quality_labels=(
            "quality_label",
            "nunique"
        )
    )
)

if (
    guard_integrity_df["n_classes"] > 1
).any():

    raise ValueError(
        "Some split protection groups contain "
        "more than one class."
    )

if (
    guard_integrity_df[
        "n_quality_labels"
    ] > 1
).any():

    raise ValueError(
        "Some split protection groups contain "
        "both Healthy and Rotten labels."
    )

if (
    guard_integrity_df["n_images"].max()
    != 19
):

    raise ValueError(
        "The maximum group size does not match "
        "the previous result."
    )


# Deterministic ordering
clean_df = (
    clean_df
    .sort_values(
        [
            "class_name",
            "split_guard_group_id",
            "duplicate_component_id",
            "dataset_id"
        ]
    )
    .reset_index(drop=True)
)


# Create ten stratified, group-aware folds
# Each group remains entirely within a single fold.
# The following folds are then selected:
# - 1 fold for validation;
# - 1 fold for testing;
# - 8 folds for training.

splitter = StratifiedGroupKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

fold_ids = np.full(
    len(clean_df),
    fill_value=-1,
    dtype=np.int16
)

dummy_features = np.zeros(
    shape=(len(clean_df), 1),
    dtype=np.uint8
)

for fold_id, (_, fold_indices) in enumerate(
    splitter.split(
        X=dummy_features,
        y=clean_df["class_name"],
        groups=clean_df[
            "split_guard_group_id"
        ]
    )
):

    fold_ids[
        fold_indices
    ] = fold_id

if (
    fold_ids < 0
).any():

    raise RuntimeError(
        "Some images were not assigned to a fold."
    )

clean_df["grouped_fold_id"] = fold_ids


# Verify that no group was split across folds
folds_per_guard_group = (
    clean_df
    .groupby(
        "split_guard_group_id"
    )[
        "grouped_fold_id"
    ]
    .nunique()
)

if (
    folds_per_guard_group > 1
).any():

    raise RuntimeError(
        "A split protection group was divided "
        "across folds."
    )


# Distribution of the ten folds
fold_summary_df = (
    clean_df
    .groupby(
        "grouped_fold_id",
        as_index=False
    )
    .agg(
        images=(
            "dataset_id",
            "size"
        ),
        guard_groups=(
            "split_guard_group_id",
            "nunique"
        ),
        classes=(
            "class_name",
            "nunique"
        )
    )
)

fold_summary_df[
    "percentage"
] = (
    100
    * fold_summary_df["images"]
    / len(clean_df)
)


# Per-fold class distribution table
fold_class_counts_df = (
    clean_df
    .groupby(
        [
            "grouped_fold_id",
            "class_name"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .sort_index()
)

class_totals = (
    clean_df[
        "class_name"
    ]
    .value_counts()
    .sort_index()
)

class_names = (
    class_totals.index.tolist()
)

target_class_counts = (
    class_totals.astype(float)
    * 0.10
)


# Select the best folds for validation and testing
# All 90 possible ordered combinations are evaluated.
# The criterion minimizes:
# - per-class deviation from 10%;
# - overall deviation from 10%;
# - size difference between validation and testing.

candidate_records = []

total_images = len(clean_df)
target_split_size = total_images * 0.10

for val_fold, test_fold in permutations(
    range(10),
    2
):

    val_counts = (
        fold_class_counts_df
        .loc[val_fold]
        .reindex(
            class_names,
            fill_value=0
        )
    )

    test_counts = (
        fold_class_counts_df
        .loc[test_fold]
        .reindex(
            class_names,
            fill_value=0
        )
    )

    val_size = int(
        val_counts.sum()
    )

    test_size = int(
        test_counts.sum()
    )

    relative_val_error = (
        (
            val_counts
            - target_class_counts
        )
        / target_class_counts.clip(
            lower=1.0
        )
    )

    relative_test_error = (
        (
            test_counts
            - target_class_counts
        )
        / target_class_counts.clip(
            lower=1.0
        )
    )

    class_balance_cost = float(
        np.mean(
            relative_val_error ** 2
        )
        +
        np.mean(
            relative_test_error ** 2
        )
    )

    global_size_cost = float(
        (
            (
                val_size
                - target_split_size
            )
            / target_split_size
        ) ** 2
        +
        (
            (
                test_size
                - target_split_size
            )
            / target_split_size
        ) ** 2
    )

    val_test_difference_cost = float(
        (
            (
                val_size
                - test_size
            )
            / target_split_size
        ) ** 2
    )

    missing_class_penalty = int(
        (val_counts == 0).sum()
        +
        (test_counts == 0).sum()
    )

    total_cost = (
        class_balance_cost
        + 3.0 * global_size_cost
        + 0.25 * val_test_difference_cost
        + 1000.0 * missing_class_penalty
    )

    candidate_records.append({
        "val_fold": val_fold,
        "test_fold": test_fold,
        "train_folds": " | ".join(
            map(
                str,
                sorted(
                    set(range(10))
                    - {
                        val_fold,
                        test_fold
                    }
                )
            )
        ),
        "val_images": val_size,
        "test_images": test_size,
        "train_images": (
            total_images
            - val_size
            - test_size
        ),
        "missing_classes_val_test": (
            missing_class_penalty
        ),
        "class_balance_cost": (
            class_balance_cost
        ),
        "global_size_cost": (
            global_size_cost
        ),
        "val_test_difference_cost": (
            val_test_difference_cost
        ),
        "total_cost": total_cost
    })

fold_pair_candidates_df = (
    pd.DataFrame(
        candidate_records
    )
    .sort_values(
        [
            "total_cost",
            "val_fold",
            "test_fold"
        ]
    )
    .reset_index(drop=True)
)

best_candidate = (
    fold_pair_candidates_df.iloc[0]
)

selected_val_fold = int(
    best_candidate["val_fold"]
)

selected_test_fold = int(
    best_candidate["test_fold"]
)


# Assign training, validation, and test splits
clean_df["split"] = "train"

clean_df.loc[
    clean_df[
        "grouped_fold_id"
    ] == selected_val_fold,
    "split"
] = "val"

clean_df.loc[
    clean_df[
        "grouped_fold_id"
    ] == selected_test_fold,
    "split"
] = "test"


# Validate group integrity across splits
splits_per_guard_group = (
    clean_df
    .groupby(
        "split_guard_group_id"
    )[
        "split"
    ]
    .nunique()
)

if (
    splits_per_guard_group > 1
).any():

    invalid_groups = (
        splits_per_guard_group[
            splits_per_guard_group > 1
        ]
    )

    raise RuntimeError(
        "Split protection groups were divided "
        "across splits:\n"
        f"{invalid_groups.head(20)}"
    )


# Identifier validations
def calculate_pairwise_overlap(
    dataframe,
    identifier_column
):

    values = {
        split_name: set(
            dataframe.loc[
                dataframe["split"]
                == split_name,
                identifier_column
            ]
        )
        for split_name in [
            "train",
            "val",
            "test"
        ]
    }

    return {
        "train_val": len(
            values["train"]
            & values["val"]
        ),
        "train_test": len(
            values["train"]
            & values["test"]
        ),
        "val_test": len(
            values["val"]
            & values["test"]
        )
    }


dataset_id_overlap = (
    calculate_pairwise_overlap(
        clean_df,
        "dataset_id"
    )
)

sha256_overlap = (
    calculate_pairwise_overlap(
        clean_df,
        "sha256"
    )
)

component_overlap = (
    calculate_pairwise_overlap(
        clean_df,
        "duplicate_component_id"
    )
)

guard_group_overlap = (
    calculate_pairwise_overlap(
        clean_df,
        "split_guard_group_id"
    )
)

overlap_results = {
    "dataset_id": dataset_id_overlap,
    "sha256": sha256_overlap,
    "duplicate_component_id": (
        component_overlap
    ),
    "split_guard_group_id": (
        guard_group_overlap
    )
}

for identifier, result in (
    overlap_results.items()
):

    if any(
        count != 0
        for count in result.values()
    ):

        raise RuntimeError(
            f"Overlap was detected in "
            f"{identifier}: {result}"
        )


# Verify all 24 classes in each split
classes_per_split = (
    clean_df
    .groupby("split")[
        "class_name"
    ]
    .nunique()
)

if not (
    classes_per_split == 24
).all():

    raise ValueError(
        "At least one split does not contain "
        "all 24 classes:\n"
        f"{classes_per_split}"
    )


# Overall summary
split_summary_df = (
    clean_df
    .groupby(
        "split",
        as_index=False
    )
    .agg(
        images=(
            "dataset_id",
            "size"
        ),
        classes=(
            "class_name",
            "nunique"
        ),
        unique_sha256=(
            "sha256",
            "nunique"
        ),
        confirmed_components=(
            "duplicate_component_id",
            "nunique"
        ),
        split_guard_groups=(
            "split_guard_group_id",
            "nunique"
        )
    )
)

split_summary_df[
    "percentage"
] = (
    100
    * split_summary_df["images"]
    / len(clean_df)
)

split_order = {
    "train": 0,
    "val": 1,
    "test": 2
}

split_summary_df[
    "split_order"
] = split_summary_df[
    "split"
].map(split_order)

split_summary_df = (
    split_summary_df
    .sort_values("split_order")
    .drop(columns="split_order")
    .reset_index(drop=True)
)


# Class distribution
split_class_distribution_df = (
    clean_df
    .groupby(
        [
            "class_name",
            "split"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

for split_name in [
    "train",
    "val",
    "test"
]:

    if (
        split_name
        not in split_class_distribution_df.columns
    ):

        split_class_distribution_df[
            split_name
        ] = 0

split_class_distribution_df[
    "total"
] = (
    split_class_distribution_df[
        [
            "train",
            "val",
            "test"
        ]
    ].sum(axis=1)
)

split_class_distribution_df[
    "train_percentage"
] = (
    100
    * split_class_distribution_df["train"]
    / split_class_distribution_df["total"]
)

split_class_distribution_df[
    "val_percentage"
] = (
    100
    * split_class_distribution_df["val"]
    / split_class_distribution_df["total"]
)

split_class_distribution_df[
    "test_percentage"
] = (
    100
    * split_class_distribution_df["test"]
    / split_class_distribution_df["total"]
)

split_class_distribution_df = (
    split_class_distribution_df[
        [
            "class_name",
            "train",
            "val",
            "test",
            "total",
            "train_percentage",
            "val_percentage",
            "test_percentage"
        ]
    ]
    .sort_values("class_name")
    .reset_index(drop=True)
)


# Overlap table
zero_overlap_records = []

for identifier, result in (
    overlap_results.items()
):

    for split_pair, overlap_count in (
        result.items()
    ):

        zero_overlap_records.append({
            "identifier": identifier,
            "split_pair": split_pair,
            "overlap_count": overlap_count
        })

zero_overlap_df = pd.DataFrame(
    zero_overlap_records
)


# Create destination paths for the clean dataset
if "CLEAN_DATASET_DIR" not in globals():

    CLEAN_DATASET_DIR = (
        WORK_ROOT
        / "clean_dataset"
    )


def create_clean_file_name(row):

    suffix = Path(
        str(row["image_path"])
    ).suffix.lower()

    allowed_suffixes = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".tif",
        ".tiff",
        ".webp"
    }

    if suffix not in allowed_suffixes:
        suffix = ".jpg"

    return (
        f"img_{int(row['dataset_id']):05d}_"
        f"{str(row['sha256'])[:16]}"
        f"{suffix}"
    )


clean_df[
    "clean_file_name"
] = clean_df.apply(
    create_clean_file_name,
    axis=1
)

clean_df[
    "clean_relative_path"
] = clean_df.apply(
    lambda row: str(
        Path(row["split"])
        / row["class_name"]
        / row["clean_file_name"]
    ),
    axis=1
)

clean_df[
    "clean_image_path"
] = clean_df[
    "clean_relative_path"
].apply(
    lambda relative_path: str(
        CLEAN_DATASET_DIR
        / relative_path
    )
)

if clean_df[
    "clean_relative_path"
].duplicated().any():

    raise ValueError(
        "Duplicate clean output paths were generated."
    )


# Sort the final manifest
clean_df["split_order"] = (
    clean_df["split"].map(
        split_order
    )
)

clean_df = (
    clean_df
    .sort_values(
        [
            "split_order",
            "class_name",
            "split_guard_group_id",
            "clean_file_name"
        ]
    )
    .drop(columns="split_order")
    .reset_index(drop=True)
)


# Save manifests and reports
CLEAN_SPLIT_MANIFEST_PATH = (
    MANIFEST_DIR
    / "12_clean_grouped_split_manifest.csv"
)

TRAIN_SPLIT_PATH = (
    SPLIT_DIR
    / "train.csv"
)

VAL_SPLIT_PATH = (
    SPLIT_DIR
    / "val.csv"
)

TEST_SPLIT_PATH = (
    SPLIT_DIR
    / "test.csv"
)

SPLIT_SUMMARY_PATH = (
    REPORT_DIR
    / "12_grouped_split_summary.csv"
)

SPLIT_CLASS_DISTRIBUTION_PATH = (
    REPORT_DIR
    / "12_grouped_split_class_distribution.csv"
)

ZERO_OVERLAP_PATH = (
    REPORT_DIR
    / "12_grouped_split_zero_overlap.csv"
)

FOLD_SUMMARY_PATH = (
    REPORT_DIR
    / "12_grouped_fold_summary.csv"
)

FOLD_PAIR_CANDIDATES_PATH = (
    REPORT_DIR
    / "12_fold_pair_selection_candidates.csv"
)

clean_df.to_csv(
    CLEAN_SPLIT_MANIFEST_PATH,
    index=False
)

clean_df[
    clean_df["split"] == "train"
].to_csv(
    TRAIN_SPLIT_PATH,
    index=False
)

clean_df[
    clean_df["split"] == "val"
].to_csv(
    VAL_SPLIT_PATH,
    index=False
)

clean_df[
    clean_df["split"] == "test"
].to_csv(
    TEST_SPLIT_PATH,
    index=False
)

split_summary_df.to_csv(
    SPLIT_SUMMARY_PATH,
    index=False
)

split_class_distribution_df.to_csv(
    SPLIT_CLASS_DISTRIBUTION_PATH,
    index=False
)

zero_overlap_df.to_csv(
    ZERO_OVERLAP_PATH,
    index=False
)

fold_summary_df.to_csv(
    FOLD_SUMMARY_PATH,
    index=False
)

fold_pair_candidates_df.to_csv(
    FOLD_PAIR_CANDIDATES_PATH,
    index=False
)


# Display results
print("=" * 80)
print("GROUP-AWARE 80/10/10 SPLIT CREATED")
print("=" * 80)

print(
    "Selected validation fold:",
    selected_val_fold
)

print(
    "Selected test fold:",
    selected_test_fold
)

print("\nOverall summary:")
display(split_summary_df)

print("\nClass distribution:")
display(split_class_distribution_df)

print("\nOverlap verification:")
display(zero_overlap_df)

print("\nDistribution of the ten folds:")
display(fold_summary_df)

print("\nComplete manifest:")
print(CLEAN_SPLIT_MANIFEST_PATH)

print("\nIndividual manifests:")
print(TRAIN_SPLIT_PATH)
print(VAL_SPLIT_PATH)
print(TEST_SPLIT_PATH)

print(
    "\nThe images have not yet been physically "
    "copied."
)

# **Cell 11 — Copy the cleaned corpus according to the final split**

In [ ]:
from pathlib import Path
import shutil

import pandas as pd
from tqdm.auto import tqdm


# Paths
CLEAN_SPLIT_MANIFEST_PATH = (
    MANIFEST_DIR
    / "12_clean_grouped_split_manifest.csv"
)

if "CLEAN_DATASET_DIR" not in globals():
    CLEAN_DATASET_DIR = (
        WORK_ROOT
        / "clean_dataset"
    )

if not CLEAN_SPLIT_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        f"Not found:\n"
        f"{CLEAN_SPLIT_MANIFEST_PATH}"
    )


# Load the final manifest
split_df = pd.read_csv(
    CLEAN_SPLIT_MANIFEST_PATH,
    low_memory=False
)

required_columns = {
    "dataset_id",
    "image_path",
    "class_name",
    "split",
    "sha256",
    "duplicate_component_id",
    "split_guard_group_id",
    "clean_file_name",
    "clean_relative_path"
}

missing_columns = (
    required_columns
    - set(split_df.columns)
)

if missing_columns:

    raise ValueError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )


# Preliminary validations
expected_split_counts = {
    "train": 22672,
    "val": 2835,
    "test": 2841
}

observed_split_counts = (
    split_df["split"]
    .value_counts()
    .to_dict()
)

if observed_split_counts != expected_split_counts:

    raise ValueError(
        "The manifest counts do not match "
        "the approved split.\n"
        f"Expected: {expected_split_counts}\n"
        f"Observed: {observed_split_counts}"
    )

if len(split_df) != 28348:

    raise ValueError(
        "Expected 28,348 images, "
        f"but found {len(split_df):,}."
    )

if split_df["class_name"].nunique() != 24:

    raise ValueError(
        "The manifest does not contain the 24 classes."
    )

if split_df["dataset_id"].duplicated().any():

    raise ValueError(
        "Duplicate dataset_id values exist."
    )

if split_df["sha256"].duplicated().any():

    raise ValueError(
        "Duplicate SHA-256 hashes exist."
    )

if split_df[
    "duplicate_component_id"
].duplicated().any():

    raise ValueError(
        "More than one image exists per "
        "confirmed component."
    )

if split_df[
    "clean_relative_path"
].duplicated().any():

    raise ValueError(
        "Duplicate destination paths exist."
    )


# Verify source files
split_df["source_exists"] = (
    split_df["image_path"]
    .apply(
        lambda path: Path(str(path)).is_file()
    )
)

missing_sources_df = (
    split_df[
        ~split_df["source_exists"]
    ]
    .copy()
)

if not missing_sources_df.empty:

    display(
        missing_sources_df[
            [
                "dataset_id",
                "class_name",
                "image_path"
            ]
        ].head(20)
    )

    raise FileNotFoundError(
        f"Could not find "
        f"{len(missing_sources_df):,} "
        "source files."
    )


# Remove a previous copy
if CLEAN_DATASET_DIR.exists():

    print(
        "Removing the previous dataset version:"
    )

    print(CLEAN_DATASET_DIR)

    shutil.rmtree(
        CLEAN_DATASET_DIR
    )

CLEAN_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Create the 72 directories:
# 3 splits × 24 classes
class_names = sorted(
    split_df["class_name"].unique()
)

for split_name in [
    "train",
    "val",
    "test"
]:

    for class_name in class_names:

        output_directory = (
            CLEAN_DATASET_DIR
            / split_name
            / class_name
        )

        output_directory.mkdir(
            parents=True,
            exist_ok=True
        )


# Copy the images
copy_error_records = []

for row in tqdm(
    split_df.itertuples(index=False),
    total=len(split_df),
    desc="Copying clean images"
):

    source_path = Path(
        str(row.image_path)
    )

    destination_path = (
        CLEAN_DATASET_DIR
        / str(row.clean_relative_path)
    )

    try:

        destination_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(
            source_path,
            destination_path
        )

    except Exception as error:

        copy_error_records.append({
            "dataset_id": row.dataset_id,
            "split": row.split,
            "class_name": row.class_name,
            "source_path": str(source_path),
            "destination_path": str(
                destination_path
            ),
            "error": (
                f"{type(error).__name__}: "
                f"{error}"
            )
        })


copy_errors_df = pd.DataFrame(
    copy_error_records
)

COPY_ERRORS_PATH = (
    REPORT_DIR
    / "13_clean_dataset_copy_errors.csv"
)

copy_errors_df.to_csv(
    COPY_ERRORS_PATH,
    index=False
)

if not copy_errors_df.empty:

    display(
        copy_errors_df.head(20)
    )

    raise RuntimeError(
        f"There were "
        f"{len(copy_errors_df):,} "
        "errors during copying."
    )


# Record the final physical paths
split_df[
    "copied_image_path"
] = split_df[
    "clean_relative_path"
].apply(
    lambda relative_path: str(
        CLEAN_DATASET_DIR
        / relative_path
    )
)

split_df[
    "copied_file_exists"
] = split_df[
    "copied_image_path"
].apply(
    lambda path: Path(path).is_file()
)

if not split_df[
    "copied_file_exists"
].all():

    missing_copies_df = (
        split_df[
            ~split_df[
                "copied_file_exists"
            ]
        ]
    )

    raise RuntimeError(
        "Not all files were found "
        "after copying:\n"
        f"{missing_copies_df.head(20)}"
    )


# Physical inventory of the clean directory
physical_records = []

for image_path in CLEAN_DATASET_DIR.rglob("*"):

    if not image_path.is_file():
        continue

    relative_path = image_path.relative_to(
        CLEAN_DATASET_DIR
    )

    if len(relative_path.parts) != 3:

        raise RuntimeError(
            "A file was found outside the "
            "split/class/image structure:\n"
            f"{relative_path}"
        )

    split_name = relative_path.parts[0]
    class_name = relative_path.parts[1]

    physical_records.append({
        "split": split_name,
        "class_name": class_name,
        "file_name": image_path.name,
        "physical_path": str(image_path),
        "file_size_bytes": (
            image_path.stat().st_size
        )
    })


physical_inventory_df = pd.DataFrame(
    physical_records
)

if len(physical_inventory_df) != 28348:

    raise RuntimeError(
        "The physical image count does not match "
        "the manifest.\n"
        f"Expected: 28.348\n"
        f"Found: "
        f"{len(physical_inventory_df):,}"
    )


# Compare the manifest and physical directory
manifest_distribution_df = (
    split_df
    .groupby(
        [
            "split",
            "class_name"
        ]
    )
    .size()
    .reset_index(
        name="manifest_images"
    )
)

physical_distribution_df = (
    physical_inventory_df
    .groupby(
        [
            "split",
            "class_name"
        ]
    )
    .size()
    .reset_index(
        name="physical_images"
    )
)

distribution_check_df = (
    manifest_distribution_df
    .merge(
        physical_distribution_df,
        on=[
            "split",
            "class_name"
        ],
        how="outer"
    )
    .fillna(0)
)

distribution_check_df[
    "count_difference"
] = (
    distribution_check_df[
        "physical_images"
    ]
    - distribution_check_df[
        "manifest_images"
    ]
)

if (
    distribution_check_df[
        "count_difference"
    ] != 0
).any():

    invalid_distribution_df = (
        distribution_check_df[
            distribution_check_df[
                "count_difference"
            ] != 0
        ]
    )

    display(invalid_distribution_df)

    raise RuntimeError(
        "The physical distribution does not match "
        "the manifest."
    )


# Physical summary by split
physical_split_summary_df = (
    physical_inventory_df
    .groupby(
        "split",
        as_index=False
    )
    .agg(
        copied_images=(
            "physical_path",
            "size"
        ),
        classes=(
            "class_name",
            "nunique"
        ),
        total_size_bytes=(
            "file_size_bytes",
            "sum"
        )
    )
)

physical_split_summary_df[
    "total_size_gb"
] = (
    physical_split_summary_df[
        "total_size_bytes"
    ]
    / (1024 ** 3)
)

split_order = {
    "train": 0,
    "val": 1,
    "test": 2
}

physical_split_summary_df[
    "split_order"
] = (
    physical_split_summary_df[
        "split"
    ].map(split_order)
)

physical_split_summary_df = (
    physical_split_summary_df
    .sort_values("split_order")
    .drop(columns="split_order")
    .reset_index(drop=True)
)


# Save final manifests
COPIED_MANIFEST_PATH = (
    MANIFEST_DIR
    / "13_clean_copied_dataset_manifest.csv"
)

PHYSICAL_INVENTORY_PATH = (
    MANIFEST_DIR
    / "13_clean_physical_inventory.csv"
)

PHYSICAL_SPLIT_SUMMARY_PATH = (
    REPORT_DIR
    / "13_clean_physical_split_summary.csv"
)

DISTRIBUTION_CHECK_PATH = (
    REPORT_DIR
    / "13_manifest_vs_physical_distribution.csv"
)

split_df.to_csv(
    COPIED_MANIFEST_PATH,
    index=False
)

physical_inventory_df.to_csv(
    PHYSICAL_INVENTORY_PATH,
    index=False
)

physical_split_summary_df.to_csv(
    PHYSICAL_SPLIT_SUMMARY_PATH,
    index=False
)

distribution_check_df.to_csv(
    DISTRIBUTION_CHECK_PATH,
    index=False
)


# Display results
print("\n" + "=" * 80)
print("CLEAN DATASET COPYING COMPLETED")
print("=" * 80)

display(
    physical_split_summary_df
)

print("\nManifest vs. physical directory verification:")

display(
    distribution_check_df[
        [
            "split",
            "class_name",
            "manifest_images",
            "physical_images",
            "count_difference"
        ]
    ]
)

print("\nClean dataset directory:")
print(CLEAN_DATASET_DIR)

print("\nPhysical manifest:")
print(COPIED_MANIFEST_PATH)

print(
    "\nTotal copied images:",
    f"{len(physical_inventory_df):,}"
)

print(
    "\nNo copying errors occurred."
)

# **Cell 12 — Final integrity check**

In [ ]:
from pathlib import Path
import hashlib

import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True


# Paths
COPIED_MANIFEST_PATH = (
    MANIFEST_DIR
    / "13_clean_copied_dataset_manifest.csv"
)

if not COPIED_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Not found:\n{COPIED_MANIFEST_PATH}"
    )


# Load physical dataset manifest
verified_df = pd.read_csv(
    COPIED_MANIFEST_PATH,
    low_memory=False
)

required_columns = {
    "dataset_id",
    "split",
    "class_name",
    "sha256",
    "duplicate_component_id",
    "split_guard_group_id",
    "copied_image_path"
}

missing_columns = (
    required_columns
    - set(verified_df.columns)
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )

if len(verified_df) != 28348:
    raise ValueError(
        "Expected 28,348 images."
    )


# SHA-256 function
def calculate_sha256(file_path, chunk_size=1024 * 1024):

    hash_object = hashlib.sha256()

    with open(file_path, "rb") as file:

        while True:

            chunk = file.read(chunk_size)

            if not chunk:
                break

            hash_object.update(chunk)

    return hash_object.hexdigest()


# Verify each file
verification_records = []

for row in tqdm(
    verified_df.itertuples(index=False),
    total=len(verified_df),
    desc="Verifying final dataset"
):

    image_path = Path(
        str(row.copied_image_path)
    )

    file_exists = image_path.is_file()
    readable = False
    recomputed_sha256 = None
    verification_error = None
    width = None
    height = None
    image_mode = None

    try:

        if not file_exists:
            raise FileNotFoundError(
                f"Does not exist: {image_path}"
            )

        # Structural verification
        with Image.open(image_path) as image:
            image.verify()

        # Full decoding verification
        with Image.open(image_path) as image:
            image.load()

            width, height = image.size
            image_mode = image.mode

        readable = True

        recomputed_sha256 = calculate_sha256(
            image_path
        )

    except Exception as error:

        verification_error = (
            f"{type(error).__name__}: {error}"
        )

    verification_records.append({
        "dataset_id": row.dataset_id,
        "file_exists": file_exists,
        "readable": readable,
        "expected_sha256": row.sha256,
        "recomputed_sha256": recomputed_sha256,
        "sha256_matches": (
            recomputed_sha256 == row.sha256
            if recomputed_sha256 is not None
            else False
        ),
        "verified_width": width,
        "verified_height": height,
        "verified_mode": image_mode,
        "verification_error": verification_error
    })


verification_df = pd.DataFrame(
    verification_records
)

verified_df = verified_df.merge(
    verification_df,
    on="dataset_id",
    how="left",
    validate="one_to_one"
)


# Identify errors
integrity_errors_df = (
    verified_df[
        (~verified_df["file_exists"])
        | (~verified_df["readable"])
        | (~verified_df["sha256_matches"])
        | verified_df[
            "verification_error"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)


# Check for global duplicates
recomputed_duplicate_count = int(
    verified_df[
        "recomputed_sha256"
    ].duplicated().sum()
)

expected_duplicate_count = int(
    verified_df[
        "sha256"
    ].duplicated().sum()
)


# Check overlap between splits
def overlap_count(dataframe, column, split_a, split_b):

    values_a = set(
        dataframe.loc[
            dataframe["split"] == split_a,
            column
        ]
    )

    values_b = set(
        dataframe.loc[
            dataframe["split"] == split_b,
            column
        ]
    )

    return len(
        values_a & values_b
    )


overlap_records = []

for identifier in [
    "dataset_id",
    "sha256",
    "recomputed_sha256",
    "duplicate_component_id",
    "split_guard_group_id"
]:

    for split_a, split_b in [
        ("train", "val"),
        ("train", "test"),
        ("val", "test")
    ]:

        overlap_records.append({
            "identifier": identifier,
            "split_pair": (
                f"{split_a}_{split_b}"
            ),
            "overlap_count": overlap_count(
                verified_df,
                identifier,
                split_a,
                split_b
            )
        })


final_overlap_df = pd.DataFrame(
    overlap_records
)


# Summary by split
verified_split_summary_df = (
    verified_df
    .groupby(
        "split",
        as_index=False
    )
    .agg(
        images=(
            "dataset_id",
            "size"
        ),
        classes=(
            "class_name",
            "nunique"
        ),
        readable_images=(
            "readable",
            "sum"
        ),
        matching_sha256=(
            "sha256_matches",
            "sum"
        ),
        unique_sha256=(
            "recomputed_sha256",
            "nunique"
        ),
        guard_groups=(
            "split_guard_group_id",
            "nunique"
        )
    )
)

split_order = {
    "train": 0,
    "val": 1,
    "test": 2
}

verified_split_summary_df[
    "split_order"
] = verified_split_summary_df[
    "split"
].map(split_order)

verified_split_summary_df = (
    verified_split_summary_df
    .sort_values("split_order")
    .drop(columns="split_order")
    .reset_index(drop=True)
)


# Final summary
final_integrity_summary_df = pd.DataFrame({
    "metric": [
        "total_images",
        "existing_files",
        "readable_images",
        "matching_sha256_files",
        "integrity_errors",
        "unique_expected_sha256",
        "unique_recomputed_sha256",
        "expected_duplicate_sha256",
        "recomputed_duplicate_sha256",
        "number_of_classes",
        "number_of_splits",
        "nonzero_overlap_checks",
        "ready_for_training"
    ],
    "value": [
        len(verified_df),
        int(
            verified_df[
                "file_exists"
            ].sum()
        ),
        int(
            verified_df[
                "readable"
            ].sum()
        ),
        int(
            verified_df[
                "sha256_matches"
            ].sum()
        ),
        len(integrity_errors_df),
        verified_df[
            "sha256"
        ].nunique(),
        verified_df[
            "recomputed_sha256"
        ].nunique(),
        expected_duplicate_count,
        recomputed_duplicate_count,
        verified_df[
            "class_name"
        ].nunique(),
        verified_df[
            "split"
        ].nunique(),
        int(
            (
                final_overlap_df[
                    "overlap_count"
                ] > 0
            ).sum()
        ),
        (
            len(integrity_errors_df) == 0
            and expected_duplicate_count == 0
            and recomputed_duplicate_count == 0
            and (
                final_overlap_df[
                    "overlap_count"
                ] == 0
            ).all()
        )
    ]
})


# Strict validation
ready_for_training = bool(
    final_integrity_summary_df.loc[
        final_integrity_summary_df[
            "metric"
        ] == "ready_for_training",
        "value"
    ].iloc[0]
)

if not ready_for_training:

    print("Integrity issues were detected.")

    display(
        integrity_errors_df.head(30)
    )

    display(
        final_overlap_df[
            final_overlap_df[
                "overlap_count"
            ] > 0
        ]
    )

    raise RuntimeError(
        "The dataset is not yet ready "
        "for training."
    )


# Save final results
FINAL_VERIFIED_MANIFEST_PATH = (
    MANIFEST_DIR
    / "14_final_verified_dataset_manifest.csv"
)

FINAL_INTEGRITY_SUMMARY_PATH = (
    REPORT_DIR
    / "14_final_integrity_summary.csv"
)

FINAL_INTEGRITY_ERRORS_PATH = (
    REPORT_DIR
    / "14_final_integrity_errors.csv"
)

FINAL_OVERLAP_PATH = (
    REPORT_DIR
    / "14_final_overlap_verification.csv"
)

FINAL_SPLIT_SUMMARY_PATH = (
    REPORT_DIR
    / "14_final_verified_split_summary.csv"
)

READY_FLAG_PATH = (
    WORK_ROOT
    / "DATASET_READY_FOR_TRAINING.txt"
)

verified_df.to_csv(
    FINAL_VERIFIED_MANIFEST_PATH,
    index=False
)

final_integrity_summary_df.to_csv(
    FINAL_INTEGRITY_SUMMARY_PATH,
    index=False
)

integrity_errors_df.to_csv(
    FINAL_INTEGRITY_ERRORS_PATH,
    index=False
)

final_overlap_df.to_csv(
    FINAL_OVERLAP_PATH,
    index=False
)

verified_split_summary_df.to_csv(
    FINAL_SPLIT_SUMMARY_PATH,
    index=False
)

READY_FLAG_PATH.write_text(
    "Dataset verified and ready for training.\n"
    "Images: 28348\n"
    "Train: 22672\n"
    "Validation: 2835\n"
    "Test: 2841\n"
    "Classes: 24\n"
    "Integrity errors: 0\n"
    "Cross-split overlap: 0\n",
    encoding="utf-8"
)


# Display results
print("\n" + "=" * 80)
print("FINAL DATASET VERIFICATION")
print("=" * 80)

display(
    final_integrity_summary_df
)

print("\nSummary by split:")
display(
    verified_split_summary_df
)

print("\nFinal overlap:")
display(
    final_overlap_df
)

print("\nVerified manifest:")
print(FINAL_VERIFIED_MANIFEST_PATH)

print("\nReadiness indicator:")
print(READY_FLAG_PATH)

print(
    "\nTHE DATASET IS READY FOR THE "
    "NINE TRAINING RUNS."
)

In [ ]:
# Create separate ZIP archives for downloading from Kaggle

from pathlib import Path
import zipfile
from tqdm.auto import tqdm

# Main directory
SOURCE_ROOT = Path(
    "/kaggle/working/fruit_dataset_clean_rebuild"
)

# Directory where the ZIP archives will be saved
ZIP_OUTPUT_DIR = Path(
    "/kaggle/working/download_zips"
)

ZIP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Directories to compress separately
FOLDERS_TO_ZIP = [
    "clean_dataset",
    "hashes",
    "manifests",
    "reports",
    "splits",
    "visual_review"
]


def create_folder_zip(
    source_folder,
    output_zip
):
    """
    Compresses a folder while preserving its name and structure.
    Uses ZIP64 to support ZIP archives larger than 4 GB.
    """

    source_folder = Path(source_folder)
    output_zip = Path(output_zip)

    if not source_folder.exists():
        print(f"Does not exist, skipping: {source_folder}")
        return

    files = [
        path
        for path in source_folder.rglob("*")
        if path.is_file()
    ]

    if output_zip.exists():
        output_zip.unlink()

    print(
        f"\nCompressing {source_folder.name}: "
        f"{len(files):,} files"
    )

    with zipfile.ZipFile(
        output_zip,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=1,
        allowZip64=True
    ) as zip_file:

        for file_path in tqdm(
            files,
            desc=source_folder.name
        ):

            # Preserve the directory structure inside the ZIP:
            # clean_dataset/train/..., etc.
            archive_name = file_path.relative_to(
                source_folder.parent
            )

            zip_file.write(
                file_path,
                arcname=archive_name
            )

    print(
        f"Created: {output_zip.name} "
        f"({output_zip.stat().st_size / (1024**3):.2f} GB)"
    )


# Create one ZIP archive per directory
for folder_name in FOLDERS_TO_ZIP:

    source_folder = (
        SOURCE_ROOT
        / folder_name
    )

    output_zip = (
        ZIP_OUTPUT_DIR
        / f"{folder_name}.zip"
    )

    create_folder_zip(
        source_folder,
        output_zip
    )


# Compress loose files in the main directory
root_files = [
    path
    for path in SOURCE_ROOT.iterdir()
    if path.is_file()
]

ROOT_FILES_ZIP = (
    ZIP_OUTPUT_DIR
    / "root_files_and_ready_flag.zip"
)

if ROOT_FILES_ZIP.exists():
    ROOT_FILES_ZIP.unlink()

with zipfile.ZipFile(
    ROOT_FILES_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True
) as zip_file:

    for file_path in root_files:

        zip_file.write(
            file_path,
            arcname=file_path.name
        )


# Display the created ZIP archives
zip_files = sorted(
    ZIP_OUTPUT_DIR.glob("*.zip")
)

print("\n" + "=" * 80)
print("ZIP ARCHIVES CREATED SUCCESSFULLY")
print("=" * 80)

total_size = 0

for zip_path in zip_files:

    size_gb = (
        zip_path.stat().st_size
        / (1024 ** 3)
    )

    total_size += zip_path.stat().st_size

    print(
        f"{zip_path.name:<40} "
        f"{size_gb:>8.3f} GB"
    )

print(
    "\nTotal size:",
    f"{total_size / (1024 ** 3):.2f} GB"
)

print("\nDownload directory:")
print(ZIP_OUTPUT_DIR)


# **TRAINING**

# **Cell 1 — Locate the dataset and define the 12 configurations**

In [ ]:
from pathlib import Path
import json
import os
import platform
import random
import sys

import numpy as np
import pandas as pd

import torch
import torchvision

try:
    import timm
except ImportError as error:
    raise ImportError(
        "The timm library was not found. "
        "Run in a separate cell: %pip install timm"
    ) from error


# General reproducibility configuration
SEED = 42

IMG_SIZE = 224
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64

NUM_WORKERS = min(
    4,
    os.cpu_count() or 2
)

MAX_EPOCHS = 20
WARMUP_EPOCHS = 2

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.05

LABEL_SMOOTHING = 0.10
MIXUP_ALPHA = 0.40
CUTMIX_ALPHA = 1.00

EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-4

PRIMARY_METRIC = "val_macro_f1"

NUM_CLASSES_EXPECTED = 24
TOTAL_IMAGES_EXPECTED = 28348

EXPECTED_SPLIT_COUNTS = {
    "train": 22672,
    "val": 2835,
    "test": 2841
}


def set_global_seed(seed):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(SEED)


# Explicitly define dataset paths

DATASET_ROOT = Path(
    "/kaggle/input/datasets/"
    "andreamencotovar/"
    "respuesta-a-revisores-2026"
)

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        "The dataset was not found at:\n"
        f"{DATASET_ROOT}"
    )

# The ZIP archives were created including the main folder,
# so Kaggle displays a nested folder with the same name.

CLEAN_DATASET_DIR = (
    DATASET_ROOT
    / "clean_dataset"
    / "clean_dataset"
)

MANIFEST_DIR_INPUT = (
    DATASET_ROOT
    / "manifests"
    / "manifests"
)

SPLIT_DIR_INPUT = (
    DATASET_ROOT
    / "splits"
    / "splits"
)

REPORT_DIR_INPUT = (
    DATASET_ROOT
    / "reports"
    / "reports"
)

HASH_DIR_INPUT = (
    DATASET_ROOT
    / "hashes"
    / "hashes"
)

VISUAL_REVIEW_DIR_INPUT = (
    DATASET_ROOT
    / "visual_review"
    / "visual_review"
)

READY_FLAG_PATH_INPUT = (
    DATASET_ROOT
    / "root_files_and_ready_flag"
    / "DATASET_READY_FOR_TRAINING.txt"
)

# Verify required paths

required_input_paths = {
    "dataset_root": DATASET_ROOT,
    "clean_dataset": CLEAN_DATASET_DIR,
    "manifests": MANIFEST_DIR_INPUT,
    "splits": SPLIT_DIR_INPUT,
    "reports": REPORT_DIR_INPUT,
    "ready_flag": READY_FLAG_PATH_INPUT
}

missing_input_paths = {
    name: path
    for name, path in required_input_paths.items()
    if not path.exists()
}

if missing_input_paths:

    print("Paths that were not found:")

    for name, path in missing_input_paths.items():
        print(f" - {name}: {path}")

    raise FileNotFoundError(
        "The dataset structure does not match "
        "the expected paths."
    )

print("=" * 80)
print("DATASET PATHS LOCATED")
print("=" * 80)

for name, path in required_input_paths.items():
    print(f"{name:<16}: {path}")

# Verify the physical dataset structure

expected_split_directories = [
    CLEAN_DATASET_DIR / "train",
    CLEAN_DATASET_DIR / "val",
    CLEAN_DATASET_DIR / "test"
]

missing_split_directories = [
    path
    for path in expected_split_directories
    if not path.is_dir()
]

if missing_split_directories:

    raise FileNotFoundError(
        "The physical directories were not found: "
        "train, val, and test:\n"
        + "\n".join(
            map(str, missing_split_directories)
        )
    )

print("\nPhysical directories found:")

for split_directory in expected_split_directories:

    class_directories = [
        path
        for path in split_directory.iterdir()
        if path.is_dir()
    ]

    print(
        f" - {split_directory.name}: "
        f"{len(class_directories)} clases"
    )

# Display available files

print("\nAvailable manifests:")

for path in sorted(
    MANIFEST_DIR_INPUT.glob("*.csv")
):
    print(" -", path.name)

print("\nAvailable splits:")

for path in sorted(
    SPLIT_DIR_INPUT.glob("*.csv")
):
    print(" -", path.name)

print("\nReadiness flag:")

print(
    READY_FLAG_PATH_INPUT.read_text(
        encoding="utf-8"
    )
)

# Select the verified manifest
manifest_candidates = [
    (
        MANIFEST_DIR_INPUT
        / "14_final_verified_dataset_manifest.csv"
    ),
    (
        MANIFEST_DIR_INPUT
        / "13_clean_copied_dataset_manifest.csv"
    ),
    (
        MANIFEST_DIR_INPUT
        / "12_clean_grouped_split_manifest.csv"
    )
]

FINAL_MANIFEST_PATH = next(
    (
        path
        for path in manifest_candidates
        if path.exists()
    ),
    None
)

if FINAL_MANIFEST_PATH is None:

    raise FileNotFoundError(
        "None of the expected final manifests "
        "was found in:\n"
        f"{MANIFEST_DIR_INPUT}"
    )


# Load and reconstruct image paths
dataset_df = pd.read_csv(
    FINAL_MANIFEST_PATH,
    low_memory=False
)

required_manifest_columns = {
    "dataset_id",
    "split",
    "class_name",
    "sha256",
    "duplicate_component_id",
    "split_guard_group_id"
}

missing_manifest_columns = (
    required_manifest_columns
    - set(dataset_df.columns)
)

if missing_manifest_columns:

    raise ValueError(
        "Required columns are missing from the manifest: "
        f"{sorted(missing_manifest_columns)}"
    )


def reconstruct_input_path(row):

    # Preferred path: the relative path created
    # during dataset reconstruction.
    if (
        "clean_relative_path" in dataset_df.columns
        and pd.notna(row.get("clean_relative_path"))
    ):

        return str(
            CLEAN_DATASET_DIR
            / str(row["clean_relative_path"])
        )

    # Alternative using the split, class, and clean filename.
    if (
        "clean_file_name" in dataset_df.columns
        and pd.notna(row.get("clean_file_name"))
    ):

        return str(
            CLEAN_DATASET_DIR
            / str(row["split"])
            / str(row["class_name"])
            / str(row["clean_file_name"])
        )

    raise ValueError(
        "The manifest contains neither clean_relative_path "
        "nor clean_file_name."
    )


dataset_df[
    "input_image_path"
] = dataset_df.apply(
    reconstruct_input_path,
    axis=1
)

dataset_df[
    "input_file_exists"
] = dataset_df[
    "input_image_path"
].apply(
    lambda path: Path(path).is_file()
)


# Strict dataset validation
if len(dataset_df) != TOTAL_IMAGES_EXPECTED:

    raise ValueError(
        "Incorrect total image count.\n"
        f"Esperada: {TOTAL_IMAGES_EXPECTED:,}\n"
        f"Encontrada: {len(dataset_df):,}"
    )

if dataset_df[
    "class_name"
].nunique() != NUM_CLASSES_EXPECTED:

    raise ValueError(
        "The manifest does not contain the "
        f"{NUM_CLASSES_EXPECTED} clases esperadas."
    )

if not dataset_df[
    "input_file_exists"
].all():

    missing_files_df = dataset_df[
        ~dataset_df[
            "input_file_exists"
        ]
    ]

    display(
        missing_files_df[
            [
                "dataset_id",
                "split",
                "class_name",
                "input_image_path"
            ]
        ].head(20)
    )

    raise FileNotFoundError(
        f"No se encontraron "
        f"{len(missing_files_df):,} imágenes."
    )

if dataset_df[
    "dataset_id"
].duplicated().any():

    raise ValueError(
        "Duplicate dataset_id values exist."
    )

if dataset_df[
    "sha256"
].duplicated().any():

    raise ValueError(
        "Duplicate SHA-256 values exist."
    )

if dataset_df[
    "duplicate_component_id"
].duplicated().any():

    raise ValueError(
        "There is more than one image per confirmed "
        "component."
    )

observed_split_counts = (
    dataset_df[
        "split"
    ]
    .value_counts()
    .to_dict()
)

if observed_split_counts != EXPECTED_SPLIT_COUNTS:

    raise ValueError(
        "The image counts per split do not match.\n"
        f"Esperadas: {EXPECTED_SPLIT_COUNTS}\n"
        f"Encontradas: {observed_split_counts}"
    )

classes_per_split = (
    dataset_df
    .groupby("split")[
        "class_name"
    ]
    .nunique()
)

if not (
    classes_per_split
    == NUM_CLASSES_EXPECTED
).all():

    raise ValueError(
        "At least one split does not contain all 24 classes:\n"
        f"{classes_per_split}"
    )


# Stable class mapping
CLASS_NAMES = sorted(
    dataset_df[
        "class_name"
    ].unique()
)

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name
    in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    index: class_name
    for class_name, index
    in CLASS_TO_INDEX.items()
}

dataset_df[
    "target"
] = dataset_df[
    "class_name"
].map(
    CLASS_TO_INDEX
)

if dataset_df[
    "target"
].isna().any():

    raise ValueError(
        "Could not assign all class "
        "indices."
    )

dataset_df[
    "target"
] = dataset_df[
    "target"
].astype(int)


# Models pretrained on ImageNet-1K
MODEL_SPECS = {
    "vit_b16": {
        "timm_model": (
            "vit_base_patch16_224.augreg_in1k"
        ),
        "architecture_family": "transformer",
        "explainability_method": (
            "transformer_gradcam_or_attention_rollout"
        )
    },

    "swin_tiny": {
        "timm_model": (
            "swin_tiny_patch4_window7_224.ms_in1k"
        ),
        "architecture_family": "transformer",
        "explainability_method": (
            "swin_gradcam"
        )
    },

    "convnext_tiny": {
        "timm_model": (
            "convnext_tiny.fb_in1k"
        ),
        "architecture_family": "cnn",
        "explainability_method": "gradcam"
    },

    "mobilenet_v2": {
        "timm_model": (
            "mobilenetv2_100.ra_in1k"
        ),
        "architecture_family": "lightweight_cnn",
        "explainability_method": "gradcam"
    }
}


# Check availability in timm
available_pretrained_models = set(
    timm.list_models(
        pretrained=True
    )
)

model_availability_records = []

for model_key, model_spec in (
    MODEL_SPECS.items()
):

    timm_model_name = model_spec[
        "timm_model"
    ]

    model_availability_records.append({
        "model_key": model_key,
        "timm_model": timm_model_name,
        "available_in_installed_timm": (
            timm_model_name
            in available_pretrained_models
        )
    })


model_availability_df = pd.DataFrame(
    model_availability_records
)

if not model_availability_df[
    "available_in_installed_timm"
].all():

    display(model_availability_df)

    unavailable_models = model_availability_df.loc[
        ~model_availability_df[
            "available_in_installed_timm"
        ],
        "timm_model"
    ].tolist()

    raise RuntimeError(
        "Some models are not available in the "
        "installed timm version:\n"
        + "\n".join(unavailable_models)
    )


# Define the 12 configurations
AUGMENTATION_SPECS = {
    "none": {
        "mixup_alpha": 0.0,
        "cutmix_alpha": 0.0
    },

    "mixup": {
        "mixup_alpha": MIXUP_ALPHA,
        "cutmix_alpha": 0.0
    },

    "cutmix": {
        "mixup_alpha": 0.0,
        "cutmix_alpha": CUTMIX_ALPHA
    }
}


experiment_records = []

configuration_number = 1

for model_key, model_spec in (
    MODEL_SPECS.items()
):

    for augmentation_name, augmentation_spec in (
        AUGMENTATION_SPECS.items()
    ):

        configuration_id = (
            f"{model_key}__{augmentation_name}"
        )

        experiment_records.append({
            "configuration_order": (
                configuration_number
            ),
            "configuration_id": (
                configuration_id
            ),
            "model_key": model_key,
            "timm_model": model_spec[
                "timm_model"
            ],
            "architecture_family": model_spec[
                "architecture_family"
            ],
            "augmentation": augmentation_name,
            "mixup_alpha": augmentation_spec[
                "mixup_alpha"
            ],
            "cutmix_alpha": augmentation_spec[
                "cutmix_alpha"
            ],
            "image_size": IMG_SIZE,
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "label_smoothing": LABEL_SMOOTHING,
            "seed": SEED,
            "selection_metric": PRIMARY_METRIC,
            "status": "pending"
        })

        configuration_number += 1


experiment_plan_df = pd.DataFrame(
    experiment_records
)

if len(experiment_plan_df) != 12:

    raise RuntimeError(
        "The experimental plan does not contain "
        "the 12 configurations."
    )


# Create the results directory structure
EXPERIMENT_ROOT = Path(
    "/kaggle/working/fruit_training_12_configs"
)

DIRECTORIES = {
    "checkpoints": (
        EXPERIMENT_ROOT
        / "checkpoints"
    ),
    "histories": (
        EXPERIMENT_ROOT
        / "histories"
    ),
    "predictions": (
        EXPERIMENT_ROOT
        / "predictions"
    ),
    "metrics": (
        EXPERIMENT_ROOT
        / "metrics"
    ),
    "calibration": (
        EXPERIMENT_ROOT
        / "calibration"
    ),
    "confusion_matrices": (
        EXPERIMENT_ROOT
        / "confusion_matrices"
    ),
    "efficiency": (
        EXPERIMENT_ROOT
        / "efficiency"
    ),
    "explainability": (
        EXPERIMENT_ROOT
        / "explainability"
    ),
    "statistics": (
        EXPERIMENT_ROOT
        / "statistics"
    ),
    "learning_curves": (
        EXPERIMENT_ROOT
        / "learning_curves"
    ),
    "logs": (
        EXPERIMENT_ROOT
        / "logs"
    ),
    "metadata": (
        EXPERIMENT_ROOT
        / "metadata"
    )
}

for directory in DIRECTORIES.values():

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# Save the runtime manifest
RUNTIME_MANIFEST_PATH = (
    DIRECTORIES["metadata"]
    / "runtime_dataset_manifest.csv"
)

EXPERIMENT_PLAN_PATH = (
    DIRECTORIES["metadata"]
    / "experiment_plan_12_configs.csv"
)

CLASS_MAPPING_PATH = (
    DIRECTORIES["metadata"]
    / "class_mapping.json"
)

GLOBAL_CONFIG_PATH = (
    DIRECTORIES["metadata"]
    / "global_training_config.json"
)

dataset_df.to_csv(
    RUNTIME_MANIFEST_PATH,
    index=False
)

experiment_plan_df.to_csv(
    EXPERIMENT_PLAN_PATH,
    index=False
)

with open(
    CLASS_MAPPING_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            "class_to_index": CLASS_TO_INDEX,
            "index_to_class": INDEX_TO_CLASS
        },
        file,
        indent=2,
        ensure_ascii=False
    )


global_config = {
    "seed": SEED,
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "max_epochs": MAX_EPOCHS,
    "warmup_epochs": WARMUP_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "mixup_alpha": MIXUP_ALPHA,
    "cutmix_alpha": CUTMIX_ALPHA,
    "early_stopping_patience": (
        EARLY_STOPPING_PATIENCE
    ),
    "primary_metric": PRIMARY_METRIC,
    "number_of_classes": (
        NUM_CLASSES_EXPECTED
    ),
    "number_of_configurations": 12,
    "dataset_root": str(DATASET_ROOT),
    "manifest_path": str(
        FINAL_MANIFEST_PATH
    )
}

with open(
    GLOBAL_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        global_config,
        file,
        indent=2,
        ensure_ascii=False
    )


# Environment information
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

environment_df = pd.DataFrame({
    "item": [
        "python",
        "platform",
        "torch",
        "torchvision",
        "timm",
        "cuda_available",
        "cuda_version",
        "device",
        "gpu_name",
        "dataset_root",
        "manifest",
        "output_root"
    ],
    "value": [
        sys.version.split()[0],
        platform.platform(),
        torch.__version__,
        torchvision.__version__,
        timm.__version__,
        torch.cuda.is_available(),
        torch.version.cuda,
        str(device),
        (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "CPU"
        ),
        str(DATASET_ROOT),
        str(FINAL_MANIFEST_PATH),
        str(EXPERIMENT_ROOT)
    ]
})

ENVIRONMENT_PATH = (
    DIRECTORIES["metadata"]
    / "environment.csv"
)

environment_df.to_csv(
    ENVIRONMENT_PATH,
    index=False
)


# Dataset summary
dataset_summary_df = (
    dataset_df
    .groupby(
        "split",
        as_index=False
    )
    .agg(
        images=(
            "dataset_id",
            "size"
        ),
        classes=(
            "class_name",
            "nunique"
        ),
        unique_sha256=(
            "sha256",
            "nunique"
        ),
        split_guard_groups=(
            "split_guard_group_id",
            "nunique"
        )
    )
)

split_order = {
    "train": 0,
    "val": 1,
    "test": 2
}

dataset_summary_df[
    "order"
] = dataset_summary_df[
    "split"
].map(split_order)

dataset_summary_df = (
    dataset_summary_df
    .sort_values("order")
    .drop(columns="order")
    .reset_index(drop=True)
)


# Display results
print("=" * 80)
print("ENVIRONMENT AND EXPERIMENTAL PLAN CONFIGURED")
print("=" * 80)

print("\nEnvironment:")
display(environment_df)

print("\nDataset located:")
display(dataset_summary_df)

print("\nModel availability:")
display(model_availability_df)

print("\nPlan of 12 configurations:")
display(
    experiment_plan_df[
        [
            "configuration_order",
            "configuration_id",
            "timm_model",
            "augmentation",
            "mixup_alpha",
            "cutmix_alpha",
            "status"
        ]
    ]
)

print("\nClass mapping:")
display(
    pd.DataFrame({
        "class_index": list(
            INDEX_TO_CLASS.keys()
        ),
        "class_name": list(
            INDEX_TO_CLASS.values()
        )
    })
)

print("\nRuntime manifest:")
print(RUNTIME_MANIFEST_PATH)

print("\nExperimental plan:")
print(EXPERIMENT_PLAN_PATH)

print("\nResults directory:")
print(EXPERIMENT_ROOT)

# **Cell 2 — Dataset, transformations, balancing, and DataLoaders**

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd

import torch
from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from PIL import Image, ImageOps, ImageFile

from torchvision import transforms
from torchvision.transforms import InterpolationMode


ImageFile.LOAD_TRUNCATED_IMAGES = True


# Data pipeline configuration
IMAGENET_MEAN = (
    0.485,
    0.456,
    0.406
)

IMAGENET_STD = (
    0.229,
    0.224,
    0.225
)

EVAL_RESIZE_SIZE = int(
    round(
        IMG_SIZE * 1.15
    )
)

RANDOM_ERASING_PROBABILITY = 0.25

CLASS_BALANCED_BETA = 0.9999

# Epochs 0 and 1:
# standard random sampling.
# From epoch 2 onward:
# WeightedRandomSampler.
REWEIGHT_AFTER_EPOCHS = 2


# Common transformations
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=IMG_SIZE,
        scale=(
            0.80,
            1.00
        ),
        ratio=(
            0.75,
            4.0 / 3.0
        ),
        interpolation=(
            InterpolationMode.BICUBIC
        ),
        antialias=True
    ),

    transforms.RandomHorizontalFlip(
        p=0.50
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ),

    transforms.RandomErasing(
        p=RANDOM_ERASING_PROBABILITY,
        scale=(
            0.02,
            0.20
        ),
        ratio=(
            0.30,
            3.30
        ),
        value="random",
        inplace=False
    )
])


evaluation_transform = transforms.Compose([
    transforms.Resize(
        size=EVAL_RESIZE_SIZE,
        interpolation=(
            InterpolationMode.BICUBIC
        ),
        antialias=True
    ),

    transforms.CenterCrop(
        size=IMG_SIZE
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# Custom dataset
class FruitCleanDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform

        required_columns = {
            "dataset_id",
            "input_image_path",
            "class_name",
            "target",
            "split",
            "sha256",
            "split_guard_group_id"
        }

        missing_columns = (
            required_columns
            - set(self.dataframe.columns)
        )

        if missing_columns:

            raise ValueError(
                "Missing columns required to build "
                "the Dataset:\n"
                f"{sorted(missing_columns)}"
            )


    def __len__(self):

        return len(
            self.dataframe
        )


    @staticmethod
    def open_rgb_image(image_path):

        with Image.open(
            image_path
        ) as image:

            image.load()

            try:

                image = ImageOps.exif_transpose(
                    image
                )

            except Exception:

                image = image.copy()

            if (
                image.mode in {
                    "RGBA",
                    "LA"
                }
                or "transparency"
                in image.info
            ):

                rgba_image = image.convert(
                    "RGBA"
                )

                background = Image.new(
                    mode="RGBA",
                    size=rgba_image.size,
                    color=(
                        255,
                        255,
                        255,
                        255
                    )
                )

                image = Image.alpha_composite(
                    background,
                    rgba_image
                ).convert("RGB")

            else:

                image = image.convert(
                    "RGB"
                )

            return image.copy()


    def __getitem__(self, index):

        row = self.dataframe.iloc[
            index
        ]

        image = self.open_rgb_image(
            row["input_image_path"]
        )

        if self.transform is not None:

            image = self.transform(
                image
            )

        return {
            "image": image,

            "target": torch.tensor(
                int(row["target"]),
                dtype=torch.long
            ),

            "dataset_id": torch.tensor(
                int(row["dataset_id"]),
                dtype=torch.long
            ),

            "class_name": str(
                row["class_name"]
            ),

            "image_path": str(
                row["input_image_path"]
            ),

            "sha256": str(
                row["sha256"]
            ),

            "split_guard_group_id": str(
                row[
                    "split_guard_group_id"
                ]
            )
        }


# Separate the three subsets
train_dataframe = (
    dataset_df[
        dataset_df["split"]
        == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

val_dataframe = (
    dataset_df[
        dataset_df["split"]
        == "val"
    ]
    .copy()
    .reset_index(drop=True)
)

test_dataframe = (
    dataset_df[
        dataset_df["split"]
        == "test"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(train_dataframe) != 22672:

    raise ValueError(
        "Incorrect training set size."
    )

if len(val_dataframe) != 2835:

    raise ValueError(
        "Incorrect validation set size."
    )

if len(test_dataframe) != 2841:

    raise ValueError(
        "Incorrect test set size."
    )


# Create Dataset objects
train_dataset = FruitCleanDataset(
    dataframe=train_dataframe,
    transform=train_transform
)

val_dataset = FruitCleanDataset(
    dataframe=val_dataframe,
    transform=evaluation_transform
)

test_dataset = FruitCleanDataset(
    dataframe=test_dataframe,
    transform=evaluation_transform
)


# Training class counts
train_class_counts = (
    train_dataframe[
        "target"
    ]
    .value_counts()
    .sort_index()
    .reindex(
        range(NUM_CLASSES_EXPECTED),
        fill_value=0
    )
)

if (
    train_class_counts <= 0
).any():

    raise ValueError(
        "At least one class has no images "
        "in the training set."
    )


# Class-Balanced Loss weights
# Effective Number of Samples:
# (1 - beta) / (1 - beta ^ n_c)

class_count_array = (
    train_class_counts
    .to_numpy(
        dtype=np.float64
    )
)

effective_number = (
    1.0
    - np.power(
        CLASS_BALANCED_BETA,
        class_count_array
    )
)

class_balanced_weights = (
    (1.0 - CLASS_BALANCED_BETA)
    / np.maximum(
        effective_number,
        1e-12
    )
)

# Normalization:
# the average weight is equal to 1.
class_balanced_weights = (
    class_balanced_weights
    / class_balanced_weights.mean()
)

CLASS_BALANCED_WEIGHTS_TENSOR = (
    torch.tensor(
        class_balanced_weights,
        dtype=torch.float32
    )
)


# Sampler weights
# Inverse frequency is used for sampling.
inverse_frequency_weights = (
    1.0
    / class_count_array
)

sample_weights = (
    train_dataframe[
        "target"
    ]
    .map(
        {
            class_index: (
                inverse_frequency_weights[
                    class_index
                ]
            )
            for class_index
            in range(
                NUM_CLASSES_EXPECTED
            )
        }
    )
    .to_numpy(
        dtype=np.float64
    )
)

SAMPLE_WEIGHTS_TENSOR = (
    torch.tensor(
        sample_weights,
        dtype=torch.double
    )
)


# Worker seeds
def seed_worker(worker_id):

    worker_seed = (
        torch.initial_seed()
        % (2 ** 32)
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


# Common DataLoader arguments
def build_loader_kwargs():

    loader_kwargs = {
        "num_workers": NUM_WORKERS,

        "pin_memory": (
            torch.cuda.is_available()
        ),

        "worker_init_fn": seed_worker,

        # Because the training loader is rebuilt
        # each epoch, persistent workers are disabled.
        "persistent_workers": False
    }

    if NUM_WORKERS > 0:

        loader_kwargs[
            "prefetch_factor"
        ] = 2

    return loader_kwargs


# Epoch-specific training DataLoader
def build_train_loader(
    epoch,
    batch_size=BATCH_SIZE
):

    epoch = int(epoch)

    generator = torch.Generator()

    generator.manual_seed(
        SEED + epoch
    )

    use_weighted_sampler = (
        epoch
        >= REWEIGHT_AFTER_EPOCHS
    )

    common_kwargs = (
        build_loader_kwargs()
    )

    if use_weighted_sampler:

        sampler = WeightedRandomSampler(
            weights=(
                SAMPLE_WEIGHTS_TENSOR
            ),

            num_samples=len(
                train_dataset
            ),

            replacement=True,

            generator=generator
        )

        train_loader = DataLoader(
            dataset=train_dataset,

            batch_size=batch_size,

            sampler=sampler,

            shuffle=False,

            drop_last=False,

            generator=generator,

            **common_kwargs
        )

    else:

        train_loader = DataLoader(
            dataset=train_dataset,

            batch_size=batch_size,

            shuffle=True,

            drop_last=False,

            generator=generator,

            **common_kwargs
        )

    return (
        train_loader,
        use_weighted_sampler
    )


# Fixed validation and test DataLoaders
evaluation_generator = (
    torch.Generator()
)

evaluation_generator.manual_seed(
    SEED
)

val_loader = DataLoader(
    dataset=val_dataset,

    batch_size=EVAL_BATCH_SIZE,

    shuffle=False,

    drop_last=False,

    generator=evaluation_generator,

    **build_loader_kwargs()
)


test_loader = DataLoader(
    dataset=test_dataset,

    batch_size=EVAL_BATCH_SIZE,

    shuffle=False,

    drop_last=False,

    generator=evaluation_generator,

    **build_loader_kwargs()
)


# Class balancing summary
class_balance_df = pd.DataFrame({
    "class_index": range(
        NUM_CLASSES_EXPECTED
    ),

    "class_name": [
        INDEX_TO_CLASS[index]
        for index in range(
            NUM_CLASSES_EXPECTED
        )
    ],

    "train_images": (
        train_class_counts.values
    ),

    "class_balanced_loss_weight": (
        class_balanced_weights
    ),

    "inverse_frequency_sampler_weight": (
        inverse_frequency_weights
    )
})


# Save pipeline configuration
CLASS_BALANCE_PATH = (
    DIRECTORIES["metadata"]
    / "class_balance.csv"
)

DATA_PIPELINE_CONFIG_PATH = (
    DIRECTORIES["metadata"]
    / "data_pipeline_config.json"
)

class_balance_df.to_csv(
    CLASS_BALANCE_PATH,
    index=False
)

data_pipeline_config = {
    "image_size": IMG_SIZE,

    "evaluation_resize_size": (
        EVAL_RESIZE_SIZE
    ),

    "imagenet_mean": list(
        IMAGENET_MEAN
    ),

    "imagenet_std": list(
        IMAGENET_STD
    ),

    "random_resized_crop_scale": [
        0.80,
        1.00
    ],

    "horizontal_flip_probability": (
        0.50
    ),

    "random_erasing_probability": (
        RANDOM_ERASING_PROBABILITY
    ),

    "class_balanced_beta": (
        CLASS_BALANCED_BETA
    ),

    "reweight_after_epochs": (
        REWEIGHT_AFTER_EPOCHS
    ),

    "train_batch_size": (
        BATCH_SIZE
    ),

    "evaluation_batch_size": (
        EVAL_BATCH_SIZE
    ),

    "num_workers": (
        NUM_WORKERS
    )
}

with open(
    DATA_PIPELINE_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        data_pipeline_config,
        file,
        indent=2,
        ensure_ascii=False
    )


# Test the DataLoaders
preview_train_loader_epoch_0, sampler_epoch_0 = (
    build_train_loader(
        epoch=0
    )
)

preview_train_loader_epoch_2, sampler_epoch_2 = (
    build_train_loader(
        epoch=2
    )
)

train_preview_batch = next(
    iter(
        preview_train_loader_epoch_0
    )
)

val_preview_batch = next(
    iter(
        val_loader
    )
)

test_preview_batch = next(
    iter(
        test_loader
    )
)


# Validate batch dimensions
expected_image_shape = (
    3,
    IMG_SIZE,
    IMG_SIZE
)

for batch_name, batch in {
    "train": train_preview_batch,
    "val": val_preview_batch,
    "test": test_preview_batch
}.items():

    observed_shape = tuple(
        batch["image"].shape[1:]
    )

    if (
        observed_shape
        != expected_image_shape
    ):

        raise ValueError(
            f"Incorrect dimensions in "
            f"{batch_name}: {observed_shape}"
        )

    targets = batch["target"]

    if targets.min() < 0:

        raise ValueError(
            f"Negative label in {batch_name}."
        )

    if (
        targets.max()
        >= NUM_CLASSES_EXPECTED
    ):

        raise ValueError(
            f"Out-of-range label "
            f"in {batch_name}."
        )


# DataLoader summary
loader_summary_df = pd.DataFrame({
    "split": [
        "train",
        "val",
        "test"
    ],

    "dataset_images": [
        len(train_dataset),
        len(val_dataset),
        len(test_dataset)
    ],

    "batch_size": [
        BATCH_SIZE,
        EVAL_BATCH_SIZE,
        EVAL_BATCH_SIZE
    ],

    "batches": [
        len(
            preview_train_loader_epoch_0
        ),
        len(val_loader),
        len(test_loader)
    ],

    "shuffle_or_sampler": [
        (
            "shuffle epochs 0-1; "
            "weighted sampler from epoch 2"
        ),
        "fixed order",
        "fixed order"
    ]
})


batch_preview_df = pd.DataFrame({
    "split": [
        "train",
        "val",
        "test"
    ],

    "batch_shape": [
        str(
            tuple(
                train_preview_batch[
                    "image"
                ].shape
            )
        ),

        str(
            tuple(
                val_preview_batch[
                    "image"
                ].shape
            )
        ),

        str(
            tuple(
                test_preview_batch[
                    "image"
                ].shape
            )
        )
    ],

    "target_shape": [
        str(
            tuple(
                train_preview_batch[
                    "target"
                ].shape
            )
        ),

        str(
            tuple(
                val_preview_batch[
                    "target"
                ].shape
            )
        ),

        str(
            tuple(
                test_preview_batch[
                    "target"
                ].shape
            )
        )
    ],

    "minimum_target": [
        int(
            train_preview_batch[
                "target"
            ].min()
        ),

        int(
            val_preview_batch[
                "target"
            ].min()
        ),

        int(
            test_preview_batch[
                "target"
            ].min()
        )
    ],

    "maximum_target": [
        int(
            train_preview_batch[
                "target"
            ].max()
        ),

        int(
            val_preview_batch[
                "target"
            ].max()
        ),

        int(
            test_preview_batch[
                "target"
            ].max()
        )
    ]
})


# Display results
print("=" * 80)
print("DATA PIPELINE CONFIGURED")
print("=" * 80)

print("\nDataLoader summary:")
display(
    loader_summary_df
)

print("\nBatch test:")
display(
    batch_preview_df
)

print("\nClass balancing:")
display(
    class_balance_df
)

print("\nSampler verification:")

print(
    "Epoch 0 uses WeightedRandomSampler:",
    sampler_epoch_0
)

print(
    "Epoch 2 uses WeightedRandomSampler:",
    sampler_epoch_2
)

print("\nConfiguration saved to:")
print(DATA_PIPELINE_CONFIG_PATH)

print("\nClass weights saved to:")
print(CLASS_BALANCE_PATH)

# **Cell 3 — Models, losses, MixUp, CutMix, and optimizer**
This cell will download and cache the pre-trained weights for the four architectures.

In [ ]:
from pathlib import Path
import gc
import json
import math

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import timm
from timm.data import Mixup


# Additional optimization settings
ADAMW_BETAS = (
    0.9,
    0.999
)

ADAMW_EPSILON = 1e-8

GRADIENT_CLIP_NORM = 1.0

MIXING_PROBABILITY = 1.0
MIXING_MODE = "batch"

USE_AMP = torch.cuda.is_available()

# torch.compile is disabled to avoid compilation differences
# across architectures and simplify checkpoint resumption.
USE_TORCH_COMPILE = False


# Create a model for an architecture
def create_experiment_model(
    model_key,
    pretrained=True
):
    """
    Creates a timm model with a 24-class classification head.
    """

    if model_key not in MODEL_SPECS:

        raise KeyError(
            f"Unrecognized model: {model_key}"
        )

    timm_model_name = MODEL_SPECS[
        model_key
    ]["timm_model"]

    try:

        model = timm.create_model(
            timm_model_name,
            pretrained=pretrained,
            num_classes=NUM_CLASSES_EXPECTED
        )

    except Exception as error:

        raise RuntimeError(
            "Could not create or download the model:\n"
            f"{timm_model_name}\n\n"
            "Check that Internet access is enabled in "
            "the Kaggle notebook settings.\n\n"
            f"Original error: {type(error).__name__}: {error}"
        ) from error

    return model


# Count model parameters
def count_model_parameters(model):

    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    frozen_parameters = (
        total_parameters
        - trainable_parameters
    )

    return {
        "total_parameters": int(
            total_parameters
        ),
        "trainable_parameters": int(
            trainable_parameters
        ),
        "frozen_parameters": int(
            frozen_parameters
        )
    }


# Weighted loss for soft targets
class WeightedSoftTargetCrossEntropy(nn.Module):
    """
    Cross-entropy for soft targets produced by MixUp or CutMix.

    Each class retains its Class-Balanced Loss weight.
    """

    def __init__(
        self,
        class_weights
    ):

        super().__init__()

        class_weights = torch.as_tensor(
            class_weights,
            dtype=torch.float32
        )

        if class_weights.ndim != 1:

            raise ValueError(
                "class_weights must be a vector."
            )

        if len(class_weights) != NUM_CLASSES_EXPECTED:

            raise ValueError(
                "Incorrect number of class weights."
            )

        self.register_buffer(
            "class_weights",
            class_weights
        )


    def forward(
        self,
        logits,
        soft_targets
    ):

        if logits.ndim != 2:

            raise ValueError(
                "Logits must have shape "
                "[batch, classes]."
            )

        if soft_targets.shape != logits.shape:

            raise ValueError(
                "Soft targets and logits "
                "must have the same shape."
            )

        log_probabilities = F.log_softmax(
            logits,
            dim=1
        )

        weighted_losses = (
            -soft_targets
            * log_probabilities
            * self.class_weights.unsqueeze(0)
        )

        loss_per_sample = weighted_losses.sum(
            dim=1
        )

        return loss_per_sample.mean()


# Build the mixing function
def build_mixing_function(
    augmentation_name
):
    """
    Returns:
    - None for the configuration without mixing;
    - Mixup for MixUp;
    - Mixup configured for CutMix only for CutMix.
    """

    augmentation_name = str(
        augmentation_name
    ).lower()

    if augmentation_name == "none":

        return None

    if augmentation_name == "mixup":

        return Mixup(
            mixup_alpha=MIXUP_ALPHA,
            cutmix_alpha=0.0,
            cutmix_minmax=None,
            prob=MIXING_PROBABILITY,
            switch_prob=0.0,
            mode=MIXING_MODE,
            label_smoothing=LABEL_SMOOTHING,
            num_classes=NUM_CLASSES_EXPECTED
        )

    if augmentation_name == "cutmix":

        return Mixup(
            mixup_alpha=0.0,
            cutmix_alpha=CUTMIX_ALPHA,
            cutmix_minmax=None,
            prob=MIXING_PROBABILITY,
            switch_prob=1.0,
            mode=MIXING_MODE,
            label_smoothing=LABEL_SMOOTHING,
            num_classes=NUM_CLASSES_EXPECTED
        )

    raise ValueError(
        "Unknown augmentation strategy: "
        f"{augmentation_name}"
    )


# Build the loss function
def build_criterion(
    augmentation_name,
    device
):
    """
    None:
        Weighted CrossEntropyLoss with label smoothing.

    MixUp/CutMix:
        Weighted cross-entropy for soft targets.
        Label smoothing is applied within Mixup.
    """

    class_weights_device = (
        CLASS_BALANCED_WEIGHTS_TENSOR
        .to(
            device=device,
            dtype=torch.float32
        )
    )

    if augmentation_name == "none":

        criterion = nn.CrossEntropyLoss(
            weight=class_weights_device,
            label_smoothing=LABEL_SMOOTHING
        )

    elif augmentation_name in {
        "mixup",
        "cutmix"
    }:

        criterion = (
            WeightedSoftTargetCrossEntropy(
                class_weights=(
                    class_weights_device
                )
            )
        )

    else:

        raise ValueError(
            "Unknown augmentation: "
            f"{augmentation_name}"
        )

    return criterion.to(device)


# Build the optimizer
def build_optimizer(model):

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        betas=ADAMW_BETAS,
        eps=ADAMW_EPSILON,
        weight_decay=WEIGHT_DECAY
    )

    return optimizer


# Warmup + cosine schedule
def calculate_learning_rate(
    epoch,
    max_epochs=MAX_EPOCHS
):
    """
    Linear warmup for two epochs followed by cosine decay
    over the remaining epochs.

    epoch uses zero-based indexing.
    """

    epoch = int(epoch)
    max_epochs = int(max_epochs)

    if epoch < 0 or epoch >= max_epochs:

        raise ValueError(
            f"Epoch out of range: {epoch}"
        )

    if epoch < WARMUP_EPOCHS:

        warmup_fraction = (
            float(epoch + 1)
            / float(WARMUP_EPOCHS)
        )

        return (
            LEARNING_RATE
            * warmup_fraction
        )

    cosine_epoch_count = (
        max_epochs
        - WARMUP_EPOCHS
    )

    if cosine_epoch_count <= 1:

        return LEARNING_RATE

    cosine_position = (
        float(
            epoch
            - WARMUP_EPOCHS
        )
        / float(
            cosine_epoch_count - 1
        )
    )

    cosine_multiplier = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi
                * cosine_position
            )
        )
    )

    return (
        LEARNING_RATE
        * cosine_multiplier
    )


def set_optimizer_learning_rate(
    optimizer,
    epoch,
    max_epochs=MAX_EPOCHS
):

    current_learning_rate = (
        calculate_learning_rate(
            epoch=epoch,
            max_epochs=max_epochs
        )
    )

    for parameter_group in (
        optimizer.param_groups
    ):

        parameter_group["lr"] = (
            current_learning_rate
        )

    return current_learning_rate


# Learning rate schedule table
learning_rate_schedule_df = pd.DataFrame({
    "epoch": np.arange(
        1,
        MAX_EPOCHS + 1
    ),

    "epoch_index": np.arange(
        0,
        MAX_EPOCHS
    ),

    "learning_rate": [
        calculate_learning_rate(
            epoch_index
        )
        for epoch_index in range(
            MAX_EPOCHS
        )
    ]
})


# Build all components for a configuration
def build_experiment_components(
    experiment_row,
    device
):
    """
    Builds the components required for one run.
    """

    if isinstance(
        experiment_row,
        pd.Series
    ):

        configuration = (
            experiment_row.to_dict()
        )

    else:

        configuration = dict(
            experiment_row
        )

    model_key = configuration[
        "model_key"
    ]

    augmentation_name = configuration[
        "augmentation"
    ]

    model = create_experiment_model(
        model_key=model_key,
        pretrained=True
    )

    model = model.to(device)

    criterion = build_criterion(
        augmentation_name=(
            augmentation_name
        ),
        device=device
    )

    mixing_function = (
        build_mixing_function(
            augmentation_name
        )
    )

    optimizer = build_optimizer(
        model
    )

    return {
        "model": model,
        "criterion": criterion,
        "mixing_function": (
            mixing_function
        ),
        "optimizer": optimizer
    }


# Test MixUp, CutMix, and losses
smoke_images = (
    train_preview_batch["image"]
    .clone()
)

smoke_targets = (
    train_preview_batch["target"]
    .clone()
)

loss_smoke_records = []

for augmentation_name in [
    "none",
    "mixup",
    "cutmix"
]:

    mixing_function = (
        build_mixing_function(
            augmentation_name
        )
    )

    if mixing_function is None:

        processed_images = (
            smoke_images.clone()
        )

        processed_targets = (
            smoke_targets.clone()
        )

        target_type = (
            "hard_labels"
        )

        expected_target_shape = (
            len(smoke_targets),
        )

    else:

        processed_images, processed_targets = (
            mixing_function(
                smoke_images.clone(),
                smoke_targets.clone()
            )
        )

        target_type = (
            "soft_labels"
        )

        expected_target_shape = (
            len(smoke_targets),
            NUM_CLASSES_EXPECTED
        )

    if tuple(
        processed_targets.shape
    ) != expected_target_shape:

        raise ValueError(
            "Incorrect target shape for "
            f"{augmentation_name}: "
            f"{tuple(processed_targets.shape)}"
        )

    random_logits = torch.randn(
        len(smoke_targets),
        NUM_CLASSES_EXPECTED,
        device=device
    )

    criterion = build_criterion(
        augmentation_name=(
            augmentation_name
        ),
        device=device
    )

    processed_targets_device = (
        processed_targets.to(device)
    )

    smoke_loss = criterion(
        random_logits,
        processed_targets_device
    )

    if not torch.isfinite(
        smoke_loss
    ):

        raise RuntimeError(
            "Loss is not finite for "
            f"{augmentation_name}."
        )

    if augmentation_name == "none":

        target_row_sum_min = 1.0
        target_row_sum_max = 1.0

    else:

        target_row_sums = (
            processed_targets.sum(
                dim=1
            )
        )

        target_row_sum_min = float(
            target_row_sums.min()
        )

        target_row_sum_max = float(
            target_row_sums.max()
        )

        if not torch.allclose(
            target_row_sums,
            torch.ones_like(
                target_row_sums
            ),
            atol=1e-5
        ):

            raise RuntimeError(
                "Soft targets do not sum to 1 "
                f"para {augmentation_name}."
            )

    loss_smoke_records.append({
        "augmentation": (
            augmentation_name
        ),
        "target_type": target_type,
        "image_shape": str(
            tuple(
                processed_images.shape
            )
        ),
        "target_shape": str(
            tuple(
                processed_targets.shape
            )
        ),
        "target_row_sum_min": (
            target_row_sum_min
        ),
        "target_row_sum_max": (
            target_row_sum_max
        ),
        "smoke_loss": float(
            smoke_loss.detach().cpu()
        ),
        "finite_loss": bool(
            torch.isfinite(
                smoke_loss
            ).item()
        )
    })


loss_smoke_test_df = pd.DataFrame(
    loss_smoke_records
)


# Download and test the four architectures
architecture_records = []

model_smoke_images = (
    train_preview_batch[
        "image"
    ][:2]
    .to(
        device,
        non_blocking=True
    )
)

for model_key, model_spec in (
    MODEL_SPECS.items()
):

    print(
        "\nPreparing model:",
        model_key
    )

    if torch.cuda.is_available():

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = create_experiment_model(
        model_key=model_key,
        pretrained=True
    )

    parameter_counts = (
        count_model_parameters(
            model
        )
    )

    model = model.to(device)
    model.eval()

    with torch.inference_mode():

        model_outputs = model(
            model_smoke_images
        )

    expected_output_shape = (
        2,
        NUM_CLASSES_EXPECTED
    )

    if tuple(
        model_outputs.shape
    ) != expected_output_shape:

        raise RuntimeError(
            f"Incorrect output for {model_key}: "
            f"{tuple(model_outputs.shape)}"
        )

    estimated_parameter_size_mb = (
        parameter_counts[
            "total_parameters"
        ]
        * 4
        / (1024 ** 2)
    )

    if torch.cuda.is_available():

        peak_cuda_memory_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    else:

        peak_cuda_memory_mb = np.nan

    architecture_records.append({
        "model_key": model_key,
        "timm_model": model_spec[
            "timm_model"
        ],
        "architecture_family": (
            model_spec[
                "architecture_family"
            ]
        ),
        "total_parameters": (
            parameter_counts[
                "total_parameters"
            ]
        ),
        "trainable_parameters": (
            parameter_counts[
                "trainable_parameters"
            ]
        ),
        "frozen_parameters": (
            parameter_counts[
                "frozen_parameters"
            ]
        ),
        "estimated_fp32_parameter_size_mb": (
            estimated_parameter_size_mb
        ),
        "smoke_batch_size": 2,
        "output_shape": str(
            tuple(
                model_outputs.shape
            )
        ),
        "peak_cuda_memory_smoke_mb": (
            peak_cuda_memory_mb
        ),
        "pretrained_loaded": True
    })

    del model
    del model_outputs

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


architecture_summary_df = pd.DataFrame(
    architecture_records
)


# Save configuration
ARCHITECTURE_SUMMARY_PATH = (
    DIRECTORIES["metadata"]
    / "architecture_summary.csv"
)

LOSS_SMOKE_TEST_PATH = (
    DIRECTORIES["metadata"]
    / "loss_and_mixing_smoke_test.csv"
)

LEARNING_RATE_SCHEDULE_PATH = (
    DIRECTORIES["metadata"]
    / "learning_rate_schedule.csv"
)

TRAINING_COMPONENTS_CONFIG_PATH = (
    DIRECTORIES["metadata"]
    / "training_components_config.json"
)

architecture_summary_df.to_csv(
    ARCHITECTURE_SUMMARY_PATH,
    index=False
)

loss_smoke_test_df.to_csv(
    LOSS_SMOKE_TEST_PATH,
    index=False
)

learning_rate_schedule_df.to_csv(
    LEARNING_RATE_SCHEDULE_PATH,
    index=False
)

training_components_config = {
    "optimizer": "AdamW",
    "learning_rate": (
        LEARNING_RATE
    ),
    "weight_decay": (
        WEIGHT_DECAY
    ),
    "adamw_betas": list(
        ADAMW_BETAS
    ),
    "adamw_epsilon": (
        ADAMW_EPSILON
    ),
    "gradient_clip_norm": (
        GRADIENT_CLIP_NORM
    ),
    "maximum_epochs": (
        MAX_EPOCHS
    ),
    "warmup_epochs": (
        WARMUP_EPOCHS
    ),
    "scheduler": (
        "linear_warmup_then_cosine"
    ),
    "label_smoothing": (
        LABEL_SMOOTHING
    ),
    "mixup_alpha": (
        MIXUP_ALPHA
    ),
    "cutmix_alpha": (
        CUTMIX_ALPHA
    ),
    "mixing_probability": (
        MIXING_PROBABILITY
    ),
    "mixing_mode": (
        MIXING_MODE
    ),
    "class_balanced_loss": True,
    "class_balanced_beta": (
        CLASS_BALANCED_BETA
    ),
    "weighted_sampler_from_epoch": (
        REWEIGHT_AFTER_EPOCHS
    ),
    "automatic_mixed_precision": (
        USE_AMP
    ),
    "torch_compile": (
        USE_TORCH_COMPILE
    )
}

with open(
    TRAINING_COMPONENTS_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        training_components_config,
        file,
        indent=2,
        ensure_ascii=False
    )


# Display results
print("\n" + "=" * 80)
print("MODELS AND TRAINING COMPONENTS CONFIGURED")
print("=" * 80)

print("\nArchitectures:")
display(
    architecture_summary_df
)

print("\nLoss and mixing test:")
display(
    loss_smoke_test_df
)

print("\nLearning rate schedule:")
display(
    learning_rate_schedule_df
)

print("\nConfiguration summary:")
print(
    json.dumps(
        training_components_config,
        indent=2,
        ensure_ascii=False
    )
)

print("\nSaved files:")
print(ARCHITECTURE_SUMMARY_PATH)
print(LOSS_SMOKE_TEST_PATH)
print(LEARNING_RATE_SCHEDULE_PATH)
print(TRAINING_COMPONENTS_CONFIG_PATH)

print(
    "\nNo training has started yet."
)

# **Cell 4 — Training, validation, and resumption engine**

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import math
import os
import random
import time
import traceback

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from tqdm.auto import tqdm


# Correct the warmup and cosine learning-rate schedule
def calculate_learning_rate(
    epoch,
    max_epochs=MAX_EPOCHS
):
    """
    Linear warmup for WARMUP_EPOCHS.

    A cosine decay follows. The final epoch
    retains a small positive learning rate instead
    of using LR=0 throughout the final epoch.

    epoch uses zero-based indexing.
    """

    epoch = int(epoch)
    max_epochs = int(max_epochs)

    if epoch < 0 or epoch >= max_epochs:
        raise ValueError(
            f"Época fuera de rango: {epoch}"
        )

    if epoch < WARMUP_EPOCHS:

        warmup_fraction = (
            float(epoch + 1)
            / float(WARMUP_EPOCHS)
        )

        return (
            LEARNING_RATE
            * warmup_fraction
        )

    cosine_epoch_count = (
        max_epochs
        - WARMUP_EPOCHS
    )

    cosine_position = (
        float(
            epoch
            - WARMUP_EPOCHS
        )
        / float(
            max(
                cosine_epoch_count,
                1
            )
        )
    )

    cosine_multiplier = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi
                * cosine_position
            )
        )
    )

    return (
        LEARNING_RATE
        * cosine_multiplier
    )


def set_optimizer_learning_rate(
    optimizer,
    epoch,
    max_epochs=MAX_EPOCHS
):

    current_learning_rate = (
        calculate_learning_rate(
            epoch=epoch,
            max_epochs=max_epochs
        )
    )

    for parameter_group in optimizer.param_groups:

        parameter_group["lr"] = (
            current_learning_rate
        )

    return current_learning_rate


corrected_learning_rate_schedule_df = pd.DataFrame({
    "epoch": np.arange(
        1,
        MAX_EPOCHS + 1
    ),

    "epoch_index": np.arange(
        MAX_EPOCHS
    ),

    "learning_rate": [
        calculate_learning_rate(
            epoch
        )
        for epoch in range(
            MAX_EPOCHS
        )
    ]
})

LEARNING_RATE_SCHEDULE_PATH = (
    DIRECTORIES["metadata"]
    / "learning_rate_schedule.csv"
)

corrected_learning_rate_schedule_df.to_csv(
    LEARNING_RATE_SCHEDULE_PATH,
    index=False
)

if (
    corrected_learning_rate_schedule_df[
        "learning_rate"
    ].iloc[-1]
    <= 0
):

    raise RuntimeError(
        "The final epoch still has "
        "a zero learning rate."
    )


# Training engine configuration
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_DEVICE_TYPE = (
    "cuda"
    if DEVICE.type == "cuda"
    else "cpu"
)

AMP_DTYPE = torch.float16

CHECKPOINT_FORMAT_VERSION = 1

TRAINING_ENGINE_CONFIG = {
    "device": str(DEVICE),
    "automatic_mixed_precision": bool(USE_AMP),
    "amp_dtype": str(AMP_DTYPE),
    "gradient_clip_norm": GRADIENT_CLIP_NORM,
    "early_stopping_patience": (
        EARLY_STOPPING_PATIENCE
    ),
    "early_stopping_min_delta": (
        EARLY_STOPPING_MIN_DELTA
    ),
    "selection_metric": PRIMARY_METRIC,
    "save_best_model": True,
    "save_last_model": True,
    "save_resume_checkpoint_each_epoch": True,
    "delete_resume_checkpoint_after_completion": True
}

TRAINING_ENGINE_CONFIG_PATH = (
    DIRECTORIES["metadata"]
    / "training_engine_config.json"
)

with open(
    TRAINING_ENGINE_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        TRAINING_ENGINE_CONFIG,
        file,
        indent=2,
        ensure_ascii=False
    )


# UTC timestamps
def utc_now_iso():

    return (
        datetime.now(timezone.utc)
        .isoformat()
    )


# Atomic saving
def atomic_torch_save(
    payload,
    output_path
):

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temporary_path = (
        output_path.parent
        / f"{output_path.name}.tmp"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    torch.save(
        payload,
        temporary_path
    )

    os.replace(
        temporary_path,
        output_path
    )


def atomic_json_save(
    payload,
    output_path
):

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temporary_path = (
        output_path.parent
        / f"{output_path.name}.tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
            default=str
        )

    os.replace(
        temporary_path,
        output_path
    )


# Random states for checkpoint resumption
def capture_rng_state():

    state = {
        "python_random_state": (
            random.getstate()
        ),

        "numpy_random_state": (
            np.random.get_state()
        ),

        "torch_cpu_rng_state": (
            torch.get_rng_state()
        )
    }

    if torch.cuda.is_available():

        state[
            "torch_cuda_rng_state_all"
        ] = torch.cuda.get_rng_state_all()

    else:

        state[
            "torch_cuda_rng_state_all"
        ] = None

    return state


def restore_rng_state(state):

    if state is None:
        return

    if (
        state.get(
            "python_random_state"
        )
        is not None
    ):

        random.setstate(
            state[
                "python_random_state"
            ]
        )

    if (
        state.get(
            "numpy_random_state"
        )
        is not None
    ):

        np.random.set_state(
            state[
                "numpy_random_state"
            ]
        )

    if (
        state.get(
            "torch_cpu_rng_state"
        )
        is not None
    ):

        torch.set_rng_state(
            state[
                "torch_cpu_rng_state"
            ]
        )

    if (
        torch.cuda.is_available()
        and state.get(
            "torch_cuda_rng_state_all"
        )
        is not None
    ):

        torch.cuda.set_rng_state_all(
            state[
                "torch_cuda_rng_state_all"
            ]
        )


# Classification metrics
def compute_classification_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=np.int64
    )

    y_pred = np.asarray(
        y_pred,
        dtype=np.int64
    )

    if len(y_true) == 0:

        raise ValueError(
            "No samples are available for metric calculation."
        )

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred
            )
        ),

        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(
                    range(
                        NUM_CLASSES_EXPECTED
                    )
                ),
                average="macro",
                zero_division=0
            )
        )
    }


# Shared validation loss function
def build_evaluation_criterion(
    device
):
    """
    The validation loss is identical across all
    12 configurations.

    It does not use MixUp, CutMix, or label smoothing.
    """

    class_weights = (
        CLASS_BALANCED_WEIGHTS_TENSOR
        .to(
            device=device,
            dtype=torch.float32
        )
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=0.0
    )

    return criterion.to(device)


# Gradient scaler
def build_grad_scaler():

    return torch.amp.GradScaler(
        "cuda",
        enabled=(
            USE_AMP
            and DEVICE.type == "cuda"
        )
    )


# Single-epoch training
def train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    mixing_function,
    scaler,
    device,
    epoch_number
):
    """
    For None, training metrics are exact.

    For MixUp and CutMix, training metrics are proxy indicators
    calculated against the dominant class of the
    soft target. Model selection is performed
    exclusively using validation Macro-F1.
    """

    model.train()

    running_loss = 0.0
    processed_samples = 0

    proxy_true_values = []
    proxy_predicted_values = []

    if torch.cuda.is_available():

        torch.cuda.reset_peak_memory_stats(
            device
        )

        torch.cuda.synchronize(
            device
        )

    epoch_start_time = time.perf_counter()

    progress_bar = tqdm(
        train_loader,
        desc=(
            f"Train epoch {epoch_number}"
        ),
        leave=False
    )

    for batch in progress_bar:

        images_cpu = batch["image"]

        hard_targets_cpu = batch[
            "target"
        ]

        if mixing_function is None:

            loss_targets_cpu = (
                hard_targets_cpu
            )

            proxy_targets_cpu = (
                hard_targets_cpu
            )

        else:

            (
                images_cpu,
                loss_targets_cpu
            ) = mixing_function(
                images_cpu,
                hard_targets_cpu
            )

            proxy_targets_cpu = (
                loss_targets_cpu.argmax(
                    dim=1
                )
            )

        images = images_cpu.to(
            device,
            non_blocking=True
        )

        loss_targets = (
            loss_targets_cpu.to(
                device,
                non_blocking=True
            )
        )

        proxy_targets = (
            proxy_targets_cpu.to(
                device,
                non_blocking=True
            )
        )

        batch_size_current = int(
            images.shape[0]
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type=AMP_DEVICE_TYPE,
            dtype=AMP_DTYPE,
            enabled=(
                USE_AMP
                and device.type == "cuda"
            )
        ):

            logits = model(
                images
            )

            loss = criterion(
                logits,
                loss_targets
            )

        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                "A non-finite loss occurred "
                f"en la época {epoch_number}."
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRADIENT_CLIP_NORM
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):

            raise RuntimeError(
                "The gradient norm is non-finite "
                f"en la época {epoch_number}."
            )

        scaler.step(
            optimizer
        )

        scaler.update()

        predictions = logits.argmax(
            dim=1
        )

        running_loss += (
            float(
                loss.detach().cpu()
            )
            * batch_size_current
        )

        processed_samples += (
            batch_size_current
        )

        proxy_true_values.extend(
            proxy_targets
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        proxy_predicted_values.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        progress_bar.set_postfix({
            "loss": (
                running_loss
                / max(
                    processed_samples,
                    1
                )
            )
        })

    if torch.cuda.is_available():

        torch.cuda.synchronize(
            device
        )

    epoch_duration_seconds = (
        time.perf_counter()
        - epoch_start_time
    )

    train_metrics = (
        compute_classification_metrics(
            proxy_true_values,
            proxy_predicted_values
        )
    )

    if torch.cuda.is_available():

        peak_memory_mb = (
            torch.cuda.max_memory_allocated(
                device
            )
            / (1024 ** 2)
        )

    else:

        peak_memory_mb = np.nan

    return {
        "train_loss": (
            running_loss
            / processed_samples
        ),

        "train_accuracy_proxy": (
            train_metrics["accuracy"]
        ),

        "train_balanced_accuracy_proxy": (
            train_metrics[
                "balanced_accuracy"
            ]
        ),

        "train_macro_f1_proxy": (
            train_metrics["macro_f1"]
        ),

        "train_samples": (
            processed_samples
        ),

        "train_seconds": (
            epoch_duration_seconds
        ),

        "train_images_per_second": (
            processed_samples
            / max(
                epoch_duration_seconds,
                1e-12
            )
        ),

        "peak_cuda_memory_mb": (
            peak_memory_mb
        )
    }


# Single-epoch validation
def evaluate_one_epoch(
    model,
    data_loader,
    criterion,
    device,
    description="Validation",
    return_logits=False
):

    model.eval()

    running_loss = 0.0
    processed_samples = 0

    true_values = []
    predicted_values = []

    logits_parts = []
    dataset_id_parts = []

    if torch.cuda.is_available():

        torch.cuda.synchronize(
            device
        )

    evaluation_start_time = (
        time.perf_counter()
    )

    with torch.inference_mode():

        progress_bar = tqdm(
            data_loader,
            desc=description,
            leave=False
        )

        for batch in progress_bar:

            images = batch["image"].to(
                device,
                non_blocking=True
            )

            targets = batch["target"].to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type=AMP_DEVICE_TYPE,
                dtype=AMP_DTYPE,
                enabled=(
                    USE_AMP
                    and device.type == "cuda"
                )
            ):

                logits = model(
                    images
                )

                loss = criterion(
                    logits,
                    targets
                )

            if not torch.isfinite(
                loss
            ):

                raise RuntimeError(
                    "A non-finite "
                    "validation loss occurred."
                )

            predictions = logits.argmax(
                dim=1
            )

            batch_size_current = int(
                images.shape[0]
            )

            running_loss += (
                float(
                    loss.detach().cpu()
                )
                * batch_size_current
            )

            processed_samples += (
                batch_size_current
            )

            true_values.extend(
                targets
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )

            predicted_values.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )

            if return_logits:

                logits_parts.append(
                    logits
                    .detach()
                    .float()
                    .cpu()
                )

                dataset_id_parts.append(
                    batch["dataset_id"]
                    .detach()
                    .cpu()
                )

    if torch.cuda.is_available():

        torch.cuda.synchronize(
            device
        )

    evaluation_duration_seconds = (
        time.perf_counter()
        - evaluation_start_time
    )

    metrics = (
        compute_classification_metrics(
            true_values,
            predicted_values
        )
    )

    result = {
        "loss": (
            running_loss
            / processed_samples
        ),

        "accuracy": (
            metrics["accuracy"]
        ),

        "balanced_accuracy": (
            metrics[
                "balanced_accuracy"
            ]
        ),

        "macro_f1": (
            metrics["macro_f1"]
        ),

        "samples": (
            processed_samples
        ),

        "seconds": (
            evaluation_duration_seconds
        ),

        "images_per_second": (
            processed_samples
            / max(
                evaluation_duration_seconds,
                1e-12
            )
        )
    }

    if return_logits:

        result["logits"] = torch.cat(
            logits_parts,
            dim=0
        )

        result["dataset_ids"] = torch.cat(
            dataset_id_parts,
            dim=0
        )

        result["targets"] = torch.tensor(
            true_values,
            dtype=torch.long
        )

        result["predictions"] = torch.tensor(
            predicted_values,
            dtype=torch.long
        )

    return result


# Move optimizer state to the device
def move_optimizer_state_to_device(
    optimizer,
    device
):

    for optimizer_state in (
        optimizer.state.values()
    ):

        for key, value in (
            optimizer_state.items()
        ):

            if torch.is_tensor(
                value
            ):

                optimizer_state[key] = (
                    value.to(
                        device
                    )
                )


# Checkpoint contents
def create_model_checkpoint_payload(
    configuration,
    model,
    epoch_index,
    best_epoch,
    best_metric,
    checkpoint_type
):

    return {
        "checkpoint_format_version": (
            CHECKPOINT_FORMAT_VERSION
        ),

        "checkpoint_type": (
            checkpoint_type
        ),

        "saved_at_utc": (
            utc_now_iso()
        ),

        "configuration": (
            configuration
        ),

        "epoch_index": int(
            epoch_index
        ),

        "epoch_number": int(
            epoch_index + 1
        ),

        "best_epoch": (
            None
            if best_epoch is None
            else int(best_epoch)
        ),

        "best_val_macro_f1": float(
            best_metric
        ),

        "model_state_dict": (
            model.state_dict()
        ),

        "class_to_index": (
            CLASS_TO_INDEX
        ),

        "index_to_class": (
            INDEX_TO_CLASS
        ),

        "torch_version": (
            torch.__version__
        ),

        "timm_version": (
            timm.__version__
        )
    }


def create_resume_checkpoint_payload(
    configuration,
    model,
    optimizer,
    scaler,
    epoch_index,
    best_epoch,
    best_metric,
    epochs_without_improvement,
    history_records
):

    return {
        "checkpoint_format_version": (
            CHECKPOINT_FORMAT_VERSION
        ),

        "checkpoint_type": (
            "resume"
        ),

        "saved_at_utc": (
            utc_now_iso()
        ),

        "configuration": (
            configuration
        ),

        "epoch_index": int(
            epoch_index
        ),

        "next_epoch_index": int(
            epoch_index + 1
        ),

        "best_epoch": (
            None
            if best_epoch is None
            else int(best_epoch)
        ),

        "best_val_macro_f1": float(
            best_metric
        ),

        "epochs_without_improvement": int(
            epochs_without_improvement
        ),

        "model_state_dict": (
            model.state_dict()
        ),

        "optimizer_state_dict": (
            optimizer.state_dict()
        ),

        "scaler_state_dict": (
            scaler.state_dict()
        ),

        "history_records": (
            history_records
        ),

        "rng_state": (
            capture_rng_state()
        )
    }


# Update the experimental plan
def update_experiment_plan_status(
    configuration_id,
    status,
    **additional_values
):

    plan_df = pd.read_csv(
        EXPERIMENT_PLAN_PATH,
        low_memory=False
    )

    matching_mask = (
        plan_df["configuration_id"]
        == configuration_id
    )

    if matching_mask.sum() != 1:

        raise ValueError(
            "A unique configuration was not found "
            f"para {configuration_id}."
        )

    plan_df.loc[
        matching_mask,
        "status"
    ] = status

    for column, value in (
        additional_values.items()
    ):

        if column not in plan_df.columns:

            plan_df[column] = np.nan

        plan_df.loc[
            matching_mask,
            column
        ] = value

    plan_df.to_csv(
        EXPERIMENT_PLAN_PATH,
        index=False
    )


# Train an experimental configuration
def train_configuration(
    configuration_id,
    force_restart=False
):
    """
    Train one complete configuration.

    Main files:
    - best.pt: best model according to validation Macro-F1.
    - last.pt: model from the last completed epoch.
    - resume.pt: model and optimizer states for resuming.
      Removed when the run finishes successfully.
    - history.csv: epoch-level learning curves.
    """

    matching_rows = (
        experiment_plan_df[
            experiment_plan_df[
                "configuration_id"
            ] == configuration_id
        ]
    )

    if len(matching_rows) != 1:

        raise ValueError(
            "Missing or duplicate configuration: "
            f"{configuration_id}"
        )

    configuration = (
        matching_rows.iloc[0]
        .to_dict()
    )

    configuration["max_epochs"] = int(
        configuration[
            "max_epochs"
        ]
    )

    model_key = str(
        configuration[
            "model_key"
        ]
    )

    augmentation_name = str(
        configuration[
            "augmentation"
        ]
    )

    checkpoint_directory = (
        DIRECTORIES["checkpoints"]
        / configuration_id
    )

    checkpoint_directory.mkdir(
        parents=True,
        exist_ok=True
    )

    best_checkpoint_path = (
        checkpoint_directory
        / "best.pt"
    )

    last_checkpoint_path = (
        checkpoint_directory
        / "last.pt"
    )

    resume_checkpoint_path = (
        checkpoint_directory
        / "resume.pt"
    )

    history_path = (
        DIRECTORIES["histories"]
        / f"{configuration_id}_history.csv"
    )

    summary_path = (
        DIRECTORIES["metrics"]
        / f"{configuration_id}_training_summary.json"
    )

    error_path = (
        DIRECTORIES["logs"]
        / f"{configuration_id}_error.txt"
    )

    if force_restart:

        for path in [
            best_checkpoint_path,
            last_checkpoint_path,
            resume_checkpoint_path,
            history_path,
            summary_path,
            error_path
        ]:

            if path.exists():
                path.unlink()

    if summary_path.exists():

        with open(
            summary_path,
            "r",
            encoding="utf-8"
        ) as file:

            existing_summary = (
                json.load(file)
            )

        if (
            existing_summary.get(
                "status"
            ) == "completed"
            and not force_restart
        ):

            print(
                f"{configuration_id} ya está completada."
            )

            return existing_summary

    set_global_seed(
        int(
            configuration["seed"]
        )
    )

    resume_available = (
        resume_checkpoint_path.exists()
        and not force_restart
    )

    print("\n" + "=" * 80)
    print(
        f"CONFIGURACIÓN: {configuration_id}"
    )
    print("=" * 80)

    print(
        "Model:",
        configuration["timm_model"]
    )

    print(
        "Augmentation:",
        augmentation_name
    )

    print(
        "Resume checkpoint available:",
        resume_available
    )

    model = create_experiment_model(
        model_key=model_key,
        pretrained=(
            not resume_available
        )
    )

    model = model.to(
        DEVICE
    )

    criterion = build_criterion(
        augmentation_name=(
            augmentation_name
        ),
        device=DEVICE
    )

    evaluation_criterion = (
        build_evaluation_criterion(
            DEVICE
        )
    )

    mixing_function = (
        build_mixing_function(
            augmentation_name
        )
    )

    optimizer = build_optimizer(
        model
    )

    scaler = build_grad_scaler()

    start_epoch = 0
    best_metric = -np.inf
    best_epoch = None
    epochs_without_improvement = 0
    history_records = []

    if resume_available:

        resume_payload = torch.load(
            resume_checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

        saved_configuration_id = (
            resume_payload[
                "configuration"
            ][
                "configuration_id"
            ]
        )

        if (
            saved_configuration_id
            != configuration_id
        ):

            raise ValueError(
                "The resume checkpoint belongs "
                "to a different configuration."
            )

        model.load_state_dict(
            resume_payload[
                "model_state_dict"
            ],
            strict=True
        )

        optimizer.load_state_dict(
            resume_payload[
                "optimizer_state_dict"
            ]
        )

        move_optimizer_state_to_device(
            optimizer,
            DEVICE
        )

        scaler.load_state_dict(
            resume_payload[
                "scaler_state_dict"
            ]
        )

        start_epoch = int(
            resume_payload[
                "next_epoch_index"
            ]
        )

        best_metric = float(
            resume_payload[
                "best_val_macro_f1"
            ]
        )

        best_epoch = resume_payload[
            "best_epoch"
        ]

        epochs_without_improvement = int(
            resume_payload[
                "epochs_without_improvement"
            ]
        )

        history_records = list(
            resume_payload[
                "history_records"
            ]
        )

        restore_rng_state(
            resume_payload.get(
                "rng_state"
            )
        )

        print(
            "Resuming from epoch:",
            start_epoch + 1
        )

    update_experiment_plan_status(
        configuration_id,
        "running",
        started_or_resumed_at_utc=(
            utc_now_iso()
        )
    )

    run_start_time = (
        time.perf_counter()
    )

    stop_reason = (
        "maximum_epochs_reached"
    )

    last_completed_epoch = (
        start_epoch - 1
    )

    try:

        for epoch_index in range(
            start_epoch,
            configuration["max_epochs"]
        ):

            epoch_number = (
                epoch_index + 1
            )

            current_learning_rate = (
                set_optimizer_learning_rate(
                    optimizer=optimizer,
                    epoch=epoch_index,
                    max_epochs=(
                        configuration[
                            "max_epochs"
                        ]
                    )
                )
            )

            (
                current_train_loader,
                weighted_sampler_used
            ) = build_train_loader(
                epoch=epoch_index,
                batch_size=int(
                    configuration[
                        "batch_size"
                    ]
                )
            )

            print("\n" + "-" * 80)

            print(
                f"{configuration_id} | "
                f"Epoch {epoch_number}/"
                f"{configuration['max_epochs']}"
            )

            print(
                f"Learning rate: "
                f"{current_learning_rate:.8f}"
            )

            print(
                "Weighted sampler:",
                weighted_sampler_used
            )

            train_result = train_one_epoch(
                model=model,
                train_loader=(
                    current_train_loader
                ),
                criterion=criterion,
                optimizer=optimizer,
                mixing_function=(
                    mixing_function
                ),
                scaler=scaler,
                device=DEVICE,
                epoch_number=(
                    epoch_number
                )
            )

            validation_result = (
                evaluate_one_epoch(
                    model=model,
                    data_loader=val_loader,
                    criterion=(
                        evaluation_criterion
                    ),
                    device=DEVICE,
                    description=(
                        f"Val epoch {epoch_number}"
                    ),
                    return_logits=False
                )
            )

            current_metric = float(
                validation_result[
                    "macro_f1"
                ]
            )

            improved = (
                current_metric
                > (
                    best_metric
                    + EARLY_STOPPING_MIN_DELTA
                )
            )

            if improved:

                best_metric = (
                    current_metric
                )

                best_epoch = (
                    epoch_number
                )

                epochs_without_improvement = 0

            else:

                epochs_without_improvement += 1

            epoch_record = {
                "configuration_id": (
                    configuration_id
                ),

                "model_key": model_key,

                "augmentation": (
                    augmentation_name
                ),

                "epoch": (
                    epoch_number
                ),

                "epoch_index": (
                    epoch_index
                ),

                "learning_rate": (
                    current_learning_rate
                ),

                "weighted_sampler_used": (
                    weighted_sampler_used
                ),

                "train_metric_interpretation": (
                    "exact"
                    if mixing_function is None
                    else "proxy_dominant_soft_target"
                ),

                **train_result,

                "val_loss": (
                    validation_result["loss"]
                ),

                "val_accuracy": (
                    validation_result[
                        "accuracy"
                    ]
                ),

                "val_balanced_accuracy": (
                    validation_result[
                        "balanced_accuracy"
                    ]
                ),

                "val_macro_f1": (
                    validation_result[
                        "macro_f1"
                    ]
                ),

                "val_samples": (
                    validation_result[
                        "samples"
                    ]
                ),

                "val_seconds": (
                    validation_result[
                        "seconds"
                    ]
                ),

                "val_images_per_second": (
                    validation_result[
                        "images_per_second"
                    ]
                ),

                "improved_best_model": (
                    improved
                ),

                "best_val_macro_f1_so_far": (
                    best_metric
                ),

                "best_epoch_so_far": (
                    best_epoch
                ),

                "epochs_without_improvement": (
                    epochs_without_improvement
                ),

                "completed_at_utc": (
                    utc_now_iso()
                )
            }

            history_records.append(
                epoch_record
            )

            history_df = pd.DataFrame(
                history_records
            )

            history_df.to_csv(
                history_path,
                index=False
            )

            if improved:

                best_payload = (
                    create_model_checkpoint_payload(
                        configuration=configuration,
                        model=model,
                        epoch_index=(
                            epoch_index
                        ),
                        best_epoch=(
                            best_epoch
                        ),
                        best_metric=(
                            best_metric
                        ),
                        checkpoint_type=(
                            "best_model"
                        )
                    )
                )

                atomic_torch_save(
                    best_payload,
                    best_checkpoint_path
                )

            resume_payload = (
                create_resume_checkpoint_payload(
                    configuration=configuration,
                    model=model,
                    optimizer=optimizer,
                    scaler=scaler,
                    epoch_index=epoch_index,
                    best_epoch=best_epoch,
                    best_metric=best_metric,
                    epochs_without_improvement=(
                        epochs_without_improvement
                    ),
                    history_records=(
                        history_records
                    )
                )
            )

            atomic_torch_save(
                resume_payload,
                resume_checkpoint_path
            )

            last_completed_epoch = (
                epoch_index
            )

            print(
                "Train loss:",
                f"{train_result['train_loss']:.6f}"
            )

            print(
                "Val loss:",
                f"{validation_result['loss']:.6f}"
            )

            print(
                "Val accuracy:",
                f"{validation_result['accuracy']:.6f}"
            )

            print(
                "Val balanced accuracy:",
                f"{validation_result['balanced_accuracy']:.6f}"
            )

            print(
                "Val Macro-F1:",
                f"{current_metric:.6f}"
            )

            print(
                "Best Val Macro-F1:",
                f"{best_metric:.6f}",
                f"(epoch {best_epoch})"
            )

            print(
                "Epochs without improvement:",
                epochs_without_improvement
            )

            if (
                epochs_without_improvement
                >= EARLY_STOPPING_PATIENCE
            ):

                stop_reason = (
                    "early_stopping"
                )

                print(
                    "\nEarly stopping activated."
                )

                break

        last_payload = (
            create_model_checkpoint_payload(
                configuration=configuration,
                model=model,
                epoch_index=(
                    last_completed_epoch
                ),
                best_epoch=best_epoch,
                best_metric=best_metric,
                checkpoint_type=(
                    "last_model"
                )
            )
        )

        atomic_torch_save(
            last_payload,
            last_checkpoint_path
        )

        total_training_seconds = (
            time.perf_counter()
            - run_start_time
        )

        completed_epochs = len(
            history_records
        )

        summary = {
            "status": "completed",

            "configuration_id": (
                configuration_id
            ),

            "model_key": model_key,

            "timm_model": (
                configuration[
                    "timm_model"
                ]
            ),

            "augmentation": (
                augmentation_name
            ),

            "completed_epochs": (
                completed_epochs
            ),

            "maximum_epochs": (
                configuration[
                    "max_epochs"
                ]
            ),

            "best_epoch": (
                best_epoch
            ),

            "best_val_macro_f1": (
                best_metric
            ),

            "stop_reason": (
                stop_reason
            ),

            "total_session_seconds": (
                total_training_seconds
            ),

            "best_checkpoint": str(
                best_checkpoint_path
            ),

            "last_checkpoint": str(
                last_checkpoint_path
            ),

            "history_path": str(
                history_path
            ),

            "completed_at_utc": (
                utc_now_iso()
            )
        }

        atomic_json_save(
            summary,
            summary_path
        )

        if resume_checkpoint_path.exists():

            resume_checkpoint_path.unlink()

        update_experiment_plan_status(
            configuration_id,
            "completed",
            completed_epochs=(
                completed_epochs
            ),
            best_epoch=(
                best_epoch
            ),
            best_val_macro_f1=(
                best_metric
            ),
            stop_reason=(
                stop_reason
            ),
            completed_at_utc=(
                utc_now_iso()
            )
        )

        print("\n" + "=" * 80)

        print(
            f"{configuration_id} COMPLETADA"
        )

        print("=" * 80)

        print(
            "Completed epochs:",
            completed_epochs
        )

        print(
            "Best epoch:",
            best_epoch
        )

        print(
            "Best validation Macro-F1:",
            f"{best_metric:.6f}"
        )

        print(
            "Stopping reason:",
            stop_reason
        )

        return summary

    except Exception as error:

        error_text = (
            f"Configuration: {configuration_id}\n"
            f"Date UTC: {utc_now_iso()}\n"
            f"Error: {type(error).__name__}: {error}\n\n"
            f"{traceback.format_exc()}"
        )

        error_path.write_text(
            error_text,
            encoding="utf-8"
        )

        update_experiment_plan_status(
            configuration_id,
            "error",
            error_type=(
                type(error).__name__
            ),
            error_message=(
                str(error)
            ),
            error_at_utc=(
                utc_now_iso()
            )
        )

        raise

    finally:

        del model
        del criterion
        del evaluation_criterion
        del optimizer
        del scaler

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


# Single-batch smoke test before training
def run_training_engine_smoke_test():

    set_global_seed(
        SEED
    )

    smoke_configuration_id = (
        "mobilenet_v2__none"
    )

    smoke_row = (
        experiment_plan_df[
            experiment_plan_df[
                "configuration_id"
            ] == smoke_configuration_id
        ]
        .iloc[0]
    )

    model = create_experiment_model(
        model_key=(
            smoke_row[
                "model_key"
            ]
        ),
        pretrained=True
    ).to(
        DEVICE
    )

    criterion = build_criterion(
        augmentation_name="none",
        device=DEVICE
    )

    evaluation_criterion = (
        build_evaluation_criterion(
            DEVICE
        )
    )

    optimizer = build_optimizer(
        model
    )

    scaler = build_grad_scaler()

    smoke_train_loader, _ = (
        build_train_loader(
            epoch=0,
            batch_size=BATCH_SIZE
        )
    )

    train_batch = next(
        iter(
            smoke_train_loader
        )
    )

    images = train_batch[
        "image"
    ].to(
        DEVICE,
        non_blocking=True
    )

    targets = train_batch[
        "target"
    ].to(
        DEVICE,
        non_blocking=True
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    if torch.cuda.is_available():

        torch.cuda.reset_peak_memory_stats()

    smoke_start = time.perf_counter()

    with torch.amp.autocast(
        device_type=AMP_DEVICE_TYPE,
        dtype=AMP_DTYPE,
        enabled=(
            USE_AMP
            and DEVICE.type == "cuda"
        )
    ):

        logits = model(
            images
        )

        train_loss = criterion(
            logits,
            targets
        )

    scaler.scale(
        train_loss
    ).backward()

    scaler.unscale_(
        optimizer
    )

    gradient_norm = (
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP_NORM
        )
    )

    scaler.step(
        optimizer
    )

    scaler.update()

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    smoke_seconds = (
        time.perf_counter()
        - smoke_start
    )

    val_batch = next(
        iter(
            val_loader
        )
    )

    val_images = val_batch[
        "image"
    ].to(
        DEVICE,
        non_blocking=True
    )

    val_targets = val_batch[
        "target"
    ].to(
        DEVICE,
        non_blocking=True
    )

    model.eval()

    with torch.inference_mode():

        with torch.amp.autocast(
            device_type=AMP_DEVICE_TYPE,
            dtype=AMP_DTYPE,
            enabled=(
                USE_AMP
                and DEVICE.type == "cuda"
            )
        ):

            val_logits = model(
                val_images
            )

            val_loss = (
                evaluation_criterion(
                    val_logits,
                    val_targets
                )
            )

    if torch.cuda.is_available():

        peak_memory_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    else:

        peak_memory_mb = np.nan

    smoke_test_df = pd.DataFrame({
        "item": [
            "configuration",
            "train_batch_shape",
            "train_logits_shape",
            "train_loss",
            "gradient_norm",
            "validation_batch_shape",
            "validation_logits_shape",
            "validation_loss",
            "elapsed_seconds",
            "peak_cuda_memory_mb",
            "finite_train_loss",
            "finite_validation_loss"
        ],

        "value": [
            smoke_configuration_id,
            str(
                tuple(
                    images.shape
                )
            ),
            str(
                tuple(
                    logits.shape
                )
            ),
            float(
                train_loss.detach().cpu()
            ),
            float(
                gradient_norm.detach().cpu()
            ),
            str(
                tuple(
                    val_images.shape
                )
            ),
            str(
                tuple(
                    val_logits.shape
                )
            ),
            float(
                val_loss.detach().cpu()
            ),
            smoke_seconds,
            peak_memory_mb,
            bool(
                torch.isfinite(
                    train_loss
                ).item()
            ),
            bool(
                torch.isfinite(
                    val_loss
                ).item()
            )
        ]
    })

    SMOKE_TEST_PATH = (
        DIRECTORIES["metadata"]
        / "training_engine_smoke_test.csv"
    )

    smoke_test_df.to_csv(
        SMOKE_TEST_PATH,
        index=False
    )

    del model
    del criterion
    del evaluation_criterion
    del optimizer
    del scaler
    del images
    del targets
    del logits
    del val_images
    del val_targets
    del val_logits

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    set_global_seed(
        SEED
    )

    return (
        smoke_test_df,
        SMOKE_TEST_PATH
    )


training_engine_smoke_df, training_engine_smoke_path = (
    run_training_engine_smoke_test()
)


# Display results
print("=" * 80)
print("TRAINING ENGINE CONFIGURED")
print("=" * 80)

print("\nCorrected learning-rate schedule:")
display(
    corrected_learning_rate_schedule_df
)

print("\nSingle-batch training smoke test:")
display(
    training_engine_smoke_df
)

print("\nTraining engine configuration:")
print(
    json.dumps(
        TRAINING_ENGINE_CONFIG,
        indent=2,
        ensure_ascii=False
    )
)

print("\nSmoke test saved to:")
print(training_engine_smoke_path)

print("\nAvailable training function:")
print(
    'train_configuration("configuration_id")'
)

print(
    "\nNo full training "
    "configuration has started yet."
)

In [ ]:
import time
import random

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from tqdm.auto import tqdm


# Use a more conservative GradScaler
AMP_INITIAL_SCALE = 1024.0
AMP_GROWTH_FACTOR = 2.0
AMP_BACKOFF_FACTOR = 0.5
AMP_GROWTH_INTERVAL = 1000


def build_grad_scaler():

    return torch.amp.GradScaler(
        "cuda",
        init_scale=AMP_INITIAL_SCALE,
        growth_factor=AMP_GROWTH_FACTOR,
        backoff_factor=AMP_BACKOFF_FACTOR,
        growth_interval=AMP_GROWTH_INTERVAL,
        enabled=(
            USE_AMP
            and DEVICE.type == "cuda"
        )
    )


# Safely update the experimental plan
def update_experiment_plan_status(
    configuration_id,
    status,
    **additional_values
):

    plan_df = pd.read_csv(
        EXPERIMENT_PLAN_PATH,
        low_memory=False
    )

    matching_mask = (
        plan_df["configuration_id"]
        == configuration_id
    )

    if matching_mask.sum() != 1:

        raise ValueError(
            "No unique configuration was found "
            f"para {configuration_id}."
        )

    # Always handle status as text.
    plan_df["status"] = (
        plan_df["status"]
        .astype("object")
    )

    plan_df.loc[
        matching_mask,
        "status"
    ] = str(status)

    for column, value in additional_values.items():

        if column not in plan_df.columns:

            if isinstance(
                value,
                (
                    str,
                    bool
                )
            ) or value is None:

                plan_df[column] = pd.Series(
                    [None] * len(plan_df),
                    dtype="object"
                )

            else:

                plan_df[column] = np.nan

        elif isinstance(
            value,
            str
        ):

            plan_df[column] = (
                plan_df[column]
                .astype("object")
            )

        plan_df.loc[
            matching_mask,
            column
        ] = value

    plan_df.to_csv(
        EXPERIMENT_PLAN_PATH,
        index=False
    )


# Corrected single-epoch training engine
def train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    mixing_function,
    scaler,
    device,
    epoch_number
):

    model.train()

    running_loss = 0.0
    processed_samples = 0

    proxy_true_values = []
    proxy_predicted_values = []

    optimizer_steps_executed = 0
    optimizer_steps_skipped = 0
    nonfinite_gradient_batches = 0

    finite_gradient_norms = []

    if torch.cuda.is_available():

        torch.cuda.reset_peak_memory_stats(
            device
        )

        torch.cuda.synchronize(
            device
        )

    epoch_start_time = time.perf_counter()

    progress_bar = tqdm(
        train_loader,
        desc=f"Train epoch {epoch_number}",
        leave=False
    )

    for batch in progress_bar:

        images_cpu = batch["image"]

        hard_targets_cpu = batch[
            "target"
        ]

        # MixUp / CutMix
        if mixing_function is None:

            loss_targets_cpu = (
                hard_targets_cpu
            )

            proxy_targets_cpu = (
                hard_targets_cpu
            )

        else:

            (
                images_cpu,
                loss_targets_cpu
            ) = mixing_function(
                images_cpu,
                hard_targets_cpu
            )

            proxy_targets_cpu = (
                loss_targets_cpu.argmax(
                    dim=1
                )
            )

        images = images_cpu.to(
            device,
            non_blocking=True
        )

        loss_targets = (
            loss_targets_cpu.to(
                device,
                non_blocking=True
            )
        )

        proxy_targets = (
            proxy_targets_cpu.to(
                device,
                non_blocking=True
            )
        )

        batch_size_current = int(
            images.shape[0]
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # Forward pass with AMP
        with torch.amp.autocast(
            device_type=AMP_DEVICE_TYPE,
            dtype=AMP_DTYPE,
            enabled=(
                USE_AMP
                and device.type == "cuda"
            )
        ):

            logits = model(
                images
            )

            loss = criterion(
                logits,
                loss_targets
            )

        if not torch.isfinite(loss):

            raise RuntimeError(
                "A non-finite loss occurred "
                f"in epoch {epoch_number}."
            )

        # Scaled backward pass
        scaler.scale(
            loss
        ).backward()

        # Unscale before inspecting or clipping.
        scaler.unscale_(
            optimizer
        )

        scale_before = float(
            scaler.get_scale()
        )

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRADIENT_CLIP_NORM,
                error_if_nonfinite=False
            )
        )

        gradient_is_finite = bool(
            torch.isfinite(
                gradient_norm
            ).item()
        )

        if gradient_is_finite:

            finite_gradient_norms.append(
                float(
                    gradient_norm
                    .detach()
                    .cpu()
                )
            )

        else:

            nonfinite_gradient_batches += 1

        # GradScaler checks for inf/NaN internally.
        # When detected, it skips optimizer.step().
        scaler.step(
            optimizer
        )

        scaler.update()

        scale_after = float(
            scaler.get_scale()
        )

        step_was_skipped = (
            not gradient_is_finite
            or scale_after < scale_before
        )

        if step_was_skipped:

            optimizer_steps_skipped += 1

        else:

            optimizer_steps_executed += 1

        # Batch metrics
        predictions = logits.argmax(
            dim=1
        )

        running_loss += (
            float(
                loss.detach().cpu()
            )
            * batch_size_current
        )

        processed_samples += (
            batch_size_current
        )

        proxy_true_values.extend(
            proxy_targets
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        proxy_predicted_values.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        progress_bar.set_postfix({
            "loss": (
                running_loss
                / max(
                    processed_samples,
                    1
                )
            ),
            "amp_scale": (
                scale_after
            ),
            "skipped": (
                optimizer_steps_skipped
            )
        })

    if torch.cuda.is_available():

        torch.cuda.synchronize(
            device
        )

    epoch_duration_seconds = (
        time.perf_counter()
        - epoch_start_time
    )

    train_metrics = (
        compute_classification_metrics(
            proxy_true_values,
            proxy_predicted_values
        )
    )

    if torch.cuda.is_available():

        peak_memory_mb = (
            torch.cuda.max_memory_allocated(
                device
            )
            / (1024 ** 2)
        )

    else:

        peak_memory_mb = np.nan

    if finite_gradient_norms:

        mean_gradient_norm = float(
            np.mean(
                finite_gradient_norms
            )
        )

        maximum_gradient_norm = float(
            np.max(
                finite_gradient_norms
            )
        )

    else:

        mean_gradient_norm = np.nan
        maximum_gradient_norm = np.nan

    return {
        "train_loss": (
            running_loss
            / processed_samples
        ),

        "train_accuracy_proxy": (
            train_metrics["accuracy"]
        ),

        "train_balanced_accuracy_proxy": (
            train_metrics[
                "balanced_accuracy"
            ]
        ),

        "train_macro_f1_proxy": (
            train_metrics["macro_f1"]
        ),

        "train_samples": (
            processed_samples
        ),

        "train_seconds": (
            epoch_duration_seconds
        ),

        "train_images_per_second": (
            processed_samples
            / max(
                epoch_duration_seconds,
                1e-12
            )
        ),

        "peak_cuda_memory_mb": (
            peak_memory_mb
        ),

        "optimizer_steps_executed": (
            optimizer_steps_executed
        ),

        "optimizer_steps_skipped": (
            optimizer_steps_skipped
        ),

        "nonfinite_gradient_batches": (
            nonfinite_gradient_batches
        ),

        "mean_gradient_norm_before_clipping": (
            mean_gradient_norm
        ),

        "maximum_gradient_norm_before_clipping": (
            maximum_gradient_norm
        ),

        "final_amp_scale": float(
            scaler.get_scale()
        )
    }


# Verify the correction
test_scaler = build_grad_scaler()

print("=" * 80)
print("TRAINING ENGINE CORRECTION APPLIED")
print("=" * 80)

print(
    "AMP enabled:",
    test_scaler.is_enabled()
)

print(
    "Initial AMP scale:",
    test_scaler.get_scale()
)

print(
    "Non-finite gradients will no longer immediately "
    "stop the run."
)

print(
    "GradScaler can skip the affected optimizer step "
    "and automatically reduce the scale."
)

print(
    "The FutureWarning in the experimental plan "
    "was also fixed."
)

In [ ]:
import gc
import torch

from torch.utils.data import DataLoader
from tqdm.auto import tqdm as _real_tqdm


# Prevent multiprocessing in Kaggle
NUM_WORKERS = 0


def build_loader_kwargs():

    return {
        "num_workers": 0,
        "pin_memory": torch.cuda.is_available(),
        "persistent_workers": False
    }


# Silent tqdm while preserving set_postfix()
def tqdm(iterable=None, *args, **kwargs):

    kwargs["disable"] = True

    return _real_tqdm(
        iterable,
        *args,
        **kwargs
    )


# Rebuild validation and test loaders
evaluation_generator = torch.Generator()

evaluation_generator.manual_seed(
    SEED
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    generator=evaluation_generator,
    **build_loader_kwargs()
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    generator=evaluation_generator,
    **build_loader_kwargs()
)


# Test training loader
safe_train_loader, sampler_used = (
    build_train_loader(
        epoch=0,
        batch_size=BATCH_SIZE
    )
)

safe_batch = next(
    iter(safe_train_loader)
)


# Verify that tqdm preserves set_postfix
test_progress_bar = tqdm(
    range(1),
    desc="Test"
)

if not hasattr(
    test_progress_bar,
    "set_postfix"
):

    raise RuntimeError(
        "The tqdm correction was not applied."
    )

test_progress_bar.set_postfix({
    "status": "ok"
})

test_progress_bar.close()


# Cleanup
del safe_train_loader
del safe_batch
del test_progress_bar

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print("=" * 80)
print("STABLE MODE CORRECTED")
print("=" * 80)

print("NUM_WORKERS:", NUM_WORKERS)
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))
print("tqdm preserves set_postfix: True")
print("Visual progress bars disabled: True")


# **Cell 5 — Train the first complete configuration**

In [ ]:
import json
from pathlib import Path

# Configuration to train
FIRST_CONFIGURATION_ID = (
    "vit_b16__mixup"
)

# force_restart=False:
# - starts from scratch if no checkpoint exists;
# - automatically resumes from resume.pt;
# - does not repeat the run if it is already completed.
first_training_summary = train_configuration(
    configuration_id=FIRST_CONFIGURATION_ID,
    force_restart=False
)

# Display summary
print("\n" + "=" * 80)
print("FIRST CONFIGURATION SUMMARY")
print("=" * 80)

print(
    json.dumps(
        first_training_summary,
        indent=2,
        ensure_ascii=False
    )
)

# Load and display history
FIRST_HISTORY_PATH = (
    DIRECTORIES["histories"]
    / f"{FIRST_CONFIGURATION_ID}_history.csv"
)

if not FIRST_HISTORY_PATH.exists():

    raise FileNotFoundError(
        f"History file not found:\n"
        f"{FIRST_HISTORY_PATH}"
    )

first_history_df = pd.read_csv(
    FIRST_HISTORY_PATH,
    low_memory=False
)

print("\nComplete history:")
display(first_history_df)

# Validate essential files
FIRST_CHECKPOINT_DIR = (
    DIRECTORIES["checkpoints"]
    / FIRST_CONFIGURATION_ID
)

FIRST_BEST_CHECKPOINT = (
    FIRST_CHECKPOINT_DIR
    / "best.pt"
)

FIRST_LAST_CHECKPOINT = (
    FIRST_CHECKPOINT_DIR
    / "last.pt"
)

required_outputs = {
    "history": FIRST_HISTORY_PATH,
    "best_checkpoint": FIRST_BEST_CHECKPOINT,
    "last_checkpoint": FIRST_LAST_CHECKPOINT
}

missing_outputs = {
    name: path
    for name, path in required_outputs.items()
    if not path.exists()
}

if missing_outputs:

    print(
        "Missing files:"
    )

    for name, path in missing_outputs.items():
        print(f" - {name}: {path}")

    raise FileNotFoundError(
        "The configuration finished without producing "
        "all essential files."
    )

# Compact summary
best_history_row = (
    first_history_df
    .sort_values(
        "val_macro_f1",
        ascending=False
    )
    .iloc[0]
)

compact_summary_df = pd.DataFrame({
    "item": [
        "configuration_id",
        "completed_epochs",
        "best_epoch",
        "best_val_macro_f1",
        "best_val_accuracy",
        "best_val_balanced_accuracy",
        "stop_reason",
        "best_checkpoint_exists",
        "last_checkpoint_exists"
    ],
    "value": [
        FIRST_CONFIGURATION_ID,
        len(first_history_df),
        int(
            best_history_row["epoch"]
        ),
        float(
            best_history_row[
                "val_macro_f1"
            ]
        ),
        float(
            best_history_row[
                "val_accuracy"
            ]
        ),
        float(
            best_history_row[
                "val_balanced_accuracy"
            ]
        ),
        first_training_summary[
            "stop_reason"
        ],
        FIRST_BEST_CHECKPOINT.exists(),
        FIRST_LAST_CHECKPOINT.exists()
    ]
})

print("\nCompact summary:")
display(compact_summary_df)

print("\nBest model checkpoint:")
print(FIRST_BEST_CHECKPOINT)

print("\nLast epoch checkpoint:")
print(FIRST_LAST_CHECKPOINT)

print("\nHistory:")
print(FIRST_HISTORY_PATH)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import zipfile

import pandas as pd


# Configuration
CONFIGURATION_ID = "vit_b16__mixup"

EXPORT_DIR = Path(
    "/kaggle/working/export_vit_b16__mixup"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ZIP_PATH = (
    EXPORT_DIR
    / f"{CONFIGURATION_ID}_complete_backup.zip"
)

BACKUP_MANIFEST_PATH = (
    EXPORT_DIR
    / f"{CONFIGURATION_ID}_backup_manifest.csv"
)

README_PATH = (
    EXPORT_DIR
    / "README_BACKUP.txt"
)

ZIP_CHECKSUM_PATH = (
    EXPORT_DIR
    / f"{ZIP_PATH.name}.sha256.txt"
)


# Required files
CHECKPOINT_DIR = (
    DIRECTORIES["checkpoints"]
    / CONFIGURATION_ID
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "best.pt"
)

LAST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "last.pt"
)

RESUME_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "resume.pt"
)

HISTORY_PATH = (
    DIRECTORIES["histories"]
    / f"{CONFIGURATION_ID}_history.csv"
)

TRAINING_SUMMARY_PATH = (
    DIRECTORIES["metrics"]
    / f"{CONFIGURATION_ID}_training_summary.json"
)

required_files = {
    "best checkpoint": BEST_CHECKPOINT_PATH,
    "last checkpoint": LAST_CHECKPOINT_PATH,
    "history": HISTORY_PATH,
    "training summary": TRAINING_SUMMARY_PATH
}

missing_required_files = {
    name: path
    for name, path in required_files.items()
    if not path.is_file()
}

if missing_required_files:

    print("Missing required files:")

    for name, path in missing_required_files.items():
        print(f" - {name}: {path}")

    raise FileNotFoundError(
        "The complete backup cannot be created."
    )


# Confirm successful completion
with open(
    TRAINING_SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as file:

    training_summary = json.load(file)

training_status = training_summary.get(
    "status"
)

if training_status != "completed":

    raise RuntimeError(
        "The configuration is not marked as completed.\n"
        f"Estado encontrado: {training_status}"
    )


# SHA-256 function
def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024
):

    sha256_hash = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            sha256_hash.update(
                chunk
            )

    return sha256_hash.hexdigest()


# Register files to include in the ZIP archive
files_to_backup = {}


def register_file(
    source_path,
    archive_path
):

    source_path = Path(
        source_path
    )

    if not source_path.is_file():
        return

    archive_path = str(
        Path(archive_path)
    )

    files_to_backup[
        archive_path
    ] = source_path


# Checkpoints
for file_path in sorted(
    CHECKPOINT_DIR.glob("*")
):

    if file_path.is_file():

        register_file(
            file_path,
            (
                Path("checkpoints")
                / CONFIGURATION_ID
                / file_path.name
            )
        )


# Training history and main summary
register_file(
    HISTORY_PATH,
    (
        Path("histories")
        / HISTORY_PATH.name
    )
)

register_file(
    TRAINING_SUMMARY_PATH,
    (
        Path("metrics")
        / TRAINING_SUMMARY_PATH.name
    )
)


# All shared metadata
METADATA_DIRECTORY = (
    DIRECTORIES["metadata"]
)

for file_path in sorted(
    METADATA_DIRECTORY.rglob("*")
):

    if file_path.is_file():

        register_file(
            file_path,
            (
                Path("metadata")
                / file_path.relative_to(
                    METADATA_DIRECTORY
                )
            )
        )


# Other existing configuration-specific results
result_directory_names = [
    "predictions",
    "metrics",
    "calibration",
    "confusion_matrices",
    "efficiency",
    "explainability",
    "statistics",
    "learning_curves",
    "logs"
]

for directory_name in result_directory_names:

    result_directory = DIRECTORIES[
        directory_name
    ]

    if not result_directory.exists():
        continue

    for file_path in sorted(
        result_directory.rglob("*")
    ):

        if (
            file_path.is_file()
            and CONFIGURATION_ID
            in file_path.name
        ):

            register_file(
                file_path,
                (
                    Path(directory_name)
                    / file_path.relative_to(
                        result_directory
                    )
                )
            )


# Final dataset manifest
FINAL_DATASET_MANIFEST = (
    MANIFEST_DIR_INPUT
    / "14_final_verified_dataset_manifest.csv"
)

register_file(
    FINAL_DATASET_MANIFEST,
    (
        Path("dataset_reference")
        / "manifests"
        / FINAL_DATASET_MANIFEST.name
    )
)


# Frozen dataset splits
for split_name in [
    "train.csv",
    "val.csv",
    "test.csv"
]:

    split_path = (
        SPLIT_DIR_INPUT
        / split_name
    )

    register_file(
        split_path,
        (
            Path("dataset_reference")
            / "splits"
            / split_name
        )
    )


# Dataset readiness indicator
register_file(
    READY_FLAG_PATH_INPUT,
    (
        Path("dataset_reference")
        / "DATASET_READY_FOR_TRAINING.txt"
    )
)


# Create backup manifest
backup_records = []

print("=" * 80)
print("CALCULATING FILE CHECKSUMS")
print("=" * 80)

for archive_path, source_path in sorted(
    files_to_backup.items()
):

    file_size_bytes = (
        source_path.stat().st_size
    )

    sha256_value = calculate_sha256(
        source_path
    )

    backup_records.append({
        "archive_path": archive_path,
        "original_path": str(
            source_path
        ),
        "size_bytes": file_size_bytes,
        "size_mb": (
            file_size_bytes
            / (1024 ** 2)
        ),
        "sha256": sha256_value
    })

    print(
        f"{archive_path:<80} "
        f"{file_size_bytes / (1024 ** 2):>10.2f} MB"
    )


backup_manifest_df = pd.DataFrame(
    backup_records
)

backup_manifest_df.to_csv(
    BACKUP_MANIFEST_PATH,
    index=False
)


# Create README
readme_text = f"""
TRAINING BACKUP
=======================

Configuration:
{CONFIGURATION_ID}

Model:
{training_summary.get("timm_model")}

Augmentation:
{training_summary.get("augmentation")}

Status:
{training_summary.get("status")}

Completed epochs:
{training_summary.get("completed_epochs")}

Best epoch:
{training_summary.get("best_epoch")}

Best validation Macro-F1:
{training_summary.get("best_val_macro_f1")}

Stopping reason:
{training_summary.get("stop_reason")}

Backup creation date:
{datetime.now(timezone.utc).isoformat()}

MAIN FILES
--------------------

checkpoints/{CONFIGURATION_ID}/best.pt
    Model selected based on the best validation Macro-F1.

checkpoints/{CONFIGURATION_ID}/last.pt
    Model from the last completed epoch.

histories/{CONFIGURATION_ID}_history.csv
    Losses, metrics, times, and learning rate per epoch.

metrics/{CONFIGURATION_ID}_training_summary.json
    Final training run summary.

metadata/
    Experimental configuration, class mapping,
    environment versions, and training parameters.

dataset_reference/
    Verified manifest and frozen dataset splits.

IMPORTANT
----------

This backup does not contain the 28,348 images.

To reproduce evaluations or subsequent analyses, also retain
the private Kaggle dataset:

revision-frutas-2026

The best.pt checkpoint must be used for testing,
calibration, confusion matrices, explainability, and efficiency.
""".strip()

README_PATH.write_text(
    readme_text,
    encoding="utf-8"
)


# Create ZIP archive
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print("\n" + "=" * 80)
print("CREATING ZIP ARCHIVE")
print("=" * 80)

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True
) as zip_file:

    for archive_path, source_path in sorted(
        files_to_backup.items()
    ):

        zip_file.write(
            source_path,
            arcname=archive_path
        )

    zip_file.write(
        BACKUP_MANIFEST_PATH,
        arcname="backup_manifest.csv"
    )

    zip_file.write(
        README_PATH,
        arcname="README_BACKUP.txt"
    )


# Calculate ZIP checksum
zip_sha256 = calculate_sha256(
    ZIP_PATH
)

ZIP_CHECKSUM_PATH.write_text(
    (
        f"{zip_sha256}  "
        f"{ZIP_PATH.name}\n"
    ),
    encoding="utf-8"
)


# Display final summary
zip_size_mb = (
    ZIP_PATH.stat().st_size
    / (1024 ** 2)
)

print("\n" + "=" * 80)
print("BACKUP CREATED SUCCESSFULLY")
print("=" * 80)

print(
    "Configuration:",
    CONFIGURATION_ID
)

print(
    "Included files:",
    len(files_to_backup) + 2
)

print(
    "ZIP size:",
    f"{zip_size_mb:.2f} MB"
)

print("\nZIP:")
print(ZIP_PATH)

print("\nChecksum SHA-256:")
print(zip_sha256)

print("\nChecksum file:")
print(ZIP_CHECKSUM_PATH)

print("\nBackup manifest:")
print(BACKUP_MANIFEST_PATH)

# **POST-TRAINING ANALYSIS**

# **Cell 1 — Check the 12 best checkpoints and summarize validation**

In [ ]:
from pathlib import Path
import pandas as pd


# Expected configurations
EXPECTED_CONFIGURATIONS = [
    "convnext_tiny__none",
    "convnext_tiny__mixup",
    "convnext_tiny__cutmix",

    "mobilenet_v2__none",
    "mobilenet_v2__mixup",
    "mobilenet_v2__cutmix",

    "swin_tiny__none",
    "swin_tiny__mixup",
    "swin_tiny__cutmix",

    "vit_b16__none",
    "vit_b16__mixup",
    "vit_b16__cutmix"
]


# Kaggle input root directory
KAGGLE_INPUT_ROOT = Path("/kaggle/input")

if not KAGGLE_INPUT_ROOT.exists():
    raise FileNotFoundError(
        "/kaggle/input does not exist."
    )


# Find directories matching each run's exact configuration name
run_records = []

RUN_DIRECTORIES = {}

for configuration_id in EXPECTED_CONFIGURATIONS:

    matching_directories = [
        path
        for path in KAGGLE_INPUT_ROOT.rglob(configuration_id)
        if path.is_dir()
    ]

    # If multiple matches are found, prioritize the backup structure
    structured_matches = []

    for path in matching_directories:

        has_backup_structure = (
            (path / "checkpoints").exists()
            and (path / "histories").exists()
            and (path / "metrics").exists()
            and (path / "metadata").exists()
        )

        if has_backup_structure:
            structured_matches.append(path)

    if len(structured_matches) == 1:

        run_directory = structured_matches[0]

    elif len(structured_matches) > 1:

        run_directory = None

    elif len(matching_directories) == 1:

        run_directory = matching_directories[0]

    else:

        run_directory = None


    # Locate essential files within each run
    if run_directory is not None:

        best_checkpoint_candidates = list(
            run_directory.rglob("best.pt")
        )

        last_checkpoint_candidates = list(
            run_directory.rglob("last.pt")
        )

        history_candidates = list(
            run_directory.rglob(
                f"{configuration_id}_history.csv"
            )
        )

        summary_candidates = list(
            run_directory.rglob(
                f"{configuration_id}_training_summary.json"
            )
        )

        best_checkpoint = (
            best_checkpoint_candidates[0]
            if len(best_checkpoint_candidates) == 1
            else None
        )

        last_checkpoint = (
            last_checkpoint_candidates[0]
            if len(last_checkpoint_candidates) == 1
            else None
        )

        history_path = (
            history_candidates[0]
            if len(history_candidates) == 1
            else None
        )

        summary_path = (
            summary_candidates[0]
            if len(summary_candidates) == 1
            else None
        )

    else:

        best_checkpoint = None
        last_checkpoint = None
        history_path = None
        summary_path = None


    # Determine whether the run backup is complete
    run_complete = all([
        run_directory is not None,
        best_checkpoint is not None,
        last_checkpoint is not None,
        history_path is not None,
        summary_path is not None
    ])


    if run_complete:

        RUN_DIRECTORIES[
            configuration_id
        ] = run_directory


    run_records.append({
        "configuration_id": configuration_id,

        "run_directory_found": (
            run_directory is not None
        ),

        "best_pt": (
            best_checkpoint is not None
        ),

        "last_pt": (
            last_checkpoint is not None
        ),

        "history_csv": (
            history_path is not None
        ),

        "training_summary_json": (
            summary_path is not None
        ),

        "complete_backup": (
            run_complete
        ),

        "run_directory": (
            str(run_directory)
            if run_directory is not None
            else None
        ),

        "best_checkpoint_path": (
            str(best_checkpoint)
            if best_checkpoint is not None
            else None
        )
    })


# Create the inventory
runs_inventory_df = pd.DataFrame(
    run_records
)


# Display results
print("=" * 80)
print("INVENTORY OF THE 12 RUNS")
print("=" * 80)

display(
    runs_inventory_df[
        [
            "configuration_id",
            "run_directory_found",
            "best_pt",
            "last_pt",
            "history_csv",
            "training_summary_json",
            "complete_backup"
        ]
    ]
)


# Summary
number_complete = int(
    runs_inventory_df[
        "complete_backup"
    ].sum()
)

print(
    "\nExpected runs:",
    len(EXPECTED_CONFIGURATIONS)
)

print(
    "Complete runs:",
    number_complete
)


# Display detected paths
print("\nDetected paths:")

for configuration_id, path in RUN_DIRECTORIES.items():

    print(
        f"{configuration_id:<28} -> {path}"
    )


# Final validation
incomplete_runs = runs_inventory_df[
    ~runs_inventory_df[
        "complete_backup"
    ]
]

if len(incomplete_runs) > 0:

    print("\nIncomplete or missing runs:")

    display(
        incomplete_runs
    )

    raise RuntimeError(
        "The 12 runs have not yet all been located correctly."
    )


print("\n" + "=" * 80)
print("ALL 12 RUNS ARE COMPLETE AND AVAILABLE")
print("=" * 80)


In [ ]:
from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import torch


# Analysis output directory
ANALYSIS_ROOT = Path(
    "/kaggle/working/fruit_post_training_analysis"
)

ANALYSIS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

TRAINING_SUMMARY_OUTPUT = (
    ANALYSIS_ROOT
    / "01_training_validation_summary.csv"
)

CHECKPOINT_INTEGRITY_OUTPUT = (
    ANALYSIS_ROOT
    / "02_best_checkpoint_integrity.csv"
)


# Tolerances
METRIC_TOLERANCE = 1e-7

# The following value was used during training:
# EARLY_STOPPING_MIN_DELTA = 1e-4
# A later epoch may therefore yield a slightly higher Macro-F1
# (less than 1e-4) without replacing best.pt.
SELECTION_MIN_DELTA = 1e-4


# Find exactly one matching file
def find_unique_file(
    run_directory,
    pattern
):
    matches = list(
        Path(run_directory).rglob(
            pattern
        )
    )

    if len(matches) != 1:

        raise RuntimeError(
            f"Expected exactly one file "
            f"'{pattern}' in:\n"
            f"{run_directory}\n"
            f"Found: {len(matches)}"
        )

    return matches[0]


# Read the 12 training runs
validation_records = []
integrity_records = []

for configuration_id in EXPECTED_CONFIGURATIONS:

    print(
        "Verifying:",
        configuration_id
    )

    run_directory = RUN_DIRECTORIES[
        configuration_id
    ]

    # Locate files
    best_checkpoint_path = (
        find_unique_file(
            run_directory,
            "best.pt"
        )
    )

    history_path = (
        find_unique_file(
            run_directory,
            f"{configuration_id}_history.csv"
        )
    )

    summary_path = (
        find_unique_file(
            run_directory,
            f"{configuration_id}_training_summary.json"
        )
    )


    # Read training summary
    with open(
        summary_path,
        "r",
        encoding="utf-8"
    ) as file:

        summary = json.load(
            file
        )


    # Read training history
    history_df = pd.read_csv(
        history_path,
        low_memory=False
    )

    if history_df.empty:

        raise RuntimeError(
            f"Empty history: {configuration_id}"
        )


    # Load checkpoint on CPU only
    checkpoint = torch.load(
        best_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )


    # Checkpoint information
    checkpoint_configuration = (
        checkpoint.get(
            "configuration",
            {}
        )
    )

    checkpoint_configuration_id = (
        checkpoint_configuration.get(
            "configuration_id"
        )
    )

    checkpoint_type = checkpoint.get(
        "checkpoint_type"
    )

    checkpoint_epoch = int(
        checkpoint.get(
            "epoch_number"
        )
    )

    checkpoint_best_epoch = int(
        checkpoint.get(
            "best_epoch"
        )
    )

    checkpoint_best_f1 = float(
        checkpoint.get(
            "best_val_macro_f1"
        )
    )


    # Training summary information
    summary_status = summary.get(
        "status"
    )

    summary_configuration_id = (
        summary.get(
            "configuration_id"
        )
    )

    summary_model_key = summary.get(
        "model_key"
    )

    summary_augmentation = summary.get(
        "augmentation"
    )

    summary_completed_epochs = int(
        summary.get(
            "completed_epochs"
        )
    )

    summary_best_epoch = int(
        summary.get(
            "best_epoch"
        )
    )

    summary_best_f1 = float(
        summary.get(
            "best_val_macro_f1"
        )
    )

    stop_reason = summary.get(
        "stop_reason"
    )


    # Find the selected epoch in the history
    best_epoch_rows = history_df[
        history_df["epoch"]
        == checkpoint_best_epoch
    ]

    if len(best_epoch_rows) != 1:

        raise RuntimeError(
            f"No unique row exists for the best "
            f"epoch of {configuration_id}."
        )

    best_history_row = (
        best_epoch_rows.iloc[0]
    )

    history_best_epoch_f1 = float(
        best_history_row[
            "val_macro_f1"
        ]
    )

    history_best_epoch_accuracy = float(
        best_history_row[
            "val_accuracy"
        ]
    )

    history_best_epoch_balanced_accuracy = float(
        best_history_row[
            "val_balanced_accuracy"
        ]
    )

    history_best_epoch_loss = float(
        best_history_row[
            "val_loss"
        ]
    )

    absolute_history_max_f1 = float(
        history_df[
            "val_macro_f1"
        ].max()
    )

    absolute_history_max_epoch = int(
        history_df.loc[
            history_df[
                "val_macro_f1"
            ].idxmax(),
            "epoch"
        ]
    )


    # Consistency checks
    checks = {
        "summary_completed": (
            summary_status
            == "completed"
        ),

        "checkpoint_is_best": (
            checkpoint_type
            == "best_model"
        ),

        "checkpoint_configuration_matches": (
            checkpoint_configuration_id
            == configuration_id
        ),

        "summary_configuration_matches": (
            summary_configuration_id
            == configuration_id
        ),

        "history_length_matches_summary": (
            len(history_df)
            == summary_completed_epochs
        ),

        "checkpoint_epoch_matches_best_epoch": (
            checkpoint_epoch
            == checkpoint_best_epoch
        ),

        "summary_best_epoch_matches_checkpoint": (
            summary_best_epoch
            == checkpoint_best_epoch
        ),

        "summary_f1_matches_checkpoint": (
            abs(
                summary_best_f1
                - checkpoint_best_f1
            )
            <= METRIC_TOLERANCE
        ),

        "history_f1_matches_checkpoint": (
            abs(
                history_best_epoch_f1
                - checkpoint_best_f1
            )
            <= METRIC_TOLERANCE
        ),

        # Allow a later improvement smaller than min_delta,
        # because it does not replace best.pt by design.
        "checkpoint_respects_selection_rule": (
            absolute_history_max_f1
            <= (
                checkpoint_best_f1
                + SELECTION_MIN_DELTA
                + METRIC_TOLERANCE
            )
        )
    }

    all_checks_passed = all(
        checks.values()
    )


    # Record the integrity audit
    integrity_record = {
        "configuration_id": (
            configuration_id
        ),

        **checks,

        "all_checks_passed": (
            all_checks_passed
        )
    }

    integrity_records.append(
        integrity_record
    )


    # Validation comparison table
    validation_records.append({
        "configuration_id": (
            configuration_id
        ),

        "model": (
            summary_model_key
        ),

        "augmentation": (
            summary_augmentation
        ),

        "completed_epochs": (
            summary_completed_epochs
        ),

        "selected_epoch": (
            checkpoint_best_epoch
        ),

        "val_loss": (
            history_best_epoch_loss
        ),

        "val_accuracy": (
            history_best_epoch_accuracy
        ),

        "val_balanced_accuracy": (
            history_best_epoch_balanced_accuracy
        ),

        "val_macro_f1": (
            checkpoint_best_f1
        ),

        "absolute_history_max_epoch": (
            absolute_history_max_epoch
        ),

        "absolute_history_max_macro_f1": (
            absolute_history_max_f1
        ),

        "stop_reason": (
            stop_reason
        ),

        "best_checkpoint_mb": (
            best_checkpoint_path.stat().st_size
            / (1024 ** 2)
        ),

        "best_checkpoint_path": (
            str(
                best_checkpoint_path
            )
        )
    })


    # Release the large checkpoint before loading the next one
    del checkpoint
    del history_df

    gc.collect()


# Create DataFrames
validation_summary_df = pd.DataFrame(
    validation_records
)

checkpoint_integrity_df = pd.DataFrame(
    integrity_records
)


# Sort by validation Macro-F1
validation_summary_df = (
    validation_summary_df
    .sort_values(
        "val_macro_f1",
        ascending=False
    )
    .reset_index(drop=True)
)

validation_summary_df.insert(
    0,
    "validation_rank",
    np.arange(
        1,
        len(validation_summary_df) + 1
    )
)


# Save results
validation_summary_df.to_csv(
    TRAINING_SUMMARY_OUTPUT,
    index=False
)

checkpoint_integrity_df.to_csv(
    CHECKPOINT_INTEGRITY_OUTPUT,
    index=False
)


# Overall verification
number_valid_checkpoints = int(
    checkpoint_integrity_df[
        "all_checks_passed"
    ].sum()
)

print("\n" + "=" * 80)
print("BEST.PT VERIFICATION")
print("=" * 80)

print(
    "Expected checkpoints:",
    len(EXPECTED_CONFIGURATIONS)
)

print(
    "Fully consistent checkpoints:",
    number_valid_checkpoints
)


if number_valid_checkpoints != len(
    EXPECTED_CONFIGURATIONS
):

    print(
        "\nConfigurations with inconsistencies:"
    )

    display(
        checkpoint_integrity_df[
            ~checkpoint_integrity_df[
                "all_checks_passed"
            ]
        ]
    )

    raise RuntimeError(
        "At least one best.pt requires review "
        "before test evaluation."
    )


# Display the compact summary
print("\n" + "=" * 80)
print("VALIDATION RESULTS FOR THE 12 RUNS")
print("=" * 80)

display(
    validation_summary_df[
        [
            "validation_rank",
            "configuration_id",
            "completed_epochs",
            "selected_epoch",
            "val_accuracy",
            "val_balanced_accuracy",
            "val_macro_f1",
            "stop_reason"
        ]
    ]
)


print("\n" + "=" * 80)
print("ALL 12 BEST.PT CHECKPOINTS ARE CONSISTENT")
print("=" * 80)

print("\nSaved files:")
print(TRAINING_SUMMARY_OUTPUT)
print(CHECKPOINT_INTEGRITY_OUTPUT)

# **Cell 2 — Locate and verify the clean test set**
This cell automatically searches for the clean dataset, confirms that it contains exactly 28,348 images—with 2,841 in the test set—verifies the presence of all 24 classes, and ensures that the actual image files are available.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# Expected values for the final dataset
EXPECTED_TOTAL_IMAGES = 28348
EXPECTED_TRAIN_IMAGES = 22672
EXPECTED_VAL_IMAGES = 2835
EXPECTED_TEST_IMAGES = 2841

EXPECTED_NUMBER_OF_CLASSES = 24


# Automatically locate the final manifest
KAGGLE_INPUT_ROOT = Path(
    "/kaggle/input"
)

manifest_candidates = list(
    KAGGLE_INPUT_ROOT.rglob(
        "14_final_verified_dataset_manifest.csv"
    )
)

print("=" * 80)
print("FINAL MANIFESTS FOUND")
print("=" * 80)

for path in manifest_candidates:
    print(path)


# Select the manifest associated with the physical clean dataset
valid_dataset_candidates = []

for manifest_path in manifest_candidates:

    # Search upward through the directory hierarchy
    # until clean_dataset is found.
    for parent in manifest_path.parents:

        candidate_clean_dirs = [
            parent / "clean_dataset",
            parent / "clean_dataset" / "clean_dataset"
        ]

        for clean_dir in candidate_clean_dirs:

            if (
                (clean_dir / "train").is_dir()
                and (clean_dir / "val").is_dir()
                and (clean_dir / "test").is_dir()
            ):

                valid_dataset_candidates.append({
                    "manifest": manifest_path,
                    "clean_dataset": clean_dir
                })

                break

        else:
            continue

        break


# Remove exact duplicates
unique_candidates = []

seen_pairs = set()

for candidate in valid_dataset_candidates:

    key = (
        str(candidate["manifest"]),
        str(candidate["clean_dataset"])
    )

    if key not in seen_pairs:

        seen_pairs.add(key)
        unique_candidates.append(candidate)


if len(unique_candidates) == 0:

    raise FileNotFoundError(
        "The clean dataset with "
        "train, val, and test was not found.\n\n"
        "Add the revision_frutas_2026 dataset as a Kaggle Input."
    )


print("\n" + "=" * 80)
print("CANDIDATES WITH PHYSICAL DATASET")
print("=" * 80)

for index, candidate in enumerate(
    unique_candidates
):

    print(
        f"\nCandidate {index + 1}"
    )

    print(
        "Manifest:",
        candidate["manifest"]
    )

    print(
        "Clean dataset:",
        candidate["clean_dataset"]
    )


# Select a candidate containing 28,348 images
selected_candidate = None
selected_manifest_df = None

for candidate in unique_candidates:

    candidate_df = pd.read_csv(
        candidate["manifest"],
        low_memory=False
    )

    if (
        len(candidate_df)
        == EXPECTED_TOTAL_IMAGES
    ):

        selected_candidate = candidate
        selected_manifest_df = candidate_df
        break


if selected_candidate is None:

    raise RuntimeError(
        "A physical dataset was found, but none "
        "contains the expected 28,348 images."
    )


FINAL_MANIFEST_PATH = (
    selected_candidate["manifest"]
)

CLEAN_DATASET_DIR = (
    selected_candidate["clean_dataset"]
)

dataset_df = (
    selected_manifest_df
    .copy()
)


# Validate required columns
required_columns = {
    "dataset_id",
    "split",
    "class_name",
    "sha256",
    "duplicate_component_id",
    "split_guard_group_id"
}

missing_columns = (
    required_columns
    - set(dataset_df.columns)
)

if missing_columns:

    raise RuntimeError(
        "Required columns are missing from the manifest:\n"
        f"{sorted(missing_columns)}"
    )


# Validate image counts by split
split_counts = (
    dataset_df["split"]
    .value_counts()
    .to_dict()
)

expected_split_counts = {
    "train": EXPECTED_TRAIN_IMAGES,
    "val": EXPECTED_VAL_IMAGES,
    "test": EXPECTED_TEST_IMAGES
}

if split_counts != expected_split_counts:

    raise RuntimeError(
        "Image counts by split do not match.\n\n"
        f"Expected: {expected_split_counts}\n"
        f"Observed: {split_counts}"
    )


# Reproduce the training class mapping
CLASS_NAMES = sorted(
    dataset_df[
        "class_name"
    ].unique()
)

if len(CLASS_NAMES) != EXPECTED_NUMBER_OF_CLASSES:

    raise RuntimeError(
        "The 24 classes were not found."
    )


CLASS_TO_INDEX = {
    class_name: index
    for index, class_name
    in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    index: class_name
    for class_name, index
    in CLASS_TO_INDEX.items()
}


dataset_df["target"] = (
    dataset_df[
        "class_name"
    ].map(
        CLASS_TO_INDEX
    )
)


# Extract only the test split
test_df = (
    dataset_df[
        dataset_df["split"]
        == "test"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(test_df) != EXPECTED_TEST_IMAGES:

    raise RuntimeError(
        "The test split does not contain exactly "
        "2,841 images."
    )


# Reconstruct physical image paths
def resolve_clean_image_path(row):

    # Standard case in the final manifest:
    # clean_relative_path
    if (
        "clean_relative_path"
        in dataset_df.columns
        and pd.notna(
            row.get(
                "clean_relative_path"
            )
        )
    ):

        relative_path = Path(
            str(
                row[
                    "clean_relative_path"
                ]
            )
        )

        # Avoid duplicating "clean_dataset"
        if (
            len(relative_path.parts) > 0
            and relative_path.parts[0]
            == "clean_dataset"
        ):

            relative_path = Path(
                *relative_path.parts[1:]
            )

        candidate = (
            CLEAN_DATASET_DIR
            / relative_path
        )

        if candidate.is_file():
            return str(candidate)


    # Alternative based on the original copied path
    if (
        "clean_path"
        in dataset_df.columns
        and pd.notna(
            row.get(
                "clean_path"
            )
        )
    ):

        old_path = Path(
            str(
                row[
                    "clean_path"
                ]
            )
        )

        # Locate the path starting at the split directory
        parts = old_path.parts

        for split_name in [
            "train",
            "val",
            "test"
        ]:

            if split_name in parts:

                split_position = (
                    parts.index(
                        split_name
                    )
                )

                relative_path = Path(
                    *parts[
                        split_position:
                    ]
                )

                candidate = (
                    CLEAN_DATASET_DIR
                    / relative_path
                )

                if candidate.is_file():
                    return str(
                        candidate
                    )


    # Alternative using the filename
    file_name_columns = [
        "clean_file_name",
        "filename",
        "file_name"
    ]

    for column in file_name_columns:

        if (
            column in dataset_df.columns
            and pd.notna(
                row.get(column)
            )
        ):

            candidate = (
                CLEAN_DATASET_DIR
                / str(row["split"])
                / str(row["class_name"])
                / str(row[column])
            )

            if candidate.is_file():

                return str(
                    candidate
                )


    return None


test_df[
    "input_image_path"
] = test_df.apply(
    resolve_clean_image_path,
    axis=1
)


# Verify physical files
test_df[
    "file_exists"
] = test_df[
    "input_image_path"
].apply(
    lambda path:
        path is not None
        and Path(path).is_file()
)


missing_test_files = test_df[
    ~test_df[
        "file_exists"
    ]
]


if len(missing_test_files) > 0:

    print(
        "\nFirst missing files:"
    )

    display(
        missing_test_files[
            [
                "dataset_id",
                "class_name",
                "input_image_path"
            ]
        ].head(20)
    )

    raise FileNotFoundError(
        f"Could not locate "
        f"{len(missing_test_files)} "
        "test images."
    )


# Check uniqueness
if test_df[
    "dataset_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate dataset_id values exist in the test set."
    )


if test_df[
    "sha256"
].duplicated().any():

    raise RuntimeError(
        "Duplicate SHA-256 hashes exist in the test set."
    )


# Test set class distribution
test_distribution_df = (
    test_df
    .groupby(
        [
            "target",
            "class_name"
        ],
        as_index=False
    )
    .agg(
        images=(
            "dataset_id",
            "size"
        )
    )
    .sort_values(
        "target"
    )
    .reset_index(drop=True)
)


# Overall verification summary
test_verification_df = pd.DataFrame({
    "item": [
        "manifest",
        "clean_dataset",
        "total_images",
        "train_images",
        "validation_images",
        "test_images",
        "classes_total",
        "classes_test",
        "test_files_found",
        "unique_test_dataset_id",
        "unique_test_sha256",
        "test_ready"
    ],

    "value": [
        str(
            FINAL_MANIFEST_PATH
        ),

        str(
            CLEAN_DATASET_DIR
        ),

        len(
            dataset_df
        ),

        split_counts[
            "train"
        ],

        split_counts[
            "val"
        ],

        split_counts[
            "test"
        ],

        dataset_df[
            "class_name"
        ].nunique(),

        test_df[
            "class_name"
        ].nunique(),

        int(
            test_df[
                "file_exists"
            ].sum()
        ),

        test_df[
            "dataset_id"
        ].nunique(),

        test_df[
            "sha256"
        ].nunique(),

        True
    ]
})


# Save the operational test manifest
TEST_MANIFEST_OUTPUT = (
    ANALYSIS_ROOT
    / "03_verified_test_manifest.csv"
)

TEST_DISTRIBUTION_OUTPUT = (
    ANALYSIS_ROOT
    / "04_test_class_distribution.csv"
)

test_df.to_csv(
    TEST_MANIFEST_OUTPUT,
    index=False
)

test_distribution_df.to_csv(
    TEST_DISTRIBUTION_OUTPUT,
    index=False
)


# Display results
print("\n" + "=" * 80)
print("CLEAN TEST SET VERIFIED")
print("=" * 80)

display(
    test_verification_df
)

print("\nClass distribution:")
display(
    test_distribution_df
)

print("\nOperational test manifest:")
print(
    TEST_MANIFEST_OUTPUT
)

print("\nSaved distribution:")
print(
    TEST_DISTRIBUTION_OUTPUT
)

# **Cell 3 — Final validation and test DataLoaders**

In [ ]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd

import torch

from torch.utils.data import (
    Dataset,
    DataLoader
)

from PIL import (
    Image,
    ImageOps,
    ImageFile
)

from torchvision import transforms
from torchvision.transforms import InterpolationMode


# Configuration identical to training

IMG_SIZE = 224

EVAL_RESIZE_SIZE = int(
    round(
        IMG_SIZE * 1.15
    )
)

EVAL_BATCH_SIZE = 64

# Important:
# multiprocessing is disabled due to previous
# Kaggle/Jupyter issues.
NUM_WORKERS = 0

ImageFile.LOAD_TRUNCATED_IMAGES = True


IMAGENET_MEAN = (
    0.485,
    0.456,
    0.406
)

IMAGENET_STD = (
    0.229,
    0.224,
    0.225
)


# Evaluation transform
# It must be exactly the same transform used for
# validation and testing during training.
# It does NOT include:
# - RandomResizedCrop
# - RandomHorizontalFlip
# - RandomErasing
# - MixUp
# - CutMix
# Therefore, evaluation is deterministic.

evaluation_transform = transforms.Compose([

    transforms.Resize(
        size=EVAL_RESIZE_SIZE,
        interpolation=InterpolationMode.BICUBIC,
        antialias=True
    ),

    transforms.CenterCrop(
        size=IMG_SIZE
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# General function for reconstructing physical paths

def resolve_evaluation_image_path(row):

    # Option 1:
    # clean_relative_path
    if (
        "clean_relative_path" in dataset_df.columns
        and pd.notna(
            row.get(
                "clean_relative_path"
            )
        )
    ):

        relative_path = Path(
            str(
                row[
                    "clean_relative_path"
                ]
            )
        )

        # Avoid:
        # clean_dataset/clean_dataset/...
        if (
            len(relative_path.parts) > 0
            and relative_path.parts[0]
            == "clean_dataset"
        ):

            relative_path = Path(
                *relative_path.parts[1:]
            )

        candidate = (
            CLEAN_DATASET_DIR
            / relative_path
        )

        if candidate.is_file():

            return str(
                candidate
            )


    # Option 2:
    # clean_path
    if (
        "clean_path" in dataset_df.columns
        and pd.notna(
            row.get(
                "clean_path"
            )
        )
    ):

        old_path = Path(
            str(
                row[
                    "clean_path"
                ]
            )
        )

        parts = old_path.parts

        for split_name in [
            "train",
            "val",
            "test"
        ]:

            if split_name in parts:

                position = parts.index(
                    split_name
                )

                relative_path = Path(
                    *parts[position:]
                )

                candidate = (
                    CLEAN_DATASET_DIR
                    / relative_path
                )

                if candidate.is_file():

                    return str(
                        candidate
                    )


    # Option 3:
    # reconstruction using the file name
    candidate_filename_columns = [
        "clean_file_name",
        "filename",
        "file_name"
    ]

    for column in candidate_filename_columns:

        if (
            column in dataset_df.columns
            and pd.notna(
                row.get(column)
            )
        ):

            candidate = (
                CLEAN_DATASET_DIR
                / str(
                    row["split"]
                )
                / str(
                    row["class_name"]
                )
                / str(
                    row[column]
                )
            )

            if candidate.is_file():

                return str(
                    candidate
                )


    return None


# Prepare validation

val_df = (
    dataset_df[
        dataset_df["split"]
        == "val"
    ]
    .copy()
    .reset_index(drop=True)
)

val_df[
    "input_image_path"
] = val_df.apply(
    resolve_evaluation_image_path,
    axis=1
)

val_df[
    "file_exists"
] = val_df[
    "input_image_path"
].apply(
    lambda path:
        path is not None
        and Path(path).is_file()
)


# Prepare test

test_df = (
    dataset_df[
        dataset_df["split"]
        == "test"
    ]
    .copy()
    .reset_index(drop=True)
)

test_df[
    "input_image_path"
] = test_df.apply(
    resolve_evaluation_image_path,
    axis=1
)

test_df[
    "file_exists"
] = test_df[
    "input_image_path"
].apply(
    lambda path:
        path is not None
        and Path(path).is_file()
)


# Strict validations

if len(val_df) != 2835:

    raise RuntimeError(
        f"Validation contains {len(val_df)} images; "
        "2835 were expected."
    )


if len(test_df) != 2841:

    raise RuntimeError(
        f"Test contains {len(test_df)} images; "
        "2841 were expected."
    )


if not val_df[
    "file_exists"
].all():

    raise FileNotFoundError(
        "Not all physical "
        "validation images were found."
    )


if not test_df[
    "file_exists"
].all():

    raise FileNotFoundError(
        "Not all physical "
        "test images were found."
    )


if (
    val_df[
        "class_name"
    ].nunique()
    != 24
):

    raise RuntimeError(
        "Validation does not contain all 24 classes."
    )


if (
    test_df[
        "class_name"
    ].nunique()
    != 24
):

    raise RuntimeError(
        "Test does not contain all 24 classes."
    )


# Evaluation dataset

class FruitEvaluationDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform
    ):

        self.dataframe = (
            dataframe
            .copy()
            .reset_index(drop=True)
        )

        self.transform = transform


    def __len__(
        self
    ):

        return len(
            self.dataframe
        )


    @staticmethod
    def open_rgb_image(
        image_path
    ):

        with Image.open(
            image_path
        ) as image:

            image.load()

            try:

                image = (
                    ImageOps.exif_transpose(
                        image
                    )
                )

            except Exception:

                image = image.copy()


            # Composite transparent images onto a white background
            if (
                image.mode in {
                    "RGBA",
                    "LA"
                }
                or "transparency"
                in image.info
            ):

                rgba_image = (
                    image.convert(
                        "RGBA"
                    )
                )

                background = (
                    Image.new(
                        mode="RGBA",
                        size=rgba_image.size,
                        color=(
                            255,
                            255,
                            255,
                            255
                        )
                    )
                )

                image = (
                    Image.alpha_composite(
                        background,
                        rgba_image
                    )
                    .convert(
                        "RGB"
                    )
                )

            else:

                image = image.convert(
                    "RGB"
                )


            return image.copy()


    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[
            index
        ]

        image = self.open_rgb_image(
            row[
                "input_image_path"
            ]
        )

        image = self.transform(
            image
        )

        return {

            "image": image,

            "target": torch.tensor(
                int(
                    row["target"]
                ),
                dtype=torch.long
            ),

            "dataset_id": torch.tensor(
                int(
                    row["dataset_id"]
                ),
                dtype=torch.long
            ),

            "class_name": str(
                row[
                    "class_name"
                ]
            ),

            "sha256": str(
                row[
                    "sha256"
                ]
            ),

            "image_path": str(
                row[
                    "input_image_path"
                ]
            )
        }


# Create datasets

val_dataset = (
    FruitEvaluationDataset(
        dataframe=val_df,
        transform=evaluation_transform
    )
)

test_dataset = (
    FruitEvaluationDataset(
        dataframe=test_df,
        transform=evaluation_transform
    )
)


# Create DataLoaders

val_loader = DataLoader(

    dataset=val_dataset,

    batch_size=EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        torch.cuda.is_available()
    ),

    drop_last=False,

    persistent_workers=False
)


test_loader = DataLoader(

    dataset=test_dataset,

    batch_size=EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        torch.cuda.is_available()
    ),

    drop_last=False,

    persistent_workers=False
)


# Test one batch

val_preview_batch = next(
    iter(
        val_loader
    )
)

test_preview_batch = next(
    iter(
        test_loader
    )
)


# Batch checks

expected_image_shape = (
    3,
    IMG_SIZE,
    IMG_SIZE
)


for split_name, batch in {

    "validation":
        val_preview_batch,

    "test":
        test_preview_batch

}.items():

    observed_shape = tuple(
        batch[
            "image"
        ].shape[1:]
    )

    if (
        observed_shape
        != expected_image_shape
    ):

        raise RuntimeError(
            f"Incorrect image dimensions in "
            f"{split_name}: "
            f"{observed_shape}"
        )


    if (
        batch[
            "target"
        ].min()
        < 0
    ):

        raise RuntimeError(
            f"Negative label in "
            f"{split_name}."
        )


    if (
        batch[
            "target"
        ].max()
        >= 24
    ):

        raise RuntimeError(
            f"Out-of-range label "
            f"en {split_name}."
        )


# Verify class mapping against checkpoints

mapping_check_records = []

for configuration_id in EXPECTED_CONFIGURATIONS:

    run_directory = RUN_DIRECTORIES[
        configuration_id
    ]

    best_checkpoint_path = (
        find_unique_file(
            run_directory,
            "best.pt"
        )
    )

    checkpoint = torch.load(
        best_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    checkpoint_class_mapping = (
        checkpoint.get(
            "class_to_index",
            {}
        )
    )

    normalized_checkpoint_mapping = {
        str(class_name):
        int(class_index)

        for class_name, class_index
        in checkpoint_class_mapping.items()
    }

    mapping_matches = (
        normalized_checkpoint_mapping
        == CLASS_TO_INDEX
    )

    mapping_check_records.append({

        "configuration_id":
            configuration_id,

        "class_mapping_matches":
            mapping_matches
    })

    del checkpoint


class_mapping_check_df = (
    pd.DataFrame(
        mapping_check_records
    )
)


if not class_mapping_check_df[
    "class_mapping_matches"
].all():

    display(
        class_mapping_check_df
    )

    raise RuntimeError(
        "Class mappings do not match "
        "across all checkpoints."
    )


gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# Summary

loader_summary_df = pd.DataFrame({

    "split": [
        "validation",
        "test"
    ],

    "images": [
        len(
            val_dataset
        ),
        len(
            test_dataset
        )
    ],

    "classes": [
        val_df[
            "class_name"
        ].nunique(),

        test_df[
            "class_name"
        ].nunique()
    ],

    "batch_size": [
        EVAL_BATCH_SIZE,
        EVAL_BATCH_SIZE
    ],

    "batches": [
        len(
            val_loader
        ),

        len(
            test_loader
        )
    ],

    "num_workers": [
        NUM_WORKERS,
        NUM_WORKERS
    ],

    "shuffle": [
        False,
        False
    ]
})


batch_summary_df = pd.DataFrame({

    "split": [
        "validation",
        "test"
    ],

    "batch_shape": [

        str(
            tuple(
                val_preview_batch[
                    "image"
                ].shape
            )
        ),

        str(
            tuple(
                test_preview_batch[
                    "image"
                ].shape
            )
        )
    ],

    "first_dataset_id": [

        int(
            val_preview_batch[
                "dataset_id"
            ][0]
        ),

        int(
            test_preview_batch[
                "dataset_id"
            ][0]
        )
    ],

    "first_target": [

        int(
            val_preview_batch[
                "target"
            ][0]
        ),

        int(
            test_preview_batch[
                "target"
            ][0]
        )
    ]
})


# Save verification results

LOADER_SUMMARY_OUTPUT = (
    ANALYSIS_ROOT
    / "05_evaluation_loader_summary.csv"
)

CLASS_MAPPING_OUTPUT = (
    ANALYSIS_ROOT
    / "06_checkpoint_class_mapping_verification.csv"
)

loader_summary_df.to_csv(
    LOADER_SUMMARY_OUTPUT,
    index=False
)

class_mapping_check_df.to_csv(
    CLASS_MAPPING_OUTPUT,
    index=False
)


# Display results

print("=" * 80)
print("EVALUATION DATALOADERS CONFIGURED")
print("=" * 80)

print("\nSummary:")
display(
    loader_summary_df
)

print("\nBatch test:")
display(
    batch_summary_df
)

print("\nClass mapping:")
display(
    class_mapping_check_df
)

print(
    "\nAll 12 checkpoints use "
    "the same mapping for the 24 classes."
)

print(
    "\nMultiprocessing disabled:",
    NUM_WORKERS == 0
)

print("\nSaved files:")
print(
    LOADER_SUMMARY_OUTPUT
)
print(
    CLASS_MAPPING_OUTPUT
)

# **Cell 4 — Final inference of the 12 best.pt**

In [ ]:
from pathlib import Path
import gc
import time

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader

import timm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)


# Configuration

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = (
    torch.cuda.is_available()
)

NUM_CLASSES = 24

PREFERRED_BATCH_SIZE = 64

OOM_FALLBACK_BATCH_SIZE = 32

VALIDATION_REPRODUCTION_TOLERANCE = 5e-4


# Result directories

RAW_OUTPUT_DIR = (
    ANALYSIS_ROOT
    / "raw_outputs"
)

PREDICTIONS_DIR = (
    ANALYSIS_ROOT
    / "predictions"
)

METRICS_DIR = (
    ANALYSIS_ROOT
    / "point_metrics"
)

for directory in [
    RAW_OUTPUT_DIR,
    PREDICTIONS_DIR,
    METRICS_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# Build loaders by batch size

def build_evaluation_loaders(
    batch_size
):

    val_loader_local = DataLoader(

        dataset=val_dataset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            torch.cuda.is_available()
        ),

        drop_last=False,

        persistent_workers=False
    )


    test_loader_local = DataLoader(

        dataset=test_dataset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            torch.cuda.is_available()
        ),

        drop_last=False,

        persistent_workers=False
    )


    return (
        val_loader_local,
        test_loader_local
    )


# Load a model from a checkpoint

def load_model_from_best_checkpoint(
    checkpoint_path
):

    checkpoint = torch.load(

        checkpoint_path,

        map_location="cpu",

        weights_only=False
    )


    configuration = checkpoint[
        "configuration"
    ]


    timm_model_name = configuration[
        "timm_model"
    ]


    model = timm.create_model(

        timm_model_name,

        pretrained=False,

        num_classes=NUM_CLASSES
    )


    model.load_state_dict(

        checkpoint[
            "model_state_dict"
        ],

        strict=True
    )


    model = model.to(
        DEVICE
    )

    model.eval()


    return (
        model,
        checkpoint
    )


# Run inference

def run_inference(
    model,
    data_loader
):

    logits_parts = []

    targets_parts = []

    dataset_id_parts = []


    if torch.cuda.is_available():

        torch.cuda.synchronize()


    start_time = (
        time.perf_counter()
    )


    with torch.inference_mode():

        for batch in data_loader:

            images = batch[
                "image"
            ].to(

                DEVICE,

                non_blocking=True
            )


            targets = batch[
                "target"
            ]


            dataset_ids = batch[
                "dataset_id"
            ]


            with torch.amp.autocast(

                device_type=(
                    "cuda"
                    if DEVICE.type == "cuda"
                    else "cpu"
                ),

                dtype=torch.float16,

                enabled=USE_AMP
            ):

                logits = model(
                    images
                )


            logits_parts.append(

                logits
                .detach()
                .float()
                .cpu()
            )


            targets_parts.append(

                targets
                .detach()
                .cpu()
            )


            dataset_id_parts.append(

                dataset_ids
                .detach()
                .cpu()
            )


    if torch.cuda.is_available():

        torch.cuda.synchronize()


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    logits = torch.cat(
        logits_parts,
        dim=0
    )


    targets = torch.cat(
        targets_parts,
        dim=0
    )


    dataset_ids = torch.cat(
        dataset_id_parts,
        dim=0
    )


    probabilities = torch.softmax(

        logits,

        dim=1
    )


    confidence, predictions = (
        probabilities.max(
            dim=1
        )
    )


    return {

        "logits":
            logits.numpy(),

        "probabilities":
            probabilities.numpy(),

        "targets":
            targets.numpy(),

        "dataset_ids":
            dataset_ids.numpy(),

        "predictions":
            predictions.numpy(),

        "confidence":
            confidence.numpy(),

        "seconds":
            elapsed_seconds
    }


# Point metrics

def compute_point_metrics(
    targets,
    predictions
):

    return {

        "accuracy": float(
            accuracy_score(
                targets,
                predictions
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                targets,
                predictions
            )
        ),

        "macro_f1": float(
            f1_score(

                targets,

                predictions,

                labels=list(
                    range(
                        NUM_CLASSES
                    )
                ),

                average="macro",

                zero_division=0
            )
        )
    }


# Build the predictions DataFrame

def build_prediction_dataframe(
    inference_result,
    reference_dataframe,
    split_name
):

    prediction_df = pd.DataFrame({

        "dataset_id":
            inference_result[
                "dataset_ids"
            ].astype(
                np.int64
            ),

        "target":
            inference_result[
                "targets"
            ].astype(
                np.int64
            ),

        "prediction":
            inference_result[
                "predictions"
            ].astype(
                np.int64
            ),

        "confidence":
            inference_result[
                "confidence"
            ].astype(
                np.float64
            )
    })


    prediction_df[
        "true_class"
    ] = prediction_df[
        "target"
    ].map(
        INDEX_TO_CLASS
    )


    prediction_df[
        "predicted_class"
    ] = prediction_df[
        "prediction"
    ].map(
        INDEX_TO_CLASS
    )


    prediction_df[
        "correct"
    ] = (

        prediction_df[
            "target"
        ]

        ==

        prediction_df[
            "prediction"
        ]
    )


    metadata_df = (

        reference_dataframe[
            [
                "dataset_id",
                "sha256",
                "class_name",
                "input_image_path"
            ]
        ]
        .copy()
    )


    prediction_df = prediction_df.merge(

        metadata_df,

        on="dataset_id",

        how="left",

        validate="one_to_one"
    )


    prediction_df[
        "split"
    ] = split_name


    return prediction_df


# Prepare stored validation reference

validation_reference = (

    validation_summary_df
    .set_index(
        "configuration_id"
    )
)


# Run all 12 configurations

all_metric_records = []

reproduction_records = []


print("=" * 80)
print("INFERENCE USING THE 12 BEST.PT CHECKPOINTS")
print("=" * 80)


for run_number, configuration_id in enumerate(

    EXPECTED_CONFIGURATIONS,

    start=1

):

    print(
        f"\n[{run_number}/12] "
        f"{configuration_id}"
    )


    run_directory = (
        RUN_DIRECTORIES[
            configuration_id
        ]
    )


    best_checkpoint_path = (
        find_unique_file(
            run_directory,
            "best.pt"
        )
    )


    # Load model

    model, checkpoint = (
        load_model_from_best_checkpoint(
            best_checkpoint_path
        )
    )


    batch_size_used = (
        PREFERRED_BATCH_SIZE
    )


    try:

        (
            current_val_loader,
            current_test_loader
        ) = build_evaluation_loaders(
            batch_size_used
        )


        val_result = run_inference(

            model,
            current_val_loader
        )


        test_result = run_inference(

            model,
            current_test_loader
        )


    except RuntimeError as error:

        is_cuda_oom = (

            "out of memory"
            in str(error).lower()
        )


        if not is_cuda_oom:

            raise


        print(
            "CUDA out of memory with batch size 64."
        )

        print(
            "Retrying with batch size 32..."
        )


        del current_val_loader
        del current_test_loader


        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


        batch_size_used = (
            OOM_FALLBACK_BATCH_SIZE
        )


        (
            current_val_loader,
            current_test_loader
        ) = build_evaluation_loaders(
            batch_size_used
        )


        val_result = run_inference(

            model,
            current_val_loader
        )


        test_result = run_inference(

            model,
            current_test_loader
        )


    # Verify sample counts

    if len(
        val_result[
            "targets"
        ]
    ) != 2835:

        raise RuntimeError(
            f"{configuration_id}: "
            "incorrect number of validation samples."
        )


    if len(
        test_result[
            "targets"
        ]
    ) != 2841:

        raise RuntimeError(
            f"{configuration_id}: "
            "incorrect number of test samples."
        )


    # Metrics

    val_metrics = compute_point_metrics(

        val_result[
            "targets"
        ],

        val_result[
            "predictions"
        ]
    )


    test_metrics = compute_point_metrics(

        test_result[
            "targets"
        ],

        test_result[
            "predictions"
        ]
    )


    # Verify validation result reproducibility

    expected_val_macro_f1 = float(

        validation_reference.loc[
            configuration_id,
            "val_macro_f1"
        ]
    )


    reproduced_val_macro_f1 = (

        val_metrics[
            "macro_f1"
        ]
    )


    absolute_difference = abs(

        reproduced_val_macro_f1

        -

        expected_val_macro_f1
    )


    validation_reproduced = (

        absolute_difference
        <=
        VALIDATION_REPRODUCTION_TOLERANCE
    )


    reproduction_records.append({

        "configuration_id":
            configuration_id,

        "expected_val_macro_f1":
            expected_val_macro_f1,

        "reproduced_val_macro_f1":
            reproduced_val_macro_f1,

        "absolute_difference":
            absolute_difference,

        "within_tolerance":
            validation_reproduced
    })


    if not validation_reproduced:

        raise RuntimeError(

            f"{configuration_id}: "
            "best.pt did not reproduce the validation Macro-F1 "
            "within the specified tolerance.\n"
            f"Expected: {expected_val_macro_f1:.6f}\n"
            f"Reproduced: {reproduced_val_macro_f1:.6f}\n"
            f"Difference: {absolute_difference:.6f}"
        )


    # Save raw arrays

    raw_output_path = (

        RAW_OUTPUT_DIR
        / f"{configuration_id}_raw_outputs.npz"
    )


    np.savez_compressed(

        raw_output_path,

        val_dataset_ids=
            val_result[
                "dataset_ids"
            ],

        val_targets=
            val_result[
                "targets"
            ],

        val_predictions=
            val_result[
                "predictions"
            ],

        val_confidence=
            val_result[
                "confidence"
            ],

        val_logits=
            val_result[
                "logits"
            ],

        val_probabilities=
            val_result[
                "probabilities"
            ],


        test_dataset_ids=
            test_result[
                "dataset_ids"
            ],

        test_targets=
            test_result[
                "targets"
            ],

        test_predictions=
            test_result[
                "predictions"
            ],

        test_confidence=
            test_result[
                "confidence"
            ],

        test_logits=
            test_result[
                "logits"
            ],

        test_probabilities=
            test_result[
                "probabilities"
            ]
    )


    # Create prediction CSV files

    val_prediction_df = (
        build_prediction_dataframe(

            val_result,

            val_df,

            "validation"
        )
    )


    test_prediction_df = (
        build_prediction_dataframe(

            test_result,

            test_df,

            "test"
        )
    )


    val_prediction_path = (

        PREDICTIONS_DIR
        / (
            f"{configuration_id}"
            "_validation_predictions.csv"
        )
    )


    test_prediction_path = (

        PREDICTIONS_DIR
        / (
            f"{configuration_id}"
            "_test_predictions.csv"
        )
    )


    val_prediction_df.to_csv(

        val_prediction_path,

        index=False
    )


    test_prediction_df.to_csv(

        test_prediction_path,

        index=False
    )


    # Global metrics records

    all_metric_records.append({

        "configuration_id":
            configuration_id,

        "model":
            checkpoint[
                "configuration"
            ][
                "model_key"
            ],

        "augmentation":
            checkpoint[
                "configuration"
            ][
                "augmentation"
            ],

        "selected_epoch":
            checkpoint[
                "best_epoch"
            ],


        "val_accuracy":
            val_metrics[
                "accuracy"
            ],

        "val_balanced_accuracy":
            val_metrics[
                "balanced_accuracy"
            ],

        "val_macro_f1":
            val_metrics[
                "macro_f1"
            ],


        "test_accuracy":
            test_metrics[
                "accuracy"
            ],

        "test_balanced_accuracy":
            test_metrics[
                "balanced_accuracy"
            ],

        "test_macro_f1":
            test_metrics[
                "macro_f1"
            ],


        "evaluation_batch_size":
            batch_size_used,

        # These are total elapsed times for this pass,
        # NOT final latency estimates.
        "val_wall_clock_seconds":
            val_result[
                "seconds"
            ],

        "test_wall_clock_seconds":
            test_result[
                "seconds"
            ],

        "raw_outputs_path":
            str(
                raw_output_path
            ),

        "test_predictions_path":
            str(
                test_prediction_path
            )
    })


    print(
        "  Val Macro-F1:",
        f"{val_metrics['macro_f1']:.6f}"
    )

    print(
        "  Test Accuracy:",
        f"{test_metrics['accuracy']:.6f}"
    )

    print(
        "  Test Balanced Accuracy:",
        f"{test_metrics['balanced_accuracy']:.6f}"
    )

    print(
        "  Test Macro-F1:",
        f"{test_metrics['macro_f1']:.6f}"
    )

    print(
        "  Batch:",
        batch_size_used
    )


    # Release GPU memory before the next model

    del model
    del checkpoint

    del val_result
    del test_result

    del val_prediction_df
    del test_prediction_df

    del current_val_loader
    del current_test_loader


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# Create global tables

point_metrics_df = pd.DataFrame(
    all_metric_records
)


reproduction_df = pd.DataFrame(
    reproduction_records
)


# Add validation ranking

validation_rank_mapping = (

    validation_summary_df
    .set_index(
        "configuration_id"
    )[
        "validation_rank"
    ]
    .to_dict()
)


point_metrics_df[
    "validation_rank"
] = point_metrics_df[
    "configuration_id"
].map(
    validation_rank_mapping
)


# Test ranking:
# descriptive only.
point_metrics_df[
    "test_rank_descriptive"
] = (

    point_metrics_df[
        "test_macro_f1"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# Methodological ordering:
# keep validation ranking first.
point_metrics_df = (

    point_metrics_df
    .sort_values(
        "validation_rank"
    )
    .reset_index(
        drop=True
    )
)


# Save tables

POINT_METRICS_OUTPUT = (

    METRICS_DIR
    / "07_all_models_point_metrics.csv"
)


REPRODUCTION_OUTPUT = (

    METRICS_DIR
    / "08_validation_reproduction_check.csv"
)


point_metrics_df.to_csv(

    POINT_METRICS_OUTPUT,

    index=False
)


reproduction_df.to_csv(

    REPRODUCTION_OUTPUT,

    index=False
)


# Global verification

if not reproduction_df[
    "within_tolerance"
].all():

    raise RuntimeError(
        "At least one checkpoint failed to reproduce "
        "the validation result."
    )


# Display results

print("\n" + "=" * 80)
print("INFERENCE COMPLETED FOR ALL 12 MODELS")
print("=" * 80)


print("\nValidation reproducibility:")

display(

    reproduction_df[
        [
            "configuration_id",
            "expected_val_macro_f1",
            "reproduced_val_macro_f1",
            "absolute_difference",
            "within_tolerance"
        ]
    ]
)


print("\nPoint metrics:")

display(

    point_metrics_df[
        [
            "validation_rank",
            "configuration_id",
            "selected_epoch",
            "val_macro_f1",
            "test_accuracy",
            "test_balanced_accuracy",
            "test_macro_f1",
            "test_rank_descriptive"
        ]
    ]
)


print("\nMain output files:")

print(
    POINT_METRICS_OUTPUT
)

print(
    REPRODUCTION_OUTPUT
)

print(
    RAW_OUTPUT_DIR
)

print(
    PREDICTIONS_DIR
)


print(
    "\nLogits and probabilities for all 12 configurations "
    "have been saved."
)

# **Cell 5 — ECE + Temperature Scaling**

In [ ]:
from pathlib import Path
import gc
import math

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F


# Configuration

N_ECE_BINS = 15

CALIBRATION_DIR = (
    ANALYSIS_ROOT
    / "calibration"
)

RELIABILITY_DIR = (
    CALIBRATION_DIR
    / "reliability_bins"
)

CALIBRATED_OUTPUT_DIR = (
    CALIBRATION_DIR
    / "calibrated_outputs"
)

for directory in [
    CALIBRATION_DIR,
    RELIABILITY_DIR,
    CALIBRATED_OUTPUT_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# Negative log-likelihood

def calculate_nll(
    logits,
    targets
):

    logits_tensor = torch.as_tensor(
        logits,
        dtype=torch.float64
    )

    targets_tensor = torch.as_tensor(
        targets,
        dtype=torch.long
    )

    nll = F.cross_entropy(
        logits_tensor,
        targets_tensor
    )

    return float(
        nll.item()
    )


# Expected calibration error (ECE)

def calculate_ece(
    probabilities,
    targets,
    n_bins=N_ECE_BINS
):

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    targets = np.asarray(
        targets,
        dtype=np.int64
    )

    predictions = probabilities.argmax(
        axis=1
    )

    confidence = probabilities.max(
        axis=1
    )

    correct = (
        predictions
        == targets
    ).astype(
        np.float64
    )


    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )


    ece = 0.0

    bin_records = []


    for bin_index in range(
        n_bins
    ):

        lower_bound = (
            bin_edges[
                bin_index
            ]
        )

        upper_bound = (
            bin_edges[
                bin_index + 1
            ]
        )


        if (
            bin_index
            == n_bins - 1
        ):

            in_bin = (
                (confidence >= lower_bound)
                &
                (confidence <= upper_bound)
            )

        else:

            in_bin = (
                (confidence >= lower_bound)
                &
                (confidence < upper_bound)
            )


        count = int(
            in_bin.sum()
        )


        if count > 0:

            mean_confidence = float(
                confidence[
                    in_bin
                ].mean()
            )

            accuracy = float(
                correct[
                    in_bin
                ].mean()
            )

            absolute_gap = abs(
                accuracy
                - mean_confidence
            )

            proportion = (
                count
                / len(targets)
            )

            ece += (
                proportion
                * absolute_gap
            )

        else:

            mean_confidence = np.nan
            accuracy = np.nan
            absolute_gap = np.nan
            proportion = 0.0


        bin_records.append({

            "bin_index":
                bin_index + 1,

            "lower_bound":
                lower_bound,

            "upper_bound":
                upper_bound,

            "count":
                count,

            "proportion":
                proportion,

            "mean_confidence":
                mean_confidence,

            "accuracy":
                accuracy,

            "absolute_gap":
                absolute_gap
        })


    reliability_df = pd.DataFrame(
        bin_records
    )


    return (
        float(ece),
        reliability_df
    )


# Fit temperature on validation data

def fit_temperature(
    validation_logits,
    validation_targets
):

    logits = torch.as_tensor(
        validation_logits,
        dtype=torch.float64
    )

    targets = torch.as_tensor(
        validation_targets,
        dtype=torch.long
    )


    # Optimizing log(T) guarantees T > 0
    log_temperature = torch.nn.Parameter(
        torch.zeros(
            1,
            dtype=torch.float64
        )
    )


    optimizer = torch.optim.LBFGS(

        [log_temperature],

        lr=0.05,

        max_iter=200,

        tolerance_grad=1e-10,

        tolerance_change=1e-12,

        line_search_fn="strong_wolfe"
    )


    def closure():

        optimizer.zero_grad()

        temperature = torch.exp(
            log_temperature
        )

        scaled_logits = (
            logits
            / temperature
        )

        loss = F.cross_entropy(
            scaled_logits,
            targets
        )

        loss.backward()

        return loss


    optimizer.step(
        closure
    )


    temperature = float(
        torch.exp(
            log_temperature
        )
        .detach()
        .item()
    )


    if (
        not np.isfinite(
            temperature
        )
        or temperature <= 0
    ):

        raise RuntimeError(
            "Temperature Scaling produced "
            "an invalid temperature."
        )


    return temperature


# Stable softmax

def softmax_numpy(
    logits
):

    logits_tensor = torch.as_tensor(
        logits,
        dtype=torch.float64
    )

    probabilities = torch.softmax(
        logits_tensor,
        dim=1
    )

    return (
        probabilities
        .numpy()
        .astype(
            np.float32
        )
    )


# Process the 12 configurations

calibration_records = []


print("=" * 80)
print("TEMPERATURE SCALING Y ECE")
print("=" * 80)


for run_number, configuration_id in enumerate(
    EXPECTED_CONFIGURATIONS,
    start=1
):

    print(
        f"\n[{run_number}/12] "
        f"{configuration_id}"
    )


    raw_output_path = (
        RAW_OUTPUT_DIR
        / f"{configuration_id}_raw_outputs.npz"
    )


    if not raw_output_path.exists():

        raise FileNotFoundError(
            f"No existe:\n"
            f"{raw_output_path}"
        )


    # Load saved outputs

    raw = np.load(
        raw_output_path
    )


    val_logits = raw[
        "val_logits"
    ].astype(
        np.float64
    )

    val_targets = raw[
        "val_targets"
    ].astype(
        np.int64
    )


    test_logits = raw[
        "test_logits"
    ].astype(
        np.float64
    )

    test_targets = raw[
        "test_targets"
    ].astype(
        np.int64
    )


    test_dataset_ids = raw[
        "test_dataset_ids"
    ].astype(
        np.int64
    )


    # Before calibration

    val_probabilities_before = (
        softmax_numpy(
            val_logits
        )
    )

    test_probabilities_before = (
        softmax_numpy(
            test_logits
        )
    )


    val_ece_before, val_bins_before = (
        calculate_ece(
            val_probabilities_before,
            val_targets
        )
    )


    test_ece_before, test_bins_before = (
        calculate_ece(
            test_probabilities_before,
            test_targets
        )
    )


    val_nll_before = (
        calculate_nll(
            val_logits,
            val_targets
        )
    )


    test_nll_before = (
        calculate_nll(
            test_logits,
            test_targets
        )
    )


    # Fit temperature using validation data ONLY

    temperature = (
        fit_temperature(
            validation_logits=
                val_logits,

            validation_targets=
                val_targets
        )
    )


    # Apply temperature scaling

    calibrated_val_logits = (
        val_logits
        / temperature
    )

    calibrated_test_logits = (
        test_logits
        / temperature
    )


    val_probabilities_after = (
        softmax_numpy(
            calibrated_val_logits
        )
    )

    test_probabilities_after = (
        softmax_numpy(
            calibrated_test_logits
        )
    )


    # After calibration

    val_ece_after, val_bins_after = (
        calculate_ece(
            val_probabilities_after,
            val_targets
        )
    )


    test_ece_after, test_bins_after = (
        calculate_ece(
            test_probabilities_after,
            test_targets
        )
    )


    val_nll_after = (
        calculate_nll(
            calibrated_val_logits,
            val_targets
        )
    )


    test_nll_after = (
        calculate_nll(
            calibrated_test_logits,
            test_targets
        )
    )


    # Verify that argmax has NOT changed

    test_predictions_before = (
        test_probabilities_before.argmax(
            axis=1
        )
    )

    test_predictions_after = (
        test_probabilities_after.argmax(
            axis=1
        )
    )


    prediction_changes = int(

        np.sum(

            test_predictions_before
            !=
            test_predictions_after
        )
    )


    if prediction_changes != 0:

        raise RuntimeError(
            f"{configuration_id}: "
            "Temperature Scaling changed "
            "predictions, which should not occur."
        )


    # Save calibrated probabilities

    calibrated_output_path = (

        CALIBRATED_OUTPUT_DIR
        / (
            f"{configuration_id}"
            "_temperature_scaled.npz"
        )
    )


    np.savez_compressed(

        calibrated_output_path,

        temperature=
            np.array(
                [temperature],
                dtype=np.float64
            ),

        test_dataset_ids=
            test_dataset_ids,

        test_targets=
            test_targets,

        test_logits_original=
            test_logits.astype(
                np.float32
            ),

        test_logits_calibrated=
            calibrated_test_logits.astype(
                np.float32
            ),

        test_probabilities_original=
            test_probabilities_before.astype(
                np.float32
            ),

        test_probabilities_calibrated=
            test_probabilities_after.astype(
                np.float32
            )
    )


    # Save reliability bins

    reliability_tables = {

        "validation_before":
            val_bins_before,

        "validation_after":
            val_bins_after,

        "test_before":
            test_bins_before,

        "test_after":
            test_bins_after
    }


    for label, reliability_df in (
        reliability_tables.items()
    ):

        reliability_df.insert(
            0,
            "configuration_id",
            configuration_id
        )

        reliability_df.insert(
            1,
            "calibration_state",
            label
        )

        reliability_path = (

            RELIABILITY_DIR
            / (
                f"{configuration_id}"
                f"__{label}.csv"
            )
        )

        reliability_df.to_csv(
            reliability_path,
            index=False
        )


    # Global record

    calibration_records.append({

        "configuration_id":
            configuration_id,

        "temperature":
            temperature,


        "val_ece_before":
            val_ece_before,

        "val_ece_after":
            val_ece_after,

        "val_ece_change":
            (
                val_ece_after
                - val_ece_before
            ),


        "test_ece_before":
            test_ece_before,

        "test_ece_after":
            test_ece_after,

        "test_ece_change":
            (
                test_ece_after
                - test_ece_before
            ),


        "val_nll_before":
            val_nll_before,

        "val_nll_after":
            val_nll_after,


        "test_nll_before":
            test_nll_before,

        "test_nll_after":
            test_nll_after,


        "test_prediction_changes":
            prediction_changes,

        "ece_bins":
            N_ECE_BINS,

        "temperature_fitted_on":
            "validation",

        "calibrated_output_path":
            str(
                calibrated_output_path
            )
    })


    print(
        "  Temperature:",
        f"{temperature:.4f}"
    )

    print(
        "  Val ECE:",
        f"{val_ece_before:.6f}",
        "->",
        f"{val_ece_after:.6f}"
    )

    print(
        "  Test ECE:",
        f"{test_ece_before:.6f}",
        "->",
        f"{test_ece_after:.6f}"
    )

    print(
        "  Test NLL:",
        f"{test_nll_before:.6f}",
        "->",
        f"{test_nll_after:.6f}"
    )


    raw.close()

    del raw

    gc.collect()


# Global summary table

calibration_summary_df = pd.DataFrame(
    calibration_records
)


# Include validation ranking

calibration_summary_df[
    "validation_rank"
] = calibration_summary_df[
    "configuration_id"
].map(
    validation_rank_mapping
)


calibration_summary_df = (

    calibration_summary_df
    .sort_values(
        "validation_rank"
    )
    .reset_index(
        drop=True
    )
)


# Save summary

CALIBRATION_SUMMARY_OUTPUT = (

    CALIBRATION_DIR
    / "09_temperature_scaling_ece_summary.csv"
)


calibration_summary_df.to_csv(

    CALIBRATION_SUMMARY_OUTPUT,

    index=False
)


# Final verification

if (
    calibration_summary_df[
        "test_prediction_changes"
    ].sum()
    != 0
):

    raise RuntimeError(
        "At least one configuration changed "
        "its predictions after Temperature Scaling."
    )


# Display results

print("\n" + "=" * 80)
print("CALIBRATION COMPLETED")
print("=" * 80)


display(

    calibration_summary_df[
        [
            "validation_rank",
            "configuration_id",
            "temperature",
            "val_ece_before",
            "val_ece_after",
            "test_ece_before",
            "test_ece_after",
            "test_nll_before",
            "test_nll_after",
            "test_prediction_changes"
        ]
    ]
)


print(
    "\nAll temperatures were fitted "
    "exclusively using validation data."
)

print(
    "Temperature Scaling did not change any "
    "test prediction."
)

print("\nSummary saved to:")
print(
    CALIBRATION_SUMMARY_OUTPUT
)

print("\nCalibrated outputs:")
print(
    CALIBRATED_OUTPUT_DIR
)

print("\nReliability-diagram data:")
print(
    RELIABILITY_DIR
)

# **Cell 6 — 95% bootstrap CI, stratified by class and group**

In [ ]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)


# Configuration

BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 42
CI_LEVEL = 0.95

LOWER_PERCENTILE = (
    100 * (1 - CI_LEVEL) / 2
)

UPPER_PERCENTILE = (
    100 * (1 + CI_LEVEL) / 2
)

BOOTSTRAP_DIR = (
    ANALYSIS_ROOT
    / "bootstrap_ci"
)

BOOTSTRAP_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Verify grouping variable

if (
    "split_guard_group_id"
    not in test_df.columns
):

    raise RuntimeError(
        "test_df does not contain split_guard_group_id."
    )


if test_df[
    "split_guard_group_id"
].isna().any():

    raise RuntimeError(
        "Some split_guard_group_id values are missing."
    )


# Create test metadata

bootstrap_metadata_df = (
    test_df[
        [
            "dataset_id",
            "target",
            "class_name",
            "split_guard_group_id"
        ]
    ]
    .copy()
)


bootstrap_metadata_df[
    "dataset_id"
] = bootstrap_metadata_df[
    "dataset_id"
].astype(
    np.int64
)


# Groups by class

class_group_dictionary = {}


for class_index in range(
    NUM_CLASSES
):

    class_subset = (
        bootstrap_metadata_df[
            bootstrap_metadata_df[
                "target"
            ] == class_index
        ]
    )


    unique_groups = (
        class_subset[
            "split_guard_group_id"
        ]
        .astype(str)
        .unique()
    )


    if len(
        unique_groups
    ) == 0:

        raise RuntimeError(
            f"Class {class_index} "
            "has no groups."
        )


    class_group_dictionary[
        class_index
    ] = unique_groups


# Group summary

group_summary_records = []


for class_index in range(
    NUM_CLASSES
):

    subset = (
        bootstrap_metadata_df[
            bootstrap_metadata_df[
                "target"
            ] == class_index
        ]
    )


    group_sizes = (
        subset
        .groupby(
            "split_guard_group_id"
        )
        .size()
    )


    group_summary_records.append({

        "target":
            class_index,

        "class_name":
            INDEX_TO_CLASS[
                class_index
            ],

        "images":
            len(
                subset
            ),

        "guard_groups":
            len(
                group_sizes
            ),

        "singleton_groups":
            int(
                (
                    group_sizes == 1
                ).sum()
            ),

        "multi_image_groups":
            int(
                (
                    group_sizes > 1
                ).sum()
            ),

        "largest_group":
            int(
                group_sizes.max()
            )
    })


bootstrap_group_summary_df = pd.DataFrame(
    group_summary_records
)


# Generate the 2,000 bootstrap samples ONCE
# Use exactly the same samples for
# all 12 configurations.
# This improves comparability across models.

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


# Global mapping:
# dataset_id -> original row in test_df
dataset_id_to_position = {

    int(dataset_id): position

    for position, dataset_id
    in enumerate(
        test_df[
            "dataset_id"
        ].to_numpy()
    )
}


# Mapping:
# group -> dataset_ids
group_to_dataset_ids = (

    bootstrap_metadata_df
    .groupby(
        "split_guard_group_id"
    )[
        "dataset_id"
    ]
    .apply(
        lambda values:
            values
            .astype(
                np.int64
            )
            .to_numpy()
    )
    .to_dict()
)


bootstrap_index_list = []


print("=" * 80)
print("GENERATING BOOTSTRAP SAMPLES")
print("=" * 80)


for bootstrap_index in range(
    BOOTSTRAP_REPLICATES
):

    sampled_positions = []


    # Independent resampling within each class

    for class_index in range(
        NUM_CLASSES
    ):

        class_groups = (
            class_group_dictionary[
                class_index
            ]
        )


        sampled_groups = rng.choice(

            class_groups,

            size=len(
                class_groups
            ),

            replace=True
        )


        for group_id in sampled_groups:

            dataset_ids = (
                group_to_dataset_ids[
                    group_id
                ]
            )


            sampled_positions.extend(

                dataset_id_to_position[
                    int(dataset_id)
                ]

                for dataset_id
                in dataset_ids
            )


    bootstrap_index_list.append(

        np.asarray(
            sampled_positions,
            dtype=np.int32
        )
    )


print(
    "Replicates generated:",
    len(
        bootstrap_index_list
    )
)


# Save bootstrap design

BOOTSTRAP_DESIGN_PATH = (
    BOOTSTRAP_DIR
    / "10_bootstrap_design.npz"
)


# Samples may have slightly different lengths
# because groups have different sizes.
np.savez_compressed(

    BOOTSTRAP_DESIGN_PATH,

    bootstrap_indices=np.array(
        bootstrap_index_list,
        dtype=object
    ),

    seed=np.array(
        [BOOTSTRAP_SEED]
    ),

    replicates=np.array(
        [BOOTSTRAP_REPLICATES]
    )
)


# NumPy NLL function

def numpy_nll(
    probabilities,
    targets
):

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    targets = np.asarray(
        targets,
        dtype=np.int64
    )


    true_probabilities = (
        probabilities[
            np.arange(
                len(targets)
            ),
            targets
        ]
    )


    true_probabilities = np.clip(

        true_probabilities,

        1e-15,

        1.0
    )


    return float(

        -np.log(
            true_probabilities
        ).mean()
    )


# Function to summarize confidence intervals

def summarize_bootstrap_metric(
    point_estimate,
    bootstrap_values
):

    bootstrap_values = np.asarray(
        bootstrap_values,
        dtype=np.float64
    )


    return {

        "estimate":
            float(
                point_estimate
            ),

        "ci_lower":
            float(
                np.percentile(
                    bootstrap_values,
                    LOWER_PERCENTILE
                )
            ),

        "ci_upper":
            float(
                np.percentile(
                    bootstrap_values,
                    UPPER_PERCENTILE
                )
            ),

        "bootstrap_mean":
            float(
                bootstrap_values.mean()
            ),

        "bootstrap_std":
            float(
                bootstrap_values.std(
                    ddof=1
                )
            )
    }


# Process the 12 configurations

bootstrap_summary_records = []

bootstrap_long_records = []


print("\n" + "=" * 80)
print("CALCULATING CONFIDENCE INTERVALS")
print("=" * 80)


for run_number, configuration_id in enumerate(

    EXPECTED_CONFIGURATIONS,

    start=1

):

    print(
        f"\n[{run_number}/12] "
        f"{configuration_id}"
    )


    # Original outputs

    raw_output_path = (

        RAW_OUTPUT_DIR
        / f"{configuration_id}_raw_outputs.npz"
    )


    raw = np.load(
        raw_output_path
    )


    raw_dataset_ids = (
        raw[
            "test_dataset_ids"
        ].astype(
            np.int64
        )
    )


    raw_targets = (
        raw[
            "test_targets"
        ].astype(
            np.int64
        )
    )


    raw_predictions = (
        raw[
            "test_predictions"
        ].astype(
            np.int64
        )
    )


    raw_probabilities = (
        raw[
            "test_probabilities"
        ].astype(
            np.float64
        )
    )


    # Verify the same ordering as test_df

    expected_dataset_ids = (

        test_df[
            "dataset_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    if not np.array_equal(
        raw_dataset_ids,
        expected_dataset_ids
    ):

        raise RuntimeError(
            f"{configuration_id}: "
            "the dataset_id ordering does not match."
        )


    # Calibrated outputs

    calibrated_path = (

        CALIBRATED_OUTPUT_DIR
        / (
            f"{configuration_id}"
            "_temperature_scaled.npz"
        )
    )


    calibrated = np.load(
        calibrated_path
    )


    calibrated_probabilities = (

        calibrated[
            "test_probabilities_calibrated"
        ]
        .astype(
            np.float64
        )
    )


    calibrated_predictions = (

        calibrated_probabilities.argmax(
            axis=1
        )
    )


    if not np.array_equal(
        raw_predictions,
        calibrated_predictions
    ):

        raise RuntimeError(
            f"{configuration_id}: "
            "calibration changed the predictions."
        )


    # Point estimates

    point_accuracy = float(

        accuracy_score(
            raw_targets,
            raw_predictions
        )
    )


    point_balanced_accuracy = float(

        balanced_accuracy_score(
            raw_targets,
            raw_predictions
        )
    )


    point_macro_f1 = float(

        f1_score(

            raw_targets,

            raw_predictions,

            labels=list(
                range(
                    NUM_CLASSES
                )
            ),

            average="macro",

            zero_division=0
        )
    )


    point_ece_before, _ = (

        calculate_ece(
            raw_probabilities,
            raw_targets
        )
    )


    point_ece_after, _ = (

        calculate_ece(
            calibrated_probabilities,
            raw_targets
        )
    )


    point_nll_before = (

        numpy_nll(
            raw_probabilities,
            raw_targets
        )
    )


    point_nll_after = (

        numpy_nll(
            calibrated_probabilities,
            raw_targets
        )
    )


    # Bootstrap

    accuracy_values = []

    balanced_accuracy_values = []

    macro_f1_values = []

    ece_before_values = []

    ece_after_values = []

    nll_before_values = []

    nll_after_values = []


    for bootstrap_number, sample_indices in enumerate(
        bootstrap_index_list,
        start=1
    ):

        sample_targets = (
            raw_targets[
                sample_indices
            ]
        )


        sample_predictions = (
            raw_predictions[
                sample_indices
            ]
        )


        sample_probabilities_before = (
            raw_probabilities[
                sample_indices
            ]
        )


        sample_probabilities_after = (
            calibrated_probabilities[
                sample_indices
            ]
        )


        # Classification

        boot_accuracy = float(

            accuracy_score(
                sample_targets,
                sample_predictions
            )
        )


        boot_balanced_accuracy = float(

            balanced_accuracy_score(
                sample_targets,
                sample_predictions
            )
        )


        boot_macro_f1 = float(

            f1_score(

                sample_targets,

                sample_predictions,

                labels=list(
                    range(
                        NUM_CLASSES
                    )
                ),

                average="macro",

                zero_division=0
            )
        )


        # Calibration

        boot_ece_before, _ = (

            calculate_ece(
                sample_probabilities_before,
                sample_targets
            )
        )


        boot_ece_after, _ = (

            calculate_ece(
                sample_probabilities_after,
                sample_targets
            )
        )


        boot_nll_before = (

            numpy_nll(
                sample_probabilities_before,
                sample_targets
            )
        )


        boot_nll_after = (

            numpy_nll(
                sample_probabilities_after,
                sample_targets
            )
        )


        accuracy_values.append(
            boot_accuracy
        )

        balanced_accuracy_values.append(
            boot_balanced_accuracy
        )

        macro_f1_values.append(
            boot_macro_f1
        )

        ece_before_values.append(
            boot_ece_before
        )

        ece_after_values.append(
            boot_ece_after
        )

        nll_before_values.append(
            boot_nll_before
        )

        nll_after_values.append(
            boot_nll_after
        )


    # Summary of each metric

    metrics_dictionary = {

        "accuracy": (
            point_accuracy,
            accuracy_values
        ),

        "balanced_accuracy": (
            point_balanced_accuracy,
            balanced_accuracy_values
        ),

        "macro_f1": (
            point_macro_f1,
            macro_f1_values
        ),

        "ece_before": (
            point_ece_before,
            ece_before_values
        ),

        "ece_after": (
            point_ece_after,
            ece_after_values
        ),

        "nll_before": (
            point_nll_before,
            nll_before_values
        ),

        "nll_after": (
            point_nll_after,
            nll_after_values
        )
    }


    model_summary_record = {
        "configuration_id":
            configuration_id
    }


    for metric_name, (
        point_estimate,
        bootstrap_values
    ) in metrics_dictionary.items():

        metric_summary = (

            summarize_bootstrap_metric(
                point_estimate,
                bootstrap_values
            )
        )


        model_summary_record[
            f"{metric_name}"
        ] = (
            metric_summary[
                "estimate"
            ]
        )


        model_summary_record[
            f"{metric_name}_ci_lower"
        ] = (
            metric_summary[
                "ci_lower"
            ]
        )


        model_summary_record[
            f"{metric_name}_ci_upper"
        ] = (
            metric_summary[
                "ci_upper"
            ]
        )


        model_summary_record[
            f"{metric_name}_bootstrap_std"
        ] = (
            metric_summary[
                "bootstrap_std"
            ]
        )


        # Long format
        for replicate_index, value in enumerate(
            bootstrap_values,
            start=1
        ):

            bootstrap_long_records.append({

                "configuration_id":
                    configuration_id,

                "replicate":
                    replicate_index,

                "metric":
                    metric_name,

                "value":
                    value
            })


    bootstrap_summary_records.append(
        model_summary_record
    )


    print(
        "  Accuracy:",
        f"{point_accuracy:.6f}",
        "[",
        f"{model_summary_record['accuracy_ci_lower']:.6f},",
        f"{model_summary_record['accuracy_ci_upper']:.6f}",
        "]"
    )


    print(
        "  Balanced Accuracy:",
        f"{point_balanced_accuracy:.6f}",
        "[",
        f"{model_summary_record['balanced_accuracy_ci_lower']:.6f},",
        f"{model_summary_record['balanced_accuracy_ci_upper']:.6f}",
        "]"
    )


    print(
        "  Macro-F1:",
        f"{point_macro_f1:.6f}",
        "[",
        f"{model_summary_record['macro_f1_ci_lower']:.6f},",
        f"{model_summary_record['macro_f1_ci_upper']:.6f}",
        "]"
    )


    print(
        "  ECE calibrated:",
        f"{point_ece_after:.6f}",
        "[",
        f"{model_summary_record['ece_after_ci_lower']:.6f},",
        f"{model_summary_record['ece_after_ci_upper']:.6f}",
        "]"
    )


    raw.close()
    calibrated.close()

    del raw
    del calibrated

    gc.collect()


# Create tables

bootstrap_summary_df = pd.DataFrame(
    bootstrap_summary_records
)


bootstrap_long_df = pd.DataFrame(
    bootstrap_long_records
)


# Add validation ranking

bootstrap_summary_df[
    "validation_rank"
] = bootstrap_summary_df[
    "configuration_id"
].map(
    validation_rank_mapping
)


bootstrap_summary_df = (

    bootstrap_summary_df
    .sort_values(
        "validation_rank"
    )
    .reset_index(
        drop=True
    )
)


# Save

BOOTSTRAP_SUMMARY_OUTPUT = (

    BOOTSTRAP_DIR
    / "11_test_bootstrap_ci_summary.csv"
)


BOOTSTRAP_LONG_OUTPUT = (

    BOOTSTRAP_DIR
    / "12_test_bootstrap_replicates_long.csv"
)


BOOTSTRAP_GROUP_OUTPUT = (

    BOOTSTRAP_DIR
    / "13_test_bootstrap_group_summary.csv"
)


bootstrap_summary_df.to_csv(

    BOOTSTRAP_SUMMARY_OUTPUT,

    index=False
)


bootstrap_long_df.to_csv(

    BOOTSTRAP_LONG_OUTPUT,

    index=False
)


bootstrap_group_summary_df.to_csv(

    BOOTSTRAP_GROUP_OUTPUT,

    index=False
)


# Display the main table

print("\n" + "=" * 80)
print("BOOTSTRAP CONFIDENCE INTERVALS COMPLETED")
print("=" * 80)


display(

    bootstrap_summary_df[
        [
            "validation_rank",
            "configuration_id",

            "accuracy",
            "accuracy_ci_lower",
            "accuracy_ci_upper",

            "balanced_accuracy",
            "balanced_accuracy_ci_lower",
            "balanced_accuracy_ci_upper",

            "macro_f1",
            "macro_f1_ci_lower",
            "macro_f1_ci_upper",

            "ece_after",
            "ece_after_ci_lower",
            "ece_after_ci_upper"
        ]
    ]
)


print("\nBootstrap design:")
print(
    "Replicates:",
    BOOTSTRAP_REPLICATES
)

print(
    "Seed:",
    BOOTSTRAP_SEED
)

print(
    "Resampling unit:",
    "split_guard_group_id"
)

print(
    "Stratification:",
    "class"
)

print(
    "CI:",
    "percentile 95%"
)


print("\nSaved files:")

print(
    BOOTSTRAP_SUMMARY_OUTPUT
)

print(
    BOOTSTRAP_LONG_OUTPUT
)

print(
    BOOTSTRAP_GROUP_OUTPUT
)

print(
    BOOTSTRAP_DESIGN_PATH
)

# **Cell 7 — Exact McNemar + Holm for the 66 comparisons**

In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import binomtest


# Configuration

ALPHA = 0.05

MCNEMAR_DIR = (
    ANALYSIS_ROOT
    / "mcnemar_holm"
)

MCNEMAR_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Load ground-truth and prediction vectors

prediction_dictionary = {}

reference_dataset_ids = None
reference_targets = None


for configuration_id in EXPECTED_CONFIGURATIONS:

    raw_output_path = (
        RAW_OUTPUT_DIR
        / f"{configuration_id}_raw_outputs.npz"
    )

    raw = np.load(
        raw_output_path
    )

    dataset_ids = (
        raw["test_dataset_ids"]
        .astype(np.int64)
    )

    targets = (
        raw["test_targets"]
        .astype(np.int64)
    )

    predictions = (
        raw["test_predictions"]
        .astype(np.int64)
    )


    # First configuration:
    # establish the reference

    if reference_dataset_ids is None:

        reference_dataset_ids = (
            dataset_ids.copy()
        )

        reference_targets = (
            targets.copy()
        )

    else:

        if not np.array_equal(
            dataset_ids,
            reference_dataset_ids
        ):

            raise RuntimeError(
                f"{configuration_id}: "
                "dataset_id is not aligned."
            )

        if not np.array_equal(
            targets,
            reference_targets
        ):

            raise RuntimeError(
                f"{configuration_id}: "
                "targets are not aligned."
            )


    prediction_dictionary[
        configuration_id
    ] = predictions.copy()

    raw.close()


# General verification

N_TEST = len(
    reference_targets
)

if N_TEST != 2841:

    raise RuntimeError(
        f"Expected 2841 samples; "
        f"found {N_TEST}."
    )


print("=" * 80)
print("ALIGNMENT FOR McNEMAR")
print("=" * 80)

print(
    "Configurations:",
    len(
        prediction_dictionary
    )
)

print(
    "Paired samples:",
    N_TEST
)

print(
    "Expected comparisons:",
    len(
        list(
            combinations(
                EXPECTED_CONFIGURATIONS,
                2
            )
        )
    )
)


# Exact McNemar test

pairwise_records = []


for model_a, model_b in combinations(
    EXPECTED_CONFIGURATIONS,
    2
):

    predictions_a = (
        prediction_dictionary[
            model_a
        ]
    )

    predictions_b = (
        prediction_dictionary[
            model_b
        ]
    )


    correct_a = (
        predictions_a
        == reference_targets
    )

    correct_b = (
        predictions_b
        == reference_targets
    )


    # 2x2 contingency table
    # both_correct
    # A correct / B wrong = b
    # A wrong / B correct = c
    # both_wrong

    both_correct = int(
        np.sum(
            correct_a
            &
            correct_b
        )
    )


    a_only_correct = int(
        np.sum(
            correct_a
            &
            ~correct_b
        )
    )


    b_only_correct = int(
        np.sum(
            ~correct_a
            &
            correct_b
        )
    )


    both_wrong = int(
        np.sum(
            ~correct_a
            &
            ~correct_b
        )
    )


    discordant_total = (
        a_only_correct
        +
        b_only_correct
    )


    # Exact McNemar test

    if discordant_total == 0:

        raw_p_value = 1.0

    else:

        raw_p_value = float(

            binomtest(

                k=min(
                    a_only_correct,
                    b_only_correct
                ),

                n=discordant_total,

                p=0.5,

                alternative="two-sided"

            ).pvalue
        )


    # Descriptive accuracy

    accuracy_a = float(
        correct_a.mean()
    )

    accuracy_b = float(
        correct_b.mean()
    )


    accuracy_difference = (
        accuracy_a
        -
        accuracy_b
    )


    # Direction of discordance

    if (
        a_only_correct
        >
        b_only_correct
    ):

        discordance_direction = (
            model_a
        )

    elif (
        b_only_correct
        >
        a_only_correct
    ):

        discordance_direction = (
            model_b
        )

    else:

        discordance_direction = (
            "tie"
        )


    pairwise_records.append({

        "model_a":
            model_a,

        "model_b":
            model_b,


        "accuracy_a":
            accuracy_a,

        "accuracy_b":
            accuracy_b,

        "accuracy_difference_a_minus_b":
            accuracy_difference,


        "both_correct":
            both_correct,

        "a_only_correct_b_wrong":
            a_only_correct,

        "a_wrong_b_only_correct":
            b_only_correct,

        "both_wrong":
            both_wrong,

        "discordant_total":
            discordant_total,


        "discordance_favors":
            discordance_direction,

        "mcnemar_exact_p_raw":
            raw_p_value
    })


# DataFrame

mcnemar_df = pd.DataFrame(
    pairwise_records
)


if len(
    mcnemar_df
) != 66:

    raise RuntimeError(
        "The 66 comparisons were not generated."
    )


# Manual Holm adjustment

def holm_adjust(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=np.float64
    )

    number_of_tests = len(
        p_values
    )


    order = np.argsort(
        p_values
    )

    sorted_p = (
        p_values[
            order
        ]
    )


    adjusted_sorted = np.empty(
        number_of_tests,
        dtype=np.float64
    )


    running_maximum = 0.0


    for rank_index, p_value in enumerate(
        sorted_p
    ):

        multiplier = (
            number_of_tests
            -
            rank_index
        )


        adjusted_value = min(
            1.0,
            p_value
            *
            multiplier
        )


        running_maximum = max(
            running_maximum,
            adjusted_value
        )


        adjusted_sorted[
            rank_index
        ] = running_maximum


    adjusted_original_order = np.empty(
        number_of_tests,
        dtype=np.float64
    )


    adjusted_original_order[
        order
    ] = adjusted_sorted


    return adjusted_original_order


mcnemar_df[
    "mcnemar_p_holm"
] = holm_adjust(

    mcnemar_df[
        "mcnemar_exact_p_raw"
    ].to_numpy()
)


mcnemar_df[
    "significant_holm_0_05"
] = (

    mcnemar_df[
        "mcnemar_p_holm"
    ]
    <
    ALPHA
)


# Interpretation of direction

def determine_significant_winner(
    row
):

    if not row[
        "significant_holm_0_05"
    ]:

        return (
            "not_significant"
        )


    if (
        row[
            "a_only_correct_b_wrong"
        ]
        >
        row[
            "a_wrong_b_only_correct"
        ]
    ):

        return row[
            "model_a"
        ]


    if (
        row[
            "a_wrong_b_only_correct"
        ]
        >
        row[
            "a_only_correct_b_wrong"
        ]
    ):

        return row[
            "model_b"
        ]


    return "tie"


mcnemar_df[
    "holm_significant_winner"
] = mcnemar_df.apply(
    determine_significant_winner,
    axis=1
)


# Add absolute differences

mcnemar_df[
    "accuracy_difference_percentage_points"
] = (

    mcnemar_df[
        "accuracy_difference_a_minus_b"
    ]
    *
    100
)


# Sort by Holm-adjusted p-value

mcnemar_df = (

    mcnemar_df
    .sort_values(
        [
            "mcnemar_p_holm",
            "mcnemar_exact_p_raw"
        ],
        ascending=True
    )
    .reset_index(
        drop=True
    )
)


# Significant comparisons

significant_mcnemar_df = (

    mcnemar_df[
        mcnemar_df[
            "significant_holm_0_05"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# Within-architecture comparisons

def architecture_from_id(
    configuration_id
):

    return (
        configuration_id
        .split("__")[0]
    )


within_architecture_df = (

    mcnemar_df[
        mcnemar_df.apply(

            lambda row:

                architecture_from_id(
                    row["model_a"]
                )
                ==
                architecture_from_id(
                    row["model_b"]
                ),

            axis=1
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# Matrix of Holm-adjusted p-values

holm_matrix = pd.DataFrame(

    np.ones(
        (
            len(
                EXPECTED_CONFIGURATIONS
            ),
            len(
                EXPECTED_CONFIGURATIONS
            )
        ),
        dtype=np.float64
    ),

    index=
        EXPECTED_CONFIGURATIONS,

    columns=
        EXPECTED_CONFIGURATIONS
)


for _, row in mcnemar_df.iterrows():

    model_a = row[
        "model_a"
    ]

    model_b = row[
        "model_b"
    ]

    adjusted_p = row[
        "mcnemar_p_holm"
    ]


    holm_matrix.loc[
        model_a,
        model_b
    ] = adjusted_p

    holm_matrix.loc[
        model_b,
        model_a
    ] = adjusted_p


# Comparisons involving the best model

BEST_CONFIGURATION = (
    "swin_tiny__cutmix"
)


best_model_comparisons_df = (

    mcnemar_df[
        (
            mcnemar_df[
                "model_a"
            ]
            ==
            BEST_CONFIGURATION
        )
        |
        (
            mcnemar_df[
                "model_b"
            ]
            ==
            BEST_CONFIGURATION
        )
    ]
    .copy()
)


# Explicitly use the best model as reference
def normalize_best_comparison(
    row
):

    if (
        row["model_a"]
        ==
        BEST_CONFIGURATION
    ):

        competitor = (
            row["model_b"]
        )

        best_only_correct = (
            row[
                "a_only_correct_b_wrong"
            ]
        )

        competitor_only_correct = (
            row[
                "a_wrong_b_only_correct"
            ]
        )

        accuracy_difference = (
            row[
                "accuracy_difference_a_minus_b"
            ]
        )

    else:

        competitor = (
            row["model_a"]
        )

        best_only_correct = (
            row[
                "a_wrong_b_only_correct"
            ]
        )

        competitor_only_correct = (
            row[
                "a_only_correct_b_wrong"
            ]
        )

        accuracy_difference = (
            -
            row[
                "accuracy_difference_a_minus_b"
            ]
        )


    return pd.Series({

        "best_model":
            BEST_CONFIGURATION,

        "competitor":
            competitor,

        "best_only_correct":
            best_only_correct,

        "competitor_only_correct":
            competitor_only_correct,

        "discordant_total":
            (
                best_only_correct
                +
                competitor_only_correct
            ),

        "accuracy_difference_best_minus_competitor":
            accuracy_difference,

        "accuracy_difference_percentage_points":
            accuracy_difference
            *
            100,

        "p_raw":
            row[
                "mcnemar_exact_p_raw"
            ],

        "p_holm":
            row[
                "mcnemar_p_holm"
            ],

        "significant_holm":
            row[
                "significant_holm_0_05"
            ]
    })


best_model_comparisons_df = (

    best_model_comparisons_df
    .apply(
        normalize_best_comparison,
        axis=1
    )
    .sort_values(
        "p_holm"
    )
    .reset_index(
        drop=True
    )
)


# Save files

MCNEMAR_ALL_OUTPUT = (
    MCNEMAR_DIR
    / "14_mcnemar_all_66_holm.csv"
)

MCNEMAR_SIGNIFICANT_OUTPUT = (
    MCNEMAR_DIR
    / "15_mcnemar_significant_holm.csv"
)

MCNEMAR_WITHIN_ARCH_OUTPUT = (
    MCNEMAR_DIR
    / "16_mcnemar_within_architecture.csv"
)

MCNEMAR_MATRIX_OUTPUT = (
    MCNEMAR_DIR
    / "17_mcnemar_holm_pvalue_matrix.csv"
)

MCNEMAR_BEST_OUTPUT = (
    MCNEMAR_DIR
    / "18_swin_cutmix_vs_all_mcnemar.csv"
)


mcnemar_df.to_csv(
    MCNEMAR_ALL_OUTPUT,
    index=False
)

significant_mcnemar_df.to_csv(
    MCNEMAR_SIGNIFICANT_OUTPUT,
    index=False
)

within_architecture_df.to_csv(
    MCNEMAR_WITHIN_ARCH_OUTPUT,
    index=False
)

holm_matrix.to_csv(
    MCNEMAR_MATRIX_OUTPUT
)

best_model_comparisons_df.to_csv(
    MCNEMAR_BEST_OUTPUT,
    index=False
)


# Summary

number_significant = int(

    mcnemar_df[
        "significant_holm_0_05"
    ].sum()
)


print("\n" + "=" * 80)
print("McNEMAR + HOLM COMPLETED")
print("=" * 80)


print(
    "\nTotal comparisons:",
    len(
        mcnemar_df
    )
)

print(
    "Alpha:",
    ALPHA
)

print(
    "Significant comparisons "
    "after Holm adjustment:",
    number_significant
)


print(
    "\nComparisons of "
    "swin_tiny__cutmix against the other models:"
)

display(
    best_model_comparisons_df
)


print(
    "\nWithin-architecture comparisons:"
)

display(

    within_architecture_df[
        [
            "model_a",
            "model_b",
            "a_only_correct_b_wrong",
            "a_wrong_b_only_correct",
            "mcnemar_exact_p_raw",
            "mcnemar_p_holm",
            "significant_holm_0_05",
            "holm_significant_winner"
        ]
    ]
)


print("\nSaved files:")

print(
    MCNEMAR_ALL_OUTPUT
)

print(
    MCNEMAR_SIGNIFICANT_OUTPUT
)

print(
    MCNEMAR_WITHIN_ARCH_OUTPUT
)

print(
    MCNEMAR_MATRIX_OUTPUT
)

print(
    MCNEMAR_BEST_OUTPUT
)

# **Cell 8 — Confusion matrices + per-class metrics**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support
)


# Directories

CONFUSION_DIR = (
    ANALYSIS_ROOT
    / "confusion_matrices"
)

CLASS_REPORT_DIR = (
    ANALYSIS_ROOT
    / "class_reports"
)

ERROR_PAIRS_DIR = (
    ANALYSIS_ROOT
    / "error_pairs"
)


for directory in [
    CONFUSION_DIR,
    CLASS_REPORT_DIR,
    ERROR_PAIRS_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# Labels

CLASS_LABELS = [
    INDEX_TO_CLASS[index]
    for index in range(NUM_CLASSES)
]


# Shorter labels for the figures
SHORT_CLASS_LABELS = [

    class_name
    .replace("__Healthy", " H")
    .replace("__Rotten", " R")

    for class_name in CLASS_LABELS
]


# Function for saving a matrix figure

def save_confusion_figure(
    matrix,
    configuration_id,
    normalized,
    output_path
):

    fig, ax = plt.subplots(
        figsize=(15, 13)
    )

    image = ax.imshow(
        matrix,
        interpolation="nearest",
        aspect="auto"
    )

    fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04
    )


    ax.set(

        xticks=np.arange(
            NUM_CLASSES
        ),

        yticks=np.arange(
            NUM_CLASSES
        ),

        xticklabels=
            SHORT_CLASS_LABELS,

        yticklabels=
            SHORT_CLASS_LABELS,

        xlabel=
            "Predicted class",

        ylabel=
            "True class",

        title=(
            f"{configuration_id}\n"
            +
            (
                "Normalized confusion matrix"
                if normalized
                else
                "Absolute confusion matrix"
            )
        )
    )


    plt.setp(
        ax.get_xticklabels(),
        rotation=90,
        ha="center",
        fontsize=8
    )

    plt.setp(
        ax.get_yticklabels(),
        fontsize=8
    )


    # Write values inside matrix cells

    if normalized:

        threshold = (
            matrix.max() / 2
            if matrix.size > 0
            else 0
        )

        for i in range(
            matrix.shape[0]
        ):

            for j in range(
                matrix.shape[1]
            ):

                value = (
                    matrix[i, j]
                )

                # Annotate nonzero cells only
                if value > 0:

                    ax.text(

                        j,
                        i,
                        f"{value:.2f}",

                        ha="center",
                        va="center",

                        fontsize=5
                    )

    else:

        for i in range(
            matrix.shape[0]
        ):

            for j in range(
                matrix.shape[1]
            ):

                value = int(
                    matrix[i, j]
                )

                if value > 0:

                    ax.text(

                        j,
                        i,
                        str(value),

                        ha="center",
                        va="center",

                        fontsize=5
                    )


    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=220,
        bbox_inches="tight"
    )

    plt.close(
        fig
    )


# Process the 12 configurations

all_class_reports = []

all_error_pairs = []

configuration_summary_records = []


print("=" * 80)
print("CONFUSION MATRICES AND PER-CLASS METRICS")
print("=" * 80)


for run_number, configuration_id in enumerate(

    EXPECTED_CONFIGURATIONS,

    start=1

):

    print(
        f"[{run_number}/12] "
        f"{configuration_id}"
    )


    # Load final predictions

    raw_output_path = (

        RAW_OUTPUT_DIR
        / f"{configuration_id}_raw_outputs.npz"
    )


    with np.load(
        raw_output_path
    ) as raw:

        targets = (
            raw[
                "test_targets"
            ]
            .astype(
                np.int64
            )
        )

        predictions = (
            raw[
                "test_predictions"
            ]
            .astype(
                np.int64
            )
        )

        dataset_ids = (
            raw[
                "test_dataset_ids"
            ]
            .astype(
                np.int64
            )
        )


    # Absolute confusion matrix

    cm_absolute = confusion_matrix(

        targets,

        predictions,

        labels=list(
            range(
                NUM_CLASSES
            )
        )
    )


    # Confusion matrix normalized by true class

    row_sums = (

        cm_absolute
        .sum(
            axis=1,
            keepdims=True
        )
    )


    cm_normalized = np.divide(

        cm_absolute,

        row_sums,

        out=np.zeros_like(
            cm_absolute,
            dtype=np.float64
        ),

        where=(
            row_sums != 0
        )
    )


    # Save confusion matrices as CSV

    cm_absolute_df = pd.DataFrame(

        cm_absolute,

        index=CLASS_LABELS,

        columns=CLASS_LABELS
    )


    cm_normalized_df = pd.DataFrame(

        cm_normalized,

        index=CLASS_LABELS,

        columns=CLASS_LABELS
    )


    absolute_csv_path = (

        CONFUSION_DIR
        / (
            f"{configuration_id}"
            "__confusion_absolute.csv"
        )
    )


    normalized_csv_path = (

        CONFUSION_DIR
        / (
            f"{configuration_id}"
            "__confusion_normalized.csv"
        )
    )


    cm_absolute_df.to_csv(
        absolute_csv_path
    )


    cm_normalized_df.to_csv(
        normalized_csv_path
    )


    # Save PNG figures

    absolute_png_path = (

        CONFUSION_DIR
        / (
            f"{configuration_id}"
            "__confusion_absolute.png"
        )
    )


    normalized_png_path = (

        CONFUSION_DIR
        / (
            f"{configuration_id}"
            "__confusion_normalized.png"
        )
    )


    save_confusion_figure(

        matrix=cm_absolute,

        configuration_id=
            configuration_id,

        normalized=False,

        output_path=
            absolute_png_path
    )


    save_confusion_figure(

        matrix=cm_normalized,

        configuration_id=
            configuration_id,

        normalized=True,

        output_path=
            normalized_png_path
    )


    # Per-class precision / recall / F1 / support

    (
        precision,
        recall,
        f1,
        support
    ) = precision_recall_fscore_support(

        targets,

        predictions,

        labels=list(
            range(
                NUM_CLASSES
            )
        ),

        zero_division=0
    )


    class_report_df = pd.DataFrame({

        "configuration_id":
            configuration_id,

        "target":
            np.arange(
                NUM_CLASSES
            ),

        "class_name":
            CLASS_LABELS,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "support":
            support.astype(
                int
            ),

        "correct":
            np.diag(
                cm_absolute
            ).astype(
                int
            ),

        "errors":
            (
                support
                -
                np.diag(
                    cm_absolute
                )
            ).astype(
                int
            )
    })


    class_report_path = (

        CLASS_REPORT_DIR
        / (
            f"{configuration_id}"
            "__class_report.csv"
        )
    )


    class_report_df.to_csv(

        class_report_path,

        index=False
    )


    all_class_reports.append(
        class_report_df
    )


    # Confusion pairs

    error_records = []


    for true_index in range(
        NUM_CLASSES
    ):

        for predicted_index in range(
            NUM_CLASSES
        ):

            if (
                true_index
                ==
                predicted_index
            ):

                continue


            count = int(

                cm_absolute[
                    true_index,
                    predicted_index
                ]
            )


            if count == 0:

                continue


            error_records.append({

                "configuration_id":
                    configuration_id,

                "true_target":
                    true_index,

                "true_class":
                    INDEX_TO_CLASS[
                        true_index
                    ],

                "predicted_target":
                    predicted_index,

                "predicted_class":
                    INDEX_TO_CLASS[
                        predicted_index
                    ],

                "count":
                    count
            })


    error_pairs_df = pd.DataFrame(
        error_records
    )


    if len(
        error_pairs_df
    ) > 0:

        error_pairs_df = (

            error_pairs_df
            .sort_values(

                [
                    "count",
                    "true_class",
                    "predicted_class"
                ],

                ascending=[
                    False,
                    True,
                    True
                ]
            )
            .reset_index(
                drop=True
            )
        )


    error_pairs_path = (

        ERROR_PAIRS_DIR
        / (
            f"{configuration_id}"
            "__error_pairs.csv"
        )
    )


    error_pairs_df.to_csv(

        error_pairs_path,

        index=False
    )


    all_error_pairs.append(
        error_pairs_df
    )


    # Error summary

    total_errors = int(
        np.sum(
            targets
            !=
            predictions
        )
    )


    worst_class_row = (

        class_report_df
        .sort_values(
            [
                "f1",
                "support"
            ],
            ascending=[
                True,
                False
            ]
        )
        .iloc[0]
    )


    configuration_summary_records.append({

        "configuration_id":
            configuration_id,

        "total_test_images":
            len(
                targets
            ),

        "correct_predictions":
            int(
                len(
                    targets
                )
                -
                total_errors
            ),

        "total_errors":
            total_errors,

        "worst_class":
            worst_class_row[
                "class_name"
            ],

        "worst_class_precision":
            float(
                worst_class_row[
                    "precision"
                ]
            ),

        "worst_class_recall":
            float(
                worst_class_row[
                    "recall"
                ]
            ),

        "worst_class_f1":
            float(
                worst_class_row[
                    "f1"
                ]
            )
    })


# Consolidate reports

all_class_reports_df = pd.concat(

    all_class_reports,

    ignore_index=True
)


non_empty_error_frames = [

    dataframe

    for dataframe in all_error_pairs

    if len(
        dataframe
    ) > 0
]


if len(
    non_empty_error_frames
) > 0:

    all_error_pairs_df = pd.concat(

        non_empty_error_frames,

        ignore_index=True
    )

else:

    all_error_pairs_df = pd.DataFrame()


configuration_error_summary_df = pd.DataFrame(

    configuration_summary_records
)


# Add validation ranking

configuration_error_summary_df[
    "validation_rank"
] = (

    configuration_error_summary_df[
        "configuration_id"
    ]
    .map(
        validation_rank_mapping
    )
)


configuration_error_summary_df = (

    configuration_error_summary_df
    .sort_values(
        "validation_rank"
    )
    .reset_index(
        drop=True
    )
)


# Save consolidated results

ALL_CLASS_REPORTS_OUTPUT = (

    CLASS_REPORT_DIR
    / "19_all_models_class_reports.csv"
)


ALL_ERROR_PAIRS_OUTPUT = (

    ERROR_PAIRS_DIR
    / "20_all_models_error_pairs.csv"
)


ERROR_SUMMARY_OUTPUT = (

    ERROR_PAIRS_DIR
    / "21_all_models_error_summary.csv"
)


all_class_reports_df.to_csv(

    ALL_CLASS_REPORTS_OUTPUT,

    index=False
)


all_error_pairs_df.to_csv(

    ALL_ERROR_PAIRS_OUTPUT,

    index=False
)


configuration_error_summary_df.to_csv(

    ERROR_SUMMARY_OUTPUT,

    index=False
)


# Display only the compact summary

print("\n" + "=" * 80)
print("CONFUSION ANALYSIS COMPLETED")
print("=" * 80)


display(

    configuration_error_summary_df[
        [
            "validation_rank",
            "configuration_id",
            "correct_predictions",
            "total_errors",
            "worst_class",
            "worst_class_f1"
        ]
    ]
)


# Most frequent errors of the best model

BEST_CONFIGURATION = (
    "swin_tiny__cutmix"
)


best_errors_df = (

    all_error_pairs_df[
        all_error_pairs_df[
            "configuration_id"
        ]
        ==
        BEST_CONFIGURATION
    ]
    .copy()
    .sort_values(
        "count",
        ascending=False
    )
    .head(15)
    .reset_index(
        drop=True
    )
)


print(
    "\nMost frequent confusions for "
    "swin_tiny__cutmix:"
)

display(
    best_errors_df
)


best_class_report_df = (

    all_class_reports_df[
        all_class_reports_df[
            "configuration_id"
        ]
        ==
        BEST_CONFIGURATION
    ]
    .copy()
    .sort_values(
        "f1",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nMost challenging classes for "
    "swin_tiny__cutmix:"
)

display(
    best_class_report_df[
        [
            "class_name",
            "precision",
            "recall",
            "f1",
            "support",
            "errors"
        ]
    ].head(10)
)


print("\nSaved files:")

print(
    CONFUSION_DIR
)

print(
    ALL_CLASS_REPORTS_OUTPUT
)

print(
    ALL_ERROR_PAIRS_OUTPUT
)

print(
    ERROR_SUMMARY_OUTPUT
)

# **Cell 9 — Latency, throughput, parameters, and GPU memory**

In [ ]:

from pathlib import Path
import gc
import platform
import time

import numpy as np
import pandas as pd
import torch

# Configuration


LATENCY_BATCH_SIZE = 1
THROUGHPUT_BATCH_SIZE = 64

LATENCY_WARMUP_ITERATIONS = 50
LATENCY_MEASURED_ITERATIONS = 300

THROUGHPUT_WARMUP_ITERATIONS = 30
THROUGHPUT_MEASURED_ITERATIONS = 100


EFFICIENCY_DIR = (
    ANALYSIS_ROOT
    / "efficiency"
)

EFFICIENCY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = (
    DEVICE.type == "cuda"
)


if DEVICE.type != "cuda":

    raise RuntimeError(
        "A GPU is required to compare latency and throughput "
        "consistently."
    )

# Hardware information


gpu_name = torch.cuda.get_device_name(
    0
)

gpu_properties = torch.cuda.get_device_properties(
    0
)

gpu_total_memory_gb = (
    gpu_properties.total_memory
    / (1024 ** 3)
)


print("=" * 80)
print("EFFICIENCY BENCHMARK")
print("=" * 80)

print(
    "GPU:",
    gpu_name
)

print(
    "GPU memory:",
    f"{gpu_total_memory_gb:.2f} GB"
)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "CUDA:",
    torch.version.cuda
)

print(
    "AMP FP16:",
    USE_AMP
)

# Contexto AMP


def forward_model(
    model,
    inputs
):

    with torch.inference_mode():

        with torch.amp.autocast(

            device_type="cuda",

            dtype=torch.float16,

            enabled=True
        ):

            outputs = model(
                inputs
            )

    return outputs

# Latency measurement (batch size 1)


def benchmark_latency(
    model
):

    dummy_input = torch.randn(

        LATENCY_BATCH_SIZE,
        3,
        IMG_SIZE,
        IMG_SIZE,

        device=DEVICE,
        dtype=torch.float32
    )

    # Warm-up
    

    for _ in range(
        LATENCY_WARMUP_ITERATIONS
    ):

        _ = forward_model(
            model,
            dummy_input
        )


    torch.cuda.synchronize()

    # Measurement
    

    latency_ms = []


    for _ in range(
        LATENCY_MEASURED_ITERATIONS
    ):

        torch.cuda.synchronize()

        start = time.perf_counter()


        _ = forward_model(
            model,
            dummy_input
        )


        torch.cuda.synchronize()

        end = time.perf_counter()


        latency_ms.append(

            (
                end
                - start
            )
            * 1000.0
        )


    latency_ms = np.asarray(
        latency_ms,
        dtype=np.float64
    )


    return {

        "latency_mean_ms":
            float(
                latency_ms.mean()
            ),

        "latency_median_ms":
            float(
                np.median(
                    latency_ms
                )
            ),

        "latency_std_ms":
            float(
                latency_ms.std(
                    ddof=1
                )
            ),

        "latency_p95_ms":
            float(
                np.percentile(
                    latency_ms,
                    95
                )
            ),

        "latency_min_ms":
            float(
                latency_ms.min()
            ),

        "latency_max_ms":
            float(
                latency_ms.max()
            ),

        "latency_all_ms":
            latency_ms
    }

# Throughput batch=64


def benchmark_throughput(
    model
):

    dummy_input = torch.randn(

        THROUGHPUT_BATCH_SIZE,
        3,
        IMG_SIZE,
        IMG_SIZE,

        device=DEVICE,
        dtype=torch.float32
    )

    # Warm-up
    

    for _ in range(
        THROUGHPUT_WARMUP_ITERATIONS
    ):

        _ = forward_model(
            model,
            dummy_input
        )


    torch.cuda.synchronize()

    # Measurement
    

    batch_times = []


    for _ in range(
        THROUGHPUT_MEASURED_ITERATIONS
    ):

        torch.cuda.synchronize()

        start = time.perf_counter()


        _ = forward_model(
            model,
            dummy_input
        )


        torch.cuda.synchronize()

        end = time.perf_counter()


        batch_times.append(
            end - start
        )


    batch_times = np.asarray(
        batch_times,
        dtype=np.float64
    )


    total_images = (
        THROUGHPUT_BATCH_SIZE
        *
        THROUGHPUT_MEASURED_ITERATIONS
    )


    total_seconds = float(
        batch_times.sum()
    )


    throughput_images_per_second = (
        total_images
        /
        total_seconds
    )


    equivalent_ms_per_image = (
        1000.0
        /
        throughput_images_per_second
    )


    return {

        "throughput_images_per_second":
            float(
                throughput_images_per_second
            ),

        "throughput_equivalent_ms_per_image":
            float(
                equivalent_ms_per_image
            ),

        "throughput_mean_batch_ms":
            float(
                batch_times.mean()
                * 1000.0
            ),

        "throughput_p95_batch_ms":
            float(
                np.percentile(
                    batch_times,
                    95
                )
                * 1000.0
            )
    }

# End-to-end usando test real


def benchmark_test_end_to_end(
    model
):

    torch.cuda.synchronize()

    start = time.perf_counter()


    number_images = 0


    with torch.inference_mode():

        for batch in test_loader:

            images = batch[
                "image"
            ].to(

                DEVICE,

                non_blocking=True
            )


            with torch.amp.autocast(

                device_type="cuda",

                dtype=torch.float16,

                enabled=True
            ):

                _ = model(
                    images
                )


            number_images += int(
                images.shape[0]
            )


    torch.cuda.synchronize()


    elapsed_seconds = (
        time.perf_counter()
        - start
    )


    return {

        "test_end_to_end_seconds":
            float(
                elapsed_seconds
            ),

        "test_end_to_end_images_per_second":
            float(
                number_images
                /
                elapsed_seconds
            ),

        "test_end_to_end_ms_per_image":
            float(
                elapsed_seconds
                * 1000.0
                /
                number_images
            )
    }

# Benchmark de las 12 configuraciones


efficiency_records = []


for run_number, configuration_id in enumerate(

    EXPECTED_CONFIGURATIONS,

    start=1

):

    print(
        f"\n[{run_number}/12] "
        f"{configuration_id}"
    )

    # Checkpoint
    

    run_directory = (
        RUN_DIRECTORIES[
            configuration_id
        ]
    )


    best_checkpoint_path = (
        find_unique_file(
            run_directory,
            "best.pt"
        )
    )

    # Clear GPU memory
    

    gc.collect()

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

    # Load model
    

    model, checkpoint = (
        load_model_from_best_checkpoint(
            best_checkpoint_path
        )
    )


    model.eval()

    # Parameters
    

    total_parameters = int(

        sum(
            parameter.numel()
            for parameter
            in model.parameters()
        )
    )


    trainable_parameters = int(

        sum(
            parameter.numel()
            for parameter
            in model.parameters()
            if parameter.requires_grad
        )
    )


    checkpoint_size_mb = float(

        best_checkpoint_path
        .stat()
        .st_size
        /
        (1024 ** 2)
    )

    # Latency
    

    latency_results = (
        benchmark_latency(
            model
        )
    )

    # Throughput
    

    throughput_results = (
        benchmark_throughput(
            model
        )
    )

    # Actual end-to-end inference
    

    end_to_end_results = (
        benchmark_test_end_to_end(
            model
        )
    )

    # Peak memory
    

    peak_memory_mb = float(

        torch.cuda.max_memory_allocated()
        /
        (1024 ** 2)
    )

    # Save individual latency measurements
    

    latency_distribution_path = (

        EFFICIENCY_DIR
        /
        (
            f"{configuration_id}"
            "__latency_distribution.csv"
        )
    )


    pd.DataFrame({

        "iteration":
            np.arange(
                1,
                len(
                    latency_results[
                        "latency_all_ms"
                    ]
                )
                + 1
            ),

        "latency_ms":
            latency_results[
                "latency_all_ms"
            ]

    }).to_csv(

        latency_distribution_path,

        index=False
    )

    # Record
    

    efficiency_records.append({

        "configuration_id":
            configuration_id,

        "model_key":
            checkpoint[
                "configuration"
            ][
                "model_key"
            ],

        "augmentation":
            checkpoint[
                "configuration"
            ][
                "augmentation"
            ],

        "total_parameters":
            total_parameters,

        "parameters_millions":
            total_parameters
            / 1e6,

        "trainable_parameters":
            trainable_parameters,

        "checkpoint_size_mb":
            checkpoint_size_mb,


        # Batch=1
        "latency_batch_size":
            LATENCY_BATCH_SIZE,

        "latency_mean_ms":
            latency_results[
                "latency_mean_ms"
            ],

        "latency_median_ms":
            latency_results[
                "latency_median_ms"
            ],

        "latency_std_ms":
            latency_results[
                "latency_std_ms"
            ],

        "latency_p95_ms":
            latency_results[
                "latency_p95_ms"
            ],


        # Batch=64
        "throughput_batch_size":
            THROUGHPUT_BATCH_SIZE,

        "throughput_images_per_second":
            throughput_results[
                "throughput_images_per_second"
            ],

        "throughput_equivalent_ms_per_image":
            throughput_results[
                "throughput_equivalent_ms_per_image"
            ],

        "throughput_mean_batch_ms":
            throughput_results[
                "throughput_mean_batch_ms"
            ],

        "throughput_p95_batch_ms":
            throughput_results[
                "throughput_p95_batch_ms"
            ],


        # End-to-end
        "test_end_to_end_seconds":
            end_to_end_results[
                "test_end_to_end_seconds"
            ],

        "test_end_to_end_images_per_second":
            end_to_end_results[
                "test_end_to_end_images_per_second"
            ],

        "test_end_to_end_ms_per_image":
            end_to_end_results[
                "test_end_to_end_ms_per_image"
            ],


        # GPU
        "peak_gpu_memory_mb":
            peak_memory_mb,

        "gpu_name":
            gpu_name,

        "amp_fp16":
            True
    })


    print(
        "  Params:",
        f"{total_parameters / 1e6:.2f} M"
    )

    print(
        "  Latency median:",
        f"{latency_results['latency_median_ms']:.3f} ms"
    )

    print(
        "  Latency P95:",
        f"{latency_results['latency_p95_ms']:.3f} ms"
    )

    print(
        "  Throughput:",
        f"{throughput_results['throughput_images_per_second']:.2f} img/s"
    )

    print(
        "  Peak GPU:",
        f"{peak_memory_mb:.2f} MB"
    )

    # Release resources
    

    del model
    del checkpoint

    gc.collect()

    torch.cuda.empty_cache()

# Tabla consolidada


efficiency_df = pd.DataFrame(
    efficiency_records
)

# Add performance metrics


performance_for_merge = (

    point_metrics_df[
        [
            "configuration_id",
            "test_accuracy",
            "test_balanced_accuracy",
            "test_macro_f1"
        ]
    ]
    .copy()
)


efficiency_df = efficiency_df.merge(

    performance_for_merge,

    on="configuration_id",

    how="left",

    validate="one_to_one"
)

# Validation ranking


efficiency_df[
    "validation_rank"
] = efficiency_df[
    "configuration_id"
].map(
    validation_rank_mapping
)


efficiency_df = (

    efficiency_df
    .sort_values(
        "validation_rank"
    )
    .reset_index(
        drop=True
    )
)

# Guardar


EFFICIENCY_OUTPUT = (

    EFFICIENCY_DIR
    / "22_all_models_efficiency_benchmark.csv"
)


efficiency_df.to_csv(

    EFFICIENCY_OUTPUT,

    index=False
)

# Hardware metadata


hardware_df = pd.DataFrame({

    "item": [
        "gpu",
        "gpu_memory_gb",
        "pytorch",
        "cuda",
        "precision",
        "image_size",
        "latency_batch",
        "latency_warmup",
        "latency_iterations",
        "throughput_batch",
        "throughput_warmup",
        "throughput_iterations"
    ],

    "value": [
        gpu_name,
        gpu_total_memory_gb,
        torch.__version__,
        torch.version.cuda,
        "AMP FP16",
        IMG_SIZE,
        LATENCY_BATCH_SIZE,
        LATENCY_WARMUP_ITERATIONS,
        LATENCY_MEASURED_ITERATIONS,
        THROUGHPUT_BATCH_SIZE,
        THROUGHPUT_WARMUP_ITERATIONS,
        THROUGHPUT_MEASURED_ITERATIONS
    ]
})


HARDWARE_OUTPUT = (

    EFFICIENCY_DIR
    / "23_efficiency_hardware_protocol.csv"
)


hardware_df.to_csv(

    HARDWARE_OUTPUT,

    index=False
)

# Mostrar resumen


print("\n" + "=" * 80)
print("EFFICIENCY BENCHMARK COMPLETED")
print("=" * 80)


display(

    efficiency_df[
        [
            "validation_rank",
            "configuration_id",
            "parameters_millions",
            "checkpoint_size_mb",
            "latency_median_ms",
            "latency_p95_ms",
            "throughput_images_per_second",
            "peak_gpu_memory_mb",
            "test_macro_f1"
        ]
    ]
)


print("\nSaved files:")

print(
    EFFICIENCY_OUTPUT
)

print(
    HARDWARE_OUTPUT
)

print(
    EFFICIENCY_DIR
)

# **Cell 10 — Consolidated performance–efficiency table by architecture**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# Output directory

SUMMARY_DIR = (
    ANALYSIS_ROOT
    / "summary_tables"
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Add architecture identifiers

efficiency_arch_df = (
    efficiency_df
    .copy()
)

efficiency_arch_df[
    "architecture"
] = (
    efficiency_arch_df[
        "configuration_id"
    ]
    .str.split("__")
    .str[0]
)


# Aggregate efficiency across three augmentation strategies
# Use the median of three measurements to reduce minor instrumental fluctuations.

architecture_efficiency_df = (

    efficiency_arch_df
    .groupby(
        "architecture",
        as_index=False
    )
    .agg(

        parameters_millions=(
            "parameters_millions",
            "median"
        ),

        checkpoint_size_mb=(
            "checkpoint_size_mb",
            "median"
        ),

        latency_median_ms=(
            "latency_median_ms",
            "median"
        ),

        latency_p95_ms=(
            "latency_p95_ms",
            "median"
        ),

        throughput_images_per_second=(
            "throughput_images_per_second",
            "median"
        ),

        peak_gpu_memory_mb=(
            "peak_gpu_memory_mb",
            "median"
        ),

        test_end_to_end_images_per_second=(
            "test_end_to_end_images_per_second",
            "median"
        ),

        test_end_to_end_ms_per_image=(
            "test_end_to_end_ms_per_image",
            "median"
        )
    )
)


# Measurement variation across augmentation strategies

efficiency_variation_df = (

    efficiency_arch_df
    .groupby(
        "architecture",
        as_index=False
    )
    .agg(

        latency_min_ms=(
            "latency_median_ms",
            "min"
        ),

        latency_max_ms=(
            "latency_median_ms",
            "max"
        ),

        throughput_min=(
            "throughput_images_per_second",
            "min"
        ),

        throughput_max=(
            "throughput_images_per_second",
            "max"
        )
    )
)


architecture_efficiency_df = (
    architecture_efficiency_df
    .merge(
        efficiency_variation_df,
        on="architecture",
        how="left",
        validate="one_to_one"
    )
)


# Select the best configuration per architecture using validation only

validation_selection_df = (
    validation_summary_df
    .copy()
)


validation_selection_df[
    "architecture"
] = (
    validation_selection_df[
        "configuration_id"
    ]
    .str.split("__")
    .str[0]
)


selected_configuration_df = (

    validation_selection_df
    .sort_values(
        "val_macro_f1",
        ascending=False
    )
    .groupby(
        "architecture",
        as_index=False
    )
    .first()
)


selected_configuration_df = (
    selected_configuration_df[
        [
            "architecture",
            "configuration_id",
            "selected_epoch",
            "val_accuracy",
            "val_balanced_accuracy",
            "val_macro_f1"
        ]
    ]
    .rename(
        columns={
            "configuration_id":
                "validation_selected_configuration"
        }
    )
)


# Merge test point estimates

test_metrics_for_merge = (
    point_metrics_df[
        [
            "configuration_id",
            "test_accuracy",
            "test_balanced_accuracy",
            "test_macro_f1"
        ]
    ]
    .copy()
)


selected_configuration_df = (
    selected_configuration_df
    .merge(

        test_metrics_for_merge,

        left_on=
            "validation_selected_configuration",

        right_on=
            "configuration_id",

        how="left",

        validate="one_to_one"
    )
    .drop(
        columns=[
            "configuration_id"
        ]
    )
)


# Merge 95% confidence intervals

bootstrap_for_merge = (
    bootstrap_summary_df[
        [
            "configuration_id",

            "accuracy_ci_lower",
            "accuracy_ci_upper",

            "balanced_accuracy_ci_lower",
            "balanced_accuracy_ci_upper",

            "macro_f1_ci_lower",
            "macro_f1_ci_upper",

            "ece_after_ci_lower",
            "ece_after_ci_upper"
        ]
    ]
    .copy()
)


selected_configuration_df = (
    selected_configuration_df
    .merge(

        bootstrap_for_merge,

        left_on=
            "validation_selected_configuration",

        right_on=
            "configuration_id",

        how="left",

        validate="one_to_one"
    )
    .drop(
        columns=[
            "configuration_id"
        ]
    )
)


# Merge calibration metrics

calibration_for_merge = (
    calibration_summary_df[
        [
            "configuration_id",
            "temperature",
            "test_ece_before",
            "test_ece_after",
            "test_nll_before",
            "test_nll_after"
        ]
    ]
    .copy()
)


selected_configuration_df = (
    selected_configuration_df
    .merge(

        calibration_for_merge,

        left_on=
            "validation_selected_configuration",

        right_on=
            "configuration_id",

        how="left",

        validate="one_to_one"
    )
    .drop(
        columns=[
            "configuration_id"
        ]
    )
)


# Combine performance and efficiency

final_architecture_comparison_df = (
    selected_configuration_df
    .merge(

        architecture_efficiency_df,

        on="architecture",

        how="left",

        validate="one_to_one"
    )
)


# Readable architecture names

architecture_display_names = {

    "convnext_tiny":
        "ConvNeXt-Tiny",

    "mobilenet_v2":
        "MobileNetV2",

    "swin_tiny":
        "Swin-Tiny",

    "vit_b16":
        "ViT-B/16"
}


final_architecture_comparison_df[
    "architecture_display"
] = (
    final_architecture_comparison_df[
        "architecture"
    ]
    .map(
        architecture_display_names
    )
)


# Extract selected augmentation

final_architecture_comparison_df[
    "selected_augmentation"
] = (
    final_architecture_comparison_df[
        "validation_selected_configuration"
    ]
    .str.split("__")
    .str[1]
)


# Sort by validation Macro-F1

final_architecture_comparison_df = (

    final_architecture_comparison_df
    .sort_values(
        "val_macro_f1",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# Build publication-ready confidence interval columns

final_architecture_comparison_df[
    "test_accuracy_ci95"
] = (

    final_architecture_comparison_df.apply(

        lambda row:

            f"{row['test_accuracy']:.4f} "
            f"[{row['accuracy_ci_lower']:.4f}, "
            f"{row['accuracy_ci_upper']:.4f}]",

        axis=1
    )
)


final_architecture_comparison_df[
    "test_balanced_accuracy_ci95"
] = (

    final_architecture_comparison_df.apply(

        lambda row:

            f"{row['test_balanced_accuracy']:.4f} "
            f"[{row['balanced_accuracy_ci_lower']:.4f}, "
            f"{row['balanced_accuracy_ci_upper']:.4f}]",

        axis=1
    )
)


final_architecture_comparison_df[
    "test_macro_f1_ci95"
] = (

    final_architecture_comparison_df.apply(

        lambda row:

            f"{row['test_macro_f1']:.4f} "
            f"[{row['macro_f1_ci_lower']:.4f}, "
            f"{row['macro_f1_ci_upper']:.4f}]",

        axis=1
    )
)


# Compact manuscript table

paper_comparison_df = (
    final_architecture_comparison_df[
        [
            "architecture_display",
            "selected_augmentation",

            "parameters_millions",

            "test_accuracy_ci95",
            "test_balanced_accuracy_ci95",
            "test_macro_f1_ci95",

            "test_ece_before",
            "test_ece_after",

            "latency_median_ms",
            "latency_p95_ms",

            "throughput_images_per_second",
            "peak_gpu_memory_mb"
        ]
    ]
    .copy()
)


paper_comparison_df.columns = [

    "Architecture",
    "Validation-selected augmentation",

    "Parameters (M)",

    "Test Accuracy [95% CI]",
    "Test Balanced Accuracy [95% CI]",
    "Test Macro-F1 [95% CI]",

    "ECE before calibration",
    "ECE after calibration",

    "Median latency (ms)",
    "P95 latency (ms)",

    "Throughput (img/s)",
    "Peak GPU memory (MB)"
]


# Save results

ARCHITECTURE_EFFICIENCY_OUTPUT = (
    SUMMARY_DIR
    / "24_architecture_efficiency_summary.csv"
)


FINAL_ARCHITECTURE_OUTPUT = (
    SUMMARY_DIR
    / "25_validation_selected_architecture_comparison.csv"
)


PAPER_TABLE_OUTPUT = (
    SUMMARY_DIR
    / "26_paper_performance_efficiency_table.csv"
)


architecture_efficiency_df.to_csv(

    ARCHITECTURE_EFFICIENCY_OUTPUT,

    index=False
)


final_architecture_comparison_df.to_csv(

    FINAL_ARCHITECTURE_OUTPUT,

    index=False
)


paper_comparison_df.to_csv(

    PAPER_TABLE_OUTPUT,

    index=False
)


# Display results

print("=" * 80)
print("CONSOLIDATED ARCHITECTURE COMPARISON")
print("=" * 80)


display(
    paper_comparison_df
)


print("\nConfiguration selected using validation:")

display(

    final_architecture_comparison_df[
        [
            "architecture_display",
            "validation_selected_configuration",
            "val_macro_f1",
            "test_macro_f1",
            "test_ece_after",
            "parameters_millions",
            "latency_median_ms",
            "throughput_images_per_second"
        ]
    ]
)


print("\nSaved files:")

print(
    ARCHITECTURE_EFFICIENCY_OUTPUT
)

print(
    FINAL_ARCHITECTURE_OUTPUT
)

print(
    PAPER_TABLE_OUTPUT
)

# **Cell 11 — Definitive learning curves**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Output directory

LEARNING_CURVES_DIR = (
    ANALYSIS_ROOT
    / "learning_curves"
)

LEARNING_CURVES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Readable labels

ARCHITECTURE_NAMES = {

    "convnext_tiny":
        "ConvNeXt-Tiny",

    "mobilenet_v2":
        "MobileNetV2",

    "swin_tiny":
        "Swin-Tiny",

    "vit_b16":
        "ViT-B/16"
}


AUGMENTATION_NAMES = {

    "none":
        "None",

    "mixup":
        "MixUp",

    "cutmix":
        "CutMix"
}


# Load the 12 training histories

history_frames = []


print("=" * 80)
print("LOADING TRAINING HISTORIES")
print("=" * 80)


for configuration_id in EXPECTED_CONFIGURATIONS:

    run_directory = (
        RUN_DIRECTORIES[
            configuration_id
        ]
    )


    history_path = (
        find_unique_file(
            run_directory,
            f"{configuration_id}_history.csv"
        )
    )


    history_df = pd.read_csv(
        history_path,
        low_memory=False
    )


    architecture = (
        configuration_id
        .split("__")[0]
    )


    augmentation = (
        configuration_id
        .split("__")[1]
    )


    history_df[
        "configuration_id"
    ] = configuration_id


    history_df[
        "architecture"
    ] = architecture


    history_df[
        "augmentation"
    ] = augmentation


    history_df[
        "architecture_display"
    ] = ARCHITECTURE_NAMES[
        architecture
    ]


    history_df[
        "augmentation_display"
    ] = AUGMENTATION_NAMES[
        augmentation
    ]


    history_frames.append(
        history_df
    )


    print(
        configuration_id,
        "->",
        len(history_df),
        "epochs"
    )


all_histories_df = pd.concat(

    history_frames,

    ignore_index=True
)


# Save consolidated training history

ALL_HISTORIES_OUTPUT = (

    LEARNING_CURVES_DIR
    / "27_all_training_histories.csv"
)


all_histories_df.to_csv(

    ALL_HISTORIES_OUTPUT,

    index=False
)


# Check required history columns

required_history_columns = [

    "epoch",
    "train_loss",
    "val_loss",
    "val_accuracy",
    "val_balanced_accuracy",
    "val_macro_f1"
]


missing_columns = [

    column

    for column
    in required_history_columns

    if column
    not in all_histories_df.columns
]


if missing_columns:

    raise RuntimeError(
        "Missing training history columns: "
        f"{missing_columns}"
    )


# Generate individual curves for each configuration

individual_curve_records = []


for configuration_id in EXPECTED_CONFIGURATIONS:

    config_df = (

        all_histories_df[
            all_histories_df[
                "configuration_id"
            ]
            ==
            configuration_id
        ]
        .copy()
        .sort_values(
            "epoch"
        )
    )


    architecture = (
        configuration_id
        .split("__")[0]
    )


    augmentation = (
        configuration_id
        .split("__")[1]
    )


    architecture_display = (
        ARCHITECTURE_NAMES[
            architecture
        ]
    )


    augmentation_display = (
        AUGMENTATION_NAMES[
            augmentation
        ]
    )


    # Best epoch according to validation

    best_row = (

        config_df
        .sort_values(
            "val_macro_f1",
            ascending=False
        )
        .iloc[0]
    )


    best_epoch = int(
        best_row[
            "epoch"
        ]
    )


    best_macro_f1 = float(
        best_row[
            "val_macro_f1"
        ]
    )


    # Figure 1: Training and validation loss

    fig, ax = plt.subplots(
        figsize=(8, 5)
    )


    ax.plot(

        config_df[
            "epoch"
        ],

        config_df[
            "train_loss"
        ],

        marker="o",

        label="Training loss"
    )


    ax.plot(

        config_df[
            "epoch"
        ],

        config_df[
            "val_loss"
        ],

        marker="o",

        label="Validation loss"
    )


    ax.set_xlabel(
        "Epoch"
    )

    ax.set_ylabel(
        "Loss"
    )

    ax.set_title(
        f"{architecture_display} + "
        f"{augmentation_display}"
    )

    ax.grid(
        alpha=0.25
    )

    ax.legend()


    fig.tight_layout()


    loss_output = (

        LEARNING_CURVES_DIR
        / (
            f"{configuration_id}"
            "__loss_curve.png"
        )
    )


    fig.savefig(

        loss_output,

        dpi=300,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    # Figure 2: Validation metrics

    fig, ax = plt.subplots(
        figsize=(8, 5)
    )


    ax.plot(

        config_df[
            "epoch"
        ],

        config_df[
            "val_accuracy"
        ],

        marker="o",

        label="Accuracy"
    )


    ax.plot(

        config_df[
            "epoch"
        ],

        config_df[
            "val_balanced_accuracy"
        ],

        marker="o",

        label="Balanced Accuracy"
    )


    ax.plot(

        config_df[
            "epoch"
        ],

        config_df[
            "val_macro_f1"
        ],

        marker="o",

        label="Macro-F1"
    )


    # Selected best epoch
    ax.axvline(

        x=best_epoch,

        linestyle="--",

        linewidth=1.5,

        label=(
            f"Selected epoch = "
            f"{best_epoch}"
        )
    )


    ax.scatter(

        [best_epoch],

        [best_macro_f1],

        s=70,

        zorder=5
    )


    ax.set_xlabel(
        "Epoch"
    )

    ax.set_ylabel(
        "Validation score"
    )


    ax.set_ylim(
        0.0,
        1.01
    )


    ax.set_title(
        f"{architecture_display} + "
        f"{augmentation_display}"
    )


    ax.grid(
        alpha=0.25
    )


    ax.legend(
        loc="lower right"
    )


    fig.tight_layout()


    metrics_output = (

        LEARNING_CURVES_DIR
        / (
            f"{configuration_id}"
            "__validation_metrics.png"
        )
    )


    fig.savefig(

        metrics_output,

        dpi=300,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    individual_curve_records.append({

        "configuration_id":
            configuration_id,

        "epochs":
            len(
                config_df
            ),

        "selected_epoch":
            best_epoch,

        "best_val_macro_f1":
            best_macro_f1,

        "loss_curve":
            str(
                loss_output
            ),

        "validation_metrics_curve":
            str(
                metrics_output
            )
    })


# Compare augmentation strategies by architecture

architecture_curve_records = []


for architecture in [

    "convnext_tiny",
    "mobilenet_v2",
    "swin_tiny",
    "vit_b16"

]:

    architecture_display = (
        ARCHITECTURE_NAMES[
            architecture
        ]
    )


    fig, ax = plt.subplots(
        figsize=(8, 5)
    )


    for augmentation in [

        "none",
        "mixup",
        "cutmix"

    ]:

        configuration_id = (
            f"{architecture}"
            f"__{augmentation}"
        )


        config_df = (

            all_histories_df[
                all_histories_df[
                    "configuration_id"
                ]
                ==
                configuration_id
            ]
            .copy()
            .sort_values(
                "epoch"
            )
        )


        ax.plot(

            config_df[
                "epoch"
            ],

            config_df[
                "val_macro_f1"
            ],

            marker="o",

            label=
                AUGMENTATION_NAMES[
                    augmentation
                ]
        )


    ax.set_xlabel(
        "Epoch"
    )

    ax.set_ylabel(
        "Validation Macro-F1"
    )


    ax.set_ylim(
        0.0,
        1.01
    )


    ax.set_title(
        f"{architecture_display}: "
        "augmentation comparison"
    )


    ax.grid(
        alpha=0.25
    )


    ax.legend(
        title="Training strategy",
        loc="lower right"
    )


    fig.tight_layout()


    architecture_output = (

        LEARNING_CURVES_DIR
        / (
            f"{architecture}"
            "__augmentation_macro_f1.png"
        )
    )


    fig.savefig(

        architecture_output,

        dpi=300,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    architecture_curve_records.append({

        "architecture":
            architecture,

        "architecture_display":
            architecture_display,

        "figure":
            str(
                architecture_output
            )
    })


# Summary tables

individual_curve_summary_df = pd.DataFrame(
    individual_curve_records
)


architecture_curve_summary_df = pd.DataFrame(
    architecture_curve_records
)


# Save figure indexes

INDIVIDUAL_CURVES_OUTPUT = (

    LEARNING_CURVES_DIR
    / "28_individual_learning_curve_index.csv"
)


ARCHITECTURE_CURVES_OUTPUT = (

    LEARNING_CURVES_DIR
    / "29_architecture_learning_curve_index.csv"
)


individual_curve_summary_df.to_csv(

    INDIVIDUAL_CURVES_OUTPUT,

    index=False
)


architecture_curve_summary_df.to_csv(

    ARCHITECTURE_CURVES_OUTPUT,

    index=False
)


# Display a compact summary

print("\n" + "=" * 80)
print("LEARNING CURVES GENERATED")
print("=" * 80)


display(

    individual_curve_summary_df[
        [
            "configuration_id",
            "epochs",
            "selected_epoch",
            "best_val_macro_f1"
        ]
    ]
)


print(
    "\nIndividual figures generated:",
    len(
        individual_curve_summary_df
    )
    * 2
)


print(
    "Architecture-level comparisons:",
    len(
        architecture_curve_summary_df
    )
)


print("\nDirectory:")
print(
    LEARNING_CURVES_DIR
)


print("\nConsolidated training history:")
print(
    ALL_HISTORIES_OUTPUT
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Reference for the actual checkpoint selection
checkpoint_selection_df = (
    validation_summary_df[
        [
            "configuration_id",
            "selected_epoch",
            "val_macro_f1"
        ]
    ]
    .copy()
    .rename(
        columns={
            "val_macro_f1":
                "selected_val_macro_f1"
        }
    )
)

checkpoint_selection = (
    checkpoint_selection_df
    .set_index(
        "configuration_id"
    )
    .to_dict(
        orient="index"
    )
)


# Regenerate individual figures with the correct selected epochs
corrected_curve_records = []

print("=" * 80)
print("CORRECTING SELECTED EPOCHS")
print("=" * 80)

for configuration_id in EXPECTED_CONFIGURATIONS:

    config_df = (
        all_histories_df[
            all_histories_df[
                "configuration_id"
            ]
            ==
            configuration_id
        ]
        .copy()
        .sort_values(
            "epoch"
        )
    )

    architecture = (
        configuration_id
        .split("__")[0]
    )

    augmentation = (
        configuration_id
        .split("__")[1]
    )

    architecture_display = (
        ARCHITECTURE_NAMES[
            architecture
        ]
    )

    augmentation_display = (
        AUGMENTATION_NAMES[
            augmentation
        ]
    )

    # Actual checkpoint selection
    selected_epoch = int(
        checkpoint_selection[
            configuration_id
        ][
            "selected_epoch"
        ]
    )

    selected_val_macro_f1 = float(
        checkpoint_selection[
            configuration_id
        ][
            "selected_val_macro_f1"
        ]
    )

    # Absolute history maximum for auditing only, not selection
    absolute_max_row = (
        config_df
        .sort_values(
            "val_macro_f1",
            ascending=False
        )
        .iloc[0]
    )

    absolute_max_epoch = int(
        absolute_max_row[
            "epoch"
        ]
    )

    absolute_max_macro_f1 = float(
        absolute_max_row[
            "val_macro_f1"
        ]
    )

    difference_absolute_vs_selected = (
        absolute_max_macro_f1
        -
        selected_val_macro_f1
    )

    # Regenerate the validation metrics curve
    fig, ax = plt.subplots(
        figsize=(8, 5)
    )

    ax.plot(
        config_df["epoch"],
        config_df["val_accuracy"],
        marker="o",
        label="Accuracy"
    )

    ax.plot(
        config_df["epoch"],
        config_df["val_balanced_accuracy"],
        marker="o",
        label="Balanced Accuracy"
    )

    ax.plot(
        config_df["epoch"],
        config_df["val_macro_f1"],
        marker="o",
        label="Macro-F1"
    )

    # Epoch actually selected during training
    ax.axvline(
        x=selected_epoch,
        linestyle="--",
        linewidth=1.5,
        label=(
            f"Selected checkpoint epoch = "
            f"{selected_epoch}"
        )
    )

    ax.scatter(
        [selected_epoch],
        [selected_val_macro_f1],
        s=70,
        zorder=5
    )

    ax.set_xlabel(
        "Epoch"
    )

    ax.set_ylabel(
        "Validation score"
    )

    ax.set_ylim(
        0.0,
        1.01
    )

    ax.set_title(
        f"{architecture_display} + "
        f"{augmentation_display}"
    )

    ax.grid(
        alpha=0.25
    )

    ax.legend(
        loc="lower right"
    )

    fig.tight_layout()

    metrics_output = (
        LEARNING_CURVES_DIR
        /
        (
            f"{configuration_id}"
            "__validation_metrics.png"
        )
    )

    # Overwrite the previous figure
    fig.savefig(
        metrics_output,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(
        fig
    )

    # Corrected audit record
    corrected_curve_records.append({

        "configuration_id":
            configuration_id,

        "epochs":
            len(
                config_df
            ),

        "selected_checkpoint_epoch":
            selected_epoch,

        "selected_checkpoint_val_macro_f1":
            selected_val_macro_f1,

        "absolute_history_max_epoch":
            absolute_max_epoch,

        "absolute_history_max_macro_f1":
            absolute_max_macro_f1,

        "absolute_minus_selected":
            difference_absolute_vs_selected,

        "same_epoch":
            (
                selected_epoch
                ==
                absolute_max_epoch
            ),

        "validation_metrics_curve":
            str(
                metrics_output
            )
    })

    print(
        f"{configuration_id:<28}",
        f"checkpoint={selected_epoch:<2}",
        f"absolute_max={absolute_max_epoch:<2}",
        f"Δ={difference_absolute_vs_selected:.8f}"
    )


# Corrected DataFrame
corrected_learning_curve_df = pd.DataFrame(
    corrected_curve_records
)


# Verify the min_delta selection rule
selection_differences_df = (
    corrected_learning_curve_df[
        ~corrected_learning_curve_df[
            "same_epoch"
        ]
    ]
    .copy()
)

if len(
    selection_differences_df
) > 0:

    invalid_differences = (
        selection_differences_df[
            selection_differences_df[
                "absolute_minus_selected"
            ]
            >
            (
                SELECTION_MIN_DELTA
                +
                METRIC_TOLERANCE
            )
        ]
    )

    if len(
        invalid_differences
    ) > 0:

        display(
            invalid_differences
        )

        raise RuntimeError(
            "A difference was found that cannot be explained "
            "by the min_delta rule."
        )


# Save the corrected figure index
CORRECTED_CURVES_OUTPUT = (
    LEARNING_CURVES_DIR
    / "28_individual_learning_curve_index.csv"
)

corrected_learning_curve_df.to_csv(
    CORRECTED_CURVES_OUTPUT,
    index=False
)


# Display the audit results
print("\n" + "=" * 80)
print("LEARNING CURVES CORRECTED")
print("=" * 80)

display(
    corrected_learning_curve_df[
        [
            "configuration_id",
            "epochs",
            "selected_checkpoint_epoch",
            "selected_checkpoint_val_macro_f1",
            "absolute_history_max_epoch",
            "absolute_history_max_macro_f1",
            "absolute_minus_selected",
            "same_epoch"
        ]
    ]
)

print(
    "\nConfigurations where the absolute maximum "
    "does not match the selected checkpoint:"
)

display(
    selection_differences_df[
        [
            "configuration_id",
            "selected_checkpoint_epoch",
            "selected_checkpoint_val_macro_f1",
            "absolute_history_max_epoch",
            "absolute_history_max_macro_f1",
            "absolute_minus_selected"
        ]
    ]
)

print(
    "\nCorrected index:"
)

print(
    CORRECTED_CURVES_OUTPUT
)


# **Cell 12 — Reliability diagrams before/after Temperature Scalin**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Directories

RELIABILITY_FIGURES_DIR = (
    CALIBRATION_DIR
    / "reliability_diagrams"
)

PAPER_RELIABILITY_DIR = (
    RELIABILITY_FIGURES_DIR
    / "paper_selected_models"
)

RELIABILITY_FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PAPER_RELIABILITY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Readable names

DISPLAY_NAMES = {

    "convnext_tiny__none":
        "ConvNeXt-Tiny + None",

    "convnext_tiny__mixup":
        "ConvNeXt-Tiny + MixUp",

    "convnext_tiny__cutmix":
        "ConvNeXt-Tiny + CutMix",

    "mobilenet_v2__none":
        "MobileNetV2 + None",

    "mobilenet_v2__mixup":
        "MobileNetV2 + MixUp",

    "mobilenet_v2__cutmix":
        "MobileNetV2 + CutMix",

    "swin_tiny__none":
        "Swin-Tiny + None",

    "swin_tiny__mixup":
        "Swin-Tiny + MixUp",

    "swin_tiny__cutmix":
        "Swin-Tiny + CutMix",

    "vit_b16__none":
        "ViT-B/16 + None",

    "vit_b16__mixup":
        "ViT-B/16 + MixUp",

    "vit_b16__cutmix":
        "ViT-B/16 + CutMix"
}


# Validation-selected configurations

PAPER_SELECTED_CONFIGURATIONS = [

    "swin_tiny__cutmix",

    "convnext_tiny__none",

    "mobilenet_v2__cutmix",

    "vit_b16__cutmix"
]


# Load reliability bins

def load_reliability_bins(
    configuration_id,
    calibration_state
):

    path = (
        RELIABILITY_DIR
        / (
            f"{configuration_id}"
            f"__{calibration_state}.csv"
        )
    )


    if not path.exists():

        raise FileNotFoundError(
            f"File not found:\n{path}"
        )


    dataframe = pd.read_csv(
        path,
        low_memory=False
    )


    return dataframe


# ECE lookup

calibration_lookup = (

    calibration_summary_df
    .set_index(
        "configuration_id"
    )
)


# Generate figures for each configuration

reliability_figure_records = []


print("=" * 80)
print("GENERATING RELIABILITY DIAGRAMS")
print("=" * 80)


for run_number, configuration_id in enumerate(
    EXPECTED_CONFIGURATIONS,
    start=1
):

    print(
        f"[{run_number}/12] "
        f"{configuration_id}"
    )


    # Test data

    before_df = load_reliability_bins(

        configuration_id,

        "test_before"
    )


    after_df = load_reliability_bins(

        configuration_id,

        "test_after"
    )


    # Bins with observations only

    before_valid = (

        before_df[
            before_df[
                "count"
            ] > 0
        ]
        .copy()
    )


    after_valid = (

        after_df[
            after_df[
                "count"
            ] > 0
        ]
        .copy()
    )


    # ECE

    ece_before = float(

        calibration_lookup.loc[
            configuration_id,
            "test_ece_before"
        ]
    )


    ece_after = float(

        calibration_lookup.loc[
            configuration_id,
            "test_ece_after"
        ]
    )


    temperature = float(

        calibration_lookup.loc[
            configuration_id,
            "temperature"
        ]
    )


    # Reliability diagram

    fig, ax = plt.subplots(
        figsize=(7, 7)
    )


    # Perfect calibration
    ax.plot(

        [0, 1],

        [0, 1],

        linestyle="--",

        linewidth=1.5,

        label="Perfect calibration"
    )


    # Before calibration
    ax.plot(

        before_valid[
            "mean_confidence"
        ],

        before_valid[
            "accuracy"
        ],

        marker="o",

        linewidth=2,

        label=(
            "Before calibration "
            f"(ECE={ece_before:.4f})"
        )
    )


    # After calibration
    ax.plot(

        after_valid[
            "mean_confidence"
        ],

        after_valid[
            "accuracy"
        ],

        marker="o",

        linewidth=2,

        label=(
            "After Temperature Scaling "
            f"(ECE={ece_after:.4f})"
        )
    )


    # Plot formatting

    ax.set_xlim(
        0.0,
        1.0
    )

    ax.set_ylim(
        0.0,
        1.0
    )


    ax.set_xlabel(
        "Mean predicted confidence"
    )

    ax.set_ylabel(
        "Empirical accuracy"
    )


    ax.set_title(
        (
            f"{DISPLAY_NAMES[configuration_id]}\n"
            f"Test reliability diagram "
            f"(T={temperature:.3f})"
        )
    )


    ax.grid(
        alpha=0.25
    )


    ax.legend(
        loc="lower right",
        fontsize=9
    )


    fig.tight_layout()


    output_path = (

        RELIABILITY_FIGURES_DIR
        / (
            f"{configuration_id}"
            "__reliability_diagram.png"
        )
    )


    fig.savefig(

        output_path,

        dpi=300,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    reliability_figure_records.append({

        "configuration_id":
            configuration_id,

        "display_name":
            DISPLAY_NAMES[
                configuration_id
            ],

        "temperature":
            temperature,

        "test_ece_before":
            ece_before,

        "test_ece_after":
            ece_after,

        "ece_reduction":
            (
                ece_before
                -
                ece_after
            ),

        "figure_path":
            str(
                output_path
            )
    })


# Create publication-ready versions
# Only configurations selected through validation

paper_figure_records = []


for configuration_id in (
    PAPER_SELECTED_CONFIGURATIONS
):

    before_df = load_reliability_bins(

        configuration_id,

        "test_before"
    )


    after_df = load_reliability_bins(

        configuration_id,

        "test_after"
    )


    before_valid = (

        before_df[
            before_df["count"] > 0
        ]
        .copy()
    )


    after_valid = (

        after_df[
            after_df["count"] > 0
        ]
        .copy()
    )


    ece_before = float(

        calibration_lookup.loc[
            configuration_id,
            "test_ece_before"
        ]
    )


    ece_after = float(

        calibration_lookup.loc[
            configuration_id,
            "test_ece_after"
        ]
    )


    # Publication-ready figure

    fig, ax = plt.subplots(
        figsize=(6.5, 6.5)
    )


    ax.plot(

        [0, 1],

        [0, 1],

        linestyle="--",

        linewidth=1.5,

        label="Perfect calibration"
    )


    ax.plot(

        before_valid[
            "mean_confidence"
        ],

        before_valid[
            "accuracy"
        ],

        marker="o",

        linewidth=2,

        label=(
            f"Before (ECE={ece_before:.4f})"
        )
    )


    ax.plot(

        after_valid[
            "mean_confidence"
        ],

        after_valid[
            "accuracy"
        ],

        marker="o",

        linewidth=2,

        label=(
            f"After TS (ECE={ece_after:.4f})"
        )
    )


    ax.set_xlim(
        0,
        1
    )

    ax.set_ylim(
        0,
        1
    )


    ax.set_xlabel(
        "Confidence"
    )

    ax.set_ylabel(
        "Accuracy"
    )


    ax.set_title(
        DISPLAY_NAMES[
            configuration_id
        ]
    )


    ax.grid(
        alpha=0.20
    )

    ax.legend(
        loc="lower right"
    )


    fig.tight_layout()


    paper_path = (

        PAPER_RELIABILITY_DIR
        / (
            f"{configuration_id}"
            "__paper_reliability.png"
        )
    )


    fig.savefig(

        paper_path,

        dpi=600,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    paper_figure_records.append({

        "configuration_id":
            configuration_id,

        "figure_path":
            str(
                paper_path
            )
    })


# Summaries

reliability_figures_df = pd.DataFrame(
    reliability_figure_records
)


paper_reliability_figures_df = pd.DataFrame(
    paper_figure_records
)


# Save indexes

RELIABILITY_INDEX_OUTPUT = (

    RELIABILITY_FIGURES_DIR
    / "30_reliability_diagram_index.csv"
)


PAPER_RELIABILITY_INDEX_OUTPUT = (

    PAPER_RELIABILITY_DIR
    / "31_paper_reliability_diagram_index.csv"
)


reliability_figures_df.to_csv(

    RELIABILITY_INDEX_OUTPUT,

    index=False
)


paper_reliability_figures_df.to_csv(

    PAPER_RELIABILITY_INDEX_OUTPUT,

    index=False
)


# Verification

if len(
    reliability_figures_df
) != 12:

    raise RuntimeError(
        "The 12 reliability diagrams were not generated."
    )


if len(
    paper_reliability_figures_df
) != 4:

    raise RuntimeError(
        "The 4 figures "
        "selected for the paper were not generated."
    )


# Display compact summary

print("\n" + "=" * 80)
print("RELIABILITY DIAGRAMS COMPLETED")
print("=" * 80)


display(

    reliability_figures_df[
        [
            "configuration_id",
            "temperature",
            "test_ece_before",
            "test_ece_after",
            "ece_reduction"
        ]
    ]
)


print(
    "\nReliability diagrams generated:",
    len(
        reliability_figures_df
    )
)


print(
    "Publication-ready figures:",
    len(
        paper_reliability_figures_df
    )
)


print("\nFull directory:")
print(
    RELIABILITY_FIGURES_DIR
)


print("\nFigures selected for the paper:")
print(
    PAPER_RELIABILITY_DIR
)


print("\nIndex:")
print(
    RELIABILITY_INDEX_OUTPUT
)

# **Cell 13 — Complete hardware specifications**

In [ ]:
from pathlib import Path
import os
import platform
import subprocess

import pandas as pd
import psutil
import torch


# Output directory
HARDWARE_DIR = (
    ANALYSIS_ROOT
    / "hardware"
)

HARDWARE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Robust CPU model detection
def get_cpu_model():

    # Linux / Kaggle: /proc/cpuinfo
    cpuinfo_path = Path(
        "/proc/cpuinfo"
    )

    if cpuinfo_path.exists():

        with open(
            cpuinfo_path,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as file:

            for line in file:

                if line.lower().startswith(
                    "model name"
                ):

                    return (
                        line.split(
                            ":",
                            1
                        )[1]
                        .strip()
                    )

    # Alternative: platform information
    processor = platform.processor()

    if processor:

        return processor

    # Last resort: lscpu
    try:

        output = subprocess.check_output(
            ["lscpu"],
            text=True
        )

        for line in output.splitlines():

            if line.startswith(
                "Model name:"
            ):

                return (
                    line.split(
                        ":",
                        1
                    )[1]
                    .strip()
                )

    except Exception:

        pass

    return "Unknown"


# CPU specifications
cpu_model = get_cpu_model()

physical_cores = psutil.cpu_count(
    logical=False
)

logical_cores = psutil.cpu_count(
    logical=True
)


# System RAM
virtual_memory = psutil.virtual_memory()

total_ram_gb = (
    virtual_memory.total
    /
    (1024 ** 3)
)

available_ram_gb = (
    virtual_memory.available
    /
    (1024 ** 3)
)


# GPU specifications
if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA is not available."
    )


gpu_name = torch.cuda.get_device_name(
    0
)

gpu_properties = torch.cuda.get_device_properties(
    0
)

gpu_total_memory_gb = (
    gpu_properties.total_memory
    /
    (1024 ** 3)
)


# System information
operating_system = (
    platform.platform()
)

python_version = (
    platform.python_version()
)


# Complete hardware specifications
hardware_specification_df = pd.DataFrame({

    "component": [

        "CPU model",
        "Physical CPU cores",
        "Logical CPU cores",
        "System RAM (GB)",
        "Available RAM at inspection (GB)",
        "GPU model",
        "GPU memory / VRAM (GB)",
        "CUDA version",
        "PyTorch version",
        "Python version",
        "Operating system",
        "Inference precision",
        "Input image size",
        "Evaluation DataLoader workers"
    ],

    "value": [

        cpu_model,
        physical_cores,
        logical_cores,
        round(
            total_ram_gb,
            3
        ),
        round(
            available_ram_gb,
            3
        ),
        gpu_name,
        round(
            gpu_total_memory_gb,
            3
        ),
        torch.version.cuda,
        torch.__version__,
        python_version,
        operating_system,
        "Automatic Mixed Precision (FP16)",
        f"{IMG_SIZE} × {IMG_SIZE}",
        NUM_WORKERS
    ]
})


# Exact latency and throughput measurement protocol
efficiency_protocol_df = pd.DataFrame({

    "measurement": [

        "Latency device",
        "Latency precision",
        "Latency batch size",
        "Latency warm-up iterations",
        "Latency measured iterations",
        "Latency synchronization",
        "Latency statistic reported",
        "Additional latency statistic",
        "Throughput device",
        "Throughput precision",
        "Throughput batch size",
        "Throughput warm-up iterations",
        "Throughput measured iterations",
        "Throughput synchronization",
        "End-to-end test images",
        "Test DataLoader workers",
        "Preprocessing for model-only latency",
        "Preprocessing for end-to-end test",
        "Inference mode"
    ],

    "value": [

        gpu_name,
        "AMP FP16",
        LATENCY_BATCH_SIZE,
        LATENCY_WARMUP_ITERATIONS,
        LATENCY_MEASURED_ITERATIONS,
        "torch.cuda.synchronize() before and after each measurement",
        "Median latency (ms)",
        "95th percentile latency (ms)",
        gpu_name,
        "AMP FP16",
        THROUGHPUT_BATCH_SIZE,
        THROUGHPUT_WARMUP_ITERATIONS,
        THROUGHPUT_MEASURED_ITERATIONS,
        "torch.cuda.synchronize()",
        len(
            test_dataset
        ),
        NUM_WORKERS,
        (
            "Excluded; synthetic 3 × 224 × 224 tensor "
            "used for model-only benchmark"
        ),
        (
            "Included image loading, deterministic resize, "
            "center crop, normalization, host-to-device transfer "
            "and model inference"
        ),
        "torch.inference_mode()"
    ]
})


# Observed GPU memory by architecture
architecture_memory_df = (

    architecture_efficiency_df[
        [
            "architecture",
            "parameters_millions",
            "checkpoint_size_mb",
            "peak_gpu_memory_mb"
        ]
    ]
    .copy()
)

architecture_memory_df[
    "architecture"
] = architecture_memory_df[
    "architecture"
].map(
    architecture_display_names
)

architecture_memory_df = (
    architecture_memory_df
    .rename(
        columns={
            "architecture":
                "Architecture",

            "parameters_millions":
                "Parameters (M)",

            "checkpoint_size_mb":
                "Checkpoint size (MB)",

            "peak_gpu_memory_mb":
                "Peak allocated GPU memory (MB)"
        }
    )
)


# Save results
HARDWARE_SPEC_OUTPUT = (
    HARDWARE_DIR
    / "32_complete_hardware_specification.csv"
)

EFFICIENCY_PROTOCOL_OUTPUT = (
    HARDWARE_DIR
    / "33_latency_throughput_protocol.csv"
)

ARCHITECTURE_MEMORY_OUTPUT = (
    HARDWARE_DIR
    / "34_architecture_gpu_memory_summary.csv"
)

hardware_specification_df.to_csv(
    HARDWARE_SPEC_OUTPUT,
    index=False
)

efficiency_protocol_df.to_csv(
    EFFICIENCY_PROTOCOL_OUTPUT,
    index=False
)

architecture_memory_df.to_csv(
    ARCHITECTURE_MEMORY_OUTPUT,
    index=False
)


# R9 verification
required_hardware_items = {
    "CPU model",
    "Physical CPU cores",
    "Logical CPU cores",
    "System RAM (GB)",
    "GPU model",
    "GPU memory / VRAM (GB)",
    "CUDA version"
}

found_items = set(
    hardware_specification_df[
        "component"
    ]
)

missing_items = (
    required_hardware_items
    -
    found_items
)

if missing_items:

    raise RuntimeError(
        "Required hardware components for R9 are missing: "
        f"{sorted(missing_items)}"
    )


# Display results
print("=" * 80)
print("HARDWARE SPECIFICATIONS - R9")
print("=" * 80)

display(
    hardware_specification_df
)

print("\n" + "=" * 80)
print("LATENCY AND THROUGHPUT PROTOCOL")
print("=" * 80)

display(
    efficiency_protocol_df
)

print("\n" + "=" * 80)
print("GPU MEMORY BY ARCHITECTURE")
print("=" * 80)

display(
    architecture_memory_df
)

print("\nSaved files:")

print(
    HARDWARE_SPEC_OUTPUT
)

print(
    EFFICIENCY_PROTOCOL_OUTPUT
)

print(
    ARCHITECTURE_MEMORY_OUTPUT
)

print(
    "\nR9 ready: GPU, VRAM, CUDA, CPU, "
    "cores, RAM, and measurement protocol documented."
)


# **Cell 14 — Reproducible selection of images for XAI**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Configuration
XAI_SEED = 42

XAI_SELECTED_CONFIGURATIONS = [
    "swin_tiny__cutmix",
    "convnext_tiny__none",
    "mobilenet_v2__cutmix",
    "vit_b16__cutmix"
]

XAI_REFERENCE_CONFIGURATION = (
    "swin_tiny__cutmix"
)

XAI_DIR = (
    ANALYSIS_ROOT
    / "explainability"
)

XAI_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Load predictions from the reference model
reference_prediction_path = (
    PREDICTIONS_DIR
    / (
        f"{XAI_REFERENCE_CONFIGURATION}"
        "_test_predictions.csv"
    )
)

reference_predictions_df = pd.read_csv(
    reference_prediction_path,
    low_memory=False
)

# Verify the reference predictions
if len(reference_predictions_df) != 2841:

    raise RuntimeError(
        "The reference prediction file "
        "does not contain 2,841 images."
    )

# Relevant classes: challenging classes, Healthy examples, and Rotten examples
# Selection is based on confidence/predictions, NOT on explanations.
TARGET_CLASSES = [
    "Bellpepper__Rotten",
    "Potato__Rotten",
    "Cucumber__Rotten",
    "Carrot__Rotten",
    "Tomato__Rotten",
    "Bellpepper__Healthy",
    "Potato__Healthy",
    "Pomegranate__Healthy"
]

# Select one correctly classified example per class
correct_cases = []

for class_name in TARGET_CLASSES:

    candidates = (
        reference_predictions_df[
            (
                reference_predictions_df[
                    "true_class"
                ] == class_name
            )
            &
            (
                reference_predictions_df[
                    "correct"
                ] == True
            )
        ]
        .copy()
    )

    if len(candidates) == 0:

        print(
            "No correctly classified case:",
            class_name
        )

        continue

    # Choose a moderately high-confidence correct case rather than the extreme maximum.
    candidates = (
        candidates
        .sort_values(
            "confidence",
            ascending=False
        )
        .reset_index(drop=True)
    )

    selected_position = min(
        len(candidates) // 3,
        len(candidates) - 1
    )

    selected = (
        candidates
        .iloc[
            selected_position
        ]
        .copy()
    )

    selected["xai_case_type"] = (
        "correct"
    )

    correct_cases.append(
        selected
    )

# Select errors made by the reference model
error_candidates = (
    reference_predictions_df[
        reference_predictions_df[
            "correct"
        ] == False
    ]
    .copy()
)

# Prioritize high-confidence errors because they are particularly informative for XAI.
error_candidates = (
    error_candidates
    .sort_values(
        "confidence",
        ascending=False
    )
    .reset_index(drop=True)
)

NUMBER_OF_ERROR_CASES = min(
    8,
    len(error_candidates)
)

selected_errors = (
    error_candidates
    .head(
        NUMBER_OF_ERROR_CASES
    )
    .copy()
)

selected_errors[
    "xai_case_type"
] = (
    "error"
)

# Consolidate selected cases
correct_cases_df = pd.DataFrame(
    correct_cases
)

xai_cases_df = pd.concat(
    [
        correct_cases_df,
        selected_errors
    ],
    ignore_index=True
)

# Remove any repeated dataset_id values
xai_cases_df = (
    xai_cases_df
    .drop_duplicates(
        subset=[
            "dataset_id"
        ]
    )
    .reset_index(drop=True)
)

# Assign fixed identifiers
xai_cases_df.insert(
    0,
    "xai_case_id",
    [
        f"XAI_{index:02d}"
        for index in range(
            1,
            len(xai_cases_df) + 1
        )
    ]
)

# Verify that image files exist
xai_cases_df[
    "image_exists"
] = (
    xai_cases_df[
        "input_image_path"
    ]
    .apply(
        lambda path:
            Path(path).is_file()
    )
)

if not xai_cases_df[
    "image_exists"
].all():

    raise FileNotFoundError(
        "At least one XAI image file does not exist."
    )

# Save the frozen selection
XAI_CASES_OUTPUT = (
    XAI_DIR
    / "35_fixed_xai_test_cases.csv"
)

xai_cases_df.to_csv(
    XAI_CASES_OUTPUT,
    index=False
)

# Summary
print("=" * 80)
print("XAI CASES FROZEN")
print("=" * 80)

print(
    "Model used only for case selection:",
    XAI_REFERENCE_CONFIGURATION
)

print(
    "Total cases:",
    len(xai_cases_df)
)

print(
    "Correct cases:",
    int(
        (
            xai_cases_df[
                "xai_case_type"
            ]
            ==
            "correct"
        ).sum()
    )
)

print(
    "Error cases:",
    int(
        (
            xai_cases_df[
                "xai_case_type"
            ]
            ==
            "error"
        ).sum()
    )
)

display(
    xai_cases_df[
        [
            "xai_case_id",
            "xai_case_type",
            "dataset_id",
            "true_class",
            "predicted_class",
            "confidence"
        ]
    ]
)

print(
    "\nThe four configurations that will use "
    "exactly the same images:"
)

for configuration_id in (
    XAI_SELECTED_CONFIGURATIONS
):
    print(
        " -",
        configuration_id
    )

print(
    "\nSelection saved to:"
)

print(
    XAI_CASES_OUTPUT
)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# Load reference-model predictions
reference_predictions_df = pd.read_csv(
    PREDICTIONS_DIR
    / (
        f"{XAI_REFERENCE_CONFIGURATION}"
        "_test_predictions.csv"
    ),
    low_memory=False
)


# Add Healthy / Rotten status
def get_quality_state(
    class_name
):

    if str(
        class_name
    ).endswith(
        "__Healthy"
    ):

        return "Healthy"

    if str(
        class_name
    ).endswith(
        "__Rotten"
    ):

        return "Rotten"

    return "Unknown"


reference_predictions_df[
    "quality_state"
] = (
    reference_predictions_df[
        "true_class"
    ]
    .apply(
        get_quality_state
    )
)


# Verify availability of all four groups
availability_df = (

    reference_predictions_df
    .groupby(
        [
            "correct",
            "quality_state"
        ]
    )
    .size()
    .reset_index(
        name="available_cases"
    )
)


print("=" * 80)
print("XAI CASE AVAILABILITY")
print("=" * 80)

display(
    availability_df
)


required_groups = [
    (True, "Healthy"),
    (True, "Rotten"),
    (False, "Healthy"),
    (False, "Rotten")
]


for correct_status, quality_state in (
    required_groups
):

    number_available = len(

        reference_predictions_df[
            (
                reference_predictions_df[
                    "correct"
                ]
                ==
                correct_status
            )
            &
            (
                reference_predictions_df[
                    "quality_state"
                ]
                ==
                quality_state
            )
        ]
    )


    if number_available == 0:

        raise RuntimeError(
            f"No cases are available for: "
            f"correct={correct_status}, "
            f"quality={quality_state}"
        )


# Case selection: use intermediate-high confidence for correct cases,
# rather than simply choosing the maximum.
# For errors, select the highest-confidence error, which is informative for XAI.
def select_primary_case(
    dataframe,
    correct_status,
    quality_state
):

    candidates = (

        dataframe[
            (
                dataframe[
                    "correct"
                ]
                ==
                correct_status
            )
            &
            (
                dataframe[
                    "quality_state"
                ]
                ==
                quality_state
            )
        ]
        .copy()
        .sort_values(
            "confidence",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    if correct_status:

        # Approximately the upper-middle percentile
        selected_position = min(
            len(candidates) // 3,
            len(candidates) - 1
        )

    else:

        # Highest-confidence error
        selected_position = 0


    return (
        candidates
        .iloc[
            selected_position
        ]
        .copy()
    )


# Select four cases
primary_cases = []


selection_specification = [

    (
        "XAI_PRIMARY_01",
        True,
        "Healthy",
        "Correct Healthy"
    ),

    (
        "XAI_PRIMARY_02",
        True,
        "Rotten",
        "Correct Rotten"
    ),

    (
        "XAI_PRIMARY_03",
        False,
        "Healthy",
        "Error Healthy"
    ),

    (
        "XAI_PRIMARY_04",
        False,
        "Rotten",
        "Error Rotten"
    )
]


for (
    xai_case_id,
    correct_status,
    quality_state,
    case_description

) in selection_specification:

    selected = select_primary_case(

        reference_predictions_df,

        correct_status=
            correct_status,

        quality_state=
            quality_state
    )


    selected[
        "xai_case_id"
    ] = xai_case_id

    selected[
        "xai_case_type"
    ] = case_description


    primary_cases.append(
        selected
    )


primary_xai_cases_df = pd.DataFrame(
    primary_cases
)


# Arrange primary columns
preferred_columns = [

    "xai_case_id",

    "xai_case_type",

    "dataset_id",

    "quality_state",

    "true_class",

    "predicted_class",

    "correct",

    "confidence",

    "input_image_path",

    "sha256"
]


existing_columns = [

    column

    for column in preferred_columns

    if column
    in primary_xai_cases_df.columns
]


primary_xai_cases_df = (
    primary_xai_cases_df[
        existing_columns
    ]
    .copy()
)


# Strict checks
if len(
    primary_xai_cases_df
) != 4:

    raise RuntimeError(
        "The primary selection must contain four cases."
    )


if (
    primary_xai_cases_df[
        "dataset_id"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate primary XAI cases were found."
    )


required_case_types = {

    "Correct Healthy",
    "Correct Rotten",
    "Error Healthy",
    "Error Rotten"
}


if set(
    primary_xai_cases_df[
        "xai_case_type"
    ]
) != required_case_types:

    raise RuntimeError(
        "The four required categories are not all represented."
    )


# Save frozen primary selection
PRIMARY_XAI_OUTPUT = (

    XAI_DIR
    / "36_primary_xai_cases_for_paper.csv"
)


primary_xai_cases_df.to_csv(

    PRIMARY_XAI_OUTPUT,

    index=False
)


# Display summary
print("\n" + "=" * 80)
print("PRIMARY XAI CASES FOR THE PAPER")
print("=" * 80)


display(

    primary_xai_cases_df[
        [
            "xai_case_id",
            "xai_case_type",
            "dataset_id",
            "true_class",
            "predicted_class",
            "confidence"
        ]
    ]
)


print(
    "\nThe four architectures will use "
    "exactly the same images:"
)


for configuration_id in (
    XAI_SELECTED_CONFIGURATIONS
):

    print(
        " -",
        configuration_id
    )


print(
    "\nPrimary selection saved to:"
)

print(
    PRIMARY_XAI_OUTPUT
)


print(
    "\nThe previous set of 16 cases remains "
    "available as a supplementary XAI pool."
)


In [ ]:
def load_model_from_checkpoint(
    configuration_id
):
    model_family = identify_model_family(
        configuration_id
    )

    timm_model_name = TIMM_MODEL_MAP[
        model_family
    ]

    run_directory = locate_run_directory(
        configuration_id
    )

    best_checkpoint_path = locate_best_checkpoint(
        run_directory=run_directory,
        configuration_id=configuration_id
    )

    # These are our own trusted checkpoints.
    # PyTorch >= 2.6 defaults to weights_only=True,
    # so False is explicitly specified here.
    checkpoint_object = torch.load(
        best_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    model_state_dict = extract_model_state_dict(
        checkpoint_object
    )

    model = timm.create_model(
        timm_model_name,
        pretrained=False,
        num_classes=len(CLASS_NAMES)
    )

    load_result = model.load_state_dict(
        model_state_dict,
        strict=False
    )

    model = model.to(
        DEVICE
    )

    model.eval()

    return {
        "model": model,
        "run_directory": run_directory,
        "best_checkpoint_path": best_checkpoint_path,
        "missing_keys": load_result.missing_keys,
        "unexpected_keys": load_result.unexpected_keys
    }


print("=" * 80)
print("CHECKPOINT LOADING CORRECTION APPLIED")
print("=" * 80)

print("torch version:", torch.__version__)
print("torch.load configured with weights_only=False")


# **Cell 15 — Generation of XAI maps and comparative figure**

In [ ]:
from pathlib import Path
import gc
import math
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from PIL import (
    Image,
    ImageOps
)

import torch
import torch.nn as nn
import torch.nn.functional as F

import timm

from torchvision import transforms
from torchvision.transforms import InterpolationMode

from IPython.display import display


# General configuration

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    DEVICE
)


# Configurations selected previously using validation ONLY

XAI_CONFIGURATIONS = [

    "swin_tiny__cutmix",

    "convnext_tiny__none",

    "mobilenet_v2__cutmix",

    "vit_b16__cutmix"
]


XAI_DISPLAY_NAMES = {

    "swin_tiny__cutmix":
        "Swin-Tiny + CutMix",

    "convnext_tiny__none":
        "ConvNeXt-Tiny + None",

    "mobilenet_v2__cutmix":
        "MobileNetV2 + CutMix",

    "vit_b16__cutmix":
        "ViT-B/16 + CutMix"
}


# EXACT TIMM models used during training

TIMM_MODEL_MAP = {

    "swin_tiny":
        "swin_tiny_patch4_window7_224.ms_in1k",

    "convnext_tiny":
        "convnext_tiny.fb_in1k",

    "mobilenet_v2":
        "mobilenetv2_100.ra_in1k",

    "vit_b16":
        "vit_base_patch16_224.augreg_in1k"
}


# Check classes already defined in the notebook

if "CLASS_NAMES" not in globals():

    if "CLASS_TO_INDEX" in globals():

        CLASS_NAMES = [

            class_name

            for class_name, class_index
            in sorted(
                CLASS_TO_INDEX.items(),
                key=lambda item: item[1]
            )
        ]

    else:

        raise RuntimeError(
            "CLASS_NAMES / CLASS_TO_INDEX "
            "are not defined in the notebook."
        )


if "CLASS_TO_INDEX" not in globals():

    CLASS_TO_INDEX = {

        class_name: index

        for index, class_name
        in enumerate(
            CLASS_NAMES
        )
    }


INDEX_TO_CLASS = {

    class_index: class_name

    for class_name, class_index
    in CLASS_TO_INDEX.items()
}


NUM_XAI_CLASSES = len(
    CLASS_NAMES
)


if NUM_XAI_CLASSES != 24:

    raise RuntimeError(
        f"Expected 24 classes but found "
        f"{NUM_XAI_CLASSES}."
    )


print(
    "Number of classes:",
    NUM_XAI_CLASSES
)


# Directories

XAI_DIR = (
    ANALYSIS_ROOT
    / "explainability"
)


XAI_MAPS_DIR = (
    XAI_DIR
    / "maps"
)


XAI_OVERLAYS_DIR = (
    XAI_DIR
    / "overlays"
)


XAI_FIGURES_DIR = (
    XAI_DIR
    / "figures"
)


for directory in [

    XAI_DIR,

    XAI_MAPS_DIR,

    XAI_OVERLAYS_DIR,

    XAI_FIGURES_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# Load the four fixed primary XAI cases

PRIMARY_XAI_CASES_PATH = (

    XAI_DIR
    / "36_primary_xai_cases_for_paper.csv"
)


if not PRIMARY_XAI_CASES_PATH.exists():

    raise FileNotFoundError(
        f"No existe:\n"
        f"{PRIMARY_XAI_CASES_PATH}"
    )


primary_xai_cases_df = pd.read_csv(

    PRIMARY_XAI_CASES_PATH,

    low_memory=False
)


# Explicit ordering for the figure

CASE_ORDER = [

    "Correct Healthy",

    "Correct Rotten",

    "Error Healthy",

    "Error Rotten"
]


primary_xai_cases_df[
    "xai_case_type"
] = pd.Categorical(

    primary_xai_cases_df[
        "xai_case_type"
    ],

    categories=CASE_ORDER,

    ordered=True
)


primary_xai_cases_df = (

    primary_xai_cases_df
    .sort_values(
        "xai_case_type"
    )
    .reset_index(
        drop=True
    )
)


if len(
    primary_xai_cases_df
) != 4:

    raise RuntimeError(
        "The primary XAI selection must "
        "contain exactly 4 images."
    )


if set(
    primary_xai_cases_df[
        "xai_case_type"
    ].astype(str)
) != set(
    CASE_ORDER
):

    raise RuntimeError(
        "The four required cases are not present: "
        "Healthy/Rotten × Correct/Error."
    )


print("\nXAI cases:")

display(

    primary_xai_cases_df[
        [
            "xai_case_id",
            "xai_case_type",
            "dataset_id",
            "true_class",
            "predicted_class",
            "confidence"
        ]
    ]
)


# Exact evaluation preprocessing
# As specified in the previously used protocol:
# Resize = round(224 × 1.15) = 258
# Bicubic
# CenterCrop = 224
# ImageNet normalization
# EXIF transpose
# Alpha composited over white

XAI_IMG_SIZE = int(
    IMG_SIZE
    if "IMG_SIZE" in globals()
    else 224
)


XAI_RESIZE_SIZE = round(
    XAI_IMG_SIZE
    * 1.15
)


IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]


IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]


xai_eval_transform = transforms.Compose([

    transforms.Resize(
        XAI_RESIZE_SIZE,
        interpolation=InterpolationMode.BICUBIC,
        antialias=True
    ),

    transforms.CenterCrop(
        XAI_IMG_SIZE
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# Robust image loading

def open_image_for_xai(
    image_path
):

    image = Image.open(
        image_path
    )


    # Preserve EXIF orientation

    image = ImageOps.exif_transpose(
        image
    )


    # Composite transparency over a white background

    if (
        image.mode in (
            "RGBA",
            "LA"
        )
        or
        "transparency" in image.info
    ):

        rgba_image = image.convert(
            "RGBA"
        )

        white_background = Image.new(
            "RGBA",
            rgba_image.size,
            (
                255,
                255,
                255,
                255
            )
        )

        image = Image.alpha_composite(
            white_background,
            rgba_image
        )

        image = image.convert(
            "RGB"
        )

    else:

        image = image.convert(
            "RGB"
        )


    return image


# Denormalize for visualization

def denormalize_tensor(
    image_tensor
):

    tensor = (
        image_tensor
        .detach()
        .cpu()
        .clone()
    )


    mean_tensor = torch.tensor(
        IMAGENET_MEAN
    ).view(
        3,
        1,
        1
    )


    std_tensor = torch.tensor(
        IMAGENET_STD
    ).view(
        3,
        1,
        1
    )


    tensor = (
        tensor
        * std_tensor
        +
        mean_tensor
    )


    tensor = torch.clamp(
        tensor,
        0.0,
        1.0
    )


    rgb_image = (

        tensor
        .permute(
            1,
            2,
            0
        )
        .numpy()
    )


    return rgb_image


# Prepare image

def prepare_xai_image(
    image_path
):

    pil_image = open_image_for_xai(
        image_path
    )


    transformed_tensor = xai_eval_transform(
        pil_image
    )


    display_image = denormalize_tensor(
        transformed_tensor
    )


    input_tensor = (

        transformed_tensor
        .unsqueeze(0)
        .to(
            DEVICE
        )
    )


    return (
        input_tensor,
        display_image
    )


# Identify model family

def identify_model_family(
    configuration_id
):

    if configuration_id.startswith(
        "convnext_tiny"
    ):

        return "convnext_tiny"


    if configuration_id.startswith(
        "mobilenet_v2"
    ):

        return "mobilenet_v2"


    if configuration_id.startswith(
        "swin_tiny"
    ):

        return "swin_tiny"


    if configuration_id.startswith(
        "vit_b16"
    ):

        return "vit_b16"


    raise ValueError(
        f"Unrecognized configuration: "
        f"{configuration_id}"
    )


# Locate checkpoint

def locate_best_checkpoint(
    configuration_id
):

    # First option:
    # use the previously verified RUN_DIRECTORIES

    if (
        "RUN_DIRECTORIES" in globals()
        and
        configuration_id in RUN_DIRECTORIES
    ):

        run_directory = Path(
            RUN_DIRECTORIES[
                configuration_id
            ]
        )


        candidates = list(
            run_directory.rglob(
                "best.pt"
            )
        )


        if len(
            candidates
        ) == 1:

            return (
                run_directory,
                candidates[0]
            )


        if len(
            candidates
        ) > 1:

            candidates = sorted(
                candidates,
                key=lambda path: len(
                    str(path)
                )
            )

            return (
                run_directory,
                candidates[0]
            )


    # Second option:
    # search under Kaggle Input

    search_roots = [

        Path(
            "/kaggle/input"
        ),

        Path(
            "/kaggle/working"
        )
    ]


    valid_candidates = []


    for root in search_roots:

        if not root.exists():

            continue


        for path in root.rglob(
            configuration_id
        ):

            if (
                path.is_dir()
                and
                path.name
                ==
                configuration_id
            ):

                checkpoint_candidates = list(
                    path.rglob(
                        "best.pt"
                    )
                )


                for checkpoint_path in checkpoint_candidates:

                    valid_candidates.append(
                        (
                            path,
                            checkpoint_path
                        )
                    )


    if len(
        valid_candidates
    ) == 0:

        raise FileNotFoundError(
            f"best.pt was not found for "
            f"{configuration_id}."
        )


    valid_candidates = sorted(

        valid_candidates,

        key=lambda item: len(
            str(
                item[1]
            )
        )
    )


    return valid_candidates[
        0
    ]


# Extract state_dict

def clean_state_dict_keys(
    state_dict
):

    cleaned = {}


    prefixes = [

        "module.",

        "model.",

        "_orig_mod."
    ]


    for key, value in state_dict.items():

        new_key = key


        prefix_removed = True


        while prefix_removed:

            prefix_removed = False


            for prefix in prefixes:

                if new_key.startswith(
                    prefix
                ):

                    new_key = new_key[
                        len(prefix):
                    ]

                    prefix_removed = True


        cleaned[
            new_key
        ] = value


    return cleaned


def extract_model_state_dict(
    checkpoint_object
):

    if not isinstance(
        checkpoint_object,
        dict
    ):

        raise RuntimeError(
            "Unexpected checkpoint format."
        )


    # Complete checkpoint

    for possible_key in [

        "model_state_dict",

        "state_dict",

        "model"
    ]:

        if (
            possible_key
            in checkpoint_object
            and
            isinstance(
                checkpoint_object[
                    possible_key
                ],
                dict
            )
        ):

            return clean_state_dict_keys(

                checkpoint_object[
                    possible_key
                ]
            )


    # Case: the file directly contains a state_dict

    if all(

        torch.is_tensor(
            value
        )

        for value
        in checkpoint_object.values()

    ):

        return clean_state_dict_keys(
            checkpoint_object
        )


    raise RuntimeError(
        "Could not identify the "
        "checkpoint state_dict."
    )


# Load model
# Important correction:
# weights_only=False is specified here.

def load_xai_model(
    configuration_id
):

    model_family = identify_model_family(
        configuration_id
    )


    timm_model_name = TIMM_MODEL_MAP[
        model_family
    ]


    (
        run_directory,
        checkpoint_path

    ) = locate_best_checkpoint(
        configuration_id
    )


    # PyTorch >= 2.6 defaults to weights_only=True.
    # These checkpoints were generated by our own
    # experiment, so we explicitly load the
    # complete checkpoint.

    checkpoint_object = torch.load(

        checkpoint_path,

        map_location="cpu",

        weights_only=False
    )


    state_dict = extract_model_state_dict(
        checkpoint_object
    )


    model = timm.create_model(

        timm_model_name,

        pretrained=False,

        num_classes=NUM_XAI_CLASSES
    )


    # Strict loading: reconstruct the exact model

    try:

        model.load_state_dict(
            state_dict,
            strict=True
        )

    except RuntimeError as error:

        print(
            "\nERROR loading state_dict for:",
            configuration_id
        )

        print(
            "Checkpoint:",
            checkpoint_path
        )

        raise error


    model = model.to(
        DEVICE
    )


    model.eval()


    return {

        "model":
            model,

        "model_family":
            model_family,

        "run_directory":
            run_directory,

        "checkpoint_path":
            checkpoint_path
    }


# Target layer selection

def get_last_conv2d_layer(
    model
):

    conv_layers = [

        module

        for module
        in model.modules()

        if isinstance(
            module,
            nn.Conv2d
        )
    ]


    if len(
        conv_layers
    ) == 0:

        raise RuntimeError(
            "No Conv2d layer was found."
        )


    return conv_layers[
        -1
    ]


def get_target_layer(
    model,
    configuration_id
):

    family = identify_model_family(
        configuration_id
    )


    # CNN: Grad-CAM on the last convolutional layer

    if family == "convnext_tiny":

        return get_last_conv2d_layer(
            model
        )


    if family == "mobilenet_v2":

        return get_last_conv2d_layer(
            model
        )


    # ViT:
    # last pre-attention normalization of the final block
    # Its output is normally:
    # [B, 197, C]

    if family == "vit_b16":

        if (
            hasattr(
                model,
                "blocks"
            )
            and
            len(
                model.blocks
            ) > 0
        ):

            last_block = model.blocks[
                -1
            ]


            if hasattr(
                last_block,
                "norm1"
            ):

                return last_block.norm1


        raise RuntimeError(
            "Target layer was not found "
            "for ViT-B/16."
        )


    # Swin:
    # last block of the final stage
    # Output may be:
    # [B,H,W,C]
    # or [B,N,C]
    # depending on the TIMM implementation.

    if family == "swin_tiny":

        if (
            hasattr(
                model,
                "layers"
            )
            and
            len(
                model.layers
            ) > 0
        ):

            last_stage = model.layers[
                -1
            ]


            if (
                hasattr(
                    last_stage,
                    "blocks"
                )
                and
                len(
                    last_stage.blocks
                ) > 0
            ):

                last_block = last_stage.blocks[
                    -1
                ]


                if hasattr(
                    last_block,
                    "norm1"
                ):

                    return last_block.norm1


        raise RuntimeError(
            "Target layer was not found "
            "for Swin-Tiny."
        )


    raise RuntimeError(
        f"No target layer is defined for "
        f"{configuration_id}."
    )


# Capture activations and gradients
# Use a forward hook and register the gradient hook
# directly on the tensor.
# This is more robust than register_full_backward_hook()
# for several Transformer modules.

def first_tensor_from_output(
    output
):

    if torch.is_tensor(
        output
    ):

        return output


    if isinstance(
        output,
        (
            tuple,
            list
        )
    ):

        for item in output:

            if torch.is_tensor(
                item
            ):

                return item


    raise RuntimeError(
        "The target layer did not return a compatible tensor."
    )


class ActivationGradientCapture:

    def __init__(
        self,
        target_layer
    ):

        self.activations = None

        self.gradients = None


        self.forward_handle = (
            target_layer
            .register_forward_hook(
                self._forward_hook
            )
        )


    def _forward_hook(
        self,
        module,
        inputs,
        output
    ):

        activation_tensor = (
            first_tensor_from_output(
                output
            )
        )


        self.activations = (
            activation_tensor
        )


        if activation_tensor.requires_grad:

            activation_tensor.register_hook(
                self._save_gradient
            )


    def _save_gradient(
        self,
        gradient
    ):

        self.gradients = gradient


    def remove(
        self
    ):

        self.forward_handle.remove()


# Convert activations to BCHW

def reshape_activation_to_bchw(
    tensor
):

    if tensor is None:

        raise RuntimeError(
            "The tensor was not captured."
        )


    # 4D:
    # CNN:
    # [B,C,H,W]
    # Swin:
    # [B,H,W,C]

    if tensor.ndim == 4:

        batch_size = tensor.shape[
            0
        ]


        dim1 = tensor.shape[
            1
        ]

        dim2 = tensor.shape[
            2
        ]

        dim3 = tensor.shape[
            3
        ]


        # BHWC:
        # C is usually the largest dimension
        if (
            dim3 > dim1
            and
            dim3 > dim2
        ):

            return (

                tensor
                .permute(
                    0,
                    3,
                    1,
                    2
                )
                .contiguous()
            )


        # assume BCHW
        return tensor


    # 3D:
    # Transformer:
    # [B,N,C]
    # ViT:
    # N=197 -> remove CLS -> 196 -> 14×14
    # Swin:
    # may be N=49 -> 7×7

    if tensor.ndim == 3:

        batch_size = tensor.shape[
            0
        ]


        dimension_1 = tensor.shape[
            1
        ]


        dimension_2 = tensor.shape[
            2
        ]


        # Detect B,C,N and convert to B,N,C

        n_candidate = dimension_1

        c_candidate = dimension_2


        square_n1 = int(
            math.sqrt(
                n_candidate
            )
        )


        square_n2 = int(
            math.sqrt(
                dimension_2
            )
        )


        n1_is_spatial = (
            square_n1
            *
            square_n1
            ==
            n_candidate
            or
            (
                n_candidate > 1
                and
                int(
                    math.sqrt(
                        n_candidate - 1
                    )
                )
                ** 2
                ==
                n_candidate - 1
            )
        )


        n2_is_spatial = (
            square_n2
            *
            square_n2
            ==
            dimension_2
            or
            (
                dimension_2 > 1
                and
                int(
                    math.sqrt(
                        dimension_2 - 1
                    )
                )
                ** 2
                ==
                dimension_2 - 1
            )
        )


        if (
            not n1_is_spatial
            and
            n2_is_spatial
        ):

            tensor = tensor.transpose(
                1,
                2
            )


        batch_size, n_tokens, channels = (
            tensor.shape
        )


        # Is N already a perfect square?

        side = int(
            math.sqrt(
                n_tokens
            )
        )


        if (
            side
            *
            side
            ==
            n_tokens
        ):

            spatial_tokens = tensor


        # Does it contain a CLS token?

        elif (

            n_tokens > 1

            and

            int(
                math.sqrt(
                    n_tokens - 1
                )
            )
            ** 2

            ==
            n_tokens - 1

        ):

            spatial_tokens = tensor[
                :,
                1:,
                :
            ]


            n_tokens = spatial_tokens.shape[
                1
            ]


            side = int(
                math.sqrt(
                    n_tokens
                )
            )


        else:

            raise RuntimeError(
                "Could not reconstruct a spatial map "
                "from activations with shape "
                f"{tuple(tensor.shape)}."
            )


        spatial_tensor = (

            spatial_tokens
            .reshape(
                batch_size,
                side,
                side,
                channels
            )
            .permute(
                0,
                3,
                1,
                2
            )
            .contiguous()
        )


        return spatial_tensor


    raise RuntimeError(
        "Unsupported activation dimensionality: "
        f"{tuple(tensor.shape)}"
    )


# Normalize CAM

def normalize_cam(
    cam_tensor
):

    cam_tensor = cam_tensor - torch.min(
        cam_tensor
    )


    maximum = torch.max(
        cam_tensor
    )


    if maximum > 1e-12:

        cam_tensor = (
            cam_tensor
            /
            maximum
        )


    return cam_tensor


# Compute Grad-CAM

def compute_gradcam(
    model,
    target_layer,
    input_tensor
):

    capture = ActivationGradientCapture(
        target_layer
    )


    try:

        model.zero_grad(
            set_to_none=True
        )


        # Gradients are required, so do NOT use
        # torch.inference_mode() or torch.no_grad()

        with torch.enable_grad():

            logits = model(
                input_tensor
            )


            probabilities = torch.softmax(
                logits,
                dim=1
            )


            predicted_index = int(

                torch.argmax(
                    probabilities,
                    dim=1
                ).item()
            )


            predicted_probability = float(

                probabilities[
                    0,
                    predicted_index
                ].detach().cpu().item()
            )


            # Explain the model-PREDICTED class

            target_score = logits[
                0,
                predicted_index
            ]


            target_score.backward()


        if capture.activations is None:

            raise RuntimeError(
                "Activations were not captured."
            )


        if capture.gradients is None:

            raise RuntimeError(
                "Gradients were not captured."
            )


        activations = reshape_activation_to_bchw(

            capture.activations
        )


        gradients = reshape_activation_to_bchw(

            capture.gradients
        )


        if (
            activations.shape
            !=
            gradients.shape
        ):

            raise RuntimeError(
                "Activations and gradients have "
                "incompatible shapes:\n"
                f"activations={tuple(activations.shape)}\n"
                f"gradients={tuple(gradients.shape)}"
            )


        # Grad-CAM weights:
        # spatial mean of gradients

        weights = gradients.mean(

            dim=(
                2,
                3
            ),

            keepdim=True
        )


        cam = (

            weights
            *
            activations

        ).sum(
            dim=1,
            keepdim=True
        )


        cam = torch.relu(
            cam
        )


        # Resize to 224×224 resolution

        cam = F.interpolate(

            cam,

            size=(
                XAI_IMG_SIZE,
                XAI_IMG_SIZE
            ),

            mode="bilinear",

            align_corners=False
        )


        cam = cam[
            0,
            0
        ]


        cam = normalize_cam(
            cam
        )


        cam_numpy = (

            cam
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )


        return {

            "cam":
                cam_numpy,

            "predicted_index":
                predicted_index,

            "predicted_class":
                INDEX_TO_CLASS[
                    predicted_index
                ],

            "predicted_probability":
                predicted_probability,

            "activation_shape":
                tuple(
                    capture.activations.shape
                ),

            "gradient_shape":
                tuple(
                    capture.gradients.shape
                )
        }


    finally:

        capture.remove()


# Create overlay

def create_xai_overlay(
    rgb_image,
    cam,
    alpha=0.42
):

    heatmap = plt.get_cmap(
        "jet"
    )(
        cam
    )[
        ...,
        :3
    ]


    overlay = (

        (
            1.0
            -
            alpha
        )
        *
        rgb_image

        +

        alpha
        *
        heatmap
    )


    overlay = np.clip(
        overlay,
        0.0,
        1.0
    )


    return (
        heatmap,
        overlay
    )


# Load previously saved predictions

def load_saved_predictions(
    configuration_id
):

    prediction_path = (

        PREDICTIONS_DIR
        /
        (
            f"{configuration_id}"
            "_test_predictions.csv"
        )
    )


    if not prediction_path.exists():

        raise FileNotFoundError(
            f"Prediction file does not exist:\n"
            f"{prediction_path}"
        )


    prediction_df = pd.read_csv(

        prediction_path,

        low_memory=False
    )


    return prediction_df


# Find image path

def get_case_image_path(
    case_row
):

    candidate_columns = [

        "input_image_path",

        "image_path",

        "file_path",

        "filepath",

        "path"
    ]


    for column_name in candidate_columns:

        if (
            column_name
            in case_row.index
            and
            pd.notna(
                case_row[
                    column_name
                ]
            )
        ):

            candidate_path = Path(
                str(
                    case_row[
                        column_name
                    ]
                )
            )


            if candidate_path.exists():

                return candidate_path


    # If the path was not saved in 36_..., recover it
    # from the Swin predictions CSV

    reference_prediction_path = (

        PREDICTIONS_DIR
        /
        (
            "swin_tiny__cutmix"
            "_test_predictions.csv"
        )
    )


    reference_df = pd.read_csv(

        reference_prediction_path,

        low_memory=False
    )


    dataset_id_value = str(
        case_row[
            "dataset_id"
        ]
    )


    matching = reference_df[
        reference_df[
            "dataset_id"
        ].astype(str)
        ==
        dataset_id_value
    ]


    if len(
        matching
    ) != 1:

        raise RuntimeError(
            "Could not recover a unique path "
            f"for dataset_id={dataset_id_value}."
        )


    for column_name in candidate_columns:

        if column_name in matching.columns:

            recovered_path = Path(
                str(
                    matching.iloc[
                        0
                    ][
                        column_name
                    ]
                )
            )


            if recovered_path.exists():

                return recovered_path


    raise FileNotFoundError(
        "Could not find the physical file for "
        f"dataset_id={dataset_id_value}."
    )


# Generate the 16 maps

xai_records = []


print("\n" + "=" * 80)

print(
    "GENERATING XAI MAPS"
)

print(
    "=" * 80
)


for model_number, configuration_id in enumerate(

    XAI_CONFIGURATIONS,

    start=1

):

    print(
        f"\n[{model_number}/"
        f"{len(XAI_CONFIGURATIONS)}] "
        f"{configuration_id}"
    )


    # Clear memory before loading each architecture

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    # Model

    model_bundle = load_xai_model(
        configuration_id
    )


    model = model_bundle[
        "model"
    ]


    target_layer = get_target_layer(

        model=model,

        configuration_id=configuration_id
    )


    checkpoint_path = model_bundle[
        "checkpoint_path"
    ]


    print(
        "Checkpoint:",
        checkpoint_path
    )


    print(
        "Target layer:",
        target_layer.__class__.__name__
    )


    # Predictions already saved during evaluation

    saved_predictions_df = (
        load_saved_predictions(
            configuration_id
        )
    )


    # Process four images

    for case_number, case_row in (

        primary_xai_cases_df
        .iterrows()

    ):

        case_id = str(
            case_row[
                "xai_case_id"
            ]
        )


        dataset_id = str(
            case_row[
                "dataset_id"
            ]
        )


        case_type = str(
            case_row[
                "xai_case_type"
            ]
        )


        true_class = str(
            case_row[
                "true_class"
            ]
        )


        image_path = get_case_image_path(
            case_row
        )


        (
            input_tensor,
            display_image

        ) = prepare_xai_image(
            image_path
        )


        # Grad-CAM

        gradcam_result = compute_gradcam(

            model=model,

            target_layer=target_layer,

            input_tensor=input_tensor
        )


        predicted_class = gradcam_result[
            "predicted_class"
        ]


        predicted_probability = (
            gradcam_result[
                "predicted_probability"
            ]
        )


        cam = gradcam_result[
            "cam"
        ]


        # Verify against previously saved prediction

        matching_saved = (

            saved_predictions_df[
                saved_predictions_df[
                    "dataset_id"
                ].astype(str)
                ==
                dataset_id
            ]
        )


        if len(
            matching_saved
        ) != 1:

            raise RuntimeError(
                f"{configuration_id}: expected one "
                f"unique saved prediction for "
                f"dataset_id={dataset_id}, "
                f"but found "
                f"{len(matching_saved)}."
            )


        saved_predicted_class = str(

            matching_saved.iloc[
                0
            ][
                "predicted_class"
            ]
        )


        prediction_matches_saved = (

            predicted_class
            ==
            saved_predicted_class
        )


        if not prediction_matches_saved:

            raise RuntimeError(

                "\nThe XAI prediction does not match "
                "the previously saved evaluation.\n"

                f"Configuration: "
                f"{configuration_id}\n"

                f"dataset_id: "
                f"{dataset_id}\n"

                f"XAI prediction: "
                f"{predicted_class}\n"

                f"Saved prediction: "
                f"{saved_predicted_class}\n\n"

                "This may indicate a difference in "
                "preprocessing or model reconstruction."
            )


        # Overlay

        heatmap_rgb, overlay = (
            create_xai_overlay(

                rgb_image=display_image,

                cam=cam,

                alpha=0.42
            )
        )


        output_stem = (

            f"{configuration_id}"
            f"__{case_id}"
        )


        # Save numerical map

        cam_npy_path = (

            XAI_MAPS_DIR
            /
            (
                f"{output_stem}"
                "__cam.npy"
            )
        )


        np.save(

            cam_npy_path,

            cam
        )


        # Save heatmap PNG

        heatmap_png_path = (

            XAI_MAPS_DIR
            /
            (
                f"{output_stem}"
                "__heatmap.png"
            )
        )


        plt.imsave(

            heatmap_png_path,

            cam,

            cmap="jet",

            vmin=0,

            vmax=1
        )


        # Save overlay PNG

        overlay_path = (

            XAI_OVERLAYS_DIR
            /
            (
                f"{output_stem}"
                "__overlay.png"
            )
        )


        plt.imsave(

            overlay_path,

            overlay
        )


        # Save record

        xai_records.append({

            "configuration_id":
                configuration_id,

            "architecture_display":
                XAI_DISPLAY_NAMES[
                    configuration_id
                ],

            "xai_case_id":
                case_id,

            "xai_case_type":
                case_type,

            "dataset_id":
                dataset_id,

            "true_class":
                true_class,

            "model_predicted_class":
                predicted_class,

            "correct_for_current_model":
                (
                    predicted_class
                    ==
                    true_class
                ),

            "model_predicted_probability":
                predicted_probability,

            "saved_predicted_class":
                saved_predicted_class,

            "prediction_matches_saved":
                prediction_matches_saved,

            "explained_class":
                predicted_class,

            "xai_method":
                (
                    "Grad-CAM"
                    if identify_model_family(
                        configuration_id
                    )
                    in {
                        "convnext_tiny",
                        "mobilenet_v2"
                    }
                    else
                    "Gradient-weighted token activation map"
                ),

            "activation_shape":
                str(
                    gradcam_result[
                        "activation_shape"
                    ]
                ),

            "gradient_shape":
                str(
                    gradcam_result[
                        "gradient_shape"
                    ]
                ),

            "input_image_path":
                str(
                    image_path
                ),

            "cam_npy_path":
                str(
                    cam_npy_path
                ),

            "heatmap_png_path":
                str(
                    heatmap_png_path
                ),

            "overlay_path":
                str(
                    overlay_path
                ),

            "checkpoint_path":
                str(
                    checkpoint_path
                )
        })


        print(
            f"   {case_id:<15} "
            f"{case_type:<17} "
            f"GT={true_class:<28} "
            f"Pred={predicted_class:<28} "
            f"p={predicted_probability:.4f}"
        )


        del input_tensor


    # Release model before moving to the next one

    del model

    del model_bundle

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# Consolidated DataFrame

xai_results_df = pd.DataFrame(
    xai_records
)


expected_records = (

    len(
        XAI_CONFIGURATIONS
    )

    *

    len(
        primary_xai_cases_df
    )
)


if len(
    xai_results_df
) != expected_records:

    raise RuntimeError(
        f"Expected {expected_records} "
        f"XAI records but obtained "
        f"{len(xai_results_df)}."
    )


if not xai_results_df[
    "prediction_matches_saved"
].all():

    raise RuntimeError(
        "At least one prediction does not match "
        "the previous evaluation."
    )


# Save summary of the 16 maps

XAI_SUMMARY_OUTPUT = (

    XAI_DIR
    /
    "38_xai_summary_records.csv"
)


xai_results_df.to_csv(

    XAI_SUMMARY_OUTPUT,

    index=False
)


# Individual figures per architecture
# 2 rows:
# - original images
# - overlays

def pretty_class_name(
    class_name
):

    return str(
        class_name
    ).replace(
        "__",
        " "
    )


for configuration_id in XAI_CONFIGURATIONS:

    model_records = (

        xai_results_df[
            xai_results_df[
                "configuration_id"
            ]
            ==
            configuration_id
        ]
        .copy()
    )


    fig, axes = plt.subplots(

        2,

        4,

        figsize=(
            16,
            7.5
        )
    )


    for column_index, case_type in enumerate(
        CASE_ORDER
    ):

        case_row = (

            primary_xai_cases_df[
                primary_xai_cases_df[
                    "xai_case_type"
                ].astype(str)
                ==
                case_type
            ]
            .iloc[
                0
            ]
        )


        result_row = (

            model_records[
                model_records[
                    "xai_case_type"
                ]
                ==
                case_type
            ]
            .iloc[
                0
            ]
        )


        image_path = Path(
            result_row[
                "input_image_path"
            ]
        )


        (
            _,
            original_image

        ) = prepare_xai_image(
            image_path
        )


        overlay_image = plt.imread(

            result_row[
                "overlay_path"
            ]
        )


        # Original

        axes[
            0,
            column_index
        ].imshow(
            original_image
        )


        axes[
            0,
            column_index
        ].axis(
            "off"
        )


        axes[
            0,
            column_index
        ].set_title(

            (
                f"{case_type}\n"
                f"GT: "
                f"{pretty_class_name(result_row['true_class'])}"
            ),

            fontsize=10
        )


        # Overlay

        axes[
            1,
            column_index
        ].imshow(
            overlay_image
        )


        axes[
            1,
            column_index
        ].axis(
            "off"
        )


        axes[
            1,
            column_index
        ].set_title(

            (
                "Pred: "
                f"{pretty_class_name(result_row['model_predicted_class'])}\n"
                f"p = "
                f"{result_row['model_predicted_probability']:.3f}"
            ),

            fontsize=9
        )


    fig.suptitle(

        (
            f"{XAI_DISPLAY_NAMES[configuration_id]} "
            "— Explainability analysis"
        ),

        fontsize=14
    )


    fig.tight_layout(
        rect=[
            0,
            0,
            1,
            0.95
        ]
    )


    architecture_figure_path = (

        XAI_FIGURES_DIR
        /
        (
            f"{configuration_id}"
            "__xai_four_cases.png"
        )
    )


    fig.savefig(

        architecture_figure_path,

        dpi=300,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# Main comparison figure for the paper
# Columns:
# Correct Healthy | Correct Rotten |
# Error Healthy   | Error Rotten
# Rows:
# Original
# Swin
# ConvNeXt
# MobileNet
# ViT

number_of_rows = (
    1
    +
    len(
        XAI_CONFIGURATIONS
    )
)


number_of_columns = len(
    CASE_ORDER
)


fig, axes = plt.subplots(

    number_of_rows,

    number_of_columns,

    figsize=(
        17,
        17
    )
)


# Row 0: original images

for column_index, case_type in enumerate(
    CASE_ORDER
):

    case_row = (

        primary_xai_cases_df[
            primary_xai_cases_df[
                "xai_case_type"
            ].astype(str)
            ==
            case_type
        ]
        .iloc[
            0
        ]
    )


    image_path = get_case_image_path(
        case_row
    )


    (
        _,
        original_image

    ) = prepare_xai_image(
        image_path
    )


    axes[
        0,
        column_index
    ].imshow(
        original_image
    )


    axes[
        0,
        column_index
    ].axis(
        "off"
    )


    axes[
        0,
        column_index
    ].set_title(

        (
            f"{case_type}\n"
            "GT: "
            f"{pretty_class_name(case_row['true_class'])}"
        ),

        fontsize=11
    )


axes[
    0,
    0
].set_ylabel(

    "Original",

    fontsize=12
)


# Rows 1–4: models

for row_index, configuration_id in enumerate(

    XAI_CONFIGURATIONS,

    start=1

):

    model_records = (

        xai_results_df[
            xai_results_df[
                "configuration_id"
            ]
            ==
            configuration_id
        ]
    )


    for column_index, case_type in enumerate(
        CASE_ORDER
    ):

        result_row = (

            model_records[
                model_records[
                    "xai_case_type"
                ]
                ==
                case_type
            ]
            .iloc[
                0
            ]
        )


        overlay_image = plt.imread(

            result_row[
                "overlay_path"
            ]
        )


        axes[
            row_index,
            column_index
        ].imshow(
            overlay_image
        )


        axes[
            row_index,
            column_index
        ].axis(
            "off"
        )


        axes[
            row_index,
            column_index
        ].set_title(

            (
                "Pred: "
                f"{pretty_class_name(result_row['model_predicted_class'])}\n"
                f"p = "
                f"{result_row['model_predicted_probability']:.3f}"
            ),

            fontsize=9
        )


    axes[
        row_index,
        0
    ].set_ylabel(

        XAI_DISPLAY_NAMES[
            configuration_id
        ],

        fontsize=11
    )


fig.suptitle(

    (
        "Explainability maps across correct and erroneous "
        "Healthy/Rotten predictions"
    ),

    fontsize=15
)


fig.tight_layout(

    rect=[
        0,
        0,
        1,
        0.965
    ]
)


PAPER_XAI_FIGURE_PATH = (

    XAI_FIGURES_DIR
    /
    "37_paper_xai_comparison_grid.png"
)


fig.savefig(

    PAPER_XAI_FIGURE_PATH,

    dpi=400,

    bbox_inches="tight"
)


plt.close(
    fig
)


# Summary table for inspection

xai_display_df = (

    xai_results_df[
        [
            "configuration_id",

            "xai_case_id",

            "xai_case_type",

            "true_class",

            "model_predicted_class",

            "correct_for_current_model",

            "model_predicted_probability",

            "xai_method",

            "activation_shape"
        ]
    ]
    .copy()
)


# Final checks

required_case_types = set(
    CASE_ORDER
)


observed_case_types = set(

    xai_results_df[
        "xai_case_type"
    ]
)


if observed_case_types != required_case_types:

    raise RuntimeError(
        "The four XAI case types "
        "were not all represented."
    )


for configuration_id in XAI_CONFIGURATIONS:

    number_of_maps = len(

        xai_results_df[
            xai_results_df[
                "configuration_id"
            ]
            ==
            configuration_id
        ]
    )


    if number_of_maps != 4:

        raise RuntimeError(
            f"{configuration_id}: "
            f"expected 4 maps but found "
            f"{number_of_maps}."
        )


if not PAPER_XAI_FIGURE_PATH.exists():

    raise FileNotFoundError(
        "The main XAI figure was not generated."
    )


# Results

print(
    "\n"
    +
    "=" * 80
)


print(
    "XAI MAPS COMPLETED"
)


print(
    "=" * 80
)


display(
    xai_display_df
)


print(
    "\nTotal maps:",
    len(
        xai_results_df
    )
)


print(
    "Predictions verified against previous evaluation:",
    bool(
        xai_results_df[
            "prediction_matches_saved"
        ].all()
    )
)


print(
    "\nMain comparison figure:"
)

print(
    PAPER_XAI_FIGURE_PATH
)


print(
    "\nFigures per architecture:"
)

print(
    XAI_FIGURES_DIR
)


print(
    "\nIndividual overlays:"
)

print(
    XAI_OVERLAYS_DIR
)


print(
    "\nNumerical maps and heatmaps:"
)

print(
    XAI_MAPS_DIR
)


print(
    "\nXAI summary:"
)

print(
    XAI_SUMMARY_OUTPUT
)


# Display only the main figure

print(
    "\nMain figure preview:"
)


display(
    Image.open(
        PAPER_XAI_FIGURE_PATH
    )
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


xai_audit_records = []


print("=" * 80)
print("NUMERICAL AUDIT OF XAI MAPS")
print("=" * 80)


for _, row in xai_results_df.iterrows():

    cam_path = Path(
        row["cam_npy_path"]
    )


    if not cam_path.exists():

        raise FileNotFoundError(
            f"File not found:\n{cam_path}"
        )


    cam = np.load(
        cam_path
    )


    finite = bool(
        np.isfinite(cam).all()
    )


    cam_min = float(
        np.min(cam)
    )


    cam_max = float(
        np.max(cam)
    )


    cam_mean = float(
        np.mean(cam)
    )


    cam_std = float(
        np.std(cam)
    )


    nonzero_fraction = float(
        np.mean(
            cam > 1e-8
        )
    )


    high_activation_fraction = float(
        np.mean(
            cam >= 0.5
        )
    )


    valid_map = bool(

        finite

        and

        cam_max > 0.0

        and

        cam_std > 1e-6

    )


    xai_audit_records.append({

        "configuration_id":
            row["configuration_id"],

        "xai_case_id":
            row["xai_case_id"],

        "xai_case_type":
            row["xai_case_type"],

        "cam_min":
            cam_min,

        "cam_max":
            cam_max,

        "cam_mean":
            cam_mean,

        "cam_std":
            cam_std,

        "nonzero_fraction":
            nonzero_fraction,

        "high_activation_fraction":
            high_activation_fraction,

        "finite":
            finite,

        "valid_map":
            valid_map
    })


xai_map_audit_df = pd.DataFrame(
    xai_audit_records
)


# Save audit results
XAI_AUDIT_OUTPUT = (
    XAI_DIR
    / "39_xai_map_numerical_audit.csv"
)


xai_map_audit_df.to_csv(
    XAI_AUDIT_OUTPUT,
    index=False
)


# Display audit results
display(
    xai_map_audit_df
)


print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)


print(
    "Maps audited:",
    len(xai_map_audit_df)
)


print(
    "Valid maps:",
    int(
        xai_map_audit_df[
            "valid_map"
        ].sum()
    )
)


print(
    "Problematic maps:",
    int(
        (
            ~xai_map_audit_df[
                "valid_map"
            ]
        ).sum()
    )
)


if not xai_map_audit_df[
    "valid_map"
].all():

    print(
        "\nWARNING: Some maps are degenerate."
    )

    display(

        xai_map_audit_df[
            ~xai_map_audit_df[
                "valid_map"
            ]
        ]

    )

else:

    print(
        "\nAll maps contain "
        "non-degenerate spatial signal."
    )


print(
    "\nAudit saved to:"
)

print(
    XAI_AUDIT_OUTPUT
)

# **Cell 16 — Refined XAI and new figure for the paper**

In [ ]:
from pathlib import Path
import gc
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from PIL import Image
from IPython.display import display


# Verify that Cell 16 outputs are available

required_objects = [

    "load_xai_model",
    "prepare_xai_image",
    "get_case_image_path",
    "compute_gradcam",
    "primary_xai_cases_df",
    "XAI_CONFIGURATIONS",
    "XAI_DISPLAY_NAMES",
    "XAI_DIR",
    "xai_results_df"
]


missing_objects = [

    object_name

    for object_name in required_objects

    if object_name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Missing objects generated by Cell 16:\n"
        f"{missing_objects}\n\n"
        "Run the complete Cell 16 first."
    )


# Refined output directories

REFINED_XAI_DIR = (
    XAI_DIR
    / "refined"
)


REFINED_MAPS_DIR = (
    REFINED_XAI_DIR
    / "maps"
)


REFINED_OVERLAYS_DIR = (
    REFINED_XAI_DIR
    / "overlays"
)


REFINED_FIGURES_DIR = (
    REFINED_XAI_DIR
    / "figures"
)


for directory in [

    REFINED_XAI_DIR,
    REFINED_MAPS_DIR,
    REFINED_OVERLAYS_DIR,
    REFINED_FIGURES_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# Fixed case ordering

CASE_ORDER = [

    "Correct Healthy",
    "Correct Rotten",
    "Error Healthy",
    "Error Rotten"
]


primary_xai_cases_df = (

    primary_xai_cases_df
    .copy()
)


primary_xai_cases_df[
    "xai_case_type"
] = (

    primary_xai_cases_df[
        "xai_case_type"
    ]
    .astype(str)
)


primary_xai_cases_df[
    "_case_order"
] = (

    primary_xai_cases_df[
        "xai_case_type"
    ]
    .map(
        {
            case_type: index
            for index, case_type
            in enumerate(CASE_ORDER)
        }
    )
)


primary_xai_cases_df = (

    primary_xai_cases_df
    .sort_values(
        "_case_order"
    )
    .drop(
        columns=[
            "_case_order"
        ]
    )
    .reset_index(
        drop=True
    )
)


# Map normalization

def normalize_refined_map(
    heatmap
):

    heatmap = np.asarray(
        heatmap,
        dtype=np.float32
    )


    heatmap = np.nan_to_num(
        heatmap,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )


    minimum = float(
        heatmap.min()
    )


    maximum = float(
        heatmap.max()
    )


    if (
        maximum
        -
        minimum
    ) < 1e-12:

        return np.zeros_like(
            heatmap
        )


    return (

        heatmap
        -
        minimum

    ) / (

        maximum
        -
        minimum
    )


# Publication-ready overlays
# "inferno" is more perceptually uniform than "jet".

def create_refined_overlay(
    original_image,
    heatmap,
    alpha=0.42
):

    heatmap = normalize_refined_map(
        heatmap
    )


    heatmap_rgb = (

        plt.get_cmap(
            "inferno"
        )(
            heatmap
        )[
            ...,
            :3
        ]
    )


    overlay = (

        (
            1.0
            -
            alpha
        )
        *
        original_image

        +

        alpha
        *
        heatmap_rgb
    )


    overlay = np.clip(
        overlay,
        0.0,
        1.0
    )


    return (
        heatmap_rgb,
        overlay
    )


# Retrieve existing CNN activation maps

def get_existing_cam(
    configuration_id,
    xai_case_id
):

    matching = (

        xai_results_df[
            (
                xai_results_df[
                    "configuration_id"
                ]
                ==
                configuration_id
            )
            &
            (
                xai_results_df[
                    "xai_case_id"
                ].astype(str)
                ==
                str(
                    xai_case_id
                )
            )
        ]
    )


    if len(
        matching
    ) != 1:

        raise RuntimeError(
            "Could not find exactly one existing map for:\n"
            f"{configuration_id}\n"
            f"{xai_case_id}"
        )


    cam_path = Path(

        matching.iloc[
            0
        ][
            "cam_npy_path"
        ]
    )


    if not cam_path.exists():

        raise FileNotFoundError(
            f"No existe:\n{cam_path}"
        )


    return np.load(
        cam_path
    )


# Swin-Tiny target layer
# Select the penultimate stage
# Use the penultimate stage instead of the final 7×7 stage:
# layers[-2].blocks[-1].norm1
# Expected approximate activation shape:
# B × 14 × 14 × C

def get_refined_swin_target_layer(
    model
):

    if not hasattr(
        model,
        "layers"
    ):

        raise RuntimeError(
            "The Swin model does not contain 'layers'."
        )


    if len(
        model.layers
    ) < 2:

        raise RuntimeError(
            "Swin does not have enough stages."
        )


    penultimate_stage = (
        model.layers[
            -2
        ]
    )


    if not hasattr(
        penultimate_stage,
        "blocks"
    ):

        raise RuntimeError(
            "The penultimate Swin stage "
            "does not contain blocks."
        )


    last_block = (
        penultimate_stage.blocks[
            -1
        ]
    )


    if not hasattr(
        last_block,
        "norm1"
    ):

        raise RuntimeError(
            "Could not find norm1 in the "
            "penultimate Swin block."
        )


    return (
        last_block.norm1
    )


# ViT-B/16
# Gradient-weighted attention rollout
# Capture the actual self-attention matrices.
# Since TIMM may use fused attention,
# temporarily disable it to access
# the attention matrices explicitly.

class AttentionCapture:

    def __init__(
        self,
        model
    ):

        self.model = model

        self.attention_tensors = []

        self.handles = []

        self.original_fused_values = []


    def _capture_attention(
        self,
        module,
        inputs,
        output
    ):

        # Expected output:
        # B × heads × tokens × tokens

        if not torch.is_tensor(
            output
        ):

            raise RuntimeError(
                "The attn_drop output "
                "is not a tensor."
            )


        output.retain_grad()


        self.attention_tensors.append(
            output
        )


    def register(
        self
    ):

        for block in self.model.blocks:

            attention_module = (
                block.attn
            )


            # Temporarily disable fused attention

            if hasattr(
                attention_module,
                "fused_attn"
            ):

                self.original_fused_values.append(
                    (
                        attention_module,
                        attention_module.fused_attn
                    )
                )


                attention_module.fused_attn = False


            # attn_drop receives the attention matrix
            # after softmax.

            handle = (

                attention_module
                .attn_drop
                .register_forward_hook(
                    self._capture_attention
                )
            )


            self.handles.append(
                handle
            )


    def remove(
        self
    ):

        for handle in self.handles:

            handle.remove()


        for (
            attention_module,
            original_value

        ) in self.original_fused_values:

            attention_module.fused_attn = (
                original_value
            )


        self.handles = []

        self.original_fused_values = []


# Compute gradient-weighted attention rollout

def compute_gradient_weighted_attention_rollout(
    model,
    input_tensor
):

    capture = AttentionCapture(
        model
    )


    capture.register()


    try:

        model.zero_grad(
            set_to_none=True
        )


        with torch.enable_grad():

            logits = model(
                input_tensor
            )


            probabilities = torch.softmax(
                logits,
                dim=1
            )


            predicted_index = int(

                torch.argmax(
                    probabilities,
                    dim=1
                ).item()
            )


            predicted_probability = float(

                probabilities[
                    0,
                    predicted_index
                ]
                .detach()
                .cpu()
                .item()
            )


            target_score = (

                logits[
                    0,
                    predicted_index
                ]
            )


            target_score.backward()


        if len(
            capture.attention_tensors
        ) == 0:

            raise RuntimeError(
                "No attention matrices were captured."
            )


        weighted_attention_matrices = []


        for attention_tensor in (
            capture.attention_tensors
        ):

            gradient = (
                attention_tensor.grad
            )


            if gradient is None:

                raise RuntimeError(
                    "No gradients were captured "
                    "for an attention matrix."
                )


            # Attention weighted by the class gradient

            weighted_attention = (

                attention_tensor
                *
                gradient

            ).clamp(
                min=0
            )


            # Average across attention heads
            weighted_attention = (

                weighted_attention
                .mean(
                    dim=1
                )[
                    0
                ]
            )


            weighted_attention_matrices.append(
                weighted_attention
            )


        # Attention rollout

        number_of_tokens = (

            weighted_attention_matrices[
                0
            ].shape[
                0
            ]
        )


        identity = torch.eye(

            number_of_tokens,

            device=weighted_attention_matrices[
                0
            ].device,

            dtype=weighted_attention_matrices[
                0
            ].dtype
        )


        joint_attention = (
            identity
        )


        for attention_matrix in (
            weighted_attention_matrices
        ):

            # Residual connection

            augmented_attention = (

                attention_matrix
                +
                identity
            )


            augmented_attention = (

                augmented_attention
                /
                augmented_attention.sum(
                    dim=-1,
                    keepdim=True
                ).clamp(
                    min=1e-12
                )
            )


            joint_attention = (

                augmented_attention
                @
                joint_attention
            )


        # CLS token to patch tokens

        cls_attention = (

            joint_attention[
                0,
                1:
            ]
        )


        number_of_patches = int(
            cls_attention.numel()
        )


        spatial_size = int(
            math.sqrt(
                number_of_patches
            )
        )


        if (
            spatial_size
            *
            spatial_size
            !=
            number_of_patches
        ):

            raise RuntimeError(
                "ViT tokens cannot be "
                "reshaped into a square map.\n"
                f"patches={number_of_patches}"
            )


        attention_map = (

            cls_attention
            .reshape(
                spatial_size,
                spatial_size
            )
        )


        # Resize to 224×224

        attention_map = (

            attention_map[
                None,
                None,
                ...,
            ]
        )


        attention_map = torch.nn.functional.interpolate(

            attention_map,

            size=(
                XAI_IMG_SIZE,
                XAI_IMG_SIZE
            ),

            mode="bilinear",

            align_corners=False
        )


        attention_map = (

            attention_map[
                0,
                0
            ]
            .detach()
            .cpu()
            .numpy()
        )


        attention_map = normalize_refined_map(
            attention_map
        )


        return {

            "map":
                attention_map,

            "predicted_index":
                predicted_index,

            "predicted_class":
                INDEX_TO_CLASS[
                    predicted_index
                ],

            "predicted_probability":
                predicted_probability,

            "num_attention_layers":
                len(
                    weighted_attention_matrices
                ),

            "native_map_shape":
                (
                    spatial_size,
                    spatial_size
                )
        }


    finally:

        capture.remove()


# Refined XAI results

refined_records = []


print("=" * 80)

print(
    "GENERATING REFINED XAI"
)

print("=" * 80)


# ConvNeXt and MobileNet
# Reuse previously computed and validated CAMs

CNN_CONFIGURATIONS = [

    "convnext_tiny__none",

    "mobilenet_v2__cutmix"
]


for configuration_id in (
    CNN_CONFIGURATIONS
):

    print(
        "\nRe-rendering:",
        configuration_id
    )


    for _, case_row in (
        primary_xai_cases_df.iterrows()
    ):

        case_id = str(
            case_row[
                "xai_case_id"
            ]
        )


        image_path = get_case_image_path(
            case_row
        )


        (
            _,
            original_image

        ) = prepare_xai_image(
            image_path
        )


        cam = get_existing_cam(

            configuration_id=
                configuration_id,

            xai_case_id=
                case_id
        )


        cam = normalize_refined_map(
            cam
        )


        (
            heatmap_rgb,
            overlay

        ) = create_refined_overlay(

            original_image=
                original_image,

            heatmap=
                cam
        )


        # Retrieve the previously validated prediction

        original_record = (

            xai_results_df[
                (
                    xai_results_df[
                        "configuration_id"
                    ]
                    ==
                    configuration_id
                )
                &
                (
                    xai_results_df[
                        "xai_case_id"
                    ].astype(str)
                    ==
                    case_id
                )
            ]
            .iloc[
                0
            ]
        )


        map_path = (

            REFINED_MAPS_DIR
            /
            (
                f"{configuration_id}"
                f"__{case_id}"
                "__refined_cam.npy"
            )
        )


        overlay_path = (

            REFINED_OVERLAYS_DIR
            /
            (
                f"{configuration_id}"
                f"__{case_id}"
                "__refined_overlay.png"
            )
        )


        np.save(
            map_path,
            cam
        )


        plt.imsave(
            overlay_path,
            overlay
        )


        refined_records.append({

            "configuration_id":
                configuration_id,

            "xai_case_id":
                case_id,

            "xai_case_type":
                str(
                    case_row[
                        "xai_case_type"
                    ]
                ),

            "true_class":
                str(
                    case_row[
                        "true_class"
                    ]
                ),

            "predicted_class":
                original_record[
                    "model_predicted_class"
                ],

            "predicted_probability":
                float(
                    original_record[
                        "model_predicted_probability"
                    ]
                ),

            "correct_for_current_model":
                bool(
                    original_record[
                        "correct_for_current_model"
                    ]
                ),

            "xai_method":
                "Grad-CAM",

            "native_resolution":
                "7x7",

            "map_path":
                str(
                    map_path
                ),

            "overlay_path":
                str(
                    overlay_path
                ),

            "input_image_path":
                str(
                    image_path
                )
        })


# Refined Swin-Tiny maps

SWIN_CONFIGURATION = (
    "swin_tiny__cutmix"
)


print(
    "\nComputing Swin-Tiny "
    "at higher spatial resolution..."
)


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


swin_bundle = load_xai_model(
    SWIN_CONFIGURATION
)


swin_model = (
    swin_bundle[
        "model"
    ]
)


swin_target_layer = (
    get_refined_swin_target_layer(
        swin_model
    )
)


for _, case_row in (
    primary_xai_cases_df.iterrows()
):

    case_id = str(
        case_row[
            "xai_case_id"
        ]
    )


    image_path = get_case_image_path(
        case_row
    )


    (
        input_tensor,
        original_image

    ) = prepare_xai_image(
        image_path
    )


    result = compute_gradcam(

        model=
            swin_model,

        target_layer=
            swin_target_layer,

        input_tensor=
            input_tensor
    )


    cam = normalize_refined_map(
        result[
            "cam"
        ]
    )


    (
        heatmap_rgb,
        overlay

    ) = create_refined_overlay(

        original_image=
            original_image,

        heatmap=
            cam
    )


    map_path = (

        REFINED_MAPS_DIR
        /
        (
            f"{SWIN_CONFIGURATION}"
            f"__{case_id}"
            "__stage3_gradcam.npy"
        )
    )


    overlay_path = (

        REFINED_OVERLAYS_DIR
        /
        (
            f"{SWIN_CONFIGURATION}"
            f"__{case_id}"
            "__stage3_overlay.png"
        )
    )


    np.save(
        map_path,
        cam
    )


    plt.imsave(
        overlay_path,
        overlay
    )


    refined_records.append({

        "configuration_id":
            SWIN_CONFIGURATION,

        "xai_case_id":
            case_id,

        "xai_case_type":
            str(
                case_row[
                    "xai_case_type"
                ]
            ),

        "true_class":
            str(
                case_row[
                    "true_class"
                ]
            ),

        "predicted_class":
            result[
                "predicted_class"
            ],

        "predicted_probability":
            float(
                result[
                    "predicted_probability"
                ]
            ),

        "correct_for_current_model":
            (
                result[
                    "predicted_class"
                ]
                ==
                str(
                    case_row[
                        "true_class"
                    ]
                )
            ),

        "xai_method":
            (
                "Grad-CAM on penultimate "
                "Swin stage"
            ),

        "native_resolution":
            str(
                result[
                    "activation_shape"
                ]
            ),

        "map_path":
            str(
                map_path
            ),

        "overlay_path":
            str(
                overlay_path
            ),

        "input_image_path":
            str(
                image_path
            )
    })


    print(
        f"  {case_id}: "
        f"activation="
        f"{result['activation_shape']} "
        f"Pred="
        f"{result['predicted_class']}"
    )


    del input_tensor


del swin_model
del swin_bundle


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ViT-B/16 maps
# Gradient-weighted attention rollout

VIT_CONFIGURATION = (
    "vit_b16__cutmix"
)


print(
    "\nComputing ViT-B/16 "
    "gradient-weighted attention rollout..."
)


vit_bundle = load_xai_model(
    VIT_CONFIGURATION
)


vit_model = (
    vit_bundle[
        "model"
    ]
)


for _, case_row in (
    primary_xai_cases_df.iterrows()
):

    case_id = str(
        case_row[
            "xai_case_id"
        ]
    )


    image_path = get_case_image_path(
        case_row
    )


    (
        input_tensor,
        original_image

    ) = prepare_xai_image(
        image_path
    )


    result = (
        compute_gradient_weighted_attention_rollout(

            model=
                vit_model,

            input_tensor=
                input_tensor
        )
    )


    attention_map = normalize_refined_map(
        result[
            "map"
        ]
    )


    (
        heatmap_rgb,
        overlay

    ) = create_refined_overlay(

        original_image=
            original_image,

        heatmap=
            attention_map
    )


    map_path = (

        REFINED_MAPS_DIR
        /
        (
            f"{VIT_CONFIGURATION}"
            f"__{case_id}"
            "__gradient_attention_rollout.npy"
        )
    )


    overlay_path = (

        REFINED_OVERLAYS_DIR
        /
        (
            f"{VIT_CONFIGURATION}"
            f"__{case_id}"
            "__gradient_attention_rollout.png"
        )
    )


    np.save(
        map_path,
        attention_map
    )


    plt.imsave(
        overlay_path,
        overlay
    )


    refined_records.append({

        "configuration_id":
            VIT_CONFIGURATION,

        "xai_case_id":
            case_id,

        "xai_case_type":
            str(
                case_row[
                    "xai_case_type"
                ]
            ),

        "true_class":
            str(
                case_row[
                    "true_class"
                ]
            ),

        "predicted_class":
            result[
                "predicted_class"
            ],

        "predicted_probability":
            float(
                result[
                    "predicted_probability"
                ]
            ),

        "correct_for_current_model":
            (
                result[
                    "predicted_class"
                ]
                ==
                str(
                    case_row[
                        "true_class"
                    ]
                )
            ),

        "xai_method":
            (
                "Gradient-weighted "
                "attention rollout"
            ),

        "native_resolution":
            str(
                result[
                    "native_map_shape"
                ]
            ),

        "attention_layers":
            int(
                result[
                    "num_attention_layers"
                ]
            ),

        "map_path":
            str(
                map_path
            ),

        "overlay_path":
            str(
                overlay_path
            ),

        "input_image_path":
            str(
                image_path
            )
    })


    print(
        f"  {case_id}: "
        f"native="
        f"{result['native_map_shape']} "
        f"layers="
        f"{result['num_attention_layers']} "
        f"Pred="
        f"{result['predicted_class']}"
    )


    del input_tensor


del vit_model
del vit_bundle


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# Consolidate results

refined_xai_df = pd.DataFrame(
    refined_records
)


expected_records = 16


if len(
    refined_xai_df
) != expected_records:

    raise RuntimeError(
        f"Se esperaban {expected_records} "
        f"mapas refinados y se obtuvieron "
        f"{len(refined_xai_df)}."
    )


# Verify predictions
# They must match Cell 16 exactly.

prediction_check_records = []


for _, row in (
    refined_xai_df.iterrows()
):

    previous_record = (

        xai_results_df[
            (
                xai_results_df[
                    "configuration_id"
                ]
                ==
                row[
                    "configuration_id"
                ]
            )
            &
            (
                xai_results_df[
                    "xai_case_id"
                ].astype(str)
                ==
                str(
                    row[
                        "xai_case_id"
                    ]
                )
            )
        ]
        .iloc[
            0
        ]
    )


    same_prediction = (

        str(
            row[
                "predicted_class"
            ]
        )
        ==
        str(
            previous_record[
                "model_predicted_class"
            ]
        )
    )


    prediction_check_records.append(
        same_prediction
    )


refined_xai_df[
    "prediction_matches_previous_evaluation"
] = (
    prediction_check_records
)


if not refined_xai_df[
    "prediction_matches_previous_evaluation"
].all():

    raise RuntimeError(
        "A refined prediction does not match "
        "the previous evaluation."
    )


# Refined numerical audit

refined_audit_records = []


for _, row in (
    refined_xai_df.iterrows()
):

    heatmap = np.load(
        row[
            "map_path"
        ]
    )


    high_activation_fraction = float(

        np.mean(
            heatmap
            >=
            0.5
        )
    )


    refined_audit_records.append({

        "configuration_id":
            row[
                "configuration_id"
            ],

        "xai_case_id":
            row[
                "xai_case_id"
            ],

        "xai_method":
            row[
                "xai_method"
            ],

        "map_min":
            float(
                heatmap.min()
            ),

        "map_max":
            float(
                heatmap.max()
            ),

        "map_mean":
            float(
                heatmap.mean()
            ),

        "map_std":
            float(
                heatmap.std()
            ),

        "nonzero_fraction":
            float(
                np.mean(
                    heatmap
                    >
                    1e-8
                )
            ),

        "high_activation_fraction":
            high_activation_fraction,

        "valid_map":
            bool(
                np.isfinite(
                    heatmap
                ).all()
                and
                heatmap.max()
                >
                0
                and
                heatmap.std()
                >
                1e-6
            )
    })


refined_audit_df = pd.DataFrame(
    refined_audit_records
)


if not refined_audit_df[
    "valid_map"
].all():

    print(
        "WARNING: a degenerate refined "
        "map was detected."
    )

    display(

        refined_audit_df[
            ~refined_audit_df[
                "valid_map"
            ]
        ]
    )


# Save results

REFINED_XAI_SUMMARY_PATH = (

    REFINED_XAI_DIR
    /
    "40_refined_xai_summary.csv"
)


REFINED_XAI_AUDIT_PATH = (

    REFINED_XAI_DIR
    /
    "41_refined_xai_numerical_audit.csv"
)


refined_xai_df.to_csv(

    REFINED_XAI_SUMMARY_PATH,

    index=False
)


refined_audit_df.to_csv(

    REFINED_XAI_AUDIT_PATH,

    index=False
)


# Short class names

def short_class_name(
    class_name
):

    return str(
        class_name
    ).replace(
        "__",
        " "
    )


# Improved final figure

ROW_ORDER = [

    "swin_tiny__cutmix",

    "convnext_tiny__none",

    "mobilenet_v2__cutmix",

    "vit_b16__cutmix"
]


ROW_LABELS = {

    "swin_tiny__cutmix":
        "Swin-Tiny\nGrad-CAM",

    "convnext_tiny__none":
        "ConvNeXt-Tiny\nGrad-CAM",

    "mobilenet_v2__cutmix":
        "MobileNetV2\nGrad-CAM",

    "vit_b16__cutmix":
        "ViT-B/16\nAttention rollout"
}


COLUMN_LABELS = {

    "Correct Healthy":
        "Reference correct — Healthy",

    "Correct Rotten":
        "Reference correct — Rotten",

    "Error Healthy":
        "Reference error — Healthy",

    "Error Rotten":
        "Reference error — Rotten"
}


number_of_rows = (
    1
    +
    len(
        ROW_ORDER
    )
)


number_of_columns = len(
    CASE_ORDER
)


fig, axes = plt.subplots(

    number_of_rows,

    number_of_columns,

    figsize=(
        16,
        17
    )
)


# Row 0 — original images

for column_index, case_type in enumerate(
    CASE_ORDER
):

    case_row = (

        primary_xai_cases_df[
            primary_xai_cases_df[
                "xai_case_type"
            ]
            ==
            case_type
        ]
        .iloc[
            0
        ]
    )


    image_path = get_case_image_path(
        case_row
    )


    (
        _,
        original_image

    ) = prepare_xai_image(
        image_path
    )


    axes[
        0,
        column_index
    ].imshow(
        original_image
    )


    axes[
        0,
        column_index
    ].axis(
        "off"
    )


    axes[
        0,
        column_index
    ].set_title(

        (
            f"{COLUMN_LABELS[case_type]}\n"
            f"Ground truth: "
            f"{short_class_name(case_row['true_class'])}"
        ),

        fontsize=11,

        pad=10
    )


# Model rows

for row_index, configuration_id in enumerate(

    ROW_ORDER,

    start=1

):

    configuration_records = (

        refined_xai_df[
            refined_xai_df[
                "configuration_id"
            ]
            ==
            configuration_id
        ]
    )


    for column_index, case_type in enumerate(
        CASE_ORDER
    ):

        result_row = (

            configuration_records[
                configuration_records[
                    "xai_case_type"
                ]
                ==
                case_type
            ]
            .iloc[
                0
            ]
        )


        overlay_image = plt.imread(

            result_row[
                "overlay_path"
            ]
        )


        axes[
            row_index,
            column_index
        ].imshow(
            overlay_image
        )


        axes[
            row_index,
            column_index
        ].axis(
            "off"
        )


        axes[
            row_index,
            column_index
        ].set_title(

            (
                "Pred: "
                f"{short_class_name(result_row['predicted_class'])}\n"
                f"p = "
                f"{result_row['predicted_probability']:.3f}"
            ),

            fontsize=9,

            pad=4
        )


# Visible row labels
# Do not use ylabel because axis('off') may hide it.

row_labels = [

    "Original",

    ROW_LABELS[
        "swin_tiny__cutmix"
    ],

    ROW_LABELS[
        "convnext_tiny__none"
    ],

    ROW_LABELS[
        "mobilenet_v2__cutmix"
    ],

    ROW_LABELS[
        "vit_b16__cutmix"
    ]
]


for row_index, row_label in enumerate(
    row_labels
):

    axes[
        row_index,
        0
    ].text(

        -0.12,

        0.50,

        row_label,

        transform=axes[
            row_index,
            0
        ].transAxes,

        rotation=90,

        va="center",

        ha="center",

        fontsize=11,

        fontweight="bold",

        clip_on=False
    )


# Title and methodological note

fig.suptitle(

    (
        "Explainability analysis across "
        "Healthy/Rotten predictions"
    ),

    fontsize=16,

    y=0.995
)


fig.text(

    0.5,

    0.008,

    (
        "Correct/error case labels refer to the "
        "Swin-Tiny + CutMix reference model used to "
        "predefine the four test examples. "
        "Each row reports the prediction of the "
        "corresponding architecture."
    ),

    ha="center",

    va="bottom",

    fontsize=9
)


fig.tight_layout(

    rect=[
        0.055,
        0.035,
        1,
        0.98
    ],

    h_pad=1.5,

    w_pad=1.0
)


# Save PNG and PDF figures

REFINED_PAPER_FIGURE_PNG = (

    REFINED_FIGURES_DIR
    /
    "42_refined_xai_paper_figure.png"
)


REFINED_PAPER_FIGURE_PDF = (

    REFINED_FIGURES_DIR
    /
    "42_refined_xai_paper_figure.pdf"
)


fig.savefig(

    REFINED_PAPER_FIGURE_PNG,

    dpi=600,

    bbox_inches="tight"
)


fig.savefig(

    REFINED_PAPER_FIGURE_PDF,

    bbox_inches="tight"
)


plt.close(
    fig
)


# Summary

print(
    "\n"
    +
    "=" * 80
)


print(
    "REFINED XAI COMPLETED"
)


print(
    "=" * 80
)


display(

    refined_xai_df[
        [
            "configuration_id",
            "xai_case_id",
            "xai_case_type",
            "true_class",
            "predicted_class",
            "correct_for_current_model",
            "predicted_probability",
            "xai_method",
            "native_resolution"
        ]
    ]
)


print(
    "\n"
    +
    "=" * 80
)


print(
    "REFINED MAP AUDIT"
)


print(
    "=" * 80
)


display(
    refined_audit_df
)


print(
    "\nValid maps:",
    int(
        refined_audit_df[
            "valid_map"
        ].sum()
    ),
    "/",
    len(
        refined_audit_df
    )
)


print(
    "\nPredictions consistent with "
    "the previous evaluation:",
    bool(
        refined_xai_df[
            "prediction_matches_previous_evaluation"
        ].all()
    )
)


print(
    "\nPNG figure:"
)

print(
    REFINED_PAPER_FIGURE_PNG
)


print(
    "\nPDF figure:"
)

print(
    REFINED_PAPER_FIGURE_PDF
)


print(
    "\nSummary:"
)

print(
    REFINED_XAI_SUMMARY_PATH
)


print(
    "\nAudit:"
)

print(
    REFINED_XAI_AUDIT_PATH
)


# Preview refined figure

print(
    "\nPreview:"
)


display(

    Image.open(
        REFINED_PAPER_FIGURE_PNG
    )
)

# **SAVE ALL**

In [ ]:
from pathlib import Path
from datetime import datetime

import hashlib
import json
import platform
import zipfile

import pandas as pd
import torch

from IPython.display import (
    display,
    FileLink
)

# Backup source directory

if "ANALYSIS_ROOT" in globals():

    BACKUP_SOURCE = Path(
        ANALYSIS_ROOT
    )

else:

    BACKUP_SOURCE = Path(
        "/kaggle/working/"
        "fruit_post_training_analysis"
    )

if not BACKUP_SOURCE.exists():

    raise FileNotFoundError(
        f"Directory does not exist:\n"
        f"{BACKUP_SOURCE}"
    )

print("=" * 80)
print("BACKUP SOURCE")
print("=" * 80)

print(
    BACKUP_SOURCE
)

# Critical files

critical_files = {

# R7 — Per-class precision, recall, and F1

    "R7_per_class_metrics":

        BACKUP_SOURCE
        / "class_reports"
        / "19_all_models_class_reports.csv",

# R8 — Calibration and reliability diagrams

    "R8_calibration_summary":

        BACKUP_SOURCE
        / "calibration"
        / "09_temperature_scaling_ece_summary.csv",

    "R8_reliability_index":

        BACKUP_SOURCE
        / "calibration"
        / "reliability_diagrams"
        / "30_reliability_diagram_index.csv",

# R9 — Hardware

    "R9_hardware":

        BACKUP_SOURCE
        / "hardware"
        / "32_complete_hardware_specification.csv",

    "R9_latency_protocol":

        BACKUP_SOURCE
        / "hardware"
        / "33_latency_throughput_protocol.csv",

    "R9_gpu_memory":

        BACKUP_SOURCE
        / "hardware"
        / "34_architecture_gpu_memory_summary.csv",

# Bootstrap

    "bootstrap_summary":

        BACKUP_SOURCE
        / "bootstrap_ci"
        / "11_test_bootstrap_ci_summary.csv",

# McNemar and Holm
# Actual filenames verified in the directory

    "mcnemar_all_66_holm":

        BACKUP_SOURCE
        / "mcnemar_holm"
        / "14_mcnemar_all_66_holm.csv",

    "mcnemar_significant_holm":

        BACKUP_SOURCE
        / "mcnemar_holm"
        / "15_mcnemar_significant_holm.csv",

    "mcnemar_within_architecture":

        BACKUP_SOURCE
        / "mcnemar_holm"
        / "16_mcnemar_within_architecture.csv",

    "mcnemar_holm_pvalue_matrix":

        BACKUP_SOURCE
        / "mcnemar_holm"
        / "17_mcnemar_holm_pvalue_matrix.csv",

    "mcnemar_swin_cutmix_vs_all":

        BACKUP_SOURCE
        / "mcnemar_holm"
        / "18_swin_cutmix_vs_all_mcnemar.csv",

# Learning curves

    "learning_histories":

        BACKUP_SOURCE
        / "learning_curves"
        / "27_all_training_histories.csv",

    "learning_curve_index":

        BACKUP_SOURCE
        / "learning_curves"
        / "28_individual_learning_curve_index.csv",

# XAI

    "xai_primary_cases":

        BACKUP_SOURCE
        / "explainability"
        / "36_primary_xai_cases_for_paper.csv",

    "xai_original_summary":

        BACKUP_SOURCE
        / "explainability"
        / "38_xai_summary_records.csv",

    "xai_original_audit":

        BACKUP_SOURCE
        / "explainability"
        / "39_xai_map_numerical_audit.csv",

    "xai_refined_summary":

        BACKUP_SOURCE
        / "explainability"
        / "refined"
        / "40_refined_xai_summary.csv",

    "xai_refined_audit":

        BACKUP_SOURCE
        / "explainability"
        / "refined"
        / "41_refined_xai_numerical_audit.csv"
}

# Final XAI figure
# Priority:
# 43 = final version with corrected headings
# 42 = previous refined version

xai_final_candidates = [

    BACKUP_SOURCE
    / "explainability"
    / "refined"
    / "figures"
    / "43_final_xai_paper_figure.png",

    BACKUP_SOURCE
    / "explainability"
    / "refined"
    / "figures"
    / "42_refined_xai_paper_figure.png"
]

existing_final_xai = [

    path

    for path
    in xai_final_candidates

    if path.exists()
]

if len(
    existing_final_xai
) == 0:

    raise FileNotFoundError(
        "No refined/final XAI figure was "
        "found."
    )

FINAL_XAI_FIGURE = (
    existing_final_xai[0]
)

critical_files[
    "xai_final_figure"
] = FINAL_XAI_FIGURE

# Include the final PDF if it exists

xai_pdf_candidates = [

    BACKUP_SOURCE
    / "explainability"
    / "refined"
    / "figures"
    / "43_final_xai_paper_figure.pdf",

    BACKUP_SOURCE
    / "explainability"
    / "refined"
    / "figures"
    / "42_refined_xai_paper_figure.pdf"
]

existing_xai_pdf = [

    path

    for path
    in xai_pdf_candidates

    if path.exists()
]

if len(
    existing_xai_pdf
) > 0:

    critical_files[
        "xai_final_figure_pdf"
    ] = existing_xai_pdf[0]

# Critical-file verification

critical_records = []

for item_name, file_path in (
    critical_files.items()
):

    critical_records.append({

        "item":
            item_name,

        "path":
            str(
                file_path
            ),

        "exists":
            file_path.exists(),

        "size_mb":
            (
                file_path.stat().st_size
                /
                (
                    1024 ** 2
                )
                if file_path.exists()
                else 0
            )
    })

critical_files_df = pd.DataFrame(
    critical_records
)

print(
    "\n"
    +
    "=" * 80
)

print(
    "CRITICAL-FILE VERIFICATION"
)

print(
    "=" * 80
)

display(
    critical_files_df
)

if not critical_files_df[
    "exists"
].all():

    missing_critical_df = (

        critical_files_df[
            ~critical_files_df[
                "exists"
            ]
        ]
    )

    print(
        "\nMISSING FILES:"
    )

    display(
        missing_critical_df
    )

    raise FileNotFoundError(
        "Critical files are missing. "
        "The ZIP will not be created."
    )

print(
    "\n All critical files are present."
)

# Recovery directory

RECOVERY_DIR = (

    BACKUP_SOURCE
    / "recovery"
)

RECOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# External persistent paths

TRAINING_RUNS_ROOT = (

    "/kaggle/input/datasets/"
    "andreamencotovar/"
    "nuevas-corridas-2026/"
    "nuevas corridas"
)

CLEAN_DATASET_ROOT = (

    "/kaggle/input/datasets/"
    "andreamencotovar/"
    "respuesta-a-revisores-2026/"
    "clean_dataset/clean_dataset"
)

FINAL_MANIFEST_PATH = (

    "/kaggle/input/datasets/"
    "andreamencotovar/"
    "respuesta-a-revisores-2026/"
    "manifests/manifests/"
    "14_final_verified_dataset_manifest.csv"
)

# Recovery metadata

backup_creation_time = (
    datetime.now().isoformat()
)

recovery_metadata = {

    "backup_created_at":
        backup_creation_time,

    "analysis_root":
        str(
            BACKUP_SOURCE
        ),

    "training_runs_root":
        TRAINING_RUNS_ROOT,

    "clean_dataset_root":
        CLEAN_DATASET_ROOT,

    "final_dataset_manifest":
        FINAL_MANIFEST_PATH,

    "dataset_total":
        28348,

    "train_images":
        22672,

    "validation_images":
        2835,

    "test_images":
        2841,

    "number_of_classes":
        24,

    "seed":
        42,

    "image_size":
        224,

    "best_overall_configuration":
        "swin_tiny__cutmix",

    "validation_selected_models": [

        "swin_tiny__cutmix",

        "convnext_tiny__none",

        "mobilenet_v2__cutmix",

        "vit_b16__cutmix"
    ],

    "python_version":
        platform.python_version(),

    "torch_version":
        torch.__version__,

    "cuda_version":
        torch.version.cuda,

    "gpu":
        (
            torch.cuda.get_device_name(
                0
            )
            if torch.cuda.is_available()
            else "No CUDA"
        ),

    "final_xai_figure":
        str(
            FINAL_XAI_FIGURE
        ),

    "critical_files_verified":
        True
}

if "EXPECTED_CONFIGURATIONS" in globals():

    recovery_metadata[
        "expected_configurations"
    ] = list(
        EXPECTED_CONFIGURATIONS
    )

if "RUN_DIRECTORIES" in globals():

    recovery_metadata[
        "run_directories"
    ] = {

        str(
            key
        ):
            str(
                value
            )

        for key, value
        in RUN_DIRECTORIES.items()
    }

METADATA_PATH = (

    RECOVERY_DIR
    / "RECOVERY_METADATA.json"
)

with open(

    METADATA_PATH,

    "w",

    encoding="utf-8"

) as file:

    json.dump(

        recovery_metadata,

        file,

        indent=4,

        ensure_ascii=False
    )

# Save critical-file status

CRITICAL_STATUS_PATH = (

    RECOVERY_DIR
    / "CRITICAL_FILES_STATUS.csv"
)

critical_files_df.to_csv(

    CRITICAL_STATUS_PATH,

    index=False
)

# Recovery README

README_TEXT = f"""
FRUIT FRESHNESS CLASSIFICATION
FINAL POST-TRAINING ANALYSIS — MASTER BACKUP
============================================================

Backup date:
{backup_creation_time}

============================================================
FINAL CLEAN DATASET
============================================================

Total images:       28,348
Train:              22,672
Validation:          2,835
Test:                2,841
Classes:                24
Seed:                   42
Input resolution:   224 x 224

============================================================
TRAINED CONFIGURATIONS
============================================================

12 total configurations:

ConvNeXt-Tiny
- None
- MixUp
- CutMix

MobileNetV2
- None
- MixUp
- CutMix

Swin-Tiny
- None
- MixUp
- CutMix

ViT-B/16
- None
- MixUp
- CutMix

============================================================
MODEL SELECTION
============================================================

Primary selection metric:
Validation Macro-F1

Best validation-selected configuration:
Swin-Tiny + CutMix

Important:
Model selection was performed using validation results,
not test results.

============================================================
ANALYSES INCLUDED
============================================================

- Raw model outputs
- Test predictions
- Point estimates
- Accuracy
- Balanced Accuracy
- Macro-F1

Calibration:
- Temperature Scaling
- ECE before calibration
- ECE after calibration
- NLL before calibration
- NLL after calibration
- Reliability bins
- Reliability diagrams

Statistical analysis:
- Stratified cluster bootstrap
- 95% confidence intervals
- Bootstrap replicates
- Exact paired McNemar tests
- Holm-Bonferroni correction
- All 66 pairwise comparisons
- Significant comparisons
- Within-architecture comparisons
- Holm p-value matrix
- Swin-Tiny + CutMix comparisons

Class-level analysis:
- Precision by class
- Recall by class
- F1-score by class
- Support by class
- Confusion matrices
- Error-pair analysis

Efficiency:
- Parameter counts
- Checkpoint size
- Latency
- P95 latency
- Throughput
- GPU memory
- Hardware specifications
- Measurement protocol

Training:
- Complete training histories
- Learning curves
- Validation-selected epochs

Explainability:
- Fixed XAI test cases
- Correct Healthy
- Correct Rotten
- Error Healthy
- Error Rotten
- Original XAI maps
- Refined XAI maps
- Grad-CAM
- Swin Grad-CAM at higher spatial resolution
- ViT gradient-weighted attention rollout
- Numerical XAI audits
- Final publication-oriented XAI figure

============================================================
REVIEWER COMMENTS COVERED
============================================================

R7
---
Per-class Precision, Recall and F1-score.

R8
---
Expected Calibration Error complemented with
reliability diagrams.

R9
---
Hardware specifications including:
- GPU
- VRAM
- CUDA
- CPU
- physical/logical cores
- RAM
- PyTorch
- inference precision
- latency protocol
- throughput protocol

============================================================
XAI CASE-SELECTION NOTE
============================================================

The four predefined cases are:

1. Correct Healthy
2. Correct Rotten
3. Error Healthy
4. Error Rotten

Correct/error labels were defined using:

Swin-Tiny + CutMix

as the reference model.

The SAME four images were evaluated with:

- Swin-Tiny
- ConvNeXt-Tiny
- MobileNetV2
- ViT-B/16

Each architecture retains its own prediction.

============================================================
PERSISTENT KAGGLE RESOURCES
============================================================

Training checkpoints:

{TRAINING_RUNS_ROOT}

Clean physical dataset:

{CLEAN_DATASET_ROOT}

Final verified manifest:

{FINAL_MANIFEST_PATH}

============================================================
IMPORTANT
============================================================

The 12 model checkpoints are NOT duplicated inside this ZIP.

They are already stored persistently in the Kaggle Dataset:

nuevas-corridas-2026

This avoids unnecessarily duplicating several large
checkpoint files.

The complete post-training analysis IS contained in this ZIP.

============================================================
HOW TO CONTINUE THE WORK
============================================================

1. Open the saved Kaggle notebook/version.

2. Attach these Kaggle datasets:

   - nuevas-corridas-2026
   - respuesta-a-revisores-2026

3. If the Kaggle working directory was lost,
   upload and extract this ZIP.

4. Restore:

   /kaggle/working/fruit_post_training_analysis

5. Recreate only the Python variables/paths needed
   for subsequent analysis.

6. DO NOT retrain the 12 configurations.

7. Use the existing best.pt checkpoints.

8. Continue directly from the CSV, NPZ, PNG and PDF
   results contained in this backup.

============================================================
FINAL WARNING
============================================================

The live Python kernel state is NOT preserved when Kaggle
ends the session.

Save a Kaggle notebook version separately before closing.
"""

README_PATH = (

    RECOVERY_DIR
    / "README_RECOVERY.txt"
)

README_PATH.write_text(

    README_TEXT,

    encoding="utf-8"
)

# SHA-256 utility

def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024
):

    hash_object = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:

                break

            hash_object.update(
                chunk
            )

    return (
        hash_object.hexdigest()
    )

# File inventory
# Build the inventory before writing the inventory file
# to avoid circular self-reference.

print(
    "\n"
    +
    "=" * 80
)

print(
    "BUILDING COMPLETE INVENTORY"
)

print(
    "=" * 80
)

files_for_inventory = sorted(

    [

        path

        for path
        in BACKUP_SOURCE.rglob("*")

        if path.is_file()

    ],

    key=lambda path:
        str(
            path
        )
)

inventory_records = []

for file_number, file_path in enumerate(

    files_for_inventory,

    start=1

):

    size_bytes = (
        file_path.stat().st_size
    )

    sha256_value = (
        calculate_sha256(
            file_path
        )
    )

    inventory_records.append({

        "file_number":
            file_number,

        "relative_path":
            str(
                file_path.relative_to(
                    BACKUP_SOURCE
                )
            ),

        "file_name":
            file_path.name,

        "extension":
            (
                file_path.suffix.lower()
                if file_path.suffix
                else "[no extension]"
            ),

        "size_bytes":
            size_bytes,

        "size_mb":
            (
                size_bytes
                /
                (
                    1024 ** 2
                )
            ),

        "sha256":
            sha256_value
    })

inventory_df = pd.DataFrame(
    inventory_records
)

INVENTORY_PATH = (

    RECOVERY_DIR
    / "COMPLETE_FILE_INVENTORY_SHA256.csv"
)

inventory_df.to_csv(

    INVENTORY_PATH,

    index=False
)

# File-type summary

inventory_summary_df = (

    inventory_df
    .groupby(
        "extension",
        dropna=False
    )
    .agg(

        files=(
            "relative_path",
            "count"
        ),

        total_mb=(
            "size_mb",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        "total_mb",
        ascending=False
    )
)

INVENTORY_SUMMARY_PATH = (

    RECOVERY_DIR
    / "FILE_TYPE_SUMMARY.csv"
)

inventory_summary_df.to_csv(

    INVENTORY_SUMMARY_PATH,

    index=False
)

# Pre-ZIP status

PRE_ZIP_STATUS_PATH = (

    RECOVERY_DIR
    / "BACKUP_CONTENT_STATUS.txt"
)

PRE_ZIP_STATUS_TEXT = f"""
BACKUP CONTENT STATUS
============================================================

Created:
{datetime.now().isoformat()}

Source:
{BACKUP_SOURCE}

Critical files:
PASSED

McNemar + Holm files:
5 / 5 present

Final XAI figure:
{FINAL_XAI_FIGURE}

File inventory with SHA-256:
{INVENTORY_PATH.name}

Checkpoints:
External persistent Kaggle Dataset
nuevas-corridas-2026

Status:
READY FOR ZIP
"""

PRE_ZIP_STATUS_PATH.write_text(

    PRE_ZIP_STATUS_TEXT,

    encoding="utf-8"
)

# Final list of files for ZIP
# Includes:
# - original results
# - README
# - metadata
# - critical-file status
# - inventory
# - file-type summary
# - backup-content status

files_to_zip = sorted(

    [

        path

        for path
        in BACKUP_SOURCE.rglob("*")

        if path.is_file()

    ],

    key=lambda path:
        str(
            path
        )
)

total_uncompressed_bytes = sum(

    path.stat().st_size

    for path
    in files_to_zip
)

total_uncompressed_mb = (

    total_uncompressed_bytes

    /
    (
        1024 ** 2
    )
)

print(
    "\nFiles to include:",
    len(
        files_to_zip
    )
)

print(
    "Total uncompressed size:",
    f"{total_uncompressed_mb:.2f} MB"
)

# Create ZIP archive

timestamp = (

    datetime.now()
    .strftime(
        "%Y%m%d_%H%M%S"
    )
)

ZIP_PATH = Path(

    "/kaggle/working/"
    f"fruit_post_training_analysis_FINAL_{timestamp}.zip"
)

print(
    "\n"
    +
    "=" * 80
)

print(
    "CREATING FINAL ZIP"
)

print(
    "=" * 80
)

with zipfile.ZipFile(

    ZIP_PATH,

    mode="w",

    compression=zipfile.ZIP_DEFLATED,

    compresslevel=6

) as zip_file:

    for file_number, file_path in enumerate(

        files_to_zip,

        start=1

    ):

        archive_name = (

            Path(
                BACKUP_SOURCE.name
            )

            /

            file_path.relative_to(
                BACKUP_SOURCE
            )
        )

        zip_file.write(

            file_path,

            arcname=str(
                archive_name
            )
        )

# Verify ZIP integrity

print(
    "\n"
    +
    "=" * 80
)

print(
    "VERIFYING ZIP INTEGRITY"
)

print(
    "=" * 80
)

with zipfile.ZipFile(

    ZIP_PATH,

    mode="r"

) as zip_file:

    corrupt_member = (
        zip_file.testzip()
    )

    zip_members = (
        zip_file.namelist()
    )

# Internal ZIP integrity test

if corrupt_member is not None:

    raise RuntimeError(

        "Corrupt ZIP. "
        "Affected file:\n"

        f"{corrupt_member}"
    )

# File count

if len(
    zip_members
) != len(
    files_to_zip
):

    raise RuntimeError(

        "The file counts do not match.\n"

        f"Original directory: "
        f"{len(files_to_zip)}\n"

        f"ZIP: "
        f"{len(zip_members)}"
    )

# Verify all critical files are included in the ZIP

zip_member_set = set(
    zip_members
)

missing_critical_in_zip = []

for item_name, file_path in (
    critical_files.items()
):

    expected_archive_path = str(

        Path(
            BACKUP_SOURCE.name
        )

        /

        file_path.relative_to(
            BACKUP_SOURCE
        )
    )

    if (
        expected_archive_path
        not in zip_member_set
    ):

        missing_critical_in_zip.append({

            "item":
                item_name,

            "expected_zip_path":
                expected_archive_path
        })

if len(
    missing_critical_in_zip
) > 0:

    missing_zip_df = pd.DataFrame(
        missing_critical_in_zip
    )

    display(
        missing_zip_df
    )

    raise RuntimeError(
        "Some critical files were not "
        "included in the ZIP."
    )

print(
    " Test interno ZIP: PASSED"
)

print(
    " File count: PASSED"
)

print(
    "Critical files in ZIP: PASSED"
)

# Verify the five McNemar files

mcnemar_expected_names = {

    "14_mcnemar_all_66_holm.csv",

    "15_mcnemar_significant_holm.csv",

    "16_mcnemar_within_architecture.csv",

    "17_mcnemar_holm_pvalue_matrix.csv",

    "18_swin_cutmix_vs_all_mcnemar.csv"
}

mcnemar_in_zip = {

    Path(
        member
    ).name

    for member
    in zip_members

    if (
        "/mcnemar_holm/"
        in member
    )
}

missing_mcnemar = (

    mcnemar_expected_names

    -

    mcnemar_in_zip
)

if len(
    missing_mcnemar
) > 0:

    raise RuntimeError(

        "McNemar files are missing from the ZIP:\n"

        +
        "\n".join(
            sorted(
                missing_mcnemar
            )
        )
    )

print(
    "McNemar + Holm 5/5 in ZIP: PASSED"
)

# Final ZIP SHA-256

print(
    "\nCalculating ZIP SHA-256..."
)

zip_sha256 = (
    calculate_sha256(
        ZIP_PATH
    )
)

zip_size_bytes = (
    ZIP_PATH.stat().st_size
)

zip_size_mb = (

    zip_size_bytes

    /
    (
        1024 ** 2
    )
)

# External final backup summary
# Save this file OUTSIDE the compressed directory
# because it contains the final ZIP SHA-256.

backup_summary_df = pd.DataFrame({

    "item": [

        "ZIP file",

        "Files included",

        "Uncompressed size (MB)",

        "ZIP size (MB)",

        "ZIP integrity test",

        "Critical files in ZIP",

        "McNemar + Holm files in ZIP",

        "Final XAI figure",

        "ZIP SHA-256"
    ],

    "value": [

        str(
            ZIP_PATH
        ),

        len(
            zip_members
        ),

        round(
            total_uncompressed_mb,
            3
        ),

        round(
            zip_size_mb,
            3
        ),

        "PASSED",

        "PASSED",

        "5/5 PASSED",

        str(
            FINAL_XAI_FIGURE
        ),

        zip_sha256
    ]
})

FINAL_BACKUP_SUMMARY_PATH = Path(

    "/kaggle/working/"
    f"fruit_post_training_analysis_FINAL_{timestamp}"
    "_SHA256.csv"
)

backup_summary_df.to_csv(

    FINAL_BACKUP_SUMMARY_PATH,

    index=False
)

# Final results

print(
    "\n"
    +
    "=" * 80
)

print(
    "FINAL BACKUP COMPLETED SUCCESSFULLY"
)

print(
    "=" * 80
)

display(
    backup_summary_df
)

print(
    "\n"
    +
    "=" * 80
)

print(
    "FILE-TYPE SUMMARY"
)

print(
    "=" * 80
)

display(
    inventory_summary_df
)

print(
    "\nZIP:"
)

print(
    ZIP_PATH
)

print(
    "\nZIP size:"
)

print(
    f"{zip_size_mb:.2f} MB"
)

print(
    "\nFiles included:"
)

print(
    len(
        zip_members
    )
)

print(
    "\nSHA-256:"
)

print(
    zip_sha256
)

print(
    "\nExternal SHA-256 summary:"
)

print(
    FINAL_BACKUP_SUMMARY_PATH
)

print(
    "\n"
    +
    "=" * 80
)

print(
    "ALL CHECKS: PASSED"
)

print(
    "=" * 80
)

print(
    "\n Critical files present"
)

print(
    " McNemar + Holm 5/5"
)

print(
    " R7 included"
)

print(
    " R8 included"
)

print(
    " R9 included"
)

print(
    " Bootstrap included"
)

print(
    "Learning curves included"
)

print(
    "Original XAI included"
)

print(
    "Refined XAI included"
)

print(
    " Final XAI figure included"
)

print(
    " SHA-256 inventory included"
)

print(
    " ZIP integrity PASSED"
)

# Download links

print(
    "\n"
    +
    "=" * 80
)

print(
    "DOWNLOAD BACKUP"
)

print(
    "=" * 80
)

print(
    "\n1. MASTER ZIP:"
)

display(

    FileLink(
        str(
            ZIP_PATH
        )
    )
)

print(
    "\n2. SHA-256 summary:"
)

display(

    FileLink(
        str(
            FINAL_BACKUP_SUMMARY_PATH
        )
    )
)

# **EXTERNAL VALIDATION**

# **Cell 1 — creation and validation of the external manifest**

In [ ]:
from pathlib import Path
import pandas as pd

# Configuration
KAGGLE_INPUT = Path("/kaggle/input")

OUTPUT_DIR = Path("/kaggle/working/external_validation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "external_validation_manifest.csv"

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

# Expected class folders
EXPECTED_FOLDERS = {
    "apple_healthy",
    "apple_rotten",
    "banana_healthy",
    "banana_rotten",
    "bell_pepper_healthy",
    "bell_pepper_rotten",
    "carrot_healthy",
    "carrot_rotten",
    "cucumber_healthy",
    "cucumber_rotten",
    "guava_healthy",
    "guava_rotten",
    "mango_healthy",
    "mango_rotten",
    "orange_healthy",
    "orange_rotten",
    "pomegranate_healthy",
    "pomegranate_rotten",
    "potato_healthy",
    "potato_rotten",
    "strawberry_healthy",
    "strawberry_rotten",
    "tomato_healthy",
    "tomato_rotten",
}

# Automatically locate the external validation dataset

def find_external_dataset_root(base_dir, expected_folders):
    """Find the directory under /kaggle/input containing the 24 external validation class folders."""
    candidates = []

    for path in base_dir.rglob("*"):
        if not path.is_dir():
            continue

        try:
            child_dirs = {
                p.name.lower()
                for p in path.iterdir()
                if p.is_dir()
            }
        except Exception:
            continue

        matches = len(expected_folders.intersection(child_dirs))

        if matches > 0:
            candidates.append((matches, path))

    if not candidates:
        raise FileNotFoundError(
            "No directory containing external validation class folders "
            "was found under /kaggle/input."
        )

    # Select the directory with the most matching class folders
    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    best_matches, best_path = candidates[0]

    if best_matches != len(expected_folders):
        print(
            f" Found {best_matches}/"
            f"{len(expected_folders)} expected folders."
        )

    return best_path


EXTERNAL_ROOT = find_external_dataset_root(
    KAGGLE_INPUT,
    EXPECTED_FOLDERS
)

print("=" * 75)
print("EXTERNAL DATASET LOCATED")
print("=" * 75)
print(EXTERNAL_ROOT)


In [ ]:
# Class name mapping
FRUIT_MAP = {
    "apple": {"display": "Apple", "model": "Apple"},
    "banana": {"display": "Banana", "model": "Banana"},
    "bell_pepper": {"display": "Bell pepper", "model": "Bellpepper"},
    "carrot": {"display": "Carrot", "model": "Carrot"},
    "cucumber": {"display": "Cucumber", "model": "Cucumber"},
    "guava": {"display": "Guava", "model": "Guava"},
    "mango": {"display": "Mango", "model": "Mango"},
    "orange": {"display": "Orange", "model": "Orange"},
    "pomegranate": {"display": "Pomegranate", "model": "Pomegranate"},
    "potato": {"display": "Potato", "model": "Potato"},
    "strawberry": {"display": "Strawberry", "model": "Strawberry"},
    "tomato": {"display": "Tomato", "model": "Tomato"},
}

# Read all images
records = []

class_folders = sorted([
    p for p in EXTERNAL_ROOT.iterdir()
    if p.is_dir()
])

for class_folder in class_folders:
    folder_name = class_folder.name.lower().strip()

    # Determine quality condition
    if folder_name.endswith("_healthy"):
        condition = "Healthy"
        fruit_key = folder_name[:-len("_healthy")]
    elif folder_name.endswith("_rotten"):
        condition = "Rotten"
        fruit_key = folder_name[:-len("_rotten")]
    else:
        print(f" Skipped folder: {class_folder.name}")
        continue

    # Validate fruit category
    if fruit_key not in FRUIT_MAP:
        raise ValueError(f"Unrecognized fruit: {fruit_key}")

    fruit_display = FRUIT_MAP[fruit_key]["display"]
    fruit_model = FRUIT_MAP[fruit_key]["model"]

    # Ground-truth label for the 24-class classifier
    true_class = f"{fruit_model}__{condition}"

    # Locate images
    images = sorted([
        p for p in class_folder.rglob("*")
        if (
            p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        )
    ])

    for image_path in images:
        records.append({
            # Neutral identifier, e.g., EXT_0000688
            "image_id": image_path.stem,
            "filename": image_path.name,

            # Ground-truth information
            "fruit": fruit_display,
            "condition": condition,
            "true_class": true_class,

            # Original directory
            "source_folder": class_folder.name,

            # Do not infer field or laboratory provenance
            "acquisition": "unknown",

            # Do not assume that each image is a distinct specimen
            "specimen_id": "",

            # Portable relative path
            "relative_path": str(
                image_path.relative_to(EXTERNAL_ROOT)
            ),

            # Absolute path inside Kaggle
            "path": str(image_path)
        })

# Construct manifest DataFrame
manifest_df = pd.DataFrame(records)

manifest_df = (
    manifest_df
    .sort_values(["true_class", "image_id"])
    .reset_index(drop=True)
)

# Basic validation
print("\n" + "=" * 75)
print("MANIFEST SUMMARY")
print("=" * 75)

print(f"Total images: {len(manifest_df):,}")
print(f"Classes found: {manifest_df['true_class'].nunique()}")
print(f"Unique IDs: {manifest_df['image_id'].nunique():,}")

# External image IDs must be unique
duplicated_ids = (
    manifest_df["image_id"]
    .duplicated()
    .sum()
)

print(f"Duplicate image_id values: {duplicated_ids}")

# Per-class distribution
class_distribution = (
    manifest_df
    .groupby(["true_class"], as_index=False)
    .size()
    .rename(columns={"size": "n_images"})
)

print("\nPer-class distribution:\n")
print(class_distribution.to_string(index=False))

# Validate expected dataset composition
EXPECTED_TOTAL = 717
EXPECTED_N_CLASSES = 24

assert len(manifest_df) == EXPECTED_TOTAL, (
    f"\n❌ Expected {EXPECTED_TOTAL} images "
    f"but found {len(manifest_df)}."
)

assert (
    manifest_df["true_class"].nunique()
    == EXPECTED_N_CLASSES
), (
    f"\n Expected {EXPECTED_N_CLASSES} classes "
    f"but found {manifest_df['true_class'].nunique()}."
)

assert duplicated_ids == 0, (
    "\n Duplicate image_id values found."
)

# Class counts
counts = (
    manifest_df
    .set_index("image_id")["true_class"]
)

class_counts = (
    manifest_df["true_class"]
    .value_counts()
)

# Tomato Rotten must contain 27 images
assert (
    class_counts.get("Tomato__Rotten", 0)
    == 27
), (
    "\n Tomato__Rotten does not contain 27 images."
)

# All remaining classes must contain 30 images
other_counts = class_counts.drop("Tomato__Rotten")

assert (other_counts == 30).all(), (
    "\n At least one of the other "
    "23 classes does not contain 30 images."
)

# Save manifest
manifest_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print(" MANIFEST CREATED SUCCESSFULLY")
print("=" * 75)

print(" 717 images")
print(" 24 classes")
print(" 23 classes with 30 images")
print(" Tomato__Rotten with 27 images")
print(" Unique external IDs")

print("\nSaved to:")
print(OUTPUT_CSV)


# **Cell 2 - verify that the 717 images are legible and record their dimensions, formats, and channels.**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from PIL import Image, ImageOps, UnidentifiedImageError

# Check manifest availability

if "manifest_df" not in globals():

    MANIFEST_PATH = Path(
        "/kaggle/working/external_validation/"
        "external_validation_manifest.csv"
    )

    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Manifest not found at:\n{MANIFEST_PATH}"
        )

    manifest_df = pd.read_csv(MANIFEST_PATH)

print("=" * 80)
print("CELL 2 - IMAGE READABILITY AUDIT")
print("=" * 80)

print(f"Images listed in the manifest: {len(manifest_df):,}")

# Image inspection function

def inspect_image(image_path):

    image_path = Path(image_path)

    result = {
        "exists": False,
        "readable": False,
        "file_size_bytes": np.nan,
        "extension": image_path.suffix.lower(),
        "pil_format": None,
        "original_width": np.nan,
        "original_height": np.nan,
        "oriented_width": np.nan,
        "oriented_height": np.nan,
        "mode": None,
        "has_alpha": False,
        "aspect_ratio": np.nan,
        "error": None
    }

    # Check file existence

    if not image_path.exists():

        result["error"] = "file_not_found"
        return result

    result["exists"] = True

    try:
        result["file_size_bytes"] = image_path.stat().st_size
    except Exception:
        pass

    # First opening:
    # verify file structure

    try:

        with Image.open(image_path) as img:

            result["pil_format"] = img.format
            result["mode"] = img.mode

            result["original_width"] = img.width
            result["original_height"] = img.height

            result["has_alpha"] = (
                img.mode in ("RGBA", "LA")
                or (
                    img.mode == "P"
                    and "transparency" in img.info
                )
            )

            # verify() checks structural integrity
            img.verify()

    except (
        UnidentifiedImageError,
        OSError,
        ValueError,
        SyntaxError
    ) as e:

        result["error"] = (
            f"{type(e).__name__}: {str(e)}"
        )

        return result

    except Exception as e:

        result["error"] = (
            f"{type(e).__name__}: {str(e)}"
        )

        return result

    # Second opening:
    # fully load image pixels
    # and correct EXIF orientation

    try:

        with Image.open(image_path) as img:

            img = ImageOps.exif_transpose(img)

            # Force full pixel decoding
            img.load()

            result["oriented_width"] = img.width
            result["oriented_height"] = img.height

            if img.height > 0:

                result["aspect_ratio"] = (
                    img.width / img.height
                )

            # Check valid image dimensions
            if img.width <= 0 or img.height <= 0:

                result["error"] = "invalid_dimensions"

                return result

            result["readable"] = True

    except Exception as e:

        result["error"] = (
            f"{type(e).__name__}: {str(e)}"
        )

        return result

    return result

# Inspect all 717 images

audit_records = []

for idx, row in manifest_df.iterrows():

    result = inspect_image(row["path"])

    audit_records.append({
        "image_id": row["image_id"],
        "true_class": row["true_class"],
        "path": row["path"],
        **result
    })

readability_df = pd.DataFrame(audit_records)

# Overall summary

total_images = len(readability_df)

existing_images = int(
    readability_df["exists"].sum()
)

readable_images = int(
    readability_df["readable"].sum()
)

unreadable_images = int(
    (~readability_df["readable"]).sum()
)

print("\n" + "=" * 80)
print("READABILITY RESULTS")
print("=" * 80)

print(f"Total audited:             {total_images:,}")
print(f"Existing files:        {existing_images:,}")
print(f"Readable files:          {readable_images:,}")
print(f"Unreadable files:       {unreadable_images:,}")

# Image format distribution

print("\n" + "=" * 80)
print("IMAGE FORMATS")
print("=" * 80)

format_summary = (
    readability_df
    .groupby(
        ["extension", "pil_format"],
        dropna=False
    )
    .size()
    .reset_index(name="n_images")
    .sort_values(
        "n_images",
        ascending=False
    )
)

print(
    format_summary
    .to_string(index=False)
)

# Color mode distribution

print("\n" + "=" * 80)
print("COLOR MODES")
print("=" * 80)

mode_summary = (
    readability_df["mode"]
    .value_counts(
        dropna=False
    )
    .rename_axis("mode")
    .reset_index(name="n_images")
)

print(
    mode_summary
    .to_string(index=False)
)

# Image dimensions

valid_dimensions = (
    readability_df[
        readability_df["readable"]
    ]
)

print("\n" + "=" * 80)
print("IMAGE DIMENSIONS")
print("=" * 80)

print(
    f"Minimum width:  "
    f"{int(valid_dimensions['oriented_width'].min())} px"
)

print(
    f"Maximum width:  "
    f"{int(valid_dimensions['oriented_width'].max())} px"
)

print(
    f"Minimum height:   "
    f"{int(valid_dimensions['oriented_height'].min())} px"
)

print(
    f"Maximum height:   "
    f"{int(valid_dimensions['oriented_height'].max())} px"
)

# Transparency / alpha channel

alpha_images = int(
    readability_df["has_alpha"].sum()
)

print("\n" + "=" * 80)
print("TRANSPARENCY")
print("=" * 80)

print(
    f"Images with alpha/transparency: "
    f"{alpha_images}"
)

# Report any detected issues

problems_df = readability_df[
    ~readability_df["readable"]
].copy()

if len(problems_df) == 0:

    print("\n No unreadable images were detected.")

else:

    print(
        f"\n Detected "
        f"{len(problems_df)} images with issues:"
    )

    display(
        problems_df[
            [
                "image_id",
                "true_class",
                "path",
                "error"
            ]
        ]
    )

# Required checks

assert total_images == 717, (
    f" Expected 717 images, "
    f"but audited {total_images}."
)

assert existing_images == 717, (
    f" Some files listed in the manifest "
    f"do not exist on disk."
)

# Save audit results

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUDIT_PATH = (
    OUTPUT_DIR /
    "external_validation_readability_audit.csv"
)

readability_df.to_csv(
    AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Create enriched manifest

columns_to_add = [
    "image_id",
    "readable",
    "file_size_bytes",
    "extension",
    "pil_format",
    "original_width",
    "original_height",
    "oriented_width",
    "oriented_height",
    "mode",
    "has_alpha",
    "aspect_ratio",
    "error"
]

manifest_step2_df = manifest_df.merge(
    readability_df[columns_to_add],
    on="image_id",
    how="left",
    validate="one_to_one"
)

MANIFEST_STEP2_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_step2.csv"
)

manifest_step2_df.to_csv(
    MANIFEST_STEP2_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Final results

print("\n" + "=" * 80)

if unreadable_images == 0:

    print(" CELL 2 COMPLETED SUCCESSFULLY")
    print(" All 717 images exist.")
    print(" All 717 images are readable.")

else:

    print(" CELL 2 COMPLETED WITH ISSUES")
    print(
        f" {unreadable_images} images "
        f"require review."
    )

print("=" * 80)

print("\nAudit saved to:")
print(AUDIT_PATH)

print("\nEnriched manifest saved to:")
print(MANIFEST_STEP2_PATH)

# **Cell 3 - calculate SHA-256 and check for exact duplicates within the external set.**

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

# Load the Cell 2 manifest if needed

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

STEP2_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_step2.csv"
)

if "manifest_step2_df" not in globals():

    if not STEP2_PATH.exists():
        raise FileNotFoundError(
            f"Not found:\n{STEP2_PATH}"
        )

    manifest_step2_df = pd.read_csv(
        STEP2_PATH
    )

print("=" * 85)
print("CELL 3 - SHA-256 AND INTERNAL EXACT DUPLICATES")
print("=" * 85)

print(
    f"Images received: "
    f"{len(manifest_step2_df):,}"
)

# Preliminary checks

assert len(manifest_step2_df) == 717, (
    " The manifest does not contain 717 images."
)

assert manifest_step2_df["image_id"].is_unique, (
    " image_id is not unique."
)

assert manifest_step2_df["path"].is_unique, (
    " Duplicate file paths were found."
)

assert manifest_step2_df["readable"].all(), (
    " Some images are marked as unreadable."
)

# SHA-256 helper

def compute_sha256(
    file_path,
    chunk_size=1024 * 1024
):

    sha = hashlib.sha256()

    with open(file_path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha.update(chunk)

    return sha.hexdigest()

# Compute SHA-256 for all images

sha_records = []

for i, row in manifest_step2_df.iterrows():

    image_path = Path(row["path"])

    try:

        sha256 = compute_sha256(
            image_path
        )

        error = None

    except Exception as e:

        sha256 = None

        error = (
            f"{type(e).__name__}: {str(e)}"
        )

    sha_records.append({
        "image_id": row["image_id"],
        "sha256": sha256,
        "sha256_error": error
    })

sha_df = pd.DataFrame(
    sha_records
)

# Merge SHA-256 hashes into the manifest

manifest_step3_df = (
    manifest_step2_df
    .merge(
        sha_df,
        on="image_id",
        how="left",
        validate="one_to_one"
    )
)

# Check for hashing errors

hash_errors = (
    manifest_step3_df[
        manifest_step3_df[
            "sha256"
        ].isna()
    ]
)

print("\n" + "=" * 85)
print("HASH RESULTS")
print("=" * 85)

print(
    f"Hashes computed successfully: "
    f"{manifest_step3_df['sha256'].notna().sum():,}"
)

print(
    f"SHA-256 errors: "
    f"{len(hash_errors):,}"
)

if len(hash_errors) > 0:

    display(
        hash_errors[
            [
                "image_id",
                "true_class",
                "path",
                "sha256_error"
            ]
        ]
    )

assert len(hash_errors) == 0, (
    " Errors occurred during SHA-256 computation."
)

# Identify exact-duplicate groups

hash_counts = (
    manifest_step3_df[
        "sha256"
    ]
    .value_counts()
)

duplicate_hashes = (
    hash_counts[
        hash_counts > 1
    ]
    .index
)

exact_duplicates_df = (
    manifest_step3_df[
        manifest_step3_df[
            "sha256"
        ].isin(
            duplicate_hashes
        )
    ]
    .copy()
)

# Overall statistics

total_images = len(
    manifest_step3_df
)

unique_sha256 = (
    manifest_step3_df[
        "sha256"
    ]
    .nunique()
)

n_duplicate_groups = len(
    duplicate_hashes
)

files_in_duplicate_groups = len(
    exact_duplicates_df
)

redundant_exact_copies = (
    total_images -
    unique_sha256
)

print("\n" + "=" * 85)
print("INTERNAL EXACT DUPLICATES")
print("=" * 85)

print(
    f"Total images:                     "
    f"{total_images:,}"
)

print(
    f"Unique SHA-256 hashes:            "
    f"{unique_sha256:,}"
)

print(
    f"Exact-duplicate groups:          "
    f"{n_duplicate_groups:,}"
)

print(
    f"Files in duplicate groups:       "
    f"{files_in_duplicate_groups:,}"
)

print(
    f"Redundant exact copies:          "
    f"{redundant_exact_copies:,}"
)

# Analyze each duplicate group

duplicate_group_records = []

if n_duplicate_groups > 0:

    grouped = (
        exact_duplicates_df
        .groupby("sha256")
    )

    for group_id, (
        sha256,
        group
    ) in enumerate(
        grouped,
        start=1
    ):

        classes = sorted(
            group[
                "true_class"
            ]
            .dropna()
            .unique()
            .tolist()
        )

        fruits = sorted(
            group[
                "fruit"
            ]
            .dropna()
            .unique()
            .tolist()
        )

        conditions = sorted(
            group[
                "condition"
            ]
            .dropna()
            .unique()
            .tolist()
        )

        cross_class = (
            len(classes) > 1
        )

        cross_fruit = (
            len(fruits) > 1
        )

        cross_condition = (
            len(conditions) > 1
        )

        for _, row in group.iterrows():

            duplicate_group_records.append({

                "exact_group_id":
                    f"EXACT_{group_id:04d}",

                "sha256":
                    sha256,

                "group_size":
                    len(group),

                "image_id":
                    row["image_id"],

                "true_class":
                    row["true_class"],

                "fruit":
                    row["fruit"],

                "condition":
                    row["condition"],

                "filename":
                    row["filename"],

                "path":
                    row["path"],

                "cross_class":
                    cross_class,

                "cross_fruit":
                    cross_fruit,

                "cross_condition":
                    cross_condition,

                "classes_in_group":
                    " | ".join(classes)
            })

exact_duplicate_groups_df = pd.DataFrame(
    duplicate_group_records
)

# Summary of duplicate-group types

if n_duplicate_groups > 0:

    group_level_summary = (
        exact_duplicate_groups_df[
            [
                "exact_group_id",
                "cross_class",
                "cross_fruit",
                "cross_condition"
            ]
        ]
        .drop_duplicates(
            "exact_group_id"
        )
    )

    same_class_groups = int(
        (
            ~group_level_summary[
                "cross_class"
            ]
        ).sum()
    )

    cross_class_groups = int(
        group_level_summary[
            "cross_class"
        ].sum()
    )

    cross_fruit_groups = int(
        group_level_summary[
            "cross_fruit"
        ].sum()
    )

    cross_condition_groups = int(
        group_level_summary[
            "cross_condition"
        ].sum()
    )

else:

    same_class_groups = 0
    cross_class_groups = 0
    cross_fruit_groups = 0
    cross_condition_groups = 0

print("\n" + "=" * 85)
print("DUPLICATE GROUP TYPES")
print("=" * 85)

print(
    f"Same class:                      "
    f"{same_class_groups}"
)

print(
    f"Across different classes:        "
    f"{cross_class_groups}"
)

print(
    f"Across different fruits:         "
    f"{cross_fruit_groups}"
)

print(
    f"Healthy ↔ Rotten:                    "
    f"{cross_condition_groups}"
)

# Display duplicates if present

if n_duplicate_groups == 0:

    print(
        "\n No internal exact duplicates were "
        "detected."
    )

else:

    print(
        "\n Exact-duplicate groups were "
        "detected."
    )

    display(
        exact_duplicate_groups_df[
            [
                "exact_group_id",
                "image_id",
                "true_class",
                "group_size",
                "cross_class",
                "cross_condition",
                "classes_in_group"
            ]
        ]
    )

# Save results

STEP3_MANIFEST_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_step3.csv"
)

EXACT_GROUPS_PATH = (
    OUTPUT_DIR /
    "external_validation_exact_duplicate_groups.csv"
)

SUMMARY_PATH = (
    OUTPUT_DIR /
    "external_validation_exact_duplicate_summary.csv"
)

manifest_step3_df.to_csv(
    STEP3_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Save the file even when no duplicates are found
if len(exact_duplicate_groups_df) > 0:

    exact_duplicate_groups_df.to_csv(
        EXACT_GROUPS_PATH,
        index=False,
        encoding="utf-8-sig"
    )

else:

    pd.DataFrame(
        columns=[
            "exact_group_id",
            "sha256",
            "group_size",
            "image_id",
            "true_class",
            "fruit",
            "condition",
            "filename",
            "path",
            "cross_class",
            "cross_fruit",
            "cross_condition",
            "classes_in_group"
        ]
    ).to_csv(
        EXACT_GROUPS_PATH,
        index=False,
        encoding="utf-8-sig"
    )

summary_df = pd.DataFrame([
    {
        "total_images":
            total_images,

        "unique_sha256":
            unique_sha256,

        "exact_duplicate_groups":
            n_duplicate_groups,

        "files_in_exact_duplicate_groups":
            files_in_duplicate_groups,

        "redundant_exact_copies":
            redundant_exact_copies,

        "same_class_groups":
            same_class_groups,

        "cross_class_groups":
            cross_class_groups,

        "cross_fruit_groups":
            cross_fruit_groups,

        "cross_condition_groups":
            cross_condition_groups
    }
])

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Final results

print("\n" + "=" * 85)
print("CELL 3 COMPLETED")
print("=" * 85)

if n_duplicate_groups == 0:

    print(
        " All 717 images have "
        "unique SHA-256 hashes."
    )

    print(
        " No internal exact duplicates were "
        "detected."
    )

else:

    print(
        f" Detected "
        f"{n_duplicate_groups} groups "
        f"of exact duplicates."
    )

    print(
        " No images have been removed "
        "yet."
    )

print("\nManifest Step 3:")
print(STEP3_MANIFEST_PATH)

print("\nDuplicate groups:")
print(EXACT_GROUPS_PATH)

print("\nSummary:")
print(SUMMARY_PATH)

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("/kaggle/working/external_validation")
STEP3_PATH = OUTPUT_DIR / "external_validation_manifest_step3.csv"

# Load the Step 3 manifest
if "manifest_step3_df" not in globals():
    if not STEP3_PATH.exists():
        raise FileNotFoundError(f"File not found:\n{STEP3_PATH}")
    manifest_step3_df = pd.read_csv(STEP3_PATH)

print("=" * 85)
print("CELL 3B - EXACT DUPLICATE REMOVAL")
print("=" * 85)
print(f"Images before deduplication: {len(manifest_step3_df):,}")

# Preliminary checks
assert len(manifest_step3_df) == 717, (
    " Expected 717 images before deduplication."
)
assert manifest_step3_df["sha256"].notna().all(), (
    " Some images have no SHA-256 hash."
)
assert manifest_step3_df["image_id"].is_unique, (
    " image_id is not unique."
)

# Determine the size of each SHA-256 group
manifest_step3_df = manifest_step3_df.copy()
manifest_step3_df["exact_group_size"] = (
    manifest_step3_df.groupby("sha256")["sha256"].transform("size")
)
manifest_step3_df["is_exact_duplicate"] = (
    manifest_step3_df["exact_group_size"] > 1
)

# Select a canonical representative using a stable image_id ordering
manifest_step3_df = (
    manifest_step3_df
    .sort_values(["sha256", "image_id"])
    .reset_index(drop=True)
)
manifest_step3_df["exact_keep"] = ~manifest_step3_df["sha256"].duplicated(
    keep="first"
)
manifest_step3_df["exact_action"] = "keep"
manifest_step3_df.loc[
    ~manifest_step3_df["exact_keep"], "exact_action"
] = "remove_exact_redundant"

# Build the cleaned manifest and the excluded-copy log
external_exact_clean_df = (
    manifest_step3_df[manifest_step3_df["exact_keep"]]
    .copy()
    .reset_index(drop=True)
)
removed_exact_df = (
    manifest_step3_df[~manifest_step3_df["exact_keep"]]
    .copy()
    .reset_index(drop=True)
)

# Deduplication summary
print("\n" + "=" * 85)
print("RESULTS")
print("=" * 85)
print(f"Original images:                  {len(manifest_step3_df):,}")
print(f"Unique images retained:          {len(external_exact_clean_df):,}")
print(f"Exact copies excluded:           {len(removed_exact_df):,}")
print(f"Final unique SHA-256 hashes:     {external_exact_clean_df['sha256'].nunique():,}")
print(f"Final classes:                   {external_exact_clean_df['true_class'].nunique()}")

# Final class distribution
class_counts_clean = (
    external_exact_clean_df
    .groupby("true_class", as_index=False)
    .size()
    .rename(columns={"size": "n_unique_images"})
    .sort_values("true_class")
    .reset_index(drop=True)
)
print("\nDistribution of unique images by class:\n")
display(class_counts_clean)

# Inspect classes affected by deduplication
counts_dict = dict(zip(
    class_counts_clean["true_class"],
    class_counts_clean["n_unique_images"]
))
print("\nClasses affected by deduplication:")
for cls in [
    "Carrot__Rotten",
    "Orange__Rotten",
    "Pomegranate__Rotten",
    "Tomato__Rotten"
]:
    print(f"{cls:25s}: {counts_dict.get(cls, 0)}")

# Validate the cleaned dataset
assert len(external_exact_clean_df) == 706, (
    " Expected 706 images after deduplication."
)
assert external_exact_clean_df["sha256"].nunique() == 706, (
    " Duplicate SHA-256 hashes remain."
)
assert external_exact_clean_df["true_class"].nunique() == 24, (
    " Not all 24 classes are represented."
)
assert external_exact_clean_df["image_id"].is_unique, (
    " Duplicate image_id values remain."
)

# Save the cleaned manifest and excluded exact duplicates
CLEAN_MANIFEST_PATH = (
    OUTPUT_DIR / "external_validation_manifest_exact_clean.csv"
)
REMOVED_PATH = (
    OUTPUT_DIR / "external_validation_removed_exact_duplicates.csv"
)
external_exact_clean_df.to_csv(
    CLEAN_MANIFEST_PATH, index=False, encoding="utf-8-sig"
)
removed_exact_df.to_csv(
    REMOVED_PATH, index=False, encoding="utf-8-sig"
)

# Final report
print("\n" + "=" * 85)
print(" CELL 3B COMPLETED SUCCESSFULLY")
print("=" * 85)
print(" 706 unique external images.")
print(" All 24 classes remain represented.")
print(" 11 exact copies excluded.")
print(" No repeated SHA-256 hashes in the cleaned manifest.")
print("\nCleaned manifest:")
print(CLEAN_MANIFEST_PATH)
print("\nExcluded exact duplicates:")
print(REMOVED_PATH)


# **Cell 4 - calculate pHash/dHash/aHash to detect highly similar images within the external dataset.**

In [ ]:
from pathlib import Path
from itertools import combinations

import pandas as pd
import numpy as np

from PIL import Image, ImageOps

# Load the exact-deduplicated manifest

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

EXACT_CLEAN_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_exact_clean.csv"
)

if "external_exact_clean_df" not in globals():

    if not EXACT_CLEAN_PATH.exists():
        raise FileNotFoundError(
            f"Not found:\n{EXACT_CLEAN_PATH}"
        )

    external_exact_clean_df = pd.read_csv(
        EXACT_CLEAN_PATH
    )


print("=" * 90)
print("CELL 4 - INTERNAL PERCEPTUAL AUDIT")
print("=" * 90)

print(
    f"Images received: "
    f"{len(external_exact_clean_df):,}"
)

assert len(external_exact_clean_df) == 706
assert external_exact_clean_df["sha256"].nunique() == 706
assert external_exact_clean_df["true_class"].nunique() == 24


# Perceptual hash implementations
# Without dedicated perceptual hashing libraries

def average_hash(img, hash_size=8):
    """64-bit aHash."""
    img = img.convert("L")
    img = img.resize(
        (hash_size, hash_size),
        Image.Resampling.LANCZOS
    )

    pixels = np.asarray(
        img,
        dtype=np.float32
    )

    avg = pixels.mean()
    bits = pixels > avg

    return bits.flatten()


def difference_hash(img, hash_size=8):
    """64-bit dHash."""
    img = img.convert("L")
    img = img.resize(
        (hash_size + 1, hash_size),
        Image.Resampling.LANCZOS
    )

    pixels = np.asarray(
        img,
        dtype=np.float32
    )

    bits = pixels[:, 1:] > pixels[:, :-1]

    return bits.flatten()


def perceptual_hash(img, hash_size=8, highfreq_factor=4):
    """64-bit pHash computed using the DCT."""

    import cv2

    size = hash_size * highfreq_factor

    img = img.convert("L")
    img = img.resize(
        (size, size),
        Image.Resampling.LANCZOS
    )

    pixels = np.asarray(
        img,
        dtype=np.float32
    )

    dct = cv2.dct(pixels)

    low_freq = dct[
        :hash_size,
        :hash_size
    ]

    # Exclude the DC coefficient when computing the median
    median = np.median(
        low_freq.flatten()[1:]
    )

    bits = low_freq > median

    return bits.flatten()


def bits_to_hex(bits):
    """Convert a 64-bit Boolean array to hexadecimal."""

    bit_string = "".join(
        "1" if b else "0"
        for b in bits
    )

    value = int(bit_string, 2)

    width = len(bits) // 4

    return f"{value:0{width}x}"


def hamming_distance(bits_a, bits_b):
    """Compute the Hamming distance between two hashes."""

    return int(
        np.count_nonzero(
            bits_a != bits_b
        )
    )


# Compute perceptual hashes

hash_records = []

bit_store = {}

for _, row in external_exact_clean_df.iterrows():

    image_id = row["image_id"]
    image_path = Path(row["path"])

    try:

        with Image.open(image_path) as img:

            img = ImageOps.exif_transpose(img)
            img.load()

            # Convert image for hashing
            rgb = img.convert("RGB")

            phash_bits = perceptual_hash(rgb)
            dhash_bits = difference_hash(rgb)
            ahash_bits = average_hash(rgb)

            bit_store[image_id] = {
                "phash": phash_bits,
                "dhash": dhash_bits,
                "ahash": ahash_bits,
            }

            hash_records.append({
                "image_id": image_id,
                "phash64": bits_to_hex(phash_bits),
                "dhash64": bits_to_hex(dhash_bits),
                "ahash64": bits_to_hex(ahash_bits),
                "perceptual_hash_error": None
            })

    except Exception as e:

        hash_records.append({
            "image_id": image_id,
            "phash64": None,
            "dhash64": None,
            "ahash64": None,
            "perceptual_hash_error":
                f"{type(e).__name__}: {str(e)}"
        })


hash_df = pd.DataFrame(hash_records)


# Merge hashes into the manifest

manifest_step4_df = (
    external_exact_clean_df
    .merge(
        hash_df,
        on="image_id",
        how="left",
        validate="one_to_one"
    )
)


# Check hashing errors

hash_errors = manifest_step4_df[
    manifest_step4_df[
        "perceptual_hash_error"
    ].notna()
].copy()


print("\n" + "=" * 90)
print("PERCEPTUAL HASHING RESULTS")
print("=" * 90)

print(
    f"Valid hashes: "
    f"{len(manifest_step4_df) - len(hash_errors):,}"
)

print(
    f"Errors: "
    f"{len(hash_errors):,}"
)


if len(hash_errors) > 0:

    display(
        hash_errors[
            [
                "image_id",
                "true_class",
                "path",
                "perceptual_hash_error"
            ]
        ]
    )


assert len(hash_errors) == 0, (
    " Errors occurred during perceptual hash computation."
)


# Summarize identical perceptual hashes

unique_phash = manifest_step4_df["phash64"].nunique()
unique_dhash = manifest_step4_df["dhash64"].nunique()
unique_ahash = manifest_step4_df["ahash64"].nunique()


phash_counts = (
    manifest_step4_df["phash64"]
    .value_counts()
)

dhash_counts = (
    manifest_step4_df["dhash64"]
    .value_counts()
)

ahash_counts = (
    manifest_step4_df["ahash64"]
    .value_counts()
)


repeated_phash = phash_counts[
    phash_counts > 1
]

repeated_dhash = dhash_counts[
    dhash_counts > 1
]

repeated_ahash = ahash_counts[
    ahash_counts > 1
]


print("\n" + "=" * 90)
print("IDENTICAL PERCEPTUAL HASHES")
print("=" * 90)

print(
    f"Unique pHashes:                     "
    f"{unique_phash:,}"
)

print(
    f"Repeated pHash groups:              "
    f"{len(repeated_phash):,}"
)

print(
    f"Unique dHashes:                     "
    f"{unique_dhash:,}"
)

print(
    f"Repeated dHash groups:              "
    f"{len(repeated_dhash):,}"
)

print(
    f"Unique aHashes:                     "
    f"{unique_ahash:,}"
)

print(
    f"Repeated aHash groups:              "
    f"{len(repeated_ahash):,}"
)


# Pairwise comparison
# 706 images → 248,865 pairs
# Computationally manageable

metadata = (
    manifest_step4_df
    .set_index("image_id")
    .to_dict("index")
)

image_ids = (
    manifest_step4_df[
        "image_id"
    ].tolist()
)


candidate_records = []

total_pairs = 0


for id_a, id_b in combinations(image_ids, 2):

    total_pairs += 1

    hashes_a = bit_store[id_a]
    hashes_b = bit_store[id_b]

    phash_dist = hamming_distance(
        hashes_a["phash"],
        hashes_b["phash"]
    )

    # Compute the other two hashes only
    # when pHash indicates sufficient similarity.
    # Broad threshold: <= 4
    # Does NOT imply an automatic duplicate decision.
    if phash_dist <= 4:

        dhash_dist = hamming_distance(
            hashes_a["dhash"],
            hashes_b["dhash"]
        )

        ahash_dist = hamming_distance(
            hashes_a["ahash"],
            hashes_b["ahash"]
        )

        row_a = metadata[id_a]
        row_b = metadata[id_b]

        same_class = (
            row_a["true_class"]
            == row_b["true_class"]
        )

        same_fruit = (
            row_a["fruit"]
            == row_b["fruit"]
        )

        same_condition = (
            row_a["condition"]
            == row_b["condition"]
        )

        # Review priority
        # Important:
        # This rule only prioritizes candidates;
        # it does NOT confirm duplication.

        if (
            phash_dist == 0
            and dhash_dist <= 2
            and ahash_dist <= 2
        ):
            review_priority = "very_high"

        elif (
            phash_dist <= 2
            and dhash_dist <= 4
        ):
            review_priority = "high"

        elif phash_dist <= 4:
            review_priority = "review"

        else:
            review_priority = "low"


        candidate_records.append({

            "image_id_a": id_a,
            "image_id_b": id_b,

            "class_a": row_a["true_class"],
            "class_b": row_b["true_class"],

            "fruit_a": row_a["fruit"],
            "fruit_b": row_b["fruit"],

            "condition_a": row_a["condition"],
            "condition_b": row_b["condition"],

            "same_class": same_class,
            "same_fruit": same_fruit,
            "same_condition": same_condition,

            "phash_distance": phash_dist,
            "dhash_distance": dhash_dist,
            "ahash_distance": ahash_dist,

            "review_priority": review_priority,

            "path_a": row_a["path"],
            "path_b": row_b["path"]
        })


candidate_pairs_df = pd.DataFrame(
    candidate_records
)


# Sort candidate pairs

if len(candidate_pairs_df) > 0:

    priority_order = {
        "very_high": 0,
        "high": 1,
        "review": 2
    }

    candidate_pairs_df[
        "_priority_order"
    ] = candidate_pairs_df[
        "review_priority"
    ].map(priority_order)

    candidate_pairs_df = (
        candidate_pairs_df
        .sort_values(
            [
                "_priority_order",
                "phash_distance",
                "dhash_distance",
                "ahash_distance"
            ]
        )
        .drop(
            columns="_priority_order"
        )
        .reset_index(drop=True)
    )


# Summarize candidate pairs

n_candidates = len(
    candidate_pairs_df
)

if n_candidates > 0:

    n_same_class = int(
        candidate_pairs_df[
            "same_class"
        ].sum()
    )

    n_cross_class = int(
        (~candidate_pairs_df[
            "same_class"
        ]).sum()
    )

    n_cross_condition = int(
        (~candidate_pairs_df[
            "same_condition"
        ]).sum()
    )

    n_cross_fruit = int(
        (~candidate_pairs_df[
            "same_fruit"
        ]).sum()
    )

else:

    n_same_class = 0
    n_cross_class = 0
    n_cross_condition = 0
    n_cross_fruit = 0


print("\n" + "=" * 90)
print("PERCEPTUALLY SIMILAR PAIRS")
print("=" * 90)

print(
    f"Pairwise comparisons:               "
    f"{total_pairs:,}"
)

print(
    f"Pairs with pHash distance <= 4:     "
    f"{n_candidates:,}"
)

print(
    f"Within-class pairs:                 "
    f"{n_same_class:,}"
)

print(
    f"Cross-class pairs:                  "
    f"{n_cross_class:,}"
)

print(
    f"Cross-fruit pairs:                  "
    f"{n_cross_fruit:,}"
)

print(
    f"Healthy ↔ Rotten pairs:             "
    f"{n_cross_condition:,}"
)


# Review priority distribution

if n_candidates > 0:

    print("\nReview priority:\n")

    priority_summary = (
        candidate_pairs_df[
            "review_priority"
        ]
        .value_counts()
        .rename_axis(
            "review_priority"
        )
        .reset_index(
            name="n_pairs"
        )
    )

    display(
        priority_summary
    )


# Show the highest-priority candidate pairs

if n_candidates == 0:

    print(
        "\n No pairs were found "
        "with pHash distance <= 4."
    )

else:

    print(
        "\nTop candidates for review:\n"
    )

    display(
        candidate_pairs_df[
            [
                "image_id_a",
                "image_id_b",
                "class_a",
                "class_b",
                "phash_distance",
                "dhash_distance",
                "ahash_distance",
                "review_priority",
                "same_class"
            ]
        ].head(50)
    )


# Save outputs

MANIFEST_STEP4_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_step4_hashes.csv"
)

CANDIDATES_PATH = (
    OUTPUT_DIR /
    "external_validation_perceptual_candidate_pairs.csv"
)

SUMMARY_PATH = (
    OUTPUT_DIR /
    "external_validation_perceptual_summary.csv"
)


manifest_step4_df.to_csv(
    MANIFEST_STEP4_PATH,
    index=False,
    encoding="utf-8-sig"
)


candidate_pairs_df.to_csv(
    CANDIDATES_PATH,
    index=False,
    encoding="utf-8-sig"
)


summary_df = pd.DataFrame([
    {
        "total_images": len(manifest_step4_df),

        "unique_phash64": unique_phash,
        "unique_dhash64": unique_dhash,
        "unique_ahash64": unique_ahash,

        "repeated_phash_groups":
            len(repeated_phash),

        "repeated_dhash_groups":
            len(repeated_dhash),

        "repeated_ahash_groups":
            len(repeated_ahash),

        "pairwise_comparisons":
            total_pairs,

        "phash_distance_le_4_pairs":
            n_candidates,

        "same_class_candidate_pairs":
            n_same_class,

        "cross_class_candidate_pairs":
            n_cross_class,

        "cross_fruit_candidate_pairs":
            n_cross_fruit,

        "cross_condition_candidate_pairs":
            n_cross_condition
    }
])


summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)


# Final summary

print("\n" + "=" * 90)
print(" CELL 4 COMPLETED")
print("=" * 90)

print(
    " Perceptual hashes were "
    "computed for all 706 images."
)

print(
    " No images were removed "
    "automatically."
)

print(
    " Similar candidate pairs were "
    "recorded for review."
)

print("\nManifest with hashes:")
print(MANIFEST_STEP4_PATH)

print("\nCandidate pairs:")
print(CANDIDATES_PATH)

print("\nSummary:")
print(SUMMARY_PATH)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image, ImageOps

# Paths
OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

CANDIDATES_PATH = (
    OUTPUT_DIR /
    "external_validation_perceptual_candidate_pairs.csv"
)

REVIEW_DIR = (
    OUTPUT_DIR /
    "perceptual_review_sheets"
)

REVIEW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Load candidate pairs
if "candidate_pairs_df" not in globals():

    if not CANDIDATES_PATH.exists():
        raise FileNotFoundError(
            f"Not found:\n{CANDIDATES_PATH}"
        )

    candidate_pairs_df = pd.read_csv(
        CANDIDATES_PATH
    )

print("=" * 90)
print("CELL 4B - VISUAL REVIEW PREPARATION")
print("=" * 90)

print(
    f"Candidate pairs: "
    f"{len(candidate_pairs_df):,}"
)

# Assign review IDs
review_df = candidate_pairs_df.copy()

review_df.insert(
    0,
    "review_id",
    [
        f"PERCEPT_{i:03d}"
        for i in range(
            1,
            len(review_df) + 1
        )
    ]
)

# Add columns for manual decisions
review_df["manual_decision"] = ""
review_df["review_notes"] = ""

# Permitted subsequent decision values:
# confirmed_duplicate, distinct_images, uncertain

# Identify special cases
review_df["special_case"] = (
    (~review_df["same_class"])
    | (~review_df["same_condition"])
    | (~review_df["same_fruit"])
)

special_df = (
    review_df[
        review_df["special_case"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("\n" + "=" * 90)
print("CROSS-CLASS / CROSS-CONDITION / CROSS-FRUIT CASES")
print("=" * 90)

print(
    f"Special cases: "
    f"{len(special_df)}"
)

if len(special_df) > 0:

    display(
        special_df[
            [
                "review_id",
                "image_id_a",
                "image_id_b",
                "class_a",
                "class_b",
                "phash_distance",
                "dhash_distance",
                "ahash_distance",
                "review_priority",
                "same_class",
                "same_fruit",
                "same_condition"
            ]
        ]
    )

# Load and orient an image for visual review
def load_review_image(path):

    with Image.open(path) as img:

        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")

        return img.copy()

# Generate a side-by-side contact sheet
def create_contact_sheet(
    subset_df,
    output_path,
    title,
    pairs_per_page=8
):

    if len(subset_df) == 0:
        return

    n_pairs = len(subset_df)

    # Each pair occupies one row, with images A and B.
    fig_height = max(
        3.2 * n_pairs,
        4
    )

    fig, axes = plt.subplots(
        n_pairs,
        2,
        figsize=(12, fig_height)
    )

    if n_pairs == 1:
        axes = np.array(
            [axes]
        )

    for row_idx, (_, row) in enumerate(
        subset_df.iterrows()
    ):

        try:
            img_a = load_review_image(
                row["path_a"]
            )

            axes[row_idx, 0].imshow(
                img_a
            )

        except Exception as e:

            axes[row_idx, 0].text(
                0.5,
                0.5,
                f"ERROR\n{e}",
                ha="center",
                va="center"
            )

        try:
            img_b = load_review_image(
                row["path_b"]
            )

            axes[row_idx, 1].imshow(
                img_b
            )

        except Exception as e:

            axes[row_idx, 1].text(
                0.5,
                0.5,
                f"ERROR\n{e}",
                ha="center",
                va="center"
            )

        title_a = (
            f"{row['review_id']} | A\n"
            f"{row['image_id_a']}\n"
            f"{row['class_a']}"
        )

        title_b = (
            f"{row['review_id']} | B\n"
            f"{row['image_id_b']}\n"
            f"{row['class_b']}"
        )

        axes[row_idx, 0].set_title(
            title_a,
            fontsize=9
        )

        axes[row_idx, 1].set_title(
            title_b,
            fontsize=9
        )

        axes[row_idx, 0].axis(
            "off"
        )

        axes[row_idx, 1].axis(
            "off"
        )

        # Hash distances displayed below each pair
        distance_text = (
            f"pHash={row['phash_distance']} | "
            f"dHash={row['dhash_distance']} | "
            f"aHash={row['ahash_distance']} | "
            f"{row['review_priority']}"
        )

        axes[row_idx, 0].text(
            0.5,
            -0.08,
            distance_text,
            transform=axes[row_idx, 0].transAxes,
            ha="center",
            va="top",
            fontsize=8
        )

    fig.suptitle(
        title,
        fontsize=14,
        y=1.0
    )

    plt.tight_layout()

    fig.savefig(
        output_path,
        dpi=160,
        bbox_inches="tight"
    )

    plt.close(fig)

# Generate review sheets in batches
PAIRS_PER_SHEET = 8

n_sheets = int(
    np.ceil(
        len(review_df)
        / PAIRS_PER_SHEET
    )
)

for sheet_idx in range(n_sheets):

    start = (
        sheet_idx *
        PAIRS_PER_SHEET
    )

    end = min(
        start + PAIRS_PER_SHEET,
        len(review_df)
    )

    subset = (
        review_df
        .iloc[start:end]
        .copy()
    )

    output_path = (
        REVIEW_DIR /
        f"perceptual_review_"
        f"{sheet_idx + 1:02d}.png"
    )

    create_contact_sheet(
        subset,
        output_path,
        title=(
            f"Perceptual duplicate review "
            f"{sheet_idx + 1}/{n_sheets}"
        )
    )

# Create a separate cross-class / cross-condition review sheet
SPECIAL_SHEET_PATH = (
    REVIEW_DIR /
    "perceptual_special_cross_class.png"
)

if len(special_df) > 0:

    create_contact_sheet(
        special_df,
        SPECIAL_SHEET_PATH,
        title=(
            "Special review: "
            "cross-class / cross-condition pairs"
        )
    )

# Save manual review queue
REVIEW_CSV_PATH = (
    OUTPUT_DIR /
    "external_validation_perceptual_review_queue.csv"
)

review_df.to_csv(
    REVIEW_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Summary
print("\n" + "=" * 90)
print("REVIEW QUEUE")
print("=" * 90)

print(
    f"Very high: "
    f"{(review_df['review_priority'] == 'very_high').sum()}"
)

print(
    f"High:      "
    f"{(review_df['review_priority'] == 'high').sum()}"
)

print(
    f"Review:    "
    f"{(review_df['review_priority'] == 'review').sum()}"
)

print(
    f"Special cases:"
    f" {review_df['special_case'].sum()}"
)

print(
    f"Sheets generated: "
    f"{n_sheets}"
)

print("\n" + "=" * 90)
print(" CELL 4B COMPLETED")
print("=" * 90)

print("\nReview queue:")
print(REVIEW_CSV_PATH)

print("\nVisual review sheets:")
print(REVIEW_DIR)

if len(special_df) > 0:
    print("\nSpecial-case sheet:")
    print(SPECIAL_SHEET_PATH)


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

# Select special cases
special_review_df = review_df[
    review_df["special_case"]
].copy().reset_index(drop=True)

print("=" * 90)
print("CELL 4C - DETAILED REVIEW OF SPECIAL CASES")
print("=" * 90)

print(
    f"Special cases to review: "
    f"{len(special_review_df)}"
)

# Image loading

def load_rgb_image(path):

    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")
        return img.copy()

# Display each pair individually
for _, row in special_review_df.iterrows():

    img_a = load_rgb_image(
        row["path_a"]
    )

    img_b = load_rgb_image(
        row["path_b"]
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 6)
    )

    # Image A
    axes[0].imshow(img_a)
    axes[0].set_title(
        f"A: {row['image_id_a']}\n"
        f"{row['class_a']}",
        fontsize=12
    )
    axes[0].axis("off")

    # Image B
    axes[1].imshow(img_b)
    axes[1].set_title(
        f"B: {row['image_id_b']}\n"
        f"{row['class_b']}",
        fontsize=12
    )
    axes[1].axis("off")

    fig.suptitle(
        f"{row['review_id']} | "
        f"pHash={row['phash_distance']} | "
        f"dHash={row['dhash_distance']} | "
        f"aHash={row['ahash_distance']}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

    print(
        f"{row['review_id']}: "
        f"{row['image_id_a']} ↔ "
        f"{row['image_id_b']}"
    )

    print(
        f"Classes: "
        f"{row['class_a']} ↔ "
        f"{row['class_b']}"
    )

    print("-" * 90)


In [ ]:
SPECIAL_REVIEW_IDS = [
    "PERCEPT_068",
    "PERCEPT_073",
    "PERCEPT_074",
    "PERCEPT_075",
]

review_df.loc[
    review_df["review_id"].isin(SPECIAL_REVIEW_IDS),
    "manual_decision"
] = "distinct_images"

review_df.loc[
    review_df["review_id"].isin(SPECIAL_REVIEW_IDS),
    "review_notes"
] = (
    "Manual visual review confirmed that the pair contains "
    "different photographs; perceptual similarity does not "
    "correspond to duplication."
)

# Verify the recorded decisions
display(
    review_df.loc[
        review_df["review_id"].isin(SPECIAL_REVIEW_IDS),
        [
            "review_id",
            "image_id_a",
            "image_id_b",
            "class_a",
            "class_b",
            "manual_decision",
            "review_notes"
        ]
    ]
)

# Save the updated review queue
REVIEW_CSV_PATH = (
    OUTPUT_DIR /
    "external_validation_perceptual_review_queue.csv"
)

review_df.to_csv(
    REVIEW_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(" The 4 special cases were recorded as distinct_images.")
print(" No images were removed.")


In [ ]:

very_high_df = (
    review_df[
        review_df["review_priority"] == "very_high"
    ]
    .copy()
    .reset_index(drop=True)
)

print("=" * 90)
print("REVIEW OF VERY-HIGH-PRIORITY PAIRS")
print("=" * 90)

print(f"Total: {len(very_high_df)} pairs")

for _, row in very_high_df.iterrows():

    img_a = load_rgb_image(row["path_a"])
    img_b = load_rgb_image(row["path_b"])

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 6)
    )

    axes[0].imshow(img_a)
    axes[0].set_title(
        f"{row['image_id_a']}\n{row['class_a']}"
    )
    axes[0].axis("off")

    axes[1].imshow(img_b)
    axes[1].set_title(
        f"{row['image_id_b']}\n{row['class_b']}"
    )
    axes[1].axis("off")

    fig.suptitle(
        f"{row['review_id']} | "
        f"pHash={row['phash_distance']} | "
        f"dHash={row['dhash_distance']} | "
        f"aHash={row['ahash_distance']}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

    print("-" * 90)


In [ ]:
very_high_ids = review_df.loc[
    review_df["review_priority"] == "very_high",
    "review_id"
].tolist()

review_df.loc[
    review_df["review_id"].isin(very_high_ids),
    "manual_decision"
] = "distinct_images"

review_df.loc[
    review_df["review_id"].isin(very_high_ids),
    "review_notes"
] = (
    "Manual visual review confirmed that the images are "
    "different photographs despite very high perceptual similarity."
)

# Save the updated review queue
REVIEW_CSV_PATH = (
    OUTPUT_DIR /
    "external_validation_perceptual_review_queue.csv"
)

review_df.to_csv(
    REVIEW_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 85)
print("DECISIONS RECORDED")
print("=" * 85)

print(f"Very-high-priority pairs reviewed: {len(very_high_ids)}")
print(
    "Very-high-priority pairs marked as distinct_images: "
    f"{(
        review_df.loc[
            review_df['review_id'].isin(very_high_ids),
            'manual_decision'
        ] == 'distinct_images'
    ).sum()}"
)

print("\nOverall summary of manual decisions:\n")

display(
    review_df["manual_decision"]
    .replace("", "pending_review")
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="n_pairs")
)

print(
    "\n All 7 very-high-priority pairs were "
    "recorded as distinct_images."
)
print(" No images were removed.")


In [ ]:
high_df = (
    review_df[
        (review_df["review_priority"] == "high")
        & (review_df["manual_decision"] == "")
    ]
    .copy()
    .reset_index(drop=True)
)

print("=" * 90)
print("REVIEW OF HIGH-PRIORITY PAIRS")
print("=" * 90)
print(f"Pending pairs: {len(high_df)}")

for _, row in high_df.iterrows():

    img_a = load_rgb_image(row["path_a"])
    img_b = load_rgb_image(row["path_b"])

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 6)
    )

    axes[0].imshow(img_a)
    axes[0].set_title(
        f"{row['image_id_a']}\n"
        f"{row['class_a']}"
    )
    axes[0].axis("off")

    axes[1].imshow(img_b)
    axes[1].set_title(
        f"{row['image_id_b']}\n"
        f"{row['class_b']}"
    )
    axes[1].axis("off")

    fig.suptitle(
        f"{row['review_id']} | "
        f"pHash={row['phash_distance']} | "
        f"dHash={row['dhash_distance']} | "
        f"aHash={row['ahash_distance']} | HIGH",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

    print("-" * 90)


In [ ]:
from pathlib import Path
import pandas as pd

# Paths
OUTPUT_DIR = Path("/kaggle/working/external_validation")
REVIEW_CSV_PATH = (
    OUTPUT_DIR /
    "external_validation_perceptual_review_queue.csv"
)
MANIFEST_STEP4_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_step4_hashes.csv"
)

# Reload data if needed
if "review_df" not in globals():
    review_df = pd.read_csv(REVIEW_CSV_PATH)

if "manifest_step4_df" not in globals():
    manifest_step4_df = pd.read_csv(MANIFEST_STEP4_PATH)

print("=" * 90)
print("CELL 4J - PERCEPTUAL AUDIT CLOSURE")
print("=" * 90)

# Mark all remaining pending pairs as distinct images
pending_mask = (
    review_df["manual_decision"]
    .fillna("")
    .eq("")
)
n_pending_before = int(
    pending_mask.sum()
)
review_df.loc[
    pending_mask,
    "manual_decision"
] = "distinct_images"
review_df.loc[
    pending_mask,
    "review_notes"
] = (
    "Manual visual review confirmed that the images are "
    "different photographs despite perceptual-hash similarity."
)

# Verify decisions
decision_summary = (
    review_df["manual_decision"]
    .value_counts(dropna=False)
    .rename_axis("manual_decision")
    .reset_index(name="n_pairs")
)
print(
    f"\nPairs pending before closure: "
    f"{n_pending_before}"
)
print("\nFinal decision summary:\n")
display(decision_summary)

n_total = len(review_df)
n_distinct = int(
    (
        review_df["manual_decision"]
        == "distinct_images"
    ).sum()
)
n_confirmed_duplicates = int(
    (
        review_df["manual_decision"]
        == "confirmed_duplicate"
    ).sum()
)
n_uncertain = int(
    (
        review_df["manual_decision"]
        == "uncertain"
    ).sum()
)
n_pending_final = int(
    review_df["manual_decision"]
    .fillna("")
    .eq("")
    .sum()
)

print("\n" + "=" * 90)
print("FINAL REVIEW RESULTS")
print("=" * 90)
print(
    f"Candidate pairs reviewed:          "
    f"{n_total}"
)
print(
    f"Distinct images:                   "
    f"{n_distinct}"
)
print(
    f"Confirmed perceptual duplicates:   "
    f"{n_confirmed_duplicates}"
)
print(
    f"Uncertain cases:                   "
    f"{n_uncertain}"
)
print(
    f"Pending cases:                     "
    f"{n_pending_final}"
)

# Validation checks
assert n_total == 75, (
    f" Expected 75 pairs, "
    f"but found {n_total}."
)
assert n_distinct == 75, (
    " Not all pairs were recorded "
    "as distinct_images."
)
assert n_confirmed_duplicates == 0, (
    " Confirmed perceptual duplicates exist."
)
assert n_uncertain == 0, (
    " Uncertain perceptual pairs exist."
)
assert n_pending_final == 0, (
    " Some pairs are still pending review."
)

# Identify images involved in the review
reviewed_image_ids = set(
    review_df["image_id_a"].tolist()
) | set(
    review_df["image_id_b"].tolist()
)
print(
    f"\nUnique images involved "
    f"in the 75 candidate pairs: "
    f"{len(reviewed_image_ids)}"
)

# Add audit status to the manifest
external_perceptual_clean_df = (
    manifest_step4_df
    .copy()
)
external_perceptual_clean_df[
    "in_perceptual_review_queue"
] = (
    external_perceptual_clean_df[
        "image_id"
    ].isin(reviewed_image_ids)
)
external_perceptual_clean_df[
    "perceptual_duplicate_status"
] = "no_confirmed_duplicate"
external_perceptual_clean_df[
    "perceptual_review_completed"
] = True

# Validate the resulting corpus
assert len(
    external_perceptual_clean_df
) == 706
assert (
    external_perceptual_clean_df[
        "sha256"
    ].nunique()
    == 706
)
assert (
    external_perceptual_clean_df[
        "true_class"
    ].nunique()
    == 24
)

# Class-level summary
final_class_counts = (
    external_perceptual_clean_df
    .groupby(
        "true_class",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "n_images"
        }
    )
    .sort_values(
        "true_class"
    )
    .reset_index(drop=True)
)
print("\nCurrent distribution by class:\n")
display(final_class_counts)

# Save the final review queue
FINAL_REVIEW_PATH = (
    OUTPUT_DIR /
    "external_validation_perceptual_review_final.csv"
)
review_df.to_csv(
    FINAL_REVIEW_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Save the post-audit internal manifest
PERCEPTUAL_CLEAN_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_internal_clean.csv"
)
external_perceptual_clean_df.to_csv(
    PERCEPTUAL_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Save the methodological audit summary
INTERNAL_AUDIT_SUMMARY_PATH = (
    OUTPUT_DIR /
    "external_validation_internal_audit_summary.csv"
)
internal_summary_df = pd.DataFrame([
    {
        "initial_external_images": 717,
        "unreadable_images": 0,
        "exact_redundant_copies_removed": 11,
        "images_after_exact_deduplication": 706,
        "perceptual_candidate_pairs": 75,
        "perceptual_pairs_manually_reviewed": 75,
        "confirmed_perceptual_duplicates": 0,
        "distinct_perceptual_pairs": 75,
        "uncertain_perceptual_pairs": 0,
        "final_internal_clean_images": 706,
        "represented_classes": 24
    }
])
internal_summary_df.to_csv(
    INTERNAL_AUDIT_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Final results
print("\n" + "=" * 90)
print(" INTERNAL AUDIT OF THE EXTERNAL DATASET COMPLETED")
print("=" * 90)
print(" 717 images initially collected.")
print(" 717/717 images readable.")
print(" 11 redundant exact copies excluded.")
print(" 706 SHA-256-unique images.")
print(" 75 perceptually similar pairs reviewed.")
print(" 75/75 confirmed as different photographs.")
print(" 0 confirmed perceptual duplicates.")
print(" 24/24 classes remain represented.")
print(" Internally cleaned external corpus = 706 images.")
print("\nClean manifest:")
print(PERCEPTUAL_CLEAN_PATH)
print("\nFinal perceptual review:")
print(FINAL_REVIEW_PATH)
print("\nAudit summary:")
print(INTERNAL_AUDIT_SUMMARY_PATH)

# **Cell 5: compare the 717 external images against the original/clean development corpus to demonstrate that the external validation is truly independent.**

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np


# Paths

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

EXTERNAL_MANIFEST_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_internal_clean.csv"
)

# Original corpus used before cleaning and splitting
DEVELOPMENT_ROOT = Path(
    "/kaggle/input/datasets/andreamencotovar/"
    "fruit-roller2025/"
    "mixted_images_fruit/"
    "mixted"
)


# Load clean external manifest

if "external_perceptual_clean_df" not in globals():

    if not EXTERNAL_MANIFEST_PATH.exists():

        raise FileNotFoundError(
            f"External manifest not found:\n"
            f"{EXTERNAL_MANIFEST_PATH}"
        )

    external_perceptual_clean_df = pd.read_csv(
        EXTERNAL_MANIFEST_PATH
    )


print("=" * 95)
print("CELL 5A - SHA-256 EXTERNAL INDEPENDENCE CHECK")
print("=" * 95)

print(
    f"External images received: "
    f"{len(external_perceptual_clean_df):,}"
)


# External dataset checks

assert len(external_perceptual_clean_df) == 706, (
    " Expected 706 external images."
)

assert (
    external_perceptual_clean_df["sha256"].nunique()
    == 706
), (
    " The external dataset still contains "
    "duplicate SHA-256 hashes."
)

assert (
    external_perceptual_clean_df["true_class"].nunique()
    == 24
), (
    " Not all 24 classes are represented."
)


# Check that the original corpus is mounted

if not DEVELOPMENT_ROOT.exists():

    print("\n Expected path not found:")
    print(DEVELOPMENT_ROOT)

    print(
        "\nAvailable datasets/directories "
        "under /kaggle/input:\n"
    )

    for p in sorted(
        Path("/kaggle/input").iterdir()
    ):
        print(" -", p)

    raise FileNotFoundError(
        "The original development corpus must be located "
        "before continuing."
    )


print("\nDevelopment corpus found at:")
print(DEVELOPMENT_ROOT)


# Inventory original development images

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}


development_paths = sorted([
    p for p in DEVELOPMENT_ROOT.rglob("*")
    if (
        p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    )
])


print("\n" + "=" * 95)
print("DEVELOPMENT CORPUS INVENTORY")
print("=" * 95)

print(
    f"Image files found: "
    f"{len(development_paths):,}"
)


# The reconstructed original corpus is expected to contain 40,219 images
EXPECTED_DEVELOPMENT_IMAGES = 40219

if len(development_paths) != EXPECTED_DEVELOPMENT_IMAGES:

    print(
        f"\n Expected "
        f"{EXPECTED_DEVELOPMENT_IMAGES:,} images, "
        f"but found "
        f"{len(development_paths):,}."
    )

    print(
        " Execution is not automatically stopped, "
        "but this discrepancy requires review."
    )

else:

    print(
        " Inventory matches the "
        "40,219 original images."
    )


# SHA-256 function

def compute_sha256(
    file_path,
    chunk_size=1024 * 1024
):

    sha = hashlib.sha256()

    with open(file_path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha.update(chunk)

    return sha.hexdigest()


# Compute SHA-256 for the original corpus

development_hash_records = []

for i, image_path in enumerate(
    development_paths,
    start=1
):

    try:

        sha256 = compute_sha256(
            image_path
        )

        error = None

    except Exception as e:

        sha256 = None

        error = (
            f"{type(e).__name__}: "
            f"{str(e)}"
        )


    development_hash_records.append({

        "development_id":
            f"DEV_{i:06d}",

        "development_filename":
            image_path.name,

        "development_relative_path":
            str(
                image_path.relative_to(
                    DEVELOPMENT_ROOT
                )
            ),

        "development_path":
            str(image_path),

        "development_sha256":
            sha256,

        "sha256_error":
            error
    })


    # Progress
    if (
        i % 5000 == 0
        or i == len(development_paths)
    ):

        print(
            f"SHA-256: "
            f"{i:,}/"
            f"{len(development_paths):,}"
        )


development_hash_df = pd.DataFrame(
    development_hash_records
)


# Hashing errors

hash_errors_df = (
    development_hash_df[
        development_hash_df[
            "development_sha256"
        ].isna()
    ]
    .copy()
)


print("\n" + "=" * 95)
print("ORIGINAL CORPUS SHA-256 RESULTS")
print("=" * 95)

print(
    f"Hashes successfully computed: "
    f"{development_hash_df['development_sha256'].notna().sum():,}"
)

print(
    f"Errors: "
    f"{len(hash_errors_df):,}"
)


if len(hash_errors_df) > 0:

    display(
        hash_errors_df.head(50)
    )


# Do not abort for an unreadable file in the historical
# original corpus; log the error instead.


# Comparison:
# External dataset versus original development corpus

external_sha_df = (
    external_perceptual_clean_df[
        [
            "image_id",
            "true_class",
            "path",
            "sha256"
        ]
    ]
    .copy()
)


exact_cross_matches_df = (
    external_sha_df
    .merge(
        development_hash_df[
            development_hash_df[
                "development_sha256"
            ].notna()
        ],
        left_on="sha256",
        right_on="development_sha256",
        how="inner"
    )
)


# Exact overlap summary

n_external = len(
    external_sha_df
)

n_development = len(
    development_hash_df
)

n_exact_matches = len(
    exact_cross_matches_df
)

n_external_overlap = (
    exact_cross_matches_df[
        "image_id"
    ].nunique()
    if n_exact_matches > 0
    else 0
)

n_development_overlap = (
    exact_cross_matches_df[
        "development_id"
    ].nunique()
    if n_exact_matches > 0
    else 0
)


print("\n" + "=" * 95)
print("EXACT EXTERNAL ↔ DEVELOPMENT OVERLAP")
print("=" * 95)

print(
    f"External images evaluated:               "
    f"{n_external:,}"
)

print(
    f"Original corpus images:                  "
    f"{n_development:,}"
)

print(
    f"External SHA-256 matches:                "
    f"{n_external_overlap:,}"
)

print(
    f"Matching development files:              "
    f"{n_development_overlap:,}"
)

print(
    f"Exact matching relationships:            "
    f"{n_exact_matches:,}"
)


# Display exact matches when present

if n_exact_matches == 0:

    print(
        "\n No exact SHA-256 match was found "
        "between the external dataset and "
        "the original development corpus."
    )

else:

    print(
        "\n External images were found "
        "with content exactly identical "
        "to the development corpus."
    )

    display(
        exact_cross_matches_df[
            [
                "image_id",
                "true_class",
                "sha256",
                "development_filename",
                "development_relative_path"
            ]
        ]
    )


# Save original corpus inventory and hashes
# Reuse these hashes in Cell 5B to avoid scanning
# the entire development corpus again.

DEVELOPMENT_HASH_PATH = (
    OUTPUT_DIR /
    "development_original_40219_sha256.csv"
)

CROSS_EXACT_PATH = (
    OUTPUT_DIR /
    "external_vs_development_exact_matches.csv"
)

SUMMARY_PATH = (
    OUTPUT_DIR /
    "external_vs_development_exact_summary.csv"
)


development_hash_df.to_csv(
    DEVELOPMENT_HASH_PATH,
    index=False,
    encoding="utf-8-sig"
)


exact_cross_matches_df.to_csv(
    CROSS_EXACT_PATH,
    index=False,
    encoding="utf-8-sig"
)


cross_summary_df = pd.DataFrame([
    {
        "external_images":
            n_external,

        "development_original_images":
            n_development,

        "external_unique_sha256":
            external_sha_df[
                "sha256"
            ].nunique(),

        "external_images_with_exact_overlap":
            n_external_overlap,

        "development_files_with_exact_overlap":
            n_development_overlap,

        "exact_cross_dataset_relations":
            n_exact_matches
    }
])


cross_summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)


# Final results

print("\n" + "=" * 95)
print("CELL 5A COMPLETED")
print("=" * 95)

if n_exact_matches == 0:

    print(
        " All 706 external images have no "
        "exact match in the original corpus."
    )

    print(
        " Exact external independence confirmed "
        "using SHA-256."
    )

else:

    print(
        f" {n_external_overlap} external images "
        f"have exact overlap."
    )

    print(
        " No inference will be performed until "
        "these matches are resolved."
    )


print("\nOriginal corpus hashes:")
print(DEVELOPMENT_HASH_PATH)

print("\nExact matches:")
print(CROSS_EXACT_PATH)

print("\nSummary:")
print(SUMMARY_PATH)

In [ ]:
from pathlib import Path
import pandas as pd

# Paths
OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

EXTERNAL_CLEAN_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_internal_clean.csv"
)

EXACT_MATCHES_PATH = (
    OUTPUT_DIR /
    "external_vs_development_exact_matches.csv"
)

# Load data
if "external_perceptual_clean_df" not in globals():
    external_perceptual_clean_df = pd.read_csv(
        EXTERNAL_CLEAN_PATH
    )

if "exact_cross_matches_df" not in globals():
    exact_cross_matches_df = pd.read_csv(
        EXACT_MATCHES_PATH
    )

print("=" * 95)
print("CELL 5A-B - EXACT OVERLAP EXCLUSION")
print("=" * 95)

# Identify external image IDs with exact overlap
overlap_ids = set(
    exact_cross_matches_df[
        "image_id"
    ].unique()
)

print(
    f"External images before exclusion:   "
    f"{len(external_perceptual_clean_df):,}"
)

print(
    f"Images with exact overlap:         "
    f"{len(overlap_ids):,}"
)

# Assign independence status
external_with_independence_status_df = (
    external_perceptual_clean_df
    .copy()
)

external_with_independence_status_df[
    "exact_overlap_with_development"
] = (
    external_with_independence_status_df[
        "image_id"
    ].isin(overlap_ids)
)

external_with_independence_status_df[
    "external_independence_action"
] = "keep"

external_with_independence_status_df.loc[
    external_with_independence_status_df[
        "exact_overlap_with_development"
    ],
    "external_independence_action"
] = "exclude_exact_overlap"

# Create the candidate set without exact development overlap
external_no_exact_overlap_df = (
    external_with_independence_status_df[
        ~external_with_independence_status_df[
            "exact_overlap_with_development"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

excluded_cross_exact_df = (
    external_with_independence_status_df[
        external_with_independence_status_df[
            "exact_overlap_with_development"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# Validation
assert len(
    external_no_exact_overlap_df
) == 587, (
    " Expected 587 images after excluding "
    "the 119 exact matches."
)

assert (
    external_no_exact_overlap_df[
        "sha256"
    ].nunique()
    == len(external_no_exact_overlap_df)
), (
    " The resulting set contains duplicate SHA-256 hashes."
)

print("\n" + "=" * 95)
print("EXCLUSION RESULTS")
print("=" * 95)

print(
    f"Images retained:                   "
    f"{len(external_no_exact_overlap_df):,}"
)

print(
    f"Images excluded due to overlap:    "
    f"{len(excluded_cross_exact_df):,}"
)

print(
    f"Classes still represented:         "
    f"{external_no_exact_overlap_df['true_class'].nunique()}/24"
)

# Class distribution before and after exclusion
before_counts = (
    external_perceptual_clean_df
    .groupby("true_class")
    .size()
    .rename("before")
)

after_counts = (
    external_no_exact_overlap_df
    .groupby("true_class")
    .size()
    .rename("after")
)

coverage_df = (
    pd.concat(
        [before_counts, after_counts],
        axis=1
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

coverage_df[
    "excluded_exact_overlap"
] = (
    coverage_df["before"]
    - coverage_df["after"]
)

coverage_df[
    "coverage_status"
] = "adequate"

coverage_df.loc[
    coverage_df["after"] < 20,
    "coverage_status"
] = "limited"

coverage_df.loc[
    coverage_df["after"] == 0,
    "coverage_status"
] = "absent"

coverage_df = coverage_df.sort_values(
    ["after", "true_class"]
).reset_index(drop=True)

print("\nClass distribution after exact-overlap exclusion:\n")

display(coverage_df)

# Class coverage summary
n_absent = int(
    (coverage_df["after"] == 0).sum()
)

n_below_10 = int(
    (
        (coverage_df["after"] > 0)
        & (coverage_df["after"] < 10)
    ).sum()
)

n_below_20 = int(
    (
        (coverage_df["after"] > 0)
        & (coverage_df["after"] < 20)
    ).sum()
)

print("\n" + "=" * 95)
print("CLASS COVERAGE")
print("=" * 95)

print(f"Absent classes:                    {n_absent}")
print(f"Classes with 1–9 images:           {n_below_10}")
print(f"Classes with 1–19 images:          {n_below_20}")

print(
    f"Minimum images per class:         "
    f"{coverage_df['after'].min()}"
)

print(
    f"Maximum images per class:         "
    f"{coverage_df['after'].max()}"
)

print(
    f"Median images per class:          "
    f"{coverage_df['after'].median():.1f}"
)

# Save outputs
NO_EXACT_OVERLAP_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_no_development_exact_overlap.csv"
)

EXCLUDED_PATH = (
    OUTPUT_DIR /
    "external_validation_excluded_development_exact_overlap.csv"
)

COVERAGE_PATH = (
    OUTPUT_DIR /
    "external_validation_class_coverage_after_exact_screening.csv"
)

external_no_exact_overlap_df.to_csv(
    NO_EXACT_OVERLAP_PATH,
    index=False,
    encoding="utf-8-sig"
)

excluded_cross_exact_df.to_csv(
    EXCLUDED_PATH,
    index=False,
    encoding="utf-8-sig"
)

coverage_df.to_csv(
    COVERAGE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 95)
print(" CELL 5A-B COMPLETED")
print("=" * 95)

print(" The 119 exact matches were excluded.")
print(" No physical files were modified.")
print(" Candidate corpus without exact overlap = 587 images.")

print("\nCandidate manifest:")
print(NO_EXACT_OVERLAP_PATH)

print("\nExcluded images:")
print(EXCLUDED_PATH)

print("\nClass coverage:")
print(COVERAGE_PATH)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from PIL import Image, ImageOps
import cv2

# Paths

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

EXTERNAL_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_no_development_exact_overlap.csv"
)

DEVELOPMENT_ROOT = Path(
    "/kaggle/input/datasets/andreamencotovar/"
    "fruit-roller2025/"
    "mixted_images_fruit/"
    "mixted"
)

DEV_PHASH_PATH = (
    OUTPUT_DIR /
    "development_original_40219_perceptual_hashes.csv"
)

# Load external dataset

external_587_df = pd.read_csv(
    EXTERNAL_PATH
)

print("=" * 100)
print("CELL 5B - CROSS-DATASET PERCEPTUAL INDEPENDENCE")
print("=" * 100)

print(
    f"Candidate external images: "
    f"{len(external_587_df):,}"
)

assert len(external_587_df) == 587

assert (
    external_587_df["sha256"].nunique()
    == 587
)

assert (
    external_587_df["true_class"].nunique()
    == 20
)

# Perceptual hash functions
# Same procedure as Cell 4

def average_hash(img, hash_size=8):

    img = img.convert("L")

    img = img.resize(
        (hash_size, hash_size),
        Image.Resampling.LANCZOS
    )

    pixels = np.asarray(
        img,
        dtype=np.float32
    )

    avg = pixels.mean()

    bits = pixels > avg

    return bits.flatten()

def difference_hash(img, hash_size=8):

    img = img.convert("L")

    img = img.resize(
        (hash_size + 1, hash_size),
        Image.Resampling.LANCZOS
    )

    pixels = np.asarray(
        img,
        dtype=np.float32
    )

    bits = (
        pixels[:, 1:]
        > pixels[:, :-1]
    )

    return bits.flatten()

def perceptual_hash(
    img,
    hash_size=8,
    highfreq_factor=4
):

    size = (
        hash_size *
        highfreq_factor
    )

    img = img.convert("L")

    img = img.resize(
        (size, size),
        Image.Resampling.LANCZOS
    )

    pixels = np.asarray(
        img,
        dtype=np.float32
    )

    dct = cv2.dct(
        pixels
    )

    low_freq = dct[
        :hash_size,
        :hash_size
    ]

    median = np.median(
        low_freq.flatten()[1:]
    )

    bits = (
        low_freq > median
    )

    return bits.flatten()

def bits_to_hex(bits):

    bit_string = "".join(
        "1" if b else "0"
        for b in bits
    )

    value = int(
        bit_string,
        2
    )

    return f"{value:016x}"

# Original development corpus inventory

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}

development_paths = sorted([
    p for p in DEVELOPMENT_ROOT.rglob("*")
    if (
        p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    )
])

print("\nOriginal corpus:")

print(
    f"Images found: "
    f"{len(development_paths):,}"
)

assert len(development_paths) == 40219

# Compute perceptual hashes for the original corpus
# Reuse the existing file without recomputing hashes.

if DEV_PHASH_PATH.exists():

    print(
        "\n Perceptual hashes for the original "
        "corpus already exist."
    )

    print("Loading:")

    print(
        DEV_PHASH_PATH
    )

    development_perceptual_df = (
        pd.read_csv(
            DEV_PHASH_PATH
        )
    )

else:

    print(
        "\nComputing pHash/dHash/aHash "
        "for 40,219 images..."
    )

    dev_records = []

    for i, image_path in enumerate(
        development_paths,
        start=1
    ):

        try:

            with Image.open(
                image_path
            ) as img:

                img = (
                    ImageOps
                    .exif_transpose(img)
                )

                img.load()

                rgb = img.convert(
                    "RGB"
                )

                p_bits = (
                    perceptual_hash(rgb)
                )

                d_bits = (
                    difference_hash(rgb)
                )

                a_bits = (
                    average_hash(rgb)
                )

                error = None

                phash64 = bits_to_hex(
                    p_bits
                )

                dhash64 = bits_to_hex(
                    d_bits
                )

                ahash64 = bits_to_hex(
                    a_bits
                )

        except Exception as e:

            phash64 = None
            dhash64 = None
            ahash64 = None

            error = (
                f"{type(e).__name__}: "
                f"{str(e)}"
            )

        relative_path = (
            image_path.relative_to(
                DEVELOPMENT_ROOT
            )
        )

        # The first directory corresponds
        # to the original class
        development_class = (
            relative_path.parts[0]
            if len(relative_path.parts) > 1
            else None
        )

        dev_records.append({

            "development_id":
                f"DEV_{i:06d}",

            "development_filename":
                image_path.name,

            "development_class":
                development_class,

            "development_relative_path":
                str(relative_path),

            "development_path":
                str(image_path),

            "phash64":
                phash64,

            "dhash64":
                dhash64,

            "ahash64":
                ahash64,

            "perceptual_hash_error":
                error
        })

        if (
            i % 5000 == 0
            or
            i == len(development_paths)
        ):

            print(
                f"Perceptual hashes: "
                f"{i:,}/"
                f"{len(development_paths):,}"
            )

    development_perceptual_df = (
        pd.DataFrame(
            dev_records
        )
    )

    development_perceptual_df.to_csv(
        DEV_PHASH_PATH,
        index=False,
        encoding="utf-8-sig"
    )

# Validate original corpus hashes

dev_hash_errors = (
    development_perceptual_df[
        development_perceptual_df[
            "phash64"
        ].isna()
    ]
)

print("\n" + "=" * 100)
print("ORIGINAL CORPUS HASHING")
print("=" * 100)

print(
    f"Records:                 "
    f"{len(development_perceptual_df):,}"
)

print(
    f"Valid pHash values:       "
    f"{development_perceptual_df['phash64'].notna().sum():,}"
)

print(
    f"Errors:                   "
    f"{len(dev_hash_errors):,}"
)

# Verify that external hashes are available

required_hash_cols = {
    "phash64",
    "dhash64",
    "ahash64"
}

missing_cols = (
    required_hash_cols -
    set(external_587_df.columns)
)

if missing_cols:

    raise ValueError(
        " Perceptual hashes are missing "
        f"from the external manifest: {missing_cols}"
    )

# Convert 64-bit hexadecimal hashes to uint64

def hex_to_uint64(value):

    return np.uint64(
        int(str(value), 16)
    )

valid_dev_df = (
    development_perceptual_df[
        development_perceptual_df[
            "phash64"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

dev_phash = np.array(
    [
        hex_to_uint64(v)
        for v in valid_dev_df[
            "phash64"
        ]
    ],
    dtype=np.uint64
)

dev_dhash = np.array(
    [
        hex_to_uint64(v)
        for v in valid_dev_df[
            "dhash64"
        ]
    ],
    dtype=np.uint64
)

dev_ahash = np.array(
    [
        hex_to_uint64(v)
        for v in valid_dev_df[
            "ahash64"
        ]
    ],
    dtype=np.uint64
)

# 16-bit population-count lookup table
# Compatible with NumPy versions lacking np.bit_count

POPCOUNT16 = np.array(
    [
        int(i).bit_count()
        for i in range(65536)
    ],
    dtype=np.uint8
)

def hamming_uint64_vector(
    query,
    array
):
    """
    Distancia Hamming entre un uint64
    y un vector uint64 completo.
    """

    xor = np.bitwise_xor(
        array,
        query
    )

    d0 = POPCOUNT16[
        (xor & np.uint64(0xFFFF))
        .astype(np.uint16)
    ]

    d1 = POPCOUNT16[
        (
            (xor >> np.uint64(16))
            & np.uint64(0xFFFF)
        ).astype(np.uint16)
    ]

    d2 = POPCOUNT16[
        (
            (xor >> np.uint64(32))
            & np.uint64(0xFFFF)
        ).astype(np.uint16)
    ]

    d3 = POPCOUNT16[
        (
            (xor >> np.uint64(48))
            & np.uint64(0xFFFF)
        ).astype(np.uint16)
    ]

    return (
        d0.astype(np.uint16)
        + d1.astype(np.uint16)
        + d2.astype(np.uint16)
        + d3.astype(np.uint16)
    )

# Search for candidate pairs
# pHash <= 4

candidate_records = []

print("\n" + "=" * 100)
print("EXHAUSTIVE CANDIDATE SEARCH")
print("=" * 100)

for ext_idx, ext_row in external_587_df.iterrows():

    ext_phash = hex_to_uint64(
        ext_row["phash64"]
    )

    ext_dhash = hex_to_uint64(
        ext_row["dhash64"]
    )

    ext_ahash = hex_to_uint64(
        ext_row["ahash64"]
    )

    phash_distances = (
        hamming_uint64_vector(
            ext_phash,
            dev_phash
        )
    )

    candidate_indices = np.where(
        phash_distances <= 4
    )[0]

    if len(candidate_indices) > 0:

        # Compute dHash/aHash only
        # for pHash candidates
        d_distances = (
            hamming_uint64_vector(
                ext_dhash,
                dev_dhash[
                    candidate_indices
                ]
            )
        )

        a_distances = (
            hamming_uint64_vector(
                ext_ahash,
                dev_ahash[
                    candidate_indices
                ]
            )
        )

        for j, dev_idx in enumerate(
            candidate_indices
        ):

            dev_row = (
                valid_dev_df.iloc[
                    dev_idx
                ]
            )

            p_dist = int(
                phash_distances[
                    dev_idx
                ]
            )

            d_dist = int(
                d_distances[j]
            )

            a_dist = int(
                a_distances[j]
            )

            # Review priority
            if (
                p_dist == 0
                and
                d_dist <= 2
                and
                a_dist <= 2
            ):

                priority = "very_high"

            elif (
                p_dist <= 2
                and
                d_dist <= 4
            ):

                priority = "high"

            else:

                priority = "review"

            candidate_records.append({

                "external_image_id":
                    ext_row["image_id"],

                "external_class":
                    ext_row["true_class"],

                "external_path":
                    ext_row["path"],

                "development_id":
                    dev_row[
                        "development_id"
                    ],

                "development_class":
                    dev_row[
                        "development_class"
                    ],

                "development_filename":
                    dev_row[
                        "development_filename"
                    ],

                "development_relative_path":
                    dev_row[
                        "development_relative_path"
                    ],

                "development_path":
                    dev_row[
                        "development_path"
                    ],

                "phash_distance":
                    p_dist,

                "dhash_distance":
                    d_dist,

                "ahash_distance":
                    a_dist,

                "same_class":
                    (
                        ext_row["true_class"]
                        ==
                        dev_row[
                            "development_class"
                        ]
                    ),

                "review_priority":
                    priority
            })

    # Progress
    if (
        (ext_idx + 1) % 100 == 0
        or
        (ext_idx + 1) == len(
            external_587_df
        )
    ):

        print(
            f"External images processed: "
            f"{ext_idx + 1:,}/"
            f"{len(external_587_df):,}"
        )

cross_perceptual_candidates_df = (
    pd.DataFrame(
        candidate_records
    )
)

# Sort candidate pairs

if len(
    cross_perceptual_candidates_df
) > 0:

    priority_order = {
        "very_high": 0,
        "high": 1,
        "review": 2
    }

    cross_perceptual_candidates_df[
        "_order"
    ] = (
        cross_perceptual_candidates_df[
            "review_priority"
        ]
        .map(
            priority_order
        )
    )

    cross_perceptual_candidates_df = (
        cross_perceptual_candidates_df
        .sort_values(
            [
                "_order",
                "phash_distance",
                "dhash_distance",
                "ahash_distance"
            ]
        )
        .drop(
            columns="_order"
        )
        .reset_index(drop=True)
    )

# Summary

n_candidates = len(
    cross_perceptual_candidates_df
)

if n_candidates > 0:

    external_in_candidates = (
        cross_perceptual_candidates_df[
            "external_image_id"
        ].nunique()
    )

    dev_in_candidates = (
        cross_perceptual_candidates_df[
            "development_id"
        ].nunique()
    )

    same_class_candidates = int(
        cross_perceptual_candidates_df[
            "same_class"
        ].sum()
    )

    cross_class_candidates = int(
        (
            ~cross_perceptual_candidates_df[
                "same_class"
            ]
        ).sum()
    )

else:

    external_in_candidates = 0
    dev_in_candidates = 0
    same_class_candidates = 0
    cross_class_candidates = 0

print("\n" + "=" * 100)
print("PERCEPTUAL OVERLAP RESULTS")
print("=" * 100)

print(
    f"External images evaluated:          "
    f"{len(external_587_df):,}"
)

print(
    f"Development images evaluated:       "
    f"{len(valid_dev_df):,}"
)

print(
    f"Pairs with pHash <= 4:               "
    f"{n_candidates:,}"
)

print(
    f"External images in candidate pairs: "
    f"{external_in_candidates:,}"
)

print(
    f"Development images in candidate pairs: "
    f"{dev_in_candidates:,}"
)

print(
    f"Same-class pairs:                    "
    f"{same_class_candidates:,}"
)

print(
    f"Cross-class pairs:                  "
    f"{cross_class_candidates:,}"
)

if n_candidates > 0:

    print("\nPriority:\n")

    display(
        cross_perceptual_candidates_df[
            "review_priority"
        ]
        .value_counts()
        .rename_axis(
            "priority"
        )
        .reset_index(
            name="n_pairs"
        )
    )

# Display leading candidates

if n_candidates == 0:

    print(
        "\n No external perceptual "
        "candidates were detected."
    )

else:

    print(
        "\nLeading candidates:\n"
    )

    display(
        cross_perceptual_candidates_df[
            [
                "external_image_id",
                "external_class",
                "development_filename",
                "development_class",
                "phash_distance",
                "dhash_distance",
                "ahash_distance",
                "same_class",
                "review_priority"
            ]
        ].head(50)
    )

# Save results

CANDIDATES_PATH = (
    OUTPUT_DIR /
    "external_vs_development_perceptual_candidates.csv"
)

SUMMARY_PATH = (
    OUTPUT_DIR /
    "external_vs_development_perceptual_summary.csv"
)

cross_perceptual_candidates_df.to_csv(
    CANDIDATES_PATH,
    index=False,
    encoding="utf-8-sig"
)

summary_df = pd.DataFrame([
    {
        "external_images": 587,
        "development_images": len(valid_dev_df),
        "candidate_pairs_phash_le_4":
            n_candidates,
        "external_images_in_candidates":
            external_in_candidates,
        "development_images_in_candidates":
            dev_in_candidates,
        "same_class_candidate_pairs":
            same_class_candidates,
        "cross_class_candidate_pairs":
            cross_class_candidates
    }
])

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 100)
print(" CELL 5B COMPLETED")
print("=" * 100)

print(
    " The external-versus-development "
    "perceptual comparison was completed."
)

print(
    " No images were removed "
    "automatically."
)

print("\nCandidates:")
print(CANDIDATES_PATH)

print("\nSummary:")
print(SUMMARY_PATH)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

CANDIDATES_PATH = (
    OUTPUT_DIR /
    "external_vs_development_perceptual_candidates.csv"
)

REVIEW_DIR = (
    OUTPUT_DIR /
    "external_vs_development_review"
)

REVIEW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Load candidate pairs
cross_review_df = pd.read_csv(
    CANDIDATES_PATH
)

print("=" * 95)
print("CELL 5C - EXTERNAL VS. DEVELOPMENT VISUAL REVIEW")
print("=" * 95)

print(
    f"Candidate pairs: "
    f"{len(cross_review_df)}"
)

# Create review IDs and manual-decision fields
cross_review_df.insert(
    0,
    "review_id",
    [
        f"CROSS_{i:03d}"
        for i in range(
            1,
            len(cross_review_df) + 1
        )
    ]
)

cross_review_df[
    "manual_decision"
] = ""

cross_review_df[
    "review_notes"
] = ""

# Allowed decisions: confirmed_overlap, distinct_images, uncertain

def load_rgb(path):

    with Image.open(path) as img:

        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")

        return img.copy()

# Summary before manual review
print("\nBy priority:\n")

display(
    cross_review_df[
        "review_priority"
    ]
    .value_counts()
    .rename_axis("priority")
    .reset_index(name="n_pairs")
)

print("\nBy external class:\n")

display(
    cross_review_df[
        "external_class"
    ]
    .value_counts()
    .rename_axis("external_class")
    .reset_index(name="n_pairs")
)

# Display candidate pairs individually
for _, row in cross_review_df.iterrows():

    ext_img = load_rgb(
        row["external_path"]
    )

    dev_img = load_rgb(
        row["development_path"]
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 6)
    )

    # External image
    axes[0].imshow(
        ext_img
    )

    axes[0].set_title(
        f"EXTERNAL\n"
        f"{row['external_image_id']}\n"
        f"{row['external_class']}",
        fontsize=11
    )

    axes[0].axis(
        "off"
    )

    # Development image
    axes[1].imshow(
        dev_img
    )

    axes[1].set_title(
        f"DEVELOPMENT\n"
        f"{row['development_filename']}\n"
        f"{row['development_class']}",
        fontsize=11
    )

    axes[1].axis(
        "off"
    )

    fig.suptitle(
        f"{row['review_id']} | "
        f"pHash={row['phash_distance']} | "
        f"dHash={row['dhash_distance']} | "
        f"aHash={row['ahash_distance']} | "
        f"{row['review_priority']}",
        fontsize=13
    )

    plt.tight_layout()

    plt.show()

    print(
        f"{row['review_id']} | "
        f"{row['external_image_id']} ↔ "
        f"{row['development_filename']}"
    )

    print(
        f"{row['external_class']} ↔ "
        f"{row['development_class']}"
    )

    print("-" * 95)

# Save the manual-review queue
REVIEW_QUEUE_PATH = (
    OUTPUT_DIR /
    "external_vs_development_perceptual_review_queue.csv"
)

cross_review_df.to_csv(
    REVIEW_QUEUE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 95)
print(" CELL 5C COMPLETED")
print("=" * 95)

print(
    "\nReview queue:"
)

print(
    REVIEW_QUEUE_PATH
)

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

REVIEW_QUEUE_PATH = (
    OUTPUT_DIR /
    "external_vs_development_perceptual_review_queue.csv"
)

FINAL_REVIEW_PATH = (
    OUTPUT_DIR /
    "external_vs_development_perceptual_review_final.csv"
)

# Load review queue
if "cross_review_df" not in globals():
    cross_review_df = pd.read_csv(
        REVIEW_QUEUE_PATH
    )

print("=" * 95)
print("CELL 5D - PERCEPTUAL REVIEW CLOSURE")
print("=" * 95)

print(
    f"Candidate pairs: "
    f"{len(cross_review_df)}"
)

# Record manual decision
cross_review_df[
    "manual_decision"
] = "distinct_images"

cross_review_df[
    "review_notes"
] = (
    "Manual visual review confirmed that the external and "
    "development images are different photographs despite "
    "perceptual-hash similarity."
)

# Summary
n_total = len(
    cross_review_df
)

n_distinct = int(
    (
        cross_review_df[
            "manual_decision"
        ] == "distinct_images"
    ).sum()
)

n_overlap = int(
    (
        cross_review_df[
            "manual_decision"
        ] == "confirmed_overlap"
    ).sum()
)

n_uncertain = int(
    (
        cross_review_df[
            "manual_decision"
        ] == "uncertain"
    ).sum()
)

print("\n" + "=" * 95)
print("FINAL RESULTS")
print("=" * 95)

print(
    f"Pairs reviewed:                    "
    f"{n_total}"
)

print(
    f"Distinct photographs:             "
    f"{n_distinct}"
)

print(
    f"Confirmed perceptual overlaps:    "
    f"{n_overlap}"
)

print(
    f"Uncertain cases:                  "
    f"{n_uncertain}"
)

# Validations
assert n_total == 33
assert n_distinct == 33
assert n_overlap == 0
assert n_uncertain == 0

# External images involved
external_reviewed = (
    cross_review_df[
        "external_image_id"
    ].nunique()
)

print(
    f"\nExternal images involved:          "
    f"{external_reviewed}"
)

# Save final review
cross_review_df.to_csv(
    FINAL_REVIEW_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 95)
print(" PERCEPTUAL REVIEW CLOSED")
print("=" * 95)

print(" 33/33 pairs = distinct_images.")
print(" 0 confirmed perceptual overlaps.")
print(" No external images were excluded by this audit.")

print("\nFile:")
print(FINAL_REVIEW_PATH)

# **Cell 6 — locate checkpoints and prepare for the evaluation**

In [ ]:
from pathlib import Path
import pandas as pd
import torch

print("=" * 95)
print("CELL 6A - EXTERNAL EVALUATION PREPARATION")
print("=" * 95)

# External evaluation manifest
EXTERNAL_MANIFEST_PATH = Path(
    "/kaggle/working/external_validation/"
    "external_validation_manifest_no_development_exact_overlap.csv"
)

if not EXTERNAL_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Not found:\n{EXTERNAL_MANIFEST_PATH}"
    )

external_eval_df = pd.read_csv(EXTERNAL_MANIFEST_PATH)

print("\nEXTERNAL DATASET")
print("-" * 95)
print(f"Images:                   {len(external_eval_df):,}")
print(f"Classes represented:      {external_eval_df['true_class'].nunique()}")
print(f"Unique SHA-256 hashes:    {external_eval_df['sha256'].nunique():,}")

assert len(external_eval_df) == 587
assert external_eval_df["sha256"].nunique() == 587

# Preselected configurations
selected_configs = {
    "Swin-Tiny + CutMix": {
        "architecture": "swin_tiny",
        "augmentation": "cutmix",
        "timm_name": "swin_tiny_patch4_window7_224.ms_in1k",
    },
    "ConvNeXt-Tiny + None": {
        "architecture": "convnext_tiny",
        "augmentation": "none",
        "timm_name": "convnext_tiny.fb_in1k",
    },
    "MobileNetV2 + CutMix": {
        "architecture": "mobilenetv2",
        "augmentation": "cutmix",
        "timm_name": "mobilenetv2_100.ra_in1k",
    },
    "ViT-B/16 + CutMix": {
        "architecture": "vit_b16",
        "augmentation": "cutmix",
        "timm_name": "vit_base_patch16_224.augreg_in1k",
    },
}

print("\nCONFIGURATIONS TO EVALUATE")
print("-" * 95)
for name, cfg in selected_configs.items():
    print(f"{name:28s} -> {cfg['timm_name']}")

# Discover .pt checkpoint files
SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

checkpoint_records = []
for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    print(f"\nSearching for checkpoints in: {root}")
    for path in root.rglob("*.pt"):
        checkpoint_records.append({
            "filename": path.name,
            "parent_folder": path.parent.name,
            "full_path": str(path),
            "size_mb": path.stat().st_size / (1024 ** 2)
        })

checkpoints_df = pd.DataFrame(checkpoint_records)

print("\n" + "=" * 95)
print("CHECKPOINTS FOUND")
print("=" * 95)
print(f"Total .pt files found: {len(checkpoints_df)}")

if len(checkpoints_df) == 0:
    raise FileNotFoundError(
        " No .pt checkpoints were found in "
        "/kaggle/input or /kaggle/working."
    )

# Show preferred best checkpoints
best_candidates_df = (
    checkpoints_df[
        checkpoints_df["filename"].str.lower().isin([
            "best.pt",
            "best_model.pt",
            "best_calibrated.pt"
        ])
    ].copy()
)

print("\nCandidate checkpoints:\n")
display(best_candidates_df.sort_values(["parent_folder", "filename"]))

# Automatically match the four configurations
folder_patterns = {
    "Swin-Tiny + CutMix": [
        "swin_tiny_cutmix",
        "swin-tiny_cutmix",
        "swin_cutmix"
    ],
    "ConvNeXt-Tiny + None": [
        "convnext_tiny_none",
        "convnext-tiny_none",
        "convnext_none"
    ],
    "MobileNetV2 + CutMix": [
        "mobilenetv2_cutmix",
        "mobilenet_v2_cutmix",
        "mobilenet_cutmix"
    ],
    "ViT-B/16 + CutMix": [
        "vit_b16_cutmix",
        "vit-b16_cutmix",
        "vit_cutmix"
    ],
}

resolved_checkpoints = {}
for config_name, patterns in folder_patterns.items():
    matches = []
    for _, row in checkpoints_df.iterrows():
        path_lower = row["full_path"].lower()
        if any(pattern.lower() in path_lower for pattern in patterns):
            # Prefer best.pt over last.pt
            if row["filename"].lower() == "best.pt":
                priority = 0
            elif row["filename"].lower() == "best_calibrated.pt":
                priority = 1
            elif row["filename"].lower() == "last.pt":
                priority = 2
            else:
                priority = 3

            matches.append((priority, row["full_path"]))

    if matches:
        matches = sorted(matches, key=lambda x: x[0])
        resolved_checkpoints[config_name] = matches[0][1]
    else:
        resolved_checkpoints[config_name] = None

# Checkpoint resolution results
resolved_df = pd.DataFrame([
    {
        "configuration": name,
        "checkpoint": path,
        "found": path is not None
    }
    for name, path in resolved_checkpoints.items()
])

print("\n" + "=" * 95)
print("SELECTED CHECKPOINTS")
print("=" * 95)
display(resolved_df)

# Inspect checkpoint structures
print("\n" + "=" * 95)
print("CHECKPOINT STRUCTURES")
print("=" * 95)

for config_name, checkpoint_path in resolved_checkpoints.items():
    print(f"\n--- {config_name} ---")
    if checkpoint_path is None:
        print(" Checkpoint not found.")
        continue

    print(f"Path: {checkpoint_path}")
    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )
        if isinstance(checkpoint, dict):
            keys = list(checkpoint.keys())
            print("Type: dict")
            print(f"Main keys: {keys[:30]}")

            # Optional metadata
            for key in [
                "epoch",
                "best_epoch",
                "best_metric",
                "val_macro_f1",
                "macro_f1",
                "model_name",
                "architecture",
                "num_classes",
                "class_names",
                "class_to_idx"
            ]:
                if key in checkpoint:
                    value = checkpoint[key]
                    print(f"{key}: {value}")
        else:
            print(f"Object type: {type(checkpoint)}")
    except Exception as e:
        print(f"❌ Inspection error: {type(e).__name__}: {e}")

print("\n" + "=" * 95)
print(" CELL 6A COMPLETED")
print("=" * 95)

In [ ]:
from pathlib import Path
import pandas as pd
import torch

print("=" * 100)
print("CELL 6A-2 - EXACT CHECKPOINT RESOLUTION")
print("=" * 100)

# Exact folders for the selected configurations
selected_checkpoint_folders = {
    "Swin-Tiny + CutMix": "swin_tiny__cutmix",
    "ConvNeXt-Tiny + None": "convnext_tiny__none",
    "MobileNetV2 + CutMix": "mobilenet_v2__cutmix",
    "ViT-B/16 + CutMix": "vit_b16__cutmix",
}

# Search exclusively for best.pt
resolved_checkpoints = {}

for config_name, folder_name in selected_checkpoint_folders.items():
    matches = checkpoints_df[
        (checkpoints_df["parent_folder"] == folder_name)
        & (checkpoints_df["filename"].str.lower() == "best.pt")
    ]

    if len(matches) == 1:
        resolved_checkpoints[config_name] = matches.iloc[0]["full_path"]
    elif len(matches) == 0:
        resolved_checkpoints[config_name] = None
        print(f" best.pt was not found for {config_name}")
    else:
        raise RuntimeError(
            f" Multiple best.pt files were found for {config_name}:\n"
            f"{matches['full_path'].tolist()}"
        )

# Display resolved checkpoints
resolved_df = pd.DataFrame([
    {
        "configuration": config_name,
        "expected_folder": selected_checkpoint_folders[config_name],
        "checkpoint": checkpoint_path,
        "found": checkpoint_path is not None,
    }
    for config_name, checkpoint_path in resolved_checkpoints.items()
])

print("\nRESOLVED CHECKPOINTS\n")
display(resolved_df)

# Verify that all four checkpoints were located
assert all(path is not None for path in resolved_checkpoints.values()), (
    " The four checkpoints could not all be located."
)
assert len(resolved_checkpoints) == 4

print("\n All four best.pt files were successfully located.")

# Inspect checkpoint structure
checkpoint_inspection = []

print("\n" + "=" * 100)
print("CHECKPOINT STRUCTURE")
print("=" * 100)

for config_name, checkpoint_path in resolved_checkpoints.items():
    print(f"\n{'-' * 100}")
    print(f"{config_name}")
    print(f"Path:\n{checkpoint_path}")

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    print(f"Object type: {type(ckpt)}")

    if isinstance(ckpt, dict):
        keys = list(ckpt.keys())
        print(f"Keys ({len(keys)}):")
        print(keys)

        # Identify possible state_dict containers
        possible_state_keys = [
            "model_state_dict",
            "state_dict",
            "model",
            "model_state",
            "net",
            "network"
        ]
        detected_state_key = None

        for key in possible_state_keys:
            if key in ckpt:
                value = ckpt[key]
                if isinstance(value, dict):
                    detected_state_key = key
                    break

        # Check whether the root dictionary itself is a state_dict
        if detected_state_key is None:
            tensor_values = sum(torch.is_tensor(v) for v in ckpt.values())
            if len(ckpt) > 0 and tensor_values / len(ckpt) > 0.8:
                detected_state_key = "__root_state_dict__"

        print(f"Detected state dict: {detected_state_key}")

        # Display relevant metadata
        metadata_to_show = [
            "epoch",
            "best_epoch",
            "best_metric",
            "best_score",
            "val_macro_f1",
            "macro_f1",
            "model_name",
            "architecture",
            "num_classes",
            "class_names",
            "classes",
            "class_to_idx",
            "idx_to_class"
        ]
        metadata_found = {}

        for key in metadata_to_show:
            if key in ckpt:
                value = ckpt[key]
                metadata_found[key] = value
                print(f"\n{key}:")
                print(value)

        checkpoint_inspection.append({
            "configuration": config_name,
            "checkpoint": checkpoint_path,
            "n_top_level_keys": len(keys),
            "state_dict_key": detected_state_key,
            "has_class_names": "class_names" in ckpt,
            "has_classes": "classes" in ckpt,
            "has_class_to_idx": "class_to_idx" in ckpt,
            "has_num_classes": "num_classes" in ckpt
        })
    else:
        checkpoint_inspection.append({
            "configuration": config_name,
            "checkpoint": checkpoint_path,
            "n_top_level_keys": None,
            "state_dict_key": None,
            "has_class_names": False,
            "has_classes": False,
            "has_class_to_idx": False,
            "has_num_classes": False
        })

# Checkpoint structure summary
checkpoint_inspection_df = pd.DataFrame(checkpoint_inspection)

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
display(checkpoint_inspection_df)

# Save paths for subsequent cells
CHECKPOINT_MAP_PATH = Path(
    "/kaggle/working/external_validation/"
    "selected_external_evaluation_checkpoints.csv"
)

resolved_df.to_csv(
    CHECKPOINT_MAP_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 100)
print(" CELL 6A-2 COMPLETED")
print("=" * 100)
print(" Only best.pt files will be used.")
print(" No last.pt files were used.")
print(" No model was selected using the external dataset.")
print("\nCheckpoint map saved to:")
print(CHECKPOINT_MAP_PATH)


# **Cell 7 — Inference from the four models**

In [ ]:
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import timm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from PIL import Image, ImageOps

# Configuration

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

EXTERNAL_MANIFEST_PATH = (
    OUTPUT_DIR /
    "external_validation_manifest_no_development_exact_overlap.csv"
)

CHECKPOINT_MAP_PATH = (
    OUTPUT_DIR /
    "selected_external_evaluation_checkpoints.csv"
)

BATCH_SIZE = 64
NUM_WORKERS = 0
NUM_CLASSES = 24

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = (
    DEVICE.type == "cuda"
)

print("=" * 100)
print("CELL 6B - EXTERNAL INFERENCE")
print("=" * 100)

print(f"Device:              {DEVICE}")
print(f"AMP FP16:            {USE_AMP}")
print(f"Batch size:          {BATCH_SIZE}")
print(f"DataLoader workers:  {NUM_WORKERS}")

# Load external manifest and checkpoint map

external_eval_df = pd.read_csv(
    EXTERNAL_MANIFEST_PATH
)

checkpoint_map_df = pd.read_csv(
    CHECKPOINT_MAP_PATH
)

print("\nEXTERNAL DATASET")
print("-" * 100)

print(
    f"Images:             "
    f"{len(external_eval_df):,}"
)

print(
    f"Classes represented:   "
    f"{external_eval_df['true_class'].nunique()}"
)

assert len(external_eval_df) == 587
assert external_eval_df["image_id"].is_unique
assert external_eval_df["sha256"].nunique() == 587

# Exact model specifications

MODEL_SPECS = {

    "Swin-Tiny + CutMix":
        "swin_tiny_patch4_window7_224.ms_in1k",

    "ConvNeXt-Tiny + None":
        "convnext_tiny.fb_in1k",

    "MobileNetV2 + CutMix":
        "mobilenetv2_100.ra_in1k",

    "ViT-B/16 + CutMix":
        "vit_base_patch16_224.augreg_in1k",
}

# Checkpoint paths

checkpoint_paths = dict(
    zip(
        checkpoint_map_df[
            "configuration"
        ],
        checkpoint_map_df[
            "checkpoint"
        ]
    )
)

for config_name in MODEL_SPECS:

    assert config_name in checkpoint_paths, (
        f" Missing checkpoint for {config_name}"
    )

    checkpoint_path = Path(
        checkpoint_paths[
            config_name
        ]
    )

    assert checkpoint_path.exists(), (
        f" File does not exist:\n{checkpoint_path}"
    )

# Normalize checkpoint class mappings

def normalize_class_mapping(ckpt):

    class_to_index = ckpt[
        "class_to_index"
    ]

    index_to_class = ckpt[
        "index_to_class"
    ]

    # class_to_index

    class_to_index = {
        str(k): int(v)
        for k, v
        in class_to_index.items()
    }

    # index_to_class
    # Keys may be integers or strings

    index_to_class = {
        int(k): str(v)
        for k, v
        in index_to_class.items()
    }

    return (
        class_to_index,
        index_to_class
    )

# Read class mappings from all four checkpoints

checkpoint_metadata = {}

reference_class_to_index = None
reference_index_to_class = None

print("\n" + "=" * 100)
print("MAPEO DE CLASES")
print("=" * 100)

for config_name in MODEL_SPECS:

    checkpoint_path = (
        checkpoint_paths[
            config_name
        ]
    )

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    class_to_index, index_to_class = (
        normalize_class_mapping(
            ckpt
        )
    )

    print(
        f"\n{config_name}"
    )

    print(
        f"Checkpoint classes: "
        f"{len(class_to_index)}"
    )

    print(
        f"Best epoch: "
        f"{ckpt.get('best_epoch')}"
    )

    print(
        f"Best val Macro-F1: "
        f"{ckpt.get('best_val_macro_f1')}"
    )

    assert len(
        class_to_index
    ) == NUM_CLASSES

    assert len(
        index_to_class
    ) == NUM_CLASSES

    # Use the first checkpoint as the reference
    if reference_class_to_index is None:

        reference_class_to_index = (
            class_to_index
        )

        reference_index_to_class = (
            index_to_class
        )

    else:

        assert (
            class_to_index
            ==
            reference_class_to_index
        ), (
            f" The class_to_index mapping for "
            f"{config_name} is different."
        )

        assert (
            index_to_class
            ==
            reference_index_to_class
        ), (
            f" The index_to_class mapping for "
            f"{config_name} is different."
        )

    checkpoint_metadata[
        config_name
    ] = {

        "best_epoch":
            ckpt.get(
                "best_epoch"
            ),

        "best_val_macro_f1":
            ckpt.get(
                "best_val_macro_f1"
            ),

        "configuration":
            ckpt.get(
                "configuration"
            )
    }

print(
    "\n All four checkpoints use "
    "exactly the same class order."
)

# Display the actual order of all 24 classes

class_mapping_df = pd.DataFrame([
    {
        "class_index": idx,
        "class_name":
            reference_index_to_class[idx]
    }

    for idx in range(
        NUM_CLASSES
    )
])

print("\nClass order recovered from training:\n")

display(
    class_mapping_df
)

CLASS_MAPPING_PATH = (
    OUTPUT_DIR /
    "external_evaluation_class_mapping.csv"
)

class_mapping_df.to_csv(
    CLASS_MAPPING_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Verify external ground truth

external_classes = sorted(
    external_eval_df[
        "true_class"
    ].unique()
)

missing_external_classes = [
    cls
    for cls in external_classes
    if cls not in reference_class_to_index
]

assert len(
    missing_external_classes
) == 0, (
    " Some external labels are missing "
    "from the training classes:\n"
    f"{missing_external_classes}"
)

external_eval_df[
    "true_index"
] = (
    external_eval_df[
        "true_class"
    ]
    .map(
        reference_class_to_index
    )
)

assert external_eval_df[
    "true_index"
].notna().all()

external_eval_df[
    "true_index"
] = (
    external_eval_df[
        "true_index"
    ].astype(int)
)

print("\n External ground truth is compatible with the model.")
print(
    f" External classes represented: "
    f"{len(external_classes)}/{NUM_CLASSES}"
)

# Evaluation preprocessing
# Resize(258)
# CenterCrop(224)
# ToTensor()
# ImageNet normalization
# NO augmentations
# NO MixUp
# NO CutMix
# NO RandomErasing

eval_transform = transforms.Compose([

    transforms.Resize(
        258,
        interpolation=InterpolationMode.BICUBIC,
        antialias=True
    ),

    transforms.CenterCrop(
        224
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

# Robust image loading

def open_image_for_model(path):

    with Image.open(path) as img:

        img = ImageOps.exif_transpose(
            img
        )

        # Transparency:
        # composite over a white background

        if (
            img.mode in ("RGBA", "LA")
            or
            (
                img.mode == "P"
                and
                "transparency" in img.info
            )
        ):

            rgba = img.convert(
                "RGBA"
            )

            background = Image.new(
                "RGBA",
                rgba.size,
                (255, 255, 255, 255)
            )

            img = Image.alpha_composite(
                background,
                rgba
            ).convert(
                "RGB"
            )

        else:

            img = img.convert(
                "RGB"
            )

        return img.copy()

# DATASET

class ExternalValidationDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.transform = transform

    def __len__(self):

        return len(
            self.df
        )

    def __getitem__(self, idx):

        row = self.df.iloc[
            idx
        ]

        image = open_image_for_model(
            row["path"]
        )

        image = self.transform(
            image
        )

        return {
            "image":
                image,

            "image_id":
                row["image_id"],

            "true_class":
                row["true_class"],

            "true_index":
                int(
                    row["true_index"]
                ),

            "path":
                row["path"]
        }

external_dataset = ExternalValidationDataset(
    external_eval_df,
    eval_transform
)

external_loader = DataLoader(

    external_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type == "cuda"
    ),

    drop_last=False
)

print("\nDataLoader ready:")
print(
    f"  images = "
    f"{len(external_dataset)}"
)
print(
    f"  batches  = "
    f"{len(external_loader)}"
)

# Model loading function

def load_selected_model(
    config_name,
    timm_name,
    checkpoint_path
):

    print(
        f"\nCreating model: "
        f"{timm_name}"
    )

    model = timm.create_model(

        timm_name,

        pretrained=False,

        num_classes=NUM_CLASSES
    )

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    state_dict = ckpt[
        "model_state_dict"
    ]

    # Support checkpoints saved using DataParallel

    if (
        len(state_dict) > 0
        and
        all(
            str(k).startswith(
                "module."
            )
            for k in state_dict.keys()
        )
    ):

        state_dict = {
            k[len("module."):]: v
            for k, v
            in state_dict.items()
        }

    # Strict loading:
    # if the architecture does not match,
    # the cell must fail.
    model.load_state_dict(
        state_dict,
        strict=True
    )

    model.to(
        DEVICE
    )

    model.eval()

    return model, ckpt

# Inference

all_prediction_records = []
all_score_records = []

print("\n" + "=" * 100)
print("STARTING INFERENCE")
print("=" * 100)

for model_number, (
    config_name,
    timm_name
) in enumerate(
    MODEL_SPECS.items(),
    start=1
):

    print(
        f"\n[{model_number}/"
        f"{len(MODEL_SPECS)}] "
        f"{config_name}"
    )

    checkpoint_path = (
        checkpoint_paths[
            config_name
        ]
    )

    model, ckpt = load_selected_model(
        config_name,
        timm_name,
        checkpoint_path
    )

    n_processed = 0

    with torch.inference_mode():

        for batch in external_loader:

            images = batch[
                "image"
            ].to(
                DEVICE,
                non_blocking=True
            )

            # AMP FP16 on GPU

            if USE_AMP:

                autocast_context = (
                    torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16
                    )
                )

            else:

                autocast_context = (
                    nullcontext()
                )

            with autocast_context:

                logits = model(
                    images
                )

            # For metrics and calibration:
            # convert back to FP32 on CPU
            logits_cpu = (
                logits
                .float()
                .cpu()
            )

            probabilities = (
                torch.softmax(
                    logits_cpu,
                    dim=1
                )
            )

            confidence, predicted_index = (
                probabilities.max(
                    dim=1
                )
            )

            true_indices = (
                batch[
                    "true_index"
                ]
                .cpu()
                .numpy()
                .astype(int)
            )

            predicted_indices = (
                predicted_index
                .numpy()
                .astype(int)
            )

            confidence_values = (
                confidence
                .numpy()
            )

            logits_np = (
                logits_cpu
                .numpy()
            )

            probabilities_np = (
                probabilities
                .numpy()
            )

            batch_size_actual = len(
                predicted_indices
            )

            # Records

            for i in range(
                batch_size_actual
            ):

                image_id = (
                    batch[
                        "image_id"
                    ][i]
                )

                true_class = (
                    batch[
                        "true_class"
                    ][i]
                )

                true_idx = int(
                    true_indices[i]
                )

                pred_idx = int(
                    predicted_indices[i]
                )

                predicted_class = (
                    reference_index_to_class[
                        pred_idx
                    ]
                )

                # Compact prediction records

                all_prediction_records.append({

                    "image_id":
                        image_id,

                    "true_class":
                        true_class,

                    "true_index":
                        true_idx,

                    "configuration":
                        config_name,

                    "timm_model":
                        timm_name,

                    "best_epoch":
                        ckpt.get(
                            "best_epoch"
                        ),

                    "predicted_class":
                        predicted_class,

                    "predicted_index":
                        pred_idx,

                    "confidence":
                        float(
                            confidence_values[i]
                        ),

                    "correct":
                        bool(
                            pred_idx
                            == true_idx
                        ),

                    "path":
                        batch[
                            "path"
                        ][i]
                })

                # Full logits and probability records

                score_record = {

                    "image_id":
                        image_id,

                    "true_class":
                        true_class,

                    "true_index":
                        true_idx,

                    "configuration":
                        config_name,

                    "predicted_class":
                        predicted_class,

                    "predicted_index":
                        pred_idx,

                    "confidence":
                        float(
                            confidence_values[i]
                        ),

                    "correct":
                        bool(
                            pred_idx
                            == true_idx
                        )
                }

                for class_idx in range(
                    NUM_CLASSES
                ):

                    class_name = (
                        reference_index_to_class[
                            class_idx
                        ]
                    )

                    safe_name = (
                        class_name
                        .replace(
                            "__",
                            "_"
                        )
                        .replace(
                            " ",
                            "_"
                        )
                    )

                    score_record[
                        f"logit__{safe_name}"
                    ] = float(
                        logits_np[
                            i,
                            class_idx
                        ]
                    )

                    score_record[
                        f"prob__{safe_name}"
                    ] = float(
                        probabilities_np[
                            i,
                            class_idx
                        ]
                    )

                all_score_records.append(
                    score_record
                )

            n_processed += (
                batch_size_actual
            )

    print(
        f"✅ Predictions: "
        f"{n_processed}/{len(external_dataset)}"
    )

    # Release GPU memory before the next model

    del model

    if DEVICE.type == "cuda":

        torch.cuda.empty_cache()

# Build DataFrames

predictions_df = pd.DataFrame(
    all_prediction_records
)

scores_df = pd.DataFrame(
    all_score_records
)

expected_predictions = (
    len(external_eval_df)
    *
    len(MODEL_SPECS)
)

assert len(
    predictions_df
) == expected_predictions

assert len(
    scores_df
) == expected_predictions

print("\n" + "=" * 100)
print("INFERENCE COMPLETED")
print("=" * 100)

print(
    f"Images:                 "
    f"{len(external_eval_df)}"
)

print(
    f"Models:                  "
    f"{len(MODEL_SPECS)}"
)

print(
    f"Total predictions:     "
    f"{len(predictions_df):,}"
)

# Check one prediction per image and model

duplicate_prediction_rows = int(
    predictions_df.duplicated(
        subset=[
            "image_id",
            "configuration"
        ]
    ).sum()
)

assert duplicate_prediction_rows == 0

counts_by_model = (
    predictions_df
    .groupby(
        "configuration"
    )
    .size()
    .rename(
        "n_predictions"
    )
    .reset_index()
)

print("\nPredictions by model:\n")

display(
    counts_by_model
)

# Descriptive preview
# Quality control only.
# Formal metrics will be computed in Cell 6C.

qc_summary = (
    predictions_df
    .groupby(
        "configuration",
        as_index=False
    )
    .agg(

        n_images=(
            "image_id",
            "size"
        ),

        n_correct=(
            "correct",
            "sum"
        ),

        mean_confidence=(
            "confidence",
            "mean"
        )
    )
)

qc_summary[
    "raw_accuracy_check"
] = (
    qc_summary[
        "n_correct"
    ]
    /
    qc_summary[
        "n_images"
    ]
)

print("\nQUICK INFERENCE QUALITY CHECK:\n")

display(
    qc_summary
)

# Save results

PREDICTIONS_PATH = (
    OUTPUT_DIR /
    "external_validation_predictions_4models.csv"
)

SCORES_PATH = (
    OUTPUT_DIR /
    "external_validation_logits_probabilities_4models.csv"
)

QC_PATH = (
    OUTPUT_DIR /
    "external_validation_inference_qc.csv"
)

predictions_df.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig"
)

scores_df.to_csv(
    SCORES_PATH,
    index=False,
    encoding="utf-8-sig"
)

qc_summary.to_csv(
    QC_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Final result

print("\n" + "=" * 100)
print(" CELL 6B COMPLETED")
print("=" * 100)

print(
    " All four models were reconstructed."
)

print(
    " State dictionaries were loaded with strict matching."
)

print(
    " No training or fine-tuning was performed."
)

print(
    " No augmentations were used during inference."
)

print(
    " Logits and probabilities were preserved "
    "for all 24 classes."
)

print(
    " One prediction was generated per image and model."
)

print("\nPredictions:")
print(PREDICTIONS_PATH)

print("\nLogits and probabilities:")
print(SCORES_PATH)

print("\nInference quality check:")
print(QC_PATH)


# **Cell 8 — Global and class-specific metrics**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report
)


# Paths

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

PREDICTIONS_PATH = (
    OUTPUT_DIR /
    "external_validation_predictions_4models.csv"
)

CLASS_MAPPING_PATH = (
    OUTPUT_DIR /
    "external_evaluation_class_mapping.csv"
)


# Load data

predictions_df = pd.read_csv(
    PREDICTIONS_PATH
)

class_mapping_df = pd.read_csv(
    CLASS_MAPPING_PATH
)


print("=" * 100)
print("CELL 6C - FORMAL EXTERNAL VALIDATION METRICS")
print("=" * 100)

print(
    f"Predicciones cargadas: "
    f"{len(predictions_df):,}"
)

print(
    f"Modelos: "
    f"{predictions_df['configuration'].nunique()}"
)


# Complete 24-class mapping

index_to_class = dict(
    zip(
        class_mapping_df["class_index"].astype(int),
        class_mapping_df["class_name"]
    )
)

class_to_index = {
    v: k
    for k, v in index_to_class.items()
}

ALL_INDICES = sorted(
    index_to_class.keys()
)

ALL_CLASSES = [
    index_to_class[i]
    for i in ALL_INDICES
]


assert len(ALL_CLASSES) == 24


# Classes represented in the external dataset

represented_indices = sorted(
    predictions_df[
        "true_index"
    ]
    .astype(int)
    .unique()
    .tolist()
)

represented_classes = [
    index_to_class[i]
    for i in represented_indices
]


unsupported_indices = sorted(
    set(ALL_INDICES)
    -
    set(represented_indices)
)

unsupported_classes = [
    index_to_class[i]
    for i in unsupported_indices
]


print("\n" + "=" * 100)
print("EXTERNAL GROUND-TRUTH COVERAGE")
print("=" * 100)

print(
    f"Classifier classes:            "
    f"{len(ALL_CLASSES)}"
)

print(
    f"Represented classes:           "
    f"{len(represented_classes)}"
)

print(
    f"Classes without external support: "
    f"{len(unsupported_classes)}"
)


print("\nClasses without external support:")

for cls in unsupported_classes:
    print(f"  - {cls}")


# Verify equal image counts per model

counts = (
    predictions_df
    .groupby("configuration")
    ["image_id"]
    .nunique()
)

print("\nImages per model:")

print(counts)


assert (
    counts == 587
).all()


# Calculate global metrics

global_records = []


for configuration, model_df in (
    predictions_df
    .groupby(
        "configuration",
        sort=False
    )
):

    model_df = (
        model_df
        .sort_values("image_id")
        .reset_index(drop=True)
    )

    y_true = (
        model_df[
            "true_index"
        ]
        .astype(int)
        .to_numpy()
    )

    y_pred = (
        model_df[
            "predicted_index"
        ]
        .astype(int)
        .to_numpy()
    )


    # Accuracy

    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    # Balanced accuracy
    # Mean recall across classes present in y_true.

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred
        )
    )


    # Macro metrics:
    # Only the 20 externally supported classes.
    # Important:
    # Predictions into the four unsupported classes still
    # count as errors and lower recall for the true class.

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=represented_indices,
            average="macro",
            zero_division=0
        )
    )


    # Weighted F1

    _, _, weighted_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=represented_indices,
            average="weighted",
            zero_division=0
        )
    )


    # Errors

    n_images = len(
        model_df
    )

    n_correct = int(
        np.sum(
            y_true == y_pred
        )
    )

    n_errors = (
        n_images -
        n_correct
    )


    # Predictions into the four classes
    # without external ground truth

    unsupported_prediction_mask = (
        np.isin(
            y_pred,
            unsupported_indices
        )
    )

    n_unsupported_predictions = int(
        unsupported_prediction_mask.sum()
    )

    unsupported_prediction_rate = (
        n_unsupported_predictions
        /
        n_images
    )


    # Confidence

    mean_confidence = (
        model_df[
            "confidence"
        ].mean()
    )

    median_confidence = (
        model_df[
            "confidence"
        ].median()
    )


    # Confidence for correct predictions only
    correct_confidence = (
        model_df.loc[
            model_df["correct"] == True,
            "confidence"
        ].mean()
    )


    # Confidence for incorrect predictions only
    error_confidence = (
        model_df.loc[
            model_df["correct"] == False,
            "confidence"
        ].mean()
    )


    global_records.append({

        "configuration":
            configuration,

        "n_images":
            n_images,

        "n_supported_classes":
            len(represented_indices),

        "n_correct":
            n_correct,

        "n_errors":
            n_errors,

        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_accuracy,

        "macro_precision_20":
            macro_precision,

        "macro_recall_20":
            macro_recall,

        "macro_f1_20":
            macro_f1,

        "weighted_f1_20":
            weighted_f1,

        "mean_confidence":
            mean_confidence,

        "median_confidence":
            median_confidence,

        "mean_confidence_correct":
            correct_confidence,

        "mean_confidence_errors":
            error_confidence,

        "unsupported_predictions":
            n_unsupported_predictions,

        "unsupported_prediction_rate":
            unsupported_prediction_rate
    })


global_metrics_df = pd.DataFrame(
    global_records
)


# Sort by Macro-F1

global_metrics_df = (
    global_metrics_df
    .sort_values(
        "macro_f1_20",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 100)
print("GLOBAL METRICS")
print("=" * 100)

display(
    global_metrics_df
)


# Percentage display for readability

display_metrics_df = (
    global_metrics_df[
        [
            "configuration",
            "n_correct",
            "n_errors",
            "accuracy",
            "balanced_accuracy",
            "macro_precision_20",
            "macro_recall_20",
            "macro_f1_20",
            "weighted_f1_20",
            "unsupported_predictions",
            "unsupported_prediction_rate"
        ]
    ]
    .copy()
)


percentage_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision_20",
    "macro_recall_20",
    "macro_f1_20",
    "weighted_f1_20",
    "unsupported_prediction_rate"
]


for col in percentage_columns:

    display_metrics_df[
        col
    ] = (
        display_metrics_df[
            col
        ] * 100
    )


print("\nMetrics expressed as percentages:\n")

display(
    display_metrics_df.round(2)
)


# Per-class metrics

per_class_records = []


for configuration, model_df in (
    predictions_df
    .groupby(
        "configuration",
        sort=False
    )
):

    y_true = (
        model_df[
            "true_index"
        ]
        .astype(int)
        .to_numpy()
    )

    y_pred = (
        model_df[
            "predicted_index"
        ]
        .astype(int)
        .to_numpy()
    )


    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=represented_indices,
            average=None,
            zero_division=0
        )
    )


    for i, class_idx in enumerate(
        represented_indices
    ):

        per_class_records.append({

            "configuration":
                configuration,

            "class_index":
                class_idx,

            "class_name":
                index_to_class[
                    class_idx
                ],

            "precision":
                float(
                    precision[i]
                ),

            "recall":
                float(
                    recall[i]
                ),

            "f1":
                float(
                    f1[i]
                ),

            "support":
                int(
                    support[i]
                )
        })


per_class_metrics_df = pd.DataFrame(
    per_class_records
)


print("\n" + "=" * 100)
print("PER-CLASS METRICS")
print("=" * 100)

display(
    per_class_metrics_df
    .sort_values(
        [
            "configuration",
            "f1"
        ],
        ascending=[
            True,
            True
        ]
    )
)


# Predictions into unsupported classes

unsupported_prediction_records = []


for configuration, model_df in (
    predictions_df
    .groupby(
        "configuration",
        sort=False
    )
):

    subset = model_df[
        model_df[
            "predicted_index"
        ].isin(
            unsupported_indices
        )
    ]


    counts_unsupported = (
        subset[
            "predicted_class"
        ]
        .value_counts()
    )


    for class_name in unsupported_classes:

        n = int(
            counts_unsupported.get(
                class_name,
                0
            )
        )

        unsupported_prediction_records.append({

            "configuration":
                configuration,

            "unsupported_predicted_class":
                class_name,

            "n_predictions":
                n,

            "percentage_of_external_set":
                (
                    n /
                    len(model_df)
                ) * 100
        })


unsupported_predictions_df = pd.DataFrame(
    unsupported_prediction_records
)


print("\n" + "=" * 100)
print("PREDICTIONS FOR CLASSES WITHOUT EXTERNAL GROUND TRUTH")
print("=" * 100)

display(
    unsupported_predictions_df[
        unsupported_predictions_df[
            "n_predictions"
        ] > 0
    ]
    .sort_values(
        [
            "configuration",
            "n_predictions"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# Most difficult classes per model

print("\n" + "=" * 100)
print("FIVE LOWEST-F1 CLASSES PER MODEL")
print("=" * 100)


worst_classes_df = (
    per_class_metrics_df
    .sort_values(
        [
            "configuration",
            "f1"
        ]
    )
    .groupby(
        "configuration",
        group_keys=False
    )
    .head(5)
    .reset_index(drop=True)
)


display(
    worst_classes_df
)


# Save outputs

GLOBAL_METRICS_PATH = (
    OUTPUT_DIR /
    "external_validation_global_metrics_4models.csv"
)

PER_CLASS_PATH = (
    OUTPUT_DIR /
    "external_validation_per_class_metrics_4models.csv"
)

UNSUPPORTED_PATH = (
    OUTPUT_DIR /
    "external_validation_unsupported_predictions_4models.csv"
)

WORST_CLASSES_PATH = (
    OUTPUT_DIR /
    "external_validation_worst_classes_4models.csv"
)


global_metrics_df.to_csv(
    GLOBAL_METRICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

per_class_metrics_df.to_csv(
    PER_CLASS_PATH,
    index=False,
    encoding="utf-8-sig"
)

unsupported_predictions_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig"
)

worst_classes_df.to_csv(
    WORST_CLASSES_PATH,
    index=False,
    encoding="utf-8-sig"
)


# Final result

print("\n" + "=" * 100)
print(" CELL 6C COMPLETED")
print("=" * 100)

print(
    " Accuracy calculated over all 587 images."
)

print(
    " Balanced accuracy calculated over "
    "the represented classes."
)

print(
    " Macro-F1 calculated over the "
    "20 classes with external support."
)

print(
    " The four classes without external support "
    "were NOT treated as ground truth."
)

print(
    " Predictions into those four classes "
    "were explicitly counted."
)

print("\nGlobal metrics:")
print(GLOBAL_METRICS_PATH)

print("\nPer-class metrics:")
print(PER_CLASS_PATH)

print("\nPredictions into unsupported classes:")
print(UNSUPPORTED_PATH)

print("\nMost difficult classes:")
print(WORST_CLASSES_PATH)


# **Cell 9 — does the model fail on the fruit, on Healthy/Rotten, or on both?**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

PREDICTIONS_PATH = (
    OUTPUT_DIR /
    "external_validation_predictions_4models.csv"
)

predictions_df = pd.read_csv(
    PREDICTIONS_PATH
)

print("=" * 100)
print("CELL 6D - EXTERNAL ERROR DECOMPOSITION")
print("=" * 100)

# Split a class label into fruit and condition.

def split_class_name(class_name):

    parts = str(class_name).split("__")

    if len(parts) != 2:
        raise ValueError(
            f"Formato de clase inesperado: {class_name}"
        )

    fruit = parts[0]
    condition = parts[1]

    return fruit, condition

# Extract fruit and condition from true and predicted labels.

true_parts = predictions_df[
    "true_class"
].apply(split_class_name)

pred_parts = predictions_df[
    "predicted_class"
].apply(split_class_name)

predictions_df[
    "true_fruit"
] = true_parts.apply(
    lambda x: x[0]
)

predictions_df[
    "true_condition"
] = true_parts.apply(
    lambda x: x[1]
)

predictions_df[
    "predicted_fruit"
] = pred_parts.apply(
    lambda x: x[0]
)

predictions_df[
    "predicted_condition"
] = pred_parts.apply(
    lambda x: x[1]
)

# Categorize correct predictions and error types.

predictions_df[
    "fruit_correct"
] = (
    predictions_df[
        "true_fruit"
    ]
    ==
    predictions_df[
        "predicted_fruit"
    ]
)

predictions_df[
    "condition_correct"
] = (
    predictions_df[
        "true_condition"
    ]
    ==
    predictions_df[
        "predicted_condition"
    ]
)

def classify_error(row):

    fruit_ok = row[
        "fruit_correct"
    ]

    condition_ok = row[
        "condition_correct"
    ]

    if fruit_ok and condition_ok:
        return "fully_correct"

    elif fruit_ok and not condition_ok:
        return "condition_only_error"

    elif not fruit_ok and condition_ok:
        return "fruit_only_error"

    else:
        return "fruit_and_condition_error"

predictions_df[
    "error_type"
] = predictions_df.apply(
    classify_error,
    axis=1
)

# Compute hierarchical performance metrics.

hierarchical_records = []

for configuration, df_model in (
    predictions_df
    .groupby(
        "configuration",
        sort=False
    )
):

    class_acc = accuracy_score(
        df_model["true_class"],
        df_model["predicted_class"]
    )

    fruit_acc = accuracy_score(
        df_model["true_fruit"],
        df_model["predicted_fruit"]
    )

    fruit_bal_acc = balanced_accuracy_score(
        df_model["true_fruit"],
        df_model["predicted_fruit"]
    )

    fruit_macro_f1 = f1_score(
        df_model["true_fruit"],
        df_model["predicted_fruit"],
        average="macro",
        zero_division=0
    )

    condition_acc = accuracy_score(
        df_model["true_condition"],
        df_model["predicted_condition"]
    )

    condition_bal_acc = balanced_accuracy_score(
        df_model["true_condition"],
        df_model["predicted_condition"]
    )

    condition_macro_f1 = f1_score(
        df_model["true_condition"],
        df_model["predicted_condition"],
        average="macro",
        zero_division=0
    )

    error_counts = (
        df_model[
            "error_type"
        ]
        .value_counts()
    )

    hierarchical_records.append({

        "configuration":
            configuration,

        "n_images":
            len(df_model),

        "class_accuracy":
            class_acc,

        "fruit_accuracy":
            fruit_acc,

        "fruit_balanced_accuracy":
            fruit_bal_acc,

        "fruit_macro_f1":
            fruit_macro_f1,

        "condition_accuracy":
            condition_acc,

        "condition_balanced_accuracy":
            condition_bal_acc,

        "condition_macro_f1":
            condition_macro_f1,

        "fully_correct":
            int(
                error_counts.get(
                    "fully_correct",
                    0
                )
            ),

        "condition_only_error":
            int(
                error_counts.get(
                    "condition_only_error",
                    0
                )
            ),

        "fruit_only_error":
            int(
                error_counts.get(
                    "fruit_only_error",
                    0
                )
            ),

        "fruit_and_condition_error":
            int(
                error_counts.get(
                    "fruit_and_condition_error",
                    0
                )
            )
    })

hierarchical_metrics_df = pd.DataFrame(
    hierarchical_records
)

hierarchical_metrics_df = (
    hierarchical_metrics_df
    .sort_values(
        "class_accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)

# Display metrics as percentages.

display_hierarchical_df = (
    hierarchical_metrics_df
    .copy()
)

metric_cols = [
    "class_accuracy",
    "fruit_accuracy",
    "fruit_balanced_accuracy",
    "fruit_macro_f1",
    "condition_accuracy",
    "condition_balanced_accuracy",
    "condition_macro_f1"
]

for col in metric_cols:

    display_hierarchical_df[
        col
    ] = (
        display_hierarchical_df[
            col
        ] * 100
    )

print("\n" + "=" * 100)
print("PERFORMANCE BY LEVEL")
print("=" * 100)

display(
    display_hierarchical_df.round(2)
)

# Summarize error-type distribution.

error_distribution_df = (
    predictions_df
    .groupby(
        [
            "configuration",
            "error_type"
        ]
    )
    .size()
    .reset_index(
        name="n"
    )
)

error_distribution_df[
    "percentage"
] = (
    error_distribution_df[
        "n"
    ]
    /
    587
    *
    100
)

print("\n" + "=" * 100)
print("ERROR TYPE")
print("=" * 100)

display(
    error_distribution_df
    .sort_values(
        [
            "configuration",
            "n"
        ],
        ascending=[
            True,
            False
        ]
    )
)

# Identify the most frequent confusion pairs.

errors_only_df = predictions_df[
    predictions_df[
        "true_class"
    ]
    !=
    predictions_df[
        "predicted_class"
    ]
].copy()

confusion_pairs_df = (
    errors_only_df
    .groupby(
        [
            "configuration",
            "true_class",
            "predicted_class"
        ]
    )
    .size()
    .reset_index(
        name="n_errors"
    )
)

top_confusion_pairs_df = (
    confusion_pairs_df
    .sort_values(
        [
            "configuration",
            "n_errors"
        ],
        ascending=[
            True,
            False
        ]
    )
    .groupby(
        "configuration",
        group_keys=False
    )
    .head(15)
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("TOP 15 MOST FREQUENT CONFUSIONS PER MODEL")
print("=" * 100)

display(
    top_confusion_pairs_df
)

# Build the Swin-Tiny + CutMix confusion matrix.

BEST_CONFIG = (
    "Swin-Tiny + CutMix"
)

swin_df = predictions_df[
    predictions_df[
        "configuration"
    ] == BEST_CONFIG
].copy()

# Use all 24 classifier classes on both axes.
all_classes = sorted(
    set(
        predictions_df[
            "true_class"
        ].unique()
    )
    |
    set(
        predictions_df[
            "predicted_class"
        ].unique()
    )
)

# Restore the original class-index order.
class_mapping_path = (
    OUTPUT_DIR /
    "external_evaluation_class_mapping.csv"
)

class_mapping_df = pd.read_csv(
    class_mapping_path
)

all_classes = (
    class_mapping_df
    .sort_values(
        "class_index"
    )[
        "class_name"
    ]
    .tolist()
)

cm = confusion_matrix(
    swin_df[
        "true_class"
    ],
    swin_df[
        "predicted_class"
    ],
    labels=all_classes
)

# Normalize the confusion matrix by true-class support.

row_sums = cm.sum(
    axis=1,
    keepdims=True
)

cm_norm = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(
        cm,
        dtype=float
    ),
    where=row_sums != 0
)

# Plot the normalized confusion matrix.

fig, ax = plt.subplots(
    figsize=(15, 13)
)

im = ax.imshow(
    cm_norm,
    aspect="auto"
)

ax.set_xticks(
    np.arange(
        len(all_classes)
    )
)

ax.set_yticks(
    np.arange(
        len(all_classes)
    )
)

ax.set_xticklabels(
    all_classes,
    rotation=90,
    fontsize=8
)

ax.set_yticklabels(
    all_classes,
    fontsize=8
)

ax.set_xlabel(
    "Predicted class"
)

ax.set_ylabel(
    "True class"
)

ax.set_title(
    "External validation confusion matrix\n"
    "Swin-Tiny + CutMix"
)

fig.colorbar(
    im,
    ax=ax,
    label="Proportion"
)

plt.tight_layout()

CM_FIG_PATH = (
    OUTPUT_DIR /
    "external_validation_confusion_matrix_swin_cutmix.png"
)

plt.savefig(
    CM_FIG_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Save absolute and normalized confusion matrices.

cm_absolute_df = pd.DataFrame(
    cm,
    index=all_classes,
    columns=all_classes
)

cm_normalized_df = pd.DataFrame(
    cm_norm,
    index=all_classes,
    columns=all_classes
)

CM_ABS_PATH = (
    OUTPUT_DIR /
    "external_validation_confusion_matrix_swin_absolute.csv"
)

CM_NORM_PATH = (
    OUTPUT_DIR /
    "external_validation_confusion_matrix_swin_normalized.csv"
)

cm_absolute_df.to_csv(
    CM_ABS_PATH,
    encoding="utf-8-sig"
)

cm_normalized_df.to_csv(
    CM_NORM_PATH,
    encoding="utf-8-sig"
)

# Save hierarchical metrics, error types, and prediction analyses.

HIERARCHICAL_PATH = (
    OUTPUT_DIR /
    "external_validation_hierarchical_metrics.csv"
)

ERROR_TYPES_PATH = (
    OUTPUT_DIR /
    "external_validation_error_types.csv"
)

CONFUSION_PAIRS_PATH = (
    OUTPUT_DIR /
    "external_validation_top_confusion_pairs.csv"
)

PREDICTIONS_ENRICHED_PATH = (
    OUTPUT_DIR /
    "external_validation_predictions_error_analysis.csv"
)

hierarchical_metrics_df.to_csv(
    HIERARCHICAL_PATH,
    index=False,
    encoding="utf-8-sig"
)

error_distribution_df.to_csv(
    ERROR_TYPES_PATH,
    index=False,
    encoding="utf-8-sig"
)

top_confusion_pairs_df.to_csv(
    CONFUSION_PAIRS_PATH,
    index=False,
    encoding="utf-8-sig"
)

predictions_df.to_csv(
    PREDICTIONS_ENRICHED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 100)
print(" CELL 6D COMPLETED")
print("=" * 100)

print(" Fruit recognition and condition classification were analyzed separately.")
print(" Errors were classified into four types.")
print(" The most frequent misclassifications were identified.")
print(" The external confusion matrix for Swin-Tiny + CutMix was generated.")

print("\nHierarchical metrics:")
print(HIERARCHICAL_PATH)

print("\nError types:")
print(ERROR_TYPES_PATH)

print("\nTop confusions:")
print(CONFUSION_PAIRS_PATH)

print("\nConfusion matrix:")
print(CM_FIG_PATH)


# **Cell 10 -BACKUP — Save absolutely everything up to this point**

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import hashlib
import pandas as pd
import os


# Paths

SOURCE_DIR = Path(
    "/kaggle/working/external_validation"
)

BACKUP_ROOT = Path(
    "/kaggle/working"
)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

BACKUP_NAME = (
    f"external_validation_backup_through_6D_{timestamp}"
)

BACKUP_DIR = (
    BACKUP_ROOT /
    BACKUP_NAME
)

ZIP_BASE = (
    BACKUP_ROOT /
    BACKUP_NAME
)


print("=" * 100)
print("COMPLETE BACKUP THROUGH CELL 6D")
print("=" * 100)


# Verify source directory

if not SOURCE_DIR.exists():

    raise FileNotFoundError(
        f"Directory not found:\n{SOURCE_DIR}"
    )


# Copy the entire external_validation directory

shutil.copytree(
    SOURCE_DIR,
    BACKUP_DIR,
    dirs_exist_ok=True
)


print(
    f"\n Directory copied:\n"
    f"{BACKUP_DIR}"
)


# Required critical files

critical_files = [

    # Final manifest used
    "external_validation_manifest_no_development_exact_overlap.csv",

    # Model configuration
    "selected_external_evaluation_checkpoints.csv",

    "external_evaluation_class_mapping.csv",

    # Cell 6B
    "external_validation_predictions_4models.csv",

    "external_validation_logits_probabilities_4models.csv",

    "external_validation_inference_qc.csv",

    # Cell 6C
    "external_validation_global_metrics_4models.csv",

    "external_validation_per_class_metrics_4models.csv",

    "external_validation_unsupported_predictions_4models.csv",

    "external_validation_worst_classes_4models.csv",

    # Cell 6D
    "external_validation_hierarchical_metrics.csv",

    "external_validation_error_types.csv",

    "external_validation_top_confusion_pairs.csv",

    "external_validation_predictions_error_analysis.csv",

    "external_validation_confusion_matrix_swin_absolute.csv",

    "external_validation_confusion_matrix_swin_normalized.csv",

    "external_validation_confusion_matrix_swin_cutmix.png",
]


verification_records = []


print("\n" + "=" * 100)
print("CRITICAL FILE VERIFICATION")
print("=" * 100)


for filename in critical_files:

    file_path = (
        BACKUP_DIR /
        filename
    )

    exists = (
        file_path.exists()
    )

    size_mb = (
        file_path.stat().st_size
        /
        (1024 ** 2)
        if exists
        else None
    )

    verification_records.append({

        "filename":
            filename,

        "exists":
            exists,

        "size_mb":
            size_mb
    })


verification_df = pd.DataFrame(
    verification_records
)


display(
    verification_df
)


missing = verification_df[
    verification_df[
        "exists"
    ] == False
]


if len(missing) > 0:

    print(
        "\n Some critical files "
        "were not found:"
    )

    display(
        missing
    )

else:

    print(
        "\n All critical files "
        "are present."
    )


# Create a complete file inventory

inventory_records = []


for path in sorted(
    BACKUP_DIR.rglob("*")
):

    if not path.is_file():
        continue


    relative_path = (
        path.relative_to(
            BACKUP_DIR
        )
    )


    inventory_records.append({

        "relative_path":
            str(relative_path),

        "filename":
            path.name,

        "extension":
            path.suffix.lower(),

        "size_bytes":
            path.stat().st_size,

        "size_mb":
            path.stat().st_size
            /
            (1024 ** 2)
    })


inventory_df = pd.DataFrame(
    inventory_records
)


INVENTORY_PATH = (
    BACKUP_DIR /
    "BACKUP_FILE_INVENTORY.csv"
)


inventory_df.to_csv(
    INVENTORY_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"\nFiles included: "
    f"{len(inventory_df)}"
)

print(
    f"Approximate total size: "
    f"{inventory_df['size_mb'].sum():.2f} MB"
)


# Compute SHA-256 hashes for backup files

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


hash_records = []


print("\nComputing backup SHA-256 hashes...")


for path in sorted(
    BACKUP_DIR.rglob("*")
):

    if (
        not path.is_file()
        or
        path.name ==
        "BACKUP_SHA256_MANIFEST.csv"
    ):

        continue


    hash_records.append({

        "relative_path":
            str(
                path.relative_to(
                    BACKUP_DIR
                )
            ),

        "sha256":
            sha256_file(
                path
            )
    })


hash_df = pd.DataFrame(
    hash_records
)


HASH_PATH = (
    BACKUP_DIR /
    "BACKUP_SHA256_MANIFEST.csv"
)


hash_df.to_csv(
    HASH_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    f" SHA-256 computed for "
    f"{len(hash_df)} files."
)


# Backup README

README_PATH = (
    BACKUP_DIR /
    "README_BACKUP.txt"
)


readme_text = f"""
EXTERNAL VALIDATION BACKUP
==========================

Created:
{datetime.now().isoformat()}

Backup scope:
All files generated in the external-validation workflow
through Cell 6D, immediately before bootstrap analysis.

External evaluation subset:
587 images
20 externally represented ground-truth classes
24 classifier output classes
4 pretrained/fine-tuned configurations evaluated

Models:
- Swin-Tiny + CutMix
- ConvNeXt-Tiny + None
- MobileNetV2 + CutMix
- ViT-B/16 + CutMix

CRITICAL FILES FOR REPRODUCING THE ANALYSIS
============================================

1. external_validation_predictions_4models.csv
   One prediction per image and model.
   587 x 4 = 2,348 prediction records.

2. external_validation_logits_probabilities_4models.csv
   Logits and probabilities for all 24 output classes.
   This is critical for calibration, ECE, NLL and further
   probability-based analyses.

3. external_evaluation_class_mapping.csv
   Original model class-index mapping.

4. external_validation_manifest_no_development_exact_overlap.csv
   External evaluation manifest used for inference.

5. external_validation_global_metrics_4models.csv
   Accuracy, Balanced Accuracy and Macro-F1-related results.

6. external_validation_per_class_metrics_4models.csv
   Per-class metrics.

7. external_validation_hierarchical_metrics.csv
   Fruit-level and Healthy/Rotten-level evaluation.

8. external_validation_error_types.csv
   Decomposition into fully correct, condition-only,
   fruit-only and fruit+condition errors.

9. external_validation_top_confusion_pairs.csv
   Most frequent external errors.

10. external_validation_predictions_error_analysis.csv
    Enriched per-image predictions for error analysis.

11. external_validation_confusion_matrix_swin_absolute.csv
12. external_validation_confusion_matrix_swin_normalized.csv
13. external_validation_confusion_matrix_swin_cutmix.png

IMPORTANT
=========

The original neural-network checkpoints are NOT duplicated in this backup
because they already reside in the persistent Kaggle input dataset.

No bootstrap results are included in this backup.
This backup represents the experiment state immediately before Cell 6E.
"""


README_PATH.write_text(
    readme_text,
    encoding="utf-8"
)


# Create ZIP archive

print(
    "\nCreating ZIP archive..."
)


zip_created = shutil.make_archive(

    str(
        ZIP_BASE
    ),

    "zip",

    root_dir=str(
        BACKUP_DIR
    )
)


ZIP_PATH = Path(
    zip_created
)


print(
    f"\n ZIP created successfully."
)

print(
    f"ZIP size: "
    f"{ZIP_PATH.stat().st_size / (1024**2):.2f} MB"
)


# Quick check of the primary prediction file

pred_path = (
    BACKUP_DIR /
    "external_validation_predictions_4models.csv"
)


if pred_path.exists():

    pred_check = pd.read_csv(
        pred_path
    )

    print("\n" + "=" * 100)
    print("PREDICTION FILE VERIFICATION")
    print("=" * 100)

    print(
        f"Rows:              "
        f"{len(pred_check):,}"
    )

    print(
        f"Unique images:     "
        f"{pred_check['image_id'].nunique():,}"
    )

    print(
        f"Models:            "
        f"{pred_check['configuration'].nunique()}"
    )


    assert len(
        pred_check
    ) == 2348

    assert pred_check[
        "image_id"
    ].nunique() == 587

    assert pred_check[
        "configuration"
    ].nunique() == 4


    print(
        " Primary prediction file integrity verified."
    )


# Final result

print("\n" + "=" * 100)
print(" BACKUP COMPLETED")
print("=" * 100)

print(
    "\nBackup directory:"
)

print(
    BACKUP_DIR
)

print(
    "\nBACKUP ZIP:"
)

print(
    ZIP_PATH
)

print(
    "\n IMPORTANT:"
)

print(
    "DOWNLOAD THIS ZIP TO YOUR COMPUTER "
    "BEFORE RUNNING THE BOOTSTRAP."
)


In [ ]:
import warnings
import gc
import torch

# Suppress the repeated sklearn warning that flooded the output.
warnings.filterwarnings(
    "ignore",
    message="y_pred contains classes not in y_true",
    category=UserWarning
)

# Release unused RAM.
gc.collect()

# Clear the GPU cache, if available.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(" Memory cleaned up.")
print(" Repeated warning suppressed.")
print(" Ready to run bootstrap safely.")


# **Cell 11 — Bootstrap 95% CI + McNemar + Holm**

In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests

# CONFIGURATION

OUTPUT_DIR = Path(
    "/kaggle/working/external_validation"
)

PREDICTIONS_PATH = (
    OUTPUT_DIR /
    "external_validation_predictions_4models.csv"
)

N_BOOTSTRAP = 2000
SEED = 42
ALPHA = 0.05

print("=" * 105)
print("CELL 6E - BOOTSTRAP + McNEMAR")
print("=" * 105)

print(f"Bootstrap replicates: {N_BOOTSTRAP:,}")
print(f"Seed:                 {SEED}")
print(f"Confidence level:     {(1-ALPHA)*100:.0f}%")

# LOAD PREDICTIONS

pred_df = pd.read_csv(
    PREDICTIONS_PATH
)

MODELS = [
    "Swin-Tiny + CutMix",
    "ConvNeXt-Tiny + None",
    "MobileNetV2 + CutMix",
    "ViT-B/16 + CutMix"
]

# VERIFY EXACT ALIGNMENT ACROSS MODELS

image_sets = {}

for model in MODELS:

    model_images = set(
        pred_df.loc[
            pred_df["configuration"] == model,
            "image_id"
        ]
    )

    image_sets[model] = model_images

reference_set = image_sets[
    MODELS[0]
]

for model in MODELS[1:]:

    assert image_sets[model] == reference_set, (
        f" Images for {model} "
        "do not match those of the reference model."
    )

print(
    f"\n All four models share "
    f"the same {len(reference_set)} images."
)

# BUILD ALIGNED TABLE

reference_df = (
    pred_df[
        pred_df["configuration"] == MODELS[0]
    ][
        [
            "image_id",
            "true_index",
            "true_class"
        ]
    ]
    .sort_values("image_id")
    .reset_index(drop=True)
)

aligned = reference_df.copy()

for model in MODELS:

    temp = (
        pred_df[
            pred_df["configuration"] == model
        ][
            [
                "image_id",
                "predicted_index",
                "correct"
            ]
        ]
        .sort_values("image_id")
        .reset_index(drop=True)
    )

    assert (
        temp["image_id"].to_numpy()
        ==
        aligned["image_id"].to_numpy()
    ).all()

    safe_model = (
        model
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("/", "_")
        .replace("-", "_")
    )

    aligned[
        f"pred__{safe_model}"
    ] = (
        temp["predicted_index"]
        .astype(int)
        .to_numpy()
    )

    aligned[
        f"correct__{safe_model}"
    ] = (
        temp["correct"]
        .astype(bool)
        .to_numpy()
    )

# REPRESENTED EXTERNAL CLASSES

represented_labels = sorted(
    aligned[
        "true_index"
    ]
    .astype(int)
    .unique()
    .tolist()
)

print(
    f"Classes represented in bootstrap: "
    f"{len(represented_labels)}"
)

# CLASS INDICES FOR STRATIFIED BOOTSTRAP

class_indices = {}

for label in represented_labels:

    idx = np.where(
        aligned[
            "true_index"
        ].to_numpy()
        == label
    )[0]

    class_indices[
        label
    ] = idx

# METRIC FUNCTION

def calculate_metrics(
    y_true,
    y_pred
):

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred
        )
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=represented_labels,
        average="macro",
        zero_division=0
    )

    return (
        accuracy,
        balanced_accuracy,
        macro_f1
    )

# OBSERVED METRICS

observed_records = []

y_true_all = (
    aligned["true_index"]
    .astype(int)
    .to_numpy()
)

model_prediction_arrays = {}

for model in MODELS:

    safe_model = (
        model
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("/", "_")
        .replace("-", "_")
    )

    y_pred = (
        aligned[
            f"pred__{safe_model}"
        ]
        .astype(int)
        .to_numpy()
    )

    model_prediction_arrays[
        model
    ] = y_pred

    acc, bal_acc, macro_f1 = (
        calculate_metrics(
            y_true_all,
            y_pred
        )
    )

    observed_records.append({

        "configuration":
            model,

        "accuracy":
            acc,

        "balanced_accuracy":
            bal_acc,

        "macro_f1":
            macro_f1
    })

observed_df = pd.DataFrame(
    observed_records
)

# STRATIFIED BOOTSTRAP
# Each class is resampled with replacement
# while preserving its original sample size.
# The same indices are applied to all four models.

rng = np.random.default_rng(
    SEED
)

bootstrap_records = []

for b in range(
    N_BOOTSTRAP
):

    sampled_parts = []

    for label in represented_labels:

        original_idx = (
            class_indices[
                label
            ]
        )

        sampled_idx = rng.choice(
            original_idx,
            size=len(original_idx),
            replace=True
        )

        sampled_parts.append(
            sampled_idx
        )

    bootstrap_idx = np.concatenate(
        sampled_parts
    )

    y_true_boot = (
        y_true_all[
            bootstrap_idx
        ]
    )

    for model in MODELS:

        y_pred_boot = (
            model_prediction_arrays[
                model
            ][
                bootstrap_idx
            ]
        )

        acc, bal_acc, macro_f1 = (
            calculate_metrics(
                y_true_boot,
                y_pred_boot
            )
        )

        bootstrap_records.append({

            "bootstrap_rep":
                b,

            "configuration":
                model,

            "accuracy":
                acc,

            "balanced_accuracy":
                bal_acc,

            "macro_f1":
                macro_f1
        })

    if (
        (b + 1) % 500 == 0
        or
        b == N_BOOTSTRAP - 1
    ):

        print(
            f"Bootstrap: "
            f"{b+1:,}/{N_BOOTSTRAP:,}"
        )

bootstrap_df = pd.DataFrame(
    bootstrap_records
)

# PERCENTILE 95% CONFIDENCE INTERVALS

ci_records = []

for model in MODELS:

    observed_row = (
        observed_df[
            observed_df[
                "configuration"
            ] == model
        ]
        .iloc[0]
    )

    boot_model = (
        bootstrap_df[
            bootstrap_df[
                "configuration"
            ] == model
        ]
    )

    for metric in [
        "accuracy",
        "balanced_accuracy",
        "macro_f1"
    ]:

        values = (
            boot_model[
                metric
            ]
            .to_numpy()
        )

        lower = np.percentile(
            values,
            2.5
        )

        upper = np.percentile(
            values,
            97.5
        )

        ci_records.append({

            "configuration":
                model,

            "metric":
                metric,

            "estimate":
                float(
                    observed_row[
                        metric
                    ]
                ),

            "ci95_lower":
                float(
                    lower
                ),

            "ci95_upper":
                float(
                    upper
                )
        })

bootstrap_ci_df = pd.DataFrame(
    ci_records
)

print("\n" + "=" * 105)
print("BOOTSTRAP 95% CI")
print("=" * 105)

bootstrap_ci_display = (
    bootstrap_ci_df
    .copy()
)

for col in [
    "estimate",
    "ci95_lower",
    "ci95_upper"
]:

    bootstrap_ci_display[
        col
    ] *= 100

display(
    bootstrap_ci_display.round(2)
)

# PAIRED BOOTSTRAP:
# MACRO-F1 DIFFERENCES

paired_difference_records = []

for model_a, model_b in combinations(
    MODELS,
    2
):

    boot_a = (
        bootstrap_df[
            bootstrap_df[
                "configuration"
            ] == model_a
        ]
        .sort_values(
            "bootstrap_rep"
        )
    )

    boot_b = (
        bootstrap_df[
            bootstrap_df[
                "configuration"
            ] == model_b
        ]
        .sort_values(
            "bootstrap_rep"
        )
    )

    diff = (
        boot_a[
            "macro_f1"
        ].to_numpy()
        -
        boot_b[
            "macro_f1"
        ].to_numpy()
    )

    obs_a = float(
        observed_df.loc[
            observed_df[
                "configuration"
            ] == model_a,
            "macro_f1"
        ].iloc[0]
    )

    obs_b = float(
        observed_df.loc[
            observed_df[
                "configuration"
            ] == model_b,
            "macro_f1"
        ].iloc[0]
    )

    paired_difference_records.append({

        "model_a":
            model_a,

        "model_b":
            model_b,

        "delta_macro_f1":
            obs_a - obs_b,

        "ci95_lower":
            np.percentile(
                diff,
                2.5
            ),

        "ci95_upper":
            np.percentile(
                diff,
                97.5
            )
    })

paired_macro_f1_df = pd.DataFrame(
    paired_difference_records
)

print("\n" + "=" * 105)
print("PAIRED MACRO-F1 DIFFERENCES")
print("=" * 105)

paired_macro_display = (
    paired_macro_f1_df.copy()
)

for col in [
    "delta_macro_f1",
    "ci95_lower",
    "ci95_upper"
]:

    paired_macro_display[
        col
    ] *= 100

display(
    paired_macro_display.round(2)
)

# EXACT McNEMAR TEST

mcnemar_records = []

for model_a, model_b in combinations(
    MODELS,
    2
):

    pred_a = (
        model_prediction_arrays[
            model_a
        ]
    )

    pred_b = (
        model_prediction_arrays[
            model_b
        ]
    )

    correct_a = (
        pred_a ==
        y_true_all
    )

    correct_b = (
        pred_b ==
        y_true_all
    )

    # b:
    # A correct, B incorrect
    b = int(
        np.sum(
            correct_a
            &
            (~correct_b)
        )
    )

    # c:
    # A incorrect, B correct
    c = int(
        np.sum(
            (~correct_a)
            &
            correct_b
        )
    )

    discordant = (
        b + c
    )

    if discordant == 0:

        p_raw = 1.0

    else:

        p_raw = (
            binomtest(
                min(b, c),
                n=discordant,
                p=0.5,
                alternative="two-sided"
            )
            .pvalue
        )

    acc_a = accuracy_score(
        y_true_all,
        pred_a
    )

    acc_b = accuracy_score(
        y_true_all,
        pred_b
    )

    mcnemar_records.append({

        "model_a":
            model_a,

        "model_b":
            model_b,

        "b_Acorrect_Bwrong":
            b,

        "c_Awrong_Bcorrect":
            c,

        "discordant":
            discordant,

        "accuracy_A":
            acc_a,

        "accuracy_B":
            acc_b,

        "delta_accuracy_pp":
            (
                acc_a -
                acc_b
            ) * 100,

        "p_raw":
            p_raw
    })

mcnemar_df = pd.DataFrame(
    mcnemar_records
)

# HOLM-BONFERRONI

reject, p_holm, _, _ = (
    multipletests(
        mcnemar_df[
            "p_raw"
        ].to_numpy(),
        alpha=ALPHA,
        method="holm"
    )
)

mcnemar_df[
    "p_holm"
] = p_holm

mcnemar_df[
    "significant_holm"
] = reject

print("\n" + "=" * 105)
print("EXACT McNEMAR + HOLM")
print("=" * 105)

display(
    mcnemar_df[
        [
            "model_a",
            "model_b",
            "b_Acorrect_Bwrong",
            "c_Awrong_Bcorrect",
            "discordant",
            "delta_accuracy_pp",
            "p_raw",
            "p_holm",
            "significant_holm"
        ]
    ].sort_values(
        "p_holm"
    )
)

# SWIN COMPARISONS AGAINST OTHER MODELS

SWIN = "Swin-Tiny + CutMix"

swin_comparisons_df = (
    mcnemar_df[
        (
            mcnemar_df[
                "model_a"
            ] == SWIN
        )
        |
        (
            mcnemar_df[
                "model_b"
            ] == SWIN
        )
    ]
    .copy()
)

print("\n" + "=" * 105)
print("SWIN-TINY + CUTMIX vs OTHER MODELS")
print("=" * 105)

display(
    swin_comparisons_df[
        [
            "model_a",
            "model_b",
            "b_Acorrect_Bwrong",
            "c_Awrong_Bcorrect",
            "delta_accuracy_pp",
            "p_holm",
            "significant_holm"
        ]
    ]
)

# SAVE RESULTS

BOOTSTRAP_RAW_PATH = (
    OUTPUT_DIR /
    "external_validation_bootstrap_2000_raw.csv"
)

BOOTSTRAP_CI_PATH = (
    OUTPUT_DIR /
    "external_validation_bootstrap_95ci.csv"
)

PAIRED_F1_PATH = (
    OUTPUT_DIR /
    "external_validation_paired_macro_f1_bootstrap.csv"
)

MCNEMAR_PATH = (
    OUTPUT_DIR /
    "external_validation_mcnemar_holm.csv"
)

bootstrap_df.to_csv(
    BOOTSTRAP_RAW_PATH,
    index=False,
    encoding="utf-8-sig"
)

bootstrap_ci_df.to_csv(
    BOOTSTRAP_CI_PATH,
    index=False,
    encoding="utf-8-sig"
)

paired_macro_f1_df.to_csv(
    PAIRED_F1_PATH,
    index=False,
    encoding="utf-8-sig"
)

mcnemar_df.to_csv(
    MCNEMAR_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 105)
print(" CELL 6E COMPLETED")
print("=" * 105)

print(
    " Stratified bootstrap: "
    f"{N_BOOTSTRAP:,} replicates."
)

print(
    " 95% CIs for Accuracy, "
    "Balanced Accuracy, and Macro-F1."
)

print(
    " Paired bootstrap of "
    "Macro-F1 differences."
)

print(
    " Exact McNemar test on the same "
    "587 images."
)

print(
    " Holm-Bonferroni applied to "
    "all six comparisons."
)

print("\nBootstrap CI:")
print(BOOTSTRAP_CI_PATH)

print("\nPaired Macro-F1 differences:")
print(PAIRED_F1_PATH)

print("\nMcNemar + Holm:")
print(MCNEMAR_PATH)

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import hashlib
import platform
import sys
import os

import pandas as pd
import numpy as np


# Paths

SOURCE_DIR = Path(
    "/kaggle/working/external_validation"
)

BACKUP_ROOT = Path(
    "/kaggle/working"
)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

BACKUP_NAME = (
    f"FINAL_external_validation_complete_{timestamp}"
)

BACKUP_DIR = (
    BACKUP_ROOT /
    BACKUP_NAME
)

ZIP_BASE = (
    BACKUP_ROOT /
    BACKUP_NAME
)


print("=" * 110)
print("FINAL BACKUP - COMPLETE EXTERNAL VALIDATION")
print("=" * 110)


if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"Directory not found:\n{SOURCE_DIR}"
    )


# Copy the entire external_validation directory

print("\nCopying the entire external_validation directory...")


shutil.copytree(
    SOURCE_DIR,
    BACKUP_DIR,
    dirs_exist_ok=True
)


print(
    f" Directory copied to:\n"
    f"{BACKUP_DIR}"
)


# Critical experiment files

critical_files = [

    # Dataset / manifests

    "external_validation_manifest_internal_clean.csv",

    "external_validation_manifest_no_development_exact_overlap.csv",

    "external_evaluation_class_mapping.csv",

    "selected_external_evaluation_checkpoints.csv",


    # Inference - Cell 6B

    "external_validation_predictions_4models.csv",

    "external_validation_logits_probabilities_4models.csv",

    "external_validation_inference_qc.csv",


    # Metrics - Cell 6C

    "external_validation_global_metrics_4models.csv",

    "external_validation_per_class_metrics_4models.csv",

    "external_validation_unsupported_predictions_4models.csv",

    "external_validation_worst_classes_4models.csv",


    # Error analysis - Cell 6D

    "external_validation_hierarchical_metrics.csv",

    "external_validation_error_types.csv",

    "external_validation_top_confusion_pairs.csv",

    "external_validation_predictions_error_analysis.csv",

    "external_validation_confusion_matrix_swin_absolute.csv",

    "external_validation_confusion_matrix_swin_normalized.csv",

    "external_validation_confusion_matrix_swin_cutmix.png",


    # Statistical analysis - Cell 6E

    "external_validation_bootstrap_2000_raw.csv",

    "external_validation_bootstrap_95ci.csv",

    "external_validation_paired_macro_f1_bootstrap.csv",

    "external_validation_mcnemar_holm.csv",
]


verification_records = []


for filename in critical_files:

    path = BACKUP_DIR / filename

    verification_records.append({

        "filename":
            filename,

        "exists":
            path.exists(),

        "size_mb":
            (
                path.stat().st_size
                / (1024 ** 2)
                if path.exists()
                else np.nan
            )
    })


verification_df = pd.DataFrame(
    verification_records
)


print("\n" + "=" * 110)
print("CRITICAL FILE VERIFICATION")
print("=" * 110)

display(
    verification_df
)


missing_df = verification_df[
    verification_df["exists"] == False
]


if len(missing_df) == 0:

    print(
        "\n All critical files "
        "are present."
    )

else:

    print(
        "\n Some listed files "
        "are missing."
    )

    print(
        "This does not stop the backup because "
        "the ENTIRE directory is being copied."
    )

    display(
        missing_df
    )


# Result integrity checks

quality_checks = []


# Predictions

pred_path = (
    BACKUP_DIR /
    "external_validation_predictions_4models.csv"
)


if pred_path.exists():

    pred_df = pd.read_csv(
        pred_path
    )

    quality_checks.append({

        "check":
            "prediction_rows",

        "observed":
            len(pred_df),

        "expected":
            2348,

        "passed":
            len(pred_df) == 2348
    })


    quality_checks.append({

        "check":
            "unique_external_images",

        "observed":
            pred_df[
                "image_id"
            ].nunique(),

        "expected":
            587,

        "passed":
            (
                pred_df[
                    "image_id"
                ].nunique()
                == 587
            )
    })


    quality_checks.append({

        "check":
            "evaluated_models",

        "observed":
            pred_df[
                "configuration"
            ].nunique(),

        "expected":
            4,

        "passed":
            (
                pred_df[
                    "configuration"
                ].nunique()
                == 4
            )
    })


# BOOTSTRAP

boot_path = (
    BACKUP_DIR /
    "external_validation_bootstrap_2000_raw.csv"
)


if boot_path.exists():

    boot_df = pd.read_csv(
        boot_path
    )


    quality_checks.append({

        "check":
            "bootstrap_replicates",

        "observed":
            boot_df[
                "bootstrap_rep"
            ].nunique(),

        "expected":
            2000,

        "passed":
            (
                boot_df[
                    "bootstrap_rep"
                ].nunique()
                == 2000
            )
    })


    quality_checks.append({

        "check":
            "bootstrap_rows",

        "observed":
            len(boot_df),

        "expected":
            8000,

        "passed":
            len(boot_df) == 8000
    })


# MCNEMAR

mcnemar_path = (
    BACKUP_DIR /
    "external_validation_mcnemar_holm.csv"
)


if mcnemar_path.exists():

    mcnemar_df = pd.read_csv(
        mcnemar_path
    )


    quality_checks.append({

        "check":
            "mcnemar_pairwise_comparisons",

        "observed":
            len(mcnemar_df),

        "expected":
            6,

        "passed":
            len(mcnemar_df) == 6
    })


quality_checks_df = pd.DataFrame(
    quality_checks
)


print("\n" + "=" * 110)
print("INTEGRITY CHECKS")
print("=" * 110)

display(
    quality_checks_df
)


QUALITY_CHECK_PATH = (
    BACKUP_DIR /
    "FINAL_INTEGRITY_CHECKS.csv"
)


quality_checks_df.to_csv(
    QUALITY_CHECK_PATH,
    index=False,
    encoding="utf-8-sig"
)


if (
    len(quality_checks_df) > 0
    and
    quality_checks_df[
        "passed"
    ].all()
):

    print(
        "\n All main integrity "
        "checks passed."
    )

else:

    print(
        "\n Review the checks before "
        "considering the backup complete."
    )


# Generate final results summary table

global_metrics_path = (
    BACKUP_DIR /
    "external_validation_global_metrics_4models.csv"
)

bootstrap_ci_path = (
    BACKUP_DIR /
    "external_validation_bootstrap_95ci.csv"
)


if (
    global_metrics_path.exists()
    and
    bootstrap_ci_path.exists()
):

    global_df = pd.read_csv(
        global_metrics_path
    )

    ci_df = pd.read_csv(
        bootstrap_ci_path
    )


    # Pivot confidence intervals to wide format

    ci_wide = (
        ci_df
        .pivot(
            index="configuration",
            columns="metric",
            values=[
                "estimate",
                "ci95_lower",
                "ci95_upper"
            ]
        )
    )


    ci_wide.columns = [
        f"{a}_{b}"
        for a, b
        in ci_wide.columns
    ]


    ci_wide = (
        ci_wide
        .reset_index()
    )


    FINAL_SUMMARY_PATH = (
        BACKUP_DIR /
        "FINAL_EXTERNAL_VALIDATION_RESULTS_SUMMARY.csv"
    )


    ci_wide.to_csv(
        FINAL_SUMMARY_PATH,
        index=False,
        encoding="utf-8-sig"
    )


    print(
        "\n Final metrics summary created."
    )


# Environment information

environment_lines = []

environment_lines.append(
    f"Python: {sys.version}"
)

environment_lines.append(
    f"Platform: {platform.platform()}"
)


try:
    import torch

    environment_lines.append(
        f"PyTorch: {torch.__version__}"
    )

    environment_lines.append(
        f"CUDA available: "
        f"{torch.cuda.is_available()}"
    )

    if torch.cuda.is_available():

        environment_lines.append(
            f"GPU: "
            f"{torch.cuda.get_device_name(0)}"
        )

except Exception as e:
    environment_lines.append(
        f"PyTorch info error: {e}"
    )


try:
    import torchvision

    environment_lines.append(
        f"torchvision: "
        f"{torchvision.__version__}"
    )

except:
    pass


try:
    import timm

    environment_lines.append(
        f"timm: "
        f"{timm.__version__}"
    )

except:
    pass


try:
    import sklearn

    environment_lines.append(
        f"scikit-learn: "
        f"{sklearn.__version__}"
    )

except:
    pass


try:
    import scipy

    environment_lines.append(
        f"SciPy: "
        f"{scipy.__version__}"
    )

except:
    pass


try:
    import statsmodels

    environment_lines.append(
        f"statsmodels: "
        f"{statsmodels.__version__}"
    )

except:
    pass


ENV_PATH = (
    BACKUP_DIR /
    "FINAL_ENVIRONMENT_INFO.txt"
)


ENV_PATH.write_text(
    "\n".join(
        environment_lines
    ),
    encoding="utf-8"
)


# Complete file inventory

inventory_records = []


for path in sorted(
    BACKUP_DIR.rglob("*")
):

    if not path.is_file():
        continue


    inventory_records.append({

        "relative_path":
            str(
                path.relative_to(
                    BACKUP_DIR
                )
            ),

        "filename":
            path.name,

        "extension":
            path.suffix.lower(),

        "size_bytes":
            path.stat().st_size,

        "size_mb":
            path.stat().st_size
            / (1024 ** 2)
    })


inventory_df = pd.DataFrame(
    inventory_records
)


INVENTORY_PATH = (
    BACKUP_DIR /
    "FINAL_FILE_INVENTORY.csv"
)


inventory_df.to_csv(
    INVENTORY_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 110)
print("FILE INVENTORY")
print("=" * 110)

print(
    f"Files included: "
    f"{len(inventory_df):,}"
)

print(
    f"Uncompressed size: "
    f"{inventory_df['size_mb'].sum():.2f} MB"
)


# SHA-256 of all files

def calculate_sha256(
    filepath,
    chunk_size=1024 * 1024
):

    sha = hashlib.sha256()

    with open(
        filepath,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            sha.update(
                chunk
            )

    return sha.hexdigest()


print(
    "\nComputing SHA-256..."
)


hash_records = []


for path in sorted(
    BACKUP_DIR.rglob("*")
):

    if not path.is_file():
        continue

    if path.name == (
        "FINAL_SHA256_MANIFEST.csv"
    ):
        continue


    hash_records.append({

        "relative_path":
            str(
                path.relative_to(
                    BACKUP_DIR
                )
            ),

        "sha256":
            calculate_sha256(
                path
            )
    })


hash_df = pd.DataFrame(
    hash_records
)


HASH_PATH = (
    BACKUP_DIR /
    "FINAL_SHA256_MANIFEST.csv"
)


hash_df.to_csv(
    HASH_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    f" SHA-256 computed for "
    f"{len(hash_df):,} files."
)


# Final README

README_PATH = (
    BACKUP_DIR /
    "README_FINAL_EXTERNAL_VALIDATION.txt"
)


readme = """
FINAL EXTERNAL VALIDATION BACKUP
================================

PURPOSE
-------
Complete backup of the external-validation experiment
through the final statistical analysis (Cell 6E).

EVALUATION SUBSET
-----------------
External images evaluated: 587
Ground-truth classes represented: 20
Original classifier output classes: 24

Classes without ground-truth support in this external subset:
- Cucumber__Healthy
- Guava__Rotten
- Pomegranate__Rotten
- Strawberry__Rotten

MODELS
------
1. Swin-Tiny + CutMix
2. ConvNeXt-Tiny + None
3. MobileNetV2 + CutMix
4. ViT-B/16 + CutMix

The four configurations were selected using the internal
validation set before external evaluation.

No model training, fine-tuning or hyperparameter selection
was performed using the external evaluation images.


MAIN EXTERNAL RESULTS
---------------------

Swin-Tiny + CutMix
Accuracy:
55.88% [95% CI 53.15-58.60]

Balanced Accuracy:
56.11% [95% CI 53.30-58.94]

Macro-F1:
53.77% [95% CI 50.64-56.63]


ConvNeXt-Tiny + None
Accuracy:
50.60% [48.04-53.15]

Balanced Accuracy:
50.85% [48.33-53.37]

Macro-F1:
47.00% [44.25-49.50]


MobileNetV2 + CutMix
Accuracy:
45.83% [42.93-48.72]

Balanced Accuracy:
45.96% [43.06-48.89]

Macro-F1:
44.74% [41.78-47.55]


ViT-B/16 + CutMix
Accuracy:
48.21% [45.66-50.77]

Balanced Accuracy:
48.50% [45.94-51.07]

Macro-F1:
44.25% [41.34-46.86]


PAIRED BOOTSTRAP MACRO-F1 DIFFERENCES
-------------------------------------

Swin vs ConvNeXt:
+6.76 percentage points
95% CI [3.14, 10.11]

Swin vs MobileNetV2:
+9.02 percentage points
95% CI [5.47, 12.52]

Swin vs ViT:
+9.52 percentage points
95% CI [5.57, 13.28]


EXACT McNEMAR + HOLM
--------------------

Swin vs ConvNeXt:
Delta Accuracy = +5.28 pp
Holm p = 0.040457

Swin vs MobileNetV2:
Delta Accuracy = +10.05 pp
Holm p ~ 0.000001

Swin vs ViT:
Delta Accuracy = +7.67 pp
Holm p = 0.000593


ERROR DECOMPOSITION - SWIN-TINY + CUTMIX
----------------------------------------

24-class accuracy:
55.88%

Fruit identification accuracy:
82.28%

Healthy/Rotten condition accuracy:
70.87%

Fully correct:
328 / 587

Condition-only errors:
155 / 587

Fruit-only errors:
88 / 587

Fruit-and-condition errors:
16 / 587


STATISTICAL ANALYSIS
--------------------

Bootstrap:
2,000 stratified bootstrap replicates
Seed = 42
95% percentile confidence intervals
Same bootstrap samples applied to all four models.

McNemar:
Exact paired McNemar test
587 aligned external images
6 pairwise model comparisons
Holm-Bonferroni correction
alpha = 0.05


IMPORTANT FILES
---------------

external_validation_predictions_4models.csv
    Individual prediction for every image and model.

external_validation_logits_probabilities_4models.csv
    Full 24-class logits and probabilities.

external_validation_global_metrics_4models.csv
    Global external metrics.

external_validation_per_class_metrics_4models.csv
    Per-class metrics.

external_validation_hierarchical_metrics.csv
    Fruit and Healthy/Rotten-level analysis.

external_validation_error_types.csv
    Error decomposition.

external_validation_confusion_matrix_swin_cutmix.png
    External confusion matrix for Swin-Tiny + CutMix.

external_validation_bootstrap_2000_raw.csv
    All 2,000 bootstrap replicates.

external_validation_bootstrap_95ci.csv
    Bootstrap confidence intervals.

external_validation_paired_macro_f1_bootstrap.csv
    Paired Macro-F1 differences.

external_validation_mcnemar_holm.csv
    Exact McNemar + Holm results.


MODEL CHECKPOINTS
-----------------

The large .pt checkpoints are NOT duplicated in this ZIP.

They remain stored in the persistent Kaggle input dataset:
nuevas-corridas-2026

The exact checkpoint paths used are preserved in:
selected_external_evaluation_checkpoints.csv


IMPORTANT METHODOLOGICAL NOTE
-----------------------------

The external subset used for the reported evaluation contains
ground truth for 20 of the original 24 classes.

The classifiers were NOT reduced to 20 outputs.
They retained all 24 original outputs.

Predictions toward classes without ground-truth representation
remain genuine classifier predictions and count as errors when
they differ from the true external label.

Macro-F1 was calculated over the 20 externally represented
ground-truth classes.
"""


README_PATH.write_text(
    readme,
    encoding="utf-8"
)


print(
     Final README created."
)


# Create final ZIP

print("\n" + "=" * 110)
print("CREATING FINAL ZIP")
print("=" * 110)


zip_created = shutil.make_archive(

    str(
        ZIP_BASE
    ),

    "zip",

    root_dir=str(
        BACKUP_DIR
    )
)


ZIP_PATH = Path(
    zip_created
)


print(
    "\n ZIP created successfully."
)


print(
    f"ZIP size: "
    f"{ZIP_PATH.stat().st_size / (1024**2):.2f} MB"
)


# SHA-256 of final ZIP

zip_sha256 = calculate_sha256(
    ZIP_PATH
)


ZIP_INFO_PATH = (
    BACKUP_ROOT /
    f"{BACKUP_NAME}_ZIP_SHA256.txt"
)


ZIP_INFO_PATH.write_text(

    f"FILE: {ZIP_PATH.name}\n"
    f"SHA256: {zip_sha256}\n",

    encoding="utf-8"
)


print(
    f"\nSHA-256 ZIP:\n"
    f"{zip_sha256}"
)


# Final result

print("\n" + "=" * 110)
print(" FINAL BACKUP COMPLETED ")
print("=" * 110)


print(
    "\nDIRECTORY:"
)

print(
    BACKUP_DIR
)


print(
    "\nZIP FINAL:"
)

print(
    ZIP_PATH
)


print(
    "\nZIP SHA-256:"
)

print(
    zip_sha256
)


print(
    "\nSHA-256 FILE:"
)

print(
    ZIP_INFO_PATH
)


print(
    "\n" + "=" * 110
)

print(
    " DOWNLOAD THE ZIP TO YOUR COMPUTER NOW."
)

print(
    " Then use 'Save Version' in Kaggle "
    "to preserve the notebook as well."
)

print(
    "=" * 110
)

# **Sensitivity Analysis**

# **Cell 1 — Post-training analysis: Locate and inventory the 12 runs**

In [ ]:

from pathlib import Path
import pandas as pd


# Expected configurations

EXPECTED_CONFIGURATIONS = [
    "convnext_tiny__none",
    "convnext_tiny__mixup",
    "convnext_tiny__cutmix",

    "mobilenet_v2__none",
    "mobilenet_v2__mixup",
    "mobilenet_v2__cutmix",

    "swin_tiny__none",
    "swin_tiny__mixup",
    "swin_tiny__cutmix",

    "vit_b16__none",
    "vit_b16__mixup",
    "vit_b16__cutmix"
]



# Kaggle input root directory

KAGGLE_INPUT_ROOT = Path("/kaggle/input")

if not KAGGLE_INPUT_ROOT.exists():
    raise FileNotFoundError(
        "/kaggle/input does not exist."
    )



# Search for directories matching each exact run name

run_records = []

RUN_DIRECTORIES = {}

for configuration_id in EXPECTED_CONFIGURATIONS:

    matching_directories = [
        path
        for path in KAGGLE_INPUT_ROOT.rglob(configuration_id)
        if path.is_dir()
    ]

   
    structured_matches = []

    for path in matching_directories:

        has_backup_structure = (
            (path / "checkpoints").exists()
            and (path / "histories").exists()
            and (path / "metrics").exists()
            and (path / "metadata").exists()
        )

        if has_backup_structure:
            structured_matches.append(path)

    if len(structured_matches) == 1:

        run_directory = structured_matches[0]

    elif len(structured_matches) > 1:

        run_directory = None

    elif len(matching_directories) == 1:

        run_directory = matching_directories[0]

    else:

        run_directory = None


    # Locate essential files within each run
  
    if run_directory is not None:

        best_checkpoint_candidates = list(
            run_directory.rglob("best.pt")
        )

        last_checkpoint_candidates = list(
            run_directory.rglob("last.pt")
        )

        history_candidates = list(
            run_directory.rglob(
                f"{configuration_id}_history.csv"
            )
        )

        summary_candidates = list(
            run_directory.rglob(
                f"{configuration_id}_training_summary.json"
            )
        )

        best_checkpoint = (
            best_checkpoint_candidates[0]
            if len(best_checkpoint_candidates) == 1
            else None
        )

        last_checkpoint = (
            last_checkpoint_candidates[0]
            if len(last_checkpoint_candidates) == 1
            else None
        )

        history_path = (
            history_candidates[0]
            if len(history_candidates) == 1
            else None
        )

        summary_path = (
            summary_candidates[0]
            if len(summary_candidates) == 1
            else None
        )

    else:

        best_checkpoint = None
        last_checkpoint = None
        history_path = None
        summary_path = None


  
    #  Determine whether the run is complete
   
    run_complete = all([
        run_directory is not None,
        best_checkpoint is not None,
        last_checkpoint is not None,
        history_path is not None,
        summary_path is not None
    ])


    if run_complete:

        RUN_DIRECTORIES[
            configuration_id
        ] = run_directory


    run_records.append({
        "configuration_id": configuration_id,

        "run_directory_found": (
            run_directory is not None
        ),

        "best_pt": (
            best_checkpoint is not None
        ),

        "last_pt": (
            last_checkpoint is not None
        ),

        "history_csv": (
            history_path is not None
        ),

        "training_summary_json": (
            summary_path is not None
        ),

        "complete_backup": (
            run_complete
        ),

        "run_directory": (
            str(run_directory)
            if run_directory is not None
            else None
        ),

        "best_checkpoint_path": (
            str(best_checkpoint)
            if best_checkpoint is not None
            else None
        )
    })


# Create inventory

runs_inventory_df = pd.DataFrame(
    run_records
)


# Display results

print("=" * 80)
print("INVENTORY OF THE 12 RUNS")
print("=" * 80)

display(
    runs_inventory_df[
        [
            "configuration_id",
            "run_directory_found",
            "best_pt",
            "last_pt",
            "history_csv",
            "training_summary_json",
            "complete_backup"
        ]
    ]
)


# Summary

number_complete = int(
    runs_inventory_df[
        "complete_backup"
    ].sum()
)

print(
    "\nExpected runs:",
    len(EXPECTED_CONFIGURATIONS)
)

print(
    "Complete runs:",
    number_complete
)

# Display detected paths

print("\nDetected paths:")

for configuration_id, path in RUN_DIRECTORIES.items():

    print(
        f"{configuration_id:<28} -> {path}"
    )


# Final validation

incomplete_runs = runs_inventory_df[
    ~runs_inventory_df[
        "complete_backup"
    ]
]

if len(incomplete_runs) > 0:

    print("\nIncomplete or missing runs:")

    display(
        incomplete_runs
    )

    raise RuntimeError(
        "The 12 runs have not yet been "
        "successfully located."
    )


print("\n" + "=" * 80)
print("ALL 12 RUNS ARE COMPLETE AND AVAILABLE")
print("=" * 80)


# **Cell 2 — Verify best.pt checkpoints and generate the validation summary**

In [ ]:

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import torch


# Analysis results directory

ANALYSIS_ROOT = Path(
    "/kaggle/working/fruit_R3_recovered_predictions"
)

ANALYSIS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

TRAINING_SUMMARY_OUTPUT = (
    ANALYSIS_ROOT
    / "01_training_validation_summary.csv"
)

CHECKPOINT_INTEGRITY_OUTPUT = (
    ANALYSIS_ROOT
    / "02_best_checkpoint_integrity.csv"
)


# Tolerances

METRIC_TOLERANCE = 1e-7

# The following value was used during training: EARLY_STOPPING_MIN_DELTA = 1e-4 Therefore, a later epoch could achieve a slightly higher Macro-F1 (< 1e-4) without replacing best.pt.
SELECTION_MIN_DELTA = 1e-4


# Function to locate exactly one matching file

def find_unique_file(
    run_directory,
    pattern
):
    matches = list(
        Path(run_directory).rglob(
            pattern
        )
    )

    if len(matches) != 1:

        raise RuntimeError(
            f"Exactly one file was expected "
            f"for '{pattern}' in:\n"
            f"{run_directory}\n"
            f"Found: {len(matches)}"
        )

    return matches[0]


# Read the 12 runs

validation_records = []
integrity_records = []

for configuration_id in EXPECTED_CONFIGURATIONS:

    print(
        "Verifying:",
        configuration_id
    )

    run_directory = RUN_DIRECTORIES[
        configuration_id
    ]

  
    # Locate files
    
    best_checkpoint_path = (
        find_unique_file(
            run_directory,
            "best.pt"
        )
    )

    history_path = (
        find_unique_file(
            run_directory,
            f"{configuration_id}_history.csv"
        )
    )

    summary_path = (
        find_unique_file(
            run_directory,
            f"{configuration_id}_training_summary.json"
        )
    )


    # Read summary
    
    with open(
        summary_path,
        "r",
        encoding="utf-8"
    ) as file:

        summary = json.load(
            file
        )



    # Read training history

    history_df = pd.read_csv(
        history_path,
        low_memory=False
    )

    if history_df.empty:

        raise RuntimeError(
            f"Empty training history: {configuration_id}"
        )


  
    # Load checkpoint on CPU only

    checkpoint = torch.load(
        best_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )


 
    # Checkpoint information
 
    checkpoint_configuration = (
        checkpoint.get(
            "configuration",
            {}
        )
    )

    checkpoint_configuration_id = (
        checkpoint_configuration.get(
            "configuration_id"
        )
    )

    checkpoint_type = checkpoint.get(
        "checkpoint_type"
    )

    checkpoint_epoch = int(
        checkpoint.get(
            "epoch_number"
        )
    )

    checkpoint_best_epoch = int(
        checkpoint.get(
            "best_epoch"
        )
    )

    checkpoint_best_f1 = float(
        checkpoint.get(
            "best_val_macro_f1"
        )
    )



    # Training summary information
    
    summary_status = summary.get(
        "status"
    )

    summary_configuration_id = (
        summary.get(
            "configuration_id"
        )
    )

    summary_model_key = summary.get(
        "model_key"
    )

    summary_augmentation = summary.get(
        "augmentation"
    )

    summary_completed_epochs = int(
        summary.get(
            "completed_epochs"
        )
    )

    summary_best_epoch = int(
        summary.get(
            "best_epoch"
        )
    )

    summary_best_f1 = float(
        summary.get(
            "best_val_macro_f1"
        )
    )

    stop_reason = summary.get(
        "stop_reason"
    )

    # Locate the saved epoch in the training history

    best_epoch_rows = history_df[
        history_df["epoch"]
        == checkpoint_best_epoch
    ]

    if len(best_epoch_rows) != 1:

        raise RuntimeError(
            f"No unique row exists for the best "
            f"epoch of {configuration_id}."
        )

    best_history_row = (
        best_epoch_rows.iloc[0]
    )

    history_best_epoch_f1 = float(
        best_history_row[
            "val_macro_f1"
        ]
    )

    history_best_epoch_accuracy = float(
        best_history_row[
            "val_accuracy"
        ]
    )

    history_best_epoch_balanced_accuracy = float(
        best_history_row[
            "val_balanced_accuracy"
        ]
    )

    history_best_epoch_loss = float(
        best_history_row[
            "val_loss"
        ]
    )

    absolute_history_max_f1 = float(
        history_df[
            "val_macro_f1"
        ].max()
    )

    absolute_history_max_epoch = int(
        history_df.loc[
            history_df[
                "val_macro_f1"
            ].idxmax(),
            "epoch"
        ]
    )


   
    # Consistency checks
  
    checks = {
        "summary_completed": (
            summary_status
            == "completed"
        ),

        "checkpoint_is_best": (
            checkpoint_type
            == "best_model"
        ),

        "checkpoint_configuration_matches": (
            checkpoint_configuration_id
            == configuration_id
        ),

        "summary_configuration_matches": (
            summary_configuration_id
            == configuration_id
        ),

        "history_length_matches_summary": (
            len(history_df)
            == summary_completed_epochs
        ),

        "checkpoint_epoch_matches_best_epoch": (
            checkpoint_epoch
            == checkpoint_best_epoch
        ),

        "summary_best_epoch_matches_checkpoint": (
            summary_best_epoch
            == checkpoint_best_epoch
        ),

        "summary_f1_matches_checkpoint": (
            abs(
                summary_best_f1
                - checkpoint_best_f1
            )
            <= METRIC_TOLERANCE
        ),

        "history_f1_matches_checkpoint": (
            abs(
                history_best_epoch_f1
                - checkpoint_best_f1
            )
            <= METRIC_TOLERANCE
        ),

        "checkpoint_respects_selection_rule": (
            absolute_history_max_f1
            <= (
                checkpoint_best_f1
                + SELECTION_MIN_DELTA
                + METRIC_TOLERANCE
            )
        )
    }

    all_checks_passed = all(
        checks.values()
    )



    # Save audit results

    integrity_record = {
        "configuration_id": (
            configuration_id
        ),

        **checks,

        "all_checks_passed": (
            all_checks_passed
        )
    }

    integrity_records.append(
        integrity_record
    )



    # Comparative validation table

    validation_records.append({
        "configuration_id": (
            configuration_id
        ),

        "model": (
            summary_model_key
        ),

        "augmentation": (
            summary_augmentation
        ),

        "completed_epochs": (
            summary_completed_epochs
        ),

        "selected_epoch": (
            checkpoint_best_epoch
        ),

        "val_loss": (
            history_best_epoch_loss
        ),

        "val_accuracy": (
            history_best_epoch_accuracy
        ),

        "val_balanced_accuracy": (
            history_best_epoch_balanced_accuracy
        ),

        "val_macro_f1": (
            checkpoint_best_f1
        ),

        "absolute_history_max_epoch": (
            absolute_history_max_epoch
        ),

        "absolute_history_max_macro_f1": (
            absolute_history_max_f1
        ),

        "stop_reason": (
            stop_reason
        ),

        "best_checkpoint_mb": (
            best_checkpoint_path.stat().st_size
            / (1024 ** 2)
        ),

        "best_checkpoint_path": (
            str(
                best_checkpoint_path
            )
        )
    })


 
    del checkpoint
    del history_df

    gc.collect()


# Create DataFrames

validation_summary_df = pd.DataFrame(
    validation_records
)

checkpoint_integrity_df = pd.DataFrame(
    integrity_records
)



# Sort by validation Macro-F1

validation_summary_df = (
    validation_summary_df
    .sort_values(
        "val_macro_f1",
        ascending=False
    )
    .reset_index(drop=True)
)

validation_summary_df.insert(
    0,
    "validation_rank",
    np.arange(
        1,
        len(validation_summary_df) + 1
    )
)


# Save results

validation_summary_df.to_csv(
    TRAINING_SUMMARY_OUTPUT,
    index=False
)

checkpoint_integrity_df.to_csv(
    CHECKPOINT_INTEGRITY_OUTPUT,
    index=False
)


# Overall verification

number_valid_checkpoints = int(
    checkpoint_integrity_df[
        "all_checks_passed"
    ].sum()
)

print("\n" + "=" * 80)
print("BEST.PT CHECKPOINT VERIFICATION")
print("=" * 80)

print(
    "Expected checkpoints:",
    len(EXPECTED_CONFIGURATIONS)
)

print(
    "Fully consistent checkpoints:",
    number_valid_checkpoints
)


if number_valid_checkpoints != len(
    EXPECTED_CONFIGURATIONS
):

    print(
        "\nConfigurations with inconsistencies:"
    )

    display(
        checkpoint_integrity_df[
            ~checkpoint_integrity_df[
                "all_checks_passed"
            ]
        ]
    )

    raise RuntimeError(
        "At least one best.pt checkpoint requires review "
        "before test evaluation."
    )



# Display compact summary

print("\n" + "=" * 80)
print("VALIDATION RESULTS FOR THE 12 RUNS")
print("=" * 80)

display(
    validation_summary_df[
        [
            "validation_rank",
            "configuration_id",
            "completed_epochs",
            "selected_epoch",
            "val_accuracy",
            "val_balanced_accuracy",
            "val_macro_f1",
            "stop_reason"
        ]
    ]
)


print("\n" + "=" * 80)
print("ALL 12 BEST.PT CHECKPOINTS ARE CONSISTENT")
print("=" * 80)

print("\nSaved files:")
print(TRAINING_SUMMARY_OUTPUT)
print(CHECKPOINT_INTEGRITY_OUTPUT)


# **Cell 3 — Locate and verify the final clean test set**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# Expected values for the final dataset

EXPECTED_TOTAL_IMAGES = 28348
EXPECTED_TRAIN_IMAGES = 22672
EXPECTED_VAL_IMAGES = 2835
EXPECTED_TEST_IMAGES = 2841

EXPECTED_NUMBER_OF_CLASSES = 24


# Automatically locate the final manifest

KAGGLE_INPUT_ROOT = Path(
    "/kaggle/input"
)

manifest_candidates = list(
    KAGGLE_INPUT_ROOT.rglob(
        "14_final_verified_dataset_manifest.csv"
    )
)

print("=" * 80)
print("FINAL MANIFESTS FOUND")
print("=" * 80)

for path in manifest_candidates:
    print(path)


# 3. Select only the manifest associated with the physical clean_dataset/train-val-test dataset

valid_dataset_candidates = []

for manifest_path in manifest_candidates:


    for parent in manifest_path.parents:

        candidate_clean_dirs = [
            parent / "clean_dataset",
            parent / "clean_dataset" / "clean_dataset"
        ]

        for clean_dir in candidate_clean_dirs:

            if (
                (clean_dir / "train").is_dir()
                and (clean_dir / "val").is_dir()
                and (clean_dir / "test").is_dir()
            ):

                valid_dataset_candidates.append({
                    "manifest": manifest_path,
                    "clean_dataset": clean_dir
                })

                break

        else:
            continue

        break


# Remove exact duplicates
unique_candidates = []

seen_pairs = set()

for candidate in valid_dataset_candidates:

    key = (
        str(candidate["manifest"]),
        str(candidate["clean_dataset"])
    )

    if key not in seen_pairs:

        seen_pairs.add(key)
        unique_candidates.append(candidate)


if len(unique_candidates) == 0:

    raise FileNotFoundError(
        "The clean dataset with "
        "train, val, and test was not found.\n\n"
        "Add the revision_frutas_2026 dataset as an Input."
    )


print("\n" + "=" * 80)
print("CANDIDATES WITH PHYSICAL DATASET")
print("=" * 80)

for index, candidate in enumerate(
    unique_candidates
):

    print(
        f"\nCandidate {index + 1}"
    )

    print(
        "Manifest:",
        candidate["manifest"]
    )

    print(
        "Clean dataset:",
        candidate["clean_dataset"]
    )


# Select a candidate matching 28,348 images

selected_candidate = None
selected_manifest_df = None

for candidate in unique_candidates:

    candidate_df = pd.read_csv(
        candidate["manifest"],
        low_memory=False
    )

    if (
        len(candidate_df)
        == EXPECTED_TOTAL_IMAGES
    ):

        selected_candidate = candidate
        selected_manifest_df = candidate_df
        break


if selected_candidate is None:

    raise RuntimeError(
        "A physical dataset was found, but none "
        "contains the expected 28,348 images."
    )


FINAL_MANIFEST_PATH = (
    selected_candidate["manifest"]
)

CLEAN_DATASET_DIR = (
    selected_candidate["clean_dataset"]
)

dataset_df = (
    selected_manifest_df
    .copy()
)


# Verify required columns

required_columns = {
    "dataset_id",
    "split",
    "class_name",
    "sha256",
    "duplicate_component_id",
    "split_guard_group_id"
}

missing_columns = (
    required_columns
    - set(dataset_df.columns)
)

if missing_columns:

    raise RuntimeError(
        "Missing columns in the manifest:\n"
        f"{sorted(missing_columns)}"
    )


# Verify image counts by split

split_counts = (
    dataset_df["split"]
    .value_counts()
    .to_dict()
)

expected_split_counts = {
    "train": EXPECTED_TRAIN_IMAGES,
    "val": EXPECTED_VAL_IMAGES,
    "test": EXPECTED_TEST_IMAGES
}

if split_counts != expected_split_counts:

    raise RuntimeError(
        "Image counts by split do not match.\n\n"
        f"Expected: {expected_split_counts}\n"
        f"Found: {split_counts}"
    )


# Class mapping identical to training

CLASS_NAMES = sorted(
    dataset_df[
        "class_name"
    ].unique()
)

if len(CLASS_NAMES) != EXPECTED_NUMBER_OF_CLASSES:

    raise RuntimeError(
        "The 24 classes were not found."
    )


CLASS_TO_INDEX = {
    class_name: index
    for index, class_name
    in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    index: class_name
    for class_name, index
    in CLASS_TO_INDEX.items()
}


dataset_df["target"] = (
    dataset_df[
        "class_name"
    ].map(
        CLASS_TO_INDEX
    )
)


# Extract the test set exclusively

test_df = (
    dataset_df[
        dataset_df["split"]
        == "test"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(test_df) != EXPECTED_TEST_IMAGES:

    raise RuntimeError(
        "The test set does not contain exactly "
        "2,841 images."
    )


# Reconstruct physical file paths

def resolve_clean_image_path(row):

    # Standard case for the final manifest:
    # clean_relative_path
    if (
        "clean_relative_path"
        in dataset_df.columns
        and pd.notna(
            row.get(
                "clean_relative_path"
            )
        )
    ):

        relative_path = Path(
            str(
                row[
                    "clean_relative_path"
                ]
            )
        )

        # Avoid duplicating "clean_dataset"
        if (
            len(relative_path.parts) > 0
            and relative_path.parts[0]
            == "clean_dataset"
        ):

            relative_path = Path(
                *relative_path.parts[1:]
            )

        candidate = (
            CLEAN_DATASET_DIR
            / relative_path
        )

        if candidate.is_file():
            return str(candidate)


    # Alternative based on the copied original path
    if (
        "clean_path"
        in dataset_df.columns
        and pd.notna(
            row.get(
                "clean_path"
            )
        )
    ):

        old_path = Path(
            str(
                row[
                    "clean_path"
                ]
            )
        )

        # Search starting from the split name
        parts = old_path.parts

        for split_name in [
            "train",
            "val",
            "test"
        ]:

            if split_name in parts:

                split_position = (
                    parts.index(
                        split_name
                    )
                )

                relative_path = Path(
                    *parts[
                        split_position:
                    ]
                )

                candidate = (
                    CLEAN_DATASET_DIR
                    / relative_path
                )

                if candidate.is_file():
                    return str(
                        candidate
                    )


    # Alternative using the file name
    file_name_columns = [
        "clean_file_name",
        "filename",
        "file_name"
    ]

    for column in file_name_columns:

        if (
            column in dataset_df.columns
            and pd.notna(
                row.get(column)
            )
        ):

            candidate = (
                CLEAN_DATASET_DIR
                / str(row["split"])
                / str(row["class_name"])
                / str(row[column])
            )

            if candidate.is_file():

                return str(
                    candidate
                )


    return None


test_df[
    "input_image_path"
] = test_df.apply(
    resolve_clean_image_path,
    axis=1
)


# Verify physical files

test_df[
    "file_exists"
] = test_df[
    "input_image_path"
].apply(
    lambda path:
        path is not None
        and Path(path).is_file()
)


missing_test_files = test_df[
    ~test_df[
        "file_exists"
    ]
]


if len(missing_test_files) > 0:

    print(
        "\nFirst missing files:"
    )

    display(
        missing_test_files[
            [
                "dataset_id",
                "class_name",
                "input_image_path"
            ]
        ].head(20)
    )

    raise FileNotFoundError(
        f"Could not locate "
        f"{len(missing_test_files)} "
        "test images."
    )


# Verify uniqueness

if test_df[
    "dataset_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate dataset_id values exist in the test set."
    )


if test_df[
    "sha256"
].duplicated().any():

    raise RuntimeError(
        "Duplicate SHA-256 hashes exist within the test set."
    )


# Test set distribution

test_distribution_df = (
    test_df
    .groupby(
        [
            "target",
            "class_name"
        ],
        as_index=False
    )
    .agg(
        images=(
            "dataset_id",
            "size"
        )
    )
    .sort_values(
        "target"
    )
    .reset_index(drop=True)
)


# Overall summary

test_verification_df = pd.DataFrame({
    "item": [
        "manifest",
        "clean_dataset",
        "total_images",
        "train_images",
        "validation_images",
        "test_images",
        "classes_total",
        "classes_test",
        "test_files_found",
        "unique_test_dataset_id",
        "unique_test_sha256",
        "test_ready"
    ],

    "value": [
        str(
            FINAL_MANIFEST_PATH
        ),

        str(
            CLEAN_DATASET_DIR
        ),

        len(
            dataset_df
        ),

        split_counts[
            "train"
        ],

        split_counts[
            "val"
        ],

        split_counts[
            "test"
        ],

        dataset_df[
            "class_name"
        ].nunique(),

        test_df[
            "class_name"
        ].nunique(),

        int(
            test_df[
                "file_exists"
            ].sum()
        ),

        test_df[
            "dataset_id"
        ].nunique(),

        test_df[
            "sha256"
        ].nunique(),

        True
    ]
})


# Save the operational test manifest

TEST_MANIFEST_OUTPUT = (
    ANALYSIS_ROOT
    / "03_verified_test_manifest.csv"
)

TEST_DISTRIBUTION_OUTPUT = (
    ANALYSIS_ROOT
    / "04_test_class_distribution.csv"
)

test_df.to_csv(
    TEST_MANIFEST_OUTPUT,
    index=False
)

test_distribution_df.to_csv(
    TEST_DISTRIBUTION_OUTPUT,
    index=False
)


# Display results

print("\n" + "=" * 80)
print("CLEAN TEST SET VERIFIED")
print("=" * 80)

display(
    test_verification_df
)

print("\nClass distribution:")
display(
    test_distribution_df
)

print("\nOperational test manifest:")
print(
    TEST_MANIFEST_OUTPUT
)

print("\nSaved distribution:")
print(
    TEST_DISTRIBUTION_OUTPUT
)

# **Cell 4 — Final validation and test DataLoaders**

In [ ]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd

import torch

from torch.utils.data import (
    Dataset,
    DataLoader
)

from PIL import (
    Image,
    ImageOps,
    ImageFile
)

from torchvision import transforms
from torchvision.transforms import InterpolationMode


# Configuration identical to training

IMG_SIZE = 224

EVAL_RESIZE_SIZE = int(
    round(
        IMG_SIZE * 1.15
    )
)

EVAL_BATCH_SIZE = 64

NUM_WORKERS = 0

ImageFile.LOAD_TRUNCATED_IMAGES = True


IMAGENET_MEAN = (
    0.485,
    0.456,
    0.406
)

IMAGENET_STD = (
    0.229,
    0.224,
    0.225
)


# Evaluation transform

evaluation_transform = transforms.Compose([

    transforms.Resize(
        size=EVAL_RESIZE_SIZE,
        interpolation=InterpolationMode.BICUBIC,
        antialias=True
    ),

    transforms.CenterCrop(
        size=IMG_SIZE
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# General function to reconstruct physical file paths


def resolve_evaluation_image_path(row):

    # clean_relative_path
 
    if (
        "clean_relative_path" in dataset_df.columns
        and pd.notna(
            row.get(
                "clean_relative_path"
            )
        )
    ):

        relative_path = Path(
            str(
                row[
                    "clean_relative_path"
                ]
            )
        )

       
        if (
            len(relative_path.parts) > 0
            and relative_path.parts[0]
            == "clean_dataset"
        ):

            relative_path = Path(
                *relative_path.parts[1:]
            )

        candidate = (
            CLEAN_DATASET_DIR
            / relative_path
        )

        if candidate.is_file():

            return str(
                candidate
            )

    
    # clean_path
    if (
        "clean_path" in dataset_df.columns
        and pd.notna(
            row.get(
                "clean_path"
            )
        )
    ):

        old_path = Path(
            str(
                row[
                    "clean_path"
                ]
            )
        )

        parts = old_path.parts

        for split_name in [
            "train",
            "val",
            "test"
        ]:

            if split_name in parts:

                position = parts.index(
                    split_name
                )

                relative_path = Path(
                    *parts[position:]
                )

                candidate = (
                    CLEAN_DATASET_DIR
                    / relative_path
                )

                if candidate.is_file():

                    return str(
                        candidate
                    )


    # reconstruct the path using the filename
    
    candidate_filename_columns = [
        "clean_file_name",
        "filename",
        "file_name"
    ]

    for column in candidate_filename_columns:

        if (
            column in dataset_df.columns
            and pd.notna(
                row.get(column)
            )
        ):

            candidate = (
                CLEAN_DATASET_DIR
                / str(
                    row["split"]
                )
                / str(
                    row["class_name"]
                )
                / str(
                    row[column]
                )
            )

            if candidate.is_file():

                return str(
                    candidate
                )


    return None


# Prepare validation data

val_df = (
    dataset_df[
        dataset_df["split"]
        == "val"
    ]
    .copy()
    .reset_index(drop=True)
)

val_df[
    "input_image_path"
] = val_df.apply(
    resolve_evaluation_image_path,
    axis=1
)

val_df[
    "file_exists"
] = val_df[
    "input_image_path"
].apply(
    lambda path:
        path is not None
        and Path(path).is_file()
)


# Prepare test data

test_df = (
    dataset_df[
        dataset_df["split"]
        == "test"
    ]
    .copy()
    .reset_index(drop=True)
)

test_df[
    "input_image_path"
] = test_df.apply(
    resolve_evaluation_image_path,
    axis=1
)

test_df[
    "file_exists"
] = test_df[
    "input_image_path"
].apply(
    lambda path:
        path is not None
        and Path(path).is_file()
)


# Strict validation checks

if len(val_df) != 2835:

    raise RuntimeError(
        f"Validation contains {len(val_df)} images; "
        "2835 were expected."
    )


if len(test_df) != 2841:

    raise RuntimeError(
        f"Test contains {len(test_df)} images; "
        "2841 were expected."
    )


if not val_df[
    "file_exists"
].all():

    raise FileNotFoundError(
        "Not all physical validation "
        "images were found."
    )


if not test_df[
    "file_exists"
].all():

    raise FileNotFoundError(
        "Not all physical test "
        "images were found."
    )


if (
    val_df[
        "class_name"
    ].nunique()
    != 24
):

    raise RuntimeError(
        "Validation does not contain all 24 classes."
    )


if (
    test_df[
        "class_name"
    ].nunique()
    != 24
):

    raise RuntimeError(
        "Test does not contain all 24 classes."
    )


# Evaluation dataset

class FruitEvaluationDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform
    ):

        self.dataframe = (
            dataframe
            .copy()
            .reset_index(drop=True)
        )

        self.transform = transform


    def __len__(
        self
    ):

        return len(
            self.dataframe
        )


    @staticmethod
    def open_rgb_image(
        image_path
    ):

        with Image.open(
            image_path
        ) as image:

            image.load()

            try:

                image = (
                    ImageOps.exif_transpose(
                        image
                    )
                )

            except Exception:

                image = image.copy()


            # Composite transparent images onto a white background
            if (
                image.mode in {
                    "RGBA",
                    "LA"
                }
                or "transparency"
                in image.info
            ):

                rgba_image = (
                    image.convert(
                        "RGBA"
                    )
                )

                background = (
                    Image.new(
                        mode="RGBA",
                        size=rgba_image.size,
                        color=(
                            255,
                            255,
                            255,
                            255
                        )
                    )
                )

                image = (
                    Image.alpha_composite(
                        background,
                        rgba_image
                    )
                    .convert(
                        "RGB"
                    )
                )

            else:

                image = image.convert(
                    "RGB"
                )


            return image.copy()


    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[
            index
        ]

        image = self.open_rgb_image(
            row[
                "input_image_path"
            ]
        )

        image = self.transform(
            image
        )

        return {

            "image": image,

            "target": torch.tensor(
                int(
                    row["target"]
                ),
                dtype=torch.long
            ),

            "dataset_id": torch.tensor(
                int(
                    row["dataset_id"]
                ),
                dtype=torch.long
            ),

            "class_name": str(
                row[
                    "class_name"
                ]
            ),

            "sha256": str(
                row[
                    "sha256"
                ]
            ),

            "image_path": str(
                row[
                    "input_image_path"
                ]
            )
        }


# Create datasets

val_dataset = (
    FruitEvaluationDataset(
        dataframe=val_df,
        transform=evaluation_transform
    )
)

test_dataset = (
    FruitEvaluationDataset(
        dataframe=test_df,
        transform=evaluation_transform
    )
)


# Create DataLoaders

val_loader = DataLoader(

    dataset=val_dataset,

    batch_size=EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        torch.cuda.is_available()
    ),

    drop_last=False,

    persistent_workers=False
)


test_loader = DataLoader(

    dataset=test_dataset,

    batch_size=EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        torch.cuda.is_available()
    ),

    drop_last=False,

    persistent_workers=False
)

# Test one batch

val_preview_batch = next(
    iter(
        val_loader
    )
)

test_preview_batch = next(
    iter(
        test_loader
    )
)


# Batch validation checks

expected_image_shape = (
    3,
    IMG_SIZE,
    IMG_SIZE
)


for split_name, batch in {

    "validation":
        val_preview_batch,

    "test":
        test_preview_batch

}.items():

    observed_shape = tuple(
        batch[
            "image"
        ].shape[1:]
    )

    if (
        observed_shape
        != expected_image_shape
    ):

        raise RuntimeError(
            f"Incorrect dimensions in "
            f"{split_name}: "
            f"{observed_shape}"
        )


    if (
        batch[
            "target"
        ].min()
        < 0
    ):

        raise RuntimeError(
            f"Negative label in "
            f"{split_name}."
        )


    if (
        batch[
            "target"
        ].max()
        >= 24
    ):

        raise RuntimeError(
            f"Out-of-range label "
            f"in {split_name}."
        )


# Verify class mapping against the checkpoints

mapping_check_records = []

for configuration_id in EXPECTED_CONFIGURATIONS:

    run_directory = RUN_DIRECTORIES[
        configuration_id
    ]

    best_checkpoint_path = (
        find_unique_file(
            run_directory,
            "best.pt"
        )
    )

    checkpoint = torch.load(
        best_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    checkpoint_class_mapping = (
        checkpoint.get(
            "class_to_index",
            {}
        )
    )

    normalized_checkpoint_mapping = {
        str(class_name):
        int(class_index)

        for class_name, class_index
        in checkpoint_class_mapping.items()
    }

    mapping_matches = (
        normalized_checkpoint_mapping
        == CLASS_TO_INDEX
    )

    mapping_check_records.append({

        "configuration_id":
            configuration_id,

        "class_mapping_matches":
            mapping_matches
    })

    del checkpoint


class_mapping_check_df = (
    pd.DataFrame(
        mapping_check_records
    )
)


if not class_mapping_check_df[
    "class_mapping_matches"
].all():

    display(
        class_mapping_check_df
    )

    raise RuntimeError(
        "The class mapping does not match "
        "across all checkpoints."
    )


gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# Summary

loader_summary_df = pd.DataFrame({

    "split": [
        "validation",
        "test"
    ],

    "images": [
        len(
            val_dataset
        ),
        len(
            test_dataset
        )
    ],

    "classes": [
        val_df[
            "class_name"
        ].nunique(),

        test_df[
            "class_name"
        ].nunique()
    ],

    "batch_size": [
        EVAL_BATCH_SIZE,
        EVAL_BATCH_SIZE
    ],

    "batches": [
        len(
            val_loader
        ),

        len(
            test_loader
        )
    ],

    "num_workers": [
        NUM_WORKERS,
        NUM_WORKERS
    ],

    "shuffle": [
        False,
        False
    ]
})


batch_summary_df = pd.DataFrame({

    "split": [
        "validation",
        "test"
    ],

    "batch_shape": [

        str(
            tuple(
                val_preview_batch[
                    "image"
                ].shape
            )
        ),

        str(
            tuple(
                test_preview_batch[
                    "image"
                ].shape
            )
        )
    ],

    "first_dataset_id": [

        int(
            val_preview_batch[
                "dataset_id"
            ][0]
        ),

        int(
            test_preview_batch[
                "dataset_id"
            ][0]
        )
    ],

    "first_target": [

        int(
            val_preview_batch[
                "target"
            ][0]
        ),

        int(
            test_preview_batch[
                "target"
            ][0]
        )
    ]
})


# Save verification results

LOADER_SUMMARY_OUTPUT = (
    ANALYSIS_ROOT
    / "05_evaluation_loader_summary.csv"
)

CLASS_MAPPING_OUTPUT = (
    ANALYSIS_ROOT
    / "06_checkpoint_class_mapping_verification.csv"
)

loader_summary_df.to_csv(
    LOADER_SUMMARY_OUTPUT,
    index=False
)

class_mapping_check_df.to_csv(
    CLASS_MAPPING_OUTPUT,
    index=False
)


# Display results

print("=" * 80)
print("EVALUATION DATALOADERS CONFIGURED")
print("=" * 80)

print("\nSummary:")
display(
    loader_summary_df
)

print("\nBatch test:")
display(
    batch_summary_df
)

print("\nClass mapping:")
display(
    class_mapping_check_df
)

print(
    "\nAll 12 checkpoints use "
    "the same mapping of the 24 classes."
)

print(
    "\nMultiprocessing disabled:",
    NUM_WORKERS == 0
)

print("\nSaved files:")
print(
    LOADER_SUMMARY_OUTPUT
)
print(
    CLASS_MAPPING_OUTPUT
)

#  **Cell 5 — Final inference using the 12 best.pt checkpoints**

In [ ]:
from pathlib import Path
import gc
import time

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader

import timm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)


# Configuration

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = (
    torch.cuda.is_available()
)

NUM_CLASSES = 24

PREFERRED_BATCH_SIZE = 64

OOM_FALLBACK_BATCH_SIZE = 32

VALIDATION_REPRODUCTION_TOLERANCE = 5e-4


#  Results directories

RAW_OUTPUT_DIR = (
    ANALYSIS_ROOT
    / "raw_outputs"
)

PREDICTIONS_DIR = (
    ANALYSIS_ROOT
    / "predictions"
)

METRICS_DIR = (
    ANALYSIS_ROOT
    / "point_metrics"
)

for directory in [
    RAW_OUTPUT_DIR,
    PREDICTIONS_DIR,
    METRICS_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# Build loaders according to batch size

def build_evaluation_loaders(
    batch_size
):

    val_loader_local = DataLoader(

        dataset=val_dataset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            torch.cuda.is_available()
        ),

        drop_last=False,

        persistent_workers=False
    )


    test_loader_local = DataLoader(

        dataset=test_dataset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            torch.cuda.is_available()
        ),

        drop_last=False,

        persistent_workers=False
    )


    return (
        val_loader_local,
        test_loader_local
    )


# Create model from checkpoint

def load_model_from_best_checkpoint(
    checkpoint_path
):

    checkpoint = torch.load(

        checkpoint_path,

        map_location="cpu",

        weights_only=False
    )


    configuration = checkpoint[
        "configuration"
    ]


    timm_model_name = configuration[
        "timm_model"
    ]


    model = timm.create_model(

        timm_model_name,

        pretrained=False,

        num_classes=NUM_CLASSES
    )


    model.load_state_dict(

        checkpoint[
            "model_state_dict"
        ],

        strict=True
    )


    model = model.to(
        DEVICE
    )

    model.eval()


    return (
        model,
        checkpoint
    )


# Run inference

def run_inference(
    model,
    data_loader
):

    logits_parts = []

    targets_parts = []

    dataset_id_parts = []


    if torch.cuda.is_available():

        torch.cuda.synchronize()


    start_time = (
        time.perf_counter()
    )


    with torch.inference_mode():

        for batch in data_loader:

            images = batch[
                "image"
            ].to(

                DEVICE,

                non_blocking=True
            )


            targets = batch[
                "target"
            ]


            dataset_ids = batch[
                "dataset_id"
            ]


            with torch.amp.autocast(

                device_type=(
                    "cuda"
                    if DEVICE.type == "cuda"
                    else "cpu"
                ),

                dtype=torch.float16,

                enabled=USE_AMP
            ):

                logits = model(
                    images
                )


            logits_parts.append(

                logits
                .detach()
                .float()
                .cpu()
            )


            targets_parts.append(

                targets
                .detach()
                .cpu()
            )


            dataset_id_parts.append(

                dataset_ids
                .detach()
                .cpu()
            )


    if torch.cuda.is_available():

        torch.cuda.synchronize()


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    logits = torch.cat(
        logits_parts,
        dim=0
    )


    targets = torch.cat(
        targets_parts,
        dim=0
    )


    dataset_ids = torch.cat(
        dataset_id_parts,
        dim=0
    )


    probabilities = torch.softmax(

        logits,

        dim=1
    )


    confidence, predictions = (
        probabilities.max(
            dim=1
        )
    )


    return {

        "logits":
            logits.numpy(),

        "probabilities":
            probabilities.numpy(),

        "targets":
            targets.numpy(),

        "dataset_ids":
            dataset_ids.numpy(),

        "predictions":
            predictions.numpy(),

        "confidence":
            confidence.numpy(),

        "seconds":
            elapsed_seconds
    }


# Point estimates

def compute_point_metrics(
    targets,
    predictions
):

    return {

        "accuracy": float(
            accuracy_score(
                targets,
                predictions
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                targets,
                predictions
            )
        ),

        "macro_f1": float(
            f1_score(

                targets,

                predictions,

                labels=list(
                    range(
                        NUM_CLASSES
                    )
                ),

                average="macro",

                zero_division=0
            )
        )
    }


# Create predictions DataFrame

def build_prediction_dataframe(
    inference_result,
    reference_dataframe,
    split_name
):

    prediction_df = pd.DataFrame({

        "dataset_id":
            inference_result[
                "dataset_ids"
            ].astype(
                np.int64
            ),

        "target":
            inference_result[
                "targets"
            ].astype(
                np.int64
            ),

        "prediction":
            inference_result[
                "predictions"
            ].astype(
                np.int64
            ),

        "confidence":
            inference_result[
                "confidence"
            ].astype(
                np.float64
            )
    })


    prediction_df[
        "true_class"
    ] = prediction_df[
        "target"
    ].map(
        INDEX_TO_CLASS
    )


    prediction_df[
        "predicted_class"
    ] = prediction_df[
        "prediction"
    ].map(
        INDEX_TO_CLASS
    )


    prediction_df[
        "correct"
    ] = (

        prediction_df[
            "target"
        ]

        ==

        prediction_df[
            "prediction"
        ]
    )


    metadata_df = (

        reference_dataframe[
            [
                "dataset_id",
                "sha256",
                "class_name",
                "input_image_path"
            ]
        ]
        .copy()
    )


    prediction_df = prediction_df.merge(

        metadata_df,

        on="dataset_id",

        how="left",

        validate="one_to_one"
    )


    prediction_df[
        "split"
    ] = split_name


    return prediction_df


#  Prepare saved validation reference

validation_reference = (

    validation_summary_df
    .set_index(
        "configuration_id"
    )
)


# Run the 12 configurations

all_metric_records = []

reproduction_records = []


print("=" * 80)
print("INFERENCE USING THE 12 BEST.PT CHECKPOINTS")
print("=" * 80)


for run_number, configuration_id in enumerate(

    EXPECTED_CONFIGURATIONS,

    start=1

):

    print(
        f"\n[{run_number}/12] "
        f"{configuration_id}"
    )


    run_directory = (
        RUN_DIRECTORIES[
            configuration_id
        ]
    )


    best_checkpoint_path = (
        find_unique_file(
            run_directory,
            "best.pt"
        )
    )


    # Load model

    model, checkpoint = (
        load_model_from_best_checkpoint(
            best_checkpoint_path
        )
    )


    batch_size_used = (
        PREFERRED_BATCH_SIZE
    )


    try:

        (
            current_val_loader,
            current_test_loader
        ) = build_evaluation_loaders(
            batch_size_used
        )


        val_result = run_inference(

            model,
            current_val_loader
        )


        test_result = run_inference(

            model,
            current_test_loader
        )


    except RuntimeError as error:

        is_cuda_oom = (

            "out of memory"
            in str(error).lower()
        )


        if not is_cuda_oom:

            raise


        print(
            "CUDA OOM with batch size 64."
        )

        print(
            "Retrying with batch size 32..."
        )


        del current_val_loader
        del current_test_loader


        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


        batch_size_used = (
            OOM_FALLBACK_BATCH_SIZE
        )


        (
            current_val_loader,
            current_test_loader
        ) = build_evaluation_loaders(
            batch_size_used
        )


        val_result = run_inference(

            model,
            current_val_loader
        )


        test_result = run_inference(

            model,
            current_test_loader
        )


    #  Verify sample counts

    if len(
        val_result[
            "targets"
        ]
    ) != 2835:

        raise RuntimeError(
            f"{configuration_id}: "
            "incorrect validation sample count."
        )


    if len(
        test_result[
            "targets"
        ]
    ) != 2841:

        raise RuntimeError(
            f"{configuration_id}: "
            "incorrect test sample count."
        )


    # Metrics

    val_metrics = compute_point_metrics(

        val_result[
            "targets"
        ],

        val_result[
            "predictions"
        ]
    )


    test_metrics = compute_point_metrics(

        test_result[
            "targets"
        ],

        test_result[
            "predictions"
        ]
    )


    # Verify reproduction of validation result

    expected_val_macro_f1 = float(

        validation_reference.loc[
            configuration_id,
            "val_macro_f1"
        ]
    )


    reproduced_val_macro_f1 = (

        val_metrics[
            "macro_f1"
        ]
    )


    absolute_difference = abs(

        reproduced_val_macro_f1

        -

        expected_val_macro_f1
    )


    validation_reproduced = (

        absolute_difference
        <=
        VALIDATION_REPRODUCTION_TOLERANCE
    )


    reproduction_records.append({

        "configuration_id":
            configuration_id,

        "expected_val_macro_f1":
            expected_val_macro_f1,

        "reproduced_val_macro_f1":
            reproduced_val_macro_f1,

        "absolute_difference":
            absolute_difference,

        "within_tolerance":
            validation_reproduced
    })


    if not validation_reproduced:

        raise RuntimeError(

            f"{configuration_id}: "
            "best.pt did not reproduce the validation Macro-F1 "
            "within the specified tolerance.\n"
            f"Expected: {expected_val_macro_f1:.6f}\n"
            f"Reproduced: {reproduced_val_macro_f1:.6f}\n"
            f"Difference: {absolute_difference:.6f}"
        )


    # Save raw arrays

    raw_output_path = (

        RAW_OUTPUT_DIR
        / f"{configuration_id}_raw_outputs.npz"
    )


    np.savez_compressed(

        raw_output_path,

        val_dataset_ids=
            val_result[
                "dataset_ids"
            ],

        val_targets=
            val_result[
                "targets"
            ],

        val_predictions=
            val_result[
                "predictions"
            ],

        val_confidence=
            val_result[
                "confidence"
            ],

        val_logits=
            val_result[
                "logits"
            ],

        val_probabilities=
            val_result[
                "probabilities"
            ],


        test_dataset_ids=
            test_result[
                "dataset_ids"
            ],

        test_targets=
            test_result[
                "targets"
            ],

        test_predictions=
            test_result[
                "predictions"
            ],

        test_confidence=
            test_result[
                "confidence"
            ],

        test_logits=
            test_result[
                "logits"
            ],

        test_probabilities=
            test_result[
                "probabilities"
            ]
    )


    # Create prediction CSV files

    val_prediction_df = (
        build_prediction_dataframe(

            val_result,

            val_df,

            "validation"
        )
    )


    test_prediction_df = (
        build_prediction_dataframe(

            test_result,

            test_df,

            "test"
        )
    )


    val_prediction_path = (

        PREDICTIONS_DIR
        / (
            f"{configuration_id}"
            "_validation_predictions.csv"
        )
    )


    test_prediction_path = (

        PREDICTIONS_DIR
        / (
            f"{configuration_id}"
            "_test_predictions.csv"
        )
    )


    val_prediction_df.to_csv(

        val_prediction_path,

        index=False
    )


    test_prediction_df.to_csv(

        test_prediction_path,

        index=False
    )


    # Global metrics record

    all_metric_records.append({

        "configuration_id":
            configuration_id,

        "model":
            checkpoint[
                "configuration"
            ][
                "model_key"
            ],

        "augmentation":
            checkpoint[
                "configuration"
            ][
                "augmentation"
            ],

        "selected_epoch":
            checkpoint[
                "best_epoch"
            ],


        "val_accuracy":
            val_metrics[
                "accuracy"
            ],

        "val_balanced_accuracy":
            val_metrics[
                "balanced_accuracy"
            ],

        "val_macro_f1":
            val_metrics[
                "macro_f1"
            ],


        "test_accuracy":
            test_metrics[
                "accuracy"
            ],

        "test_balanced_accuracy":
            test_metrics[
                "balanced_accuracy"
            ],

        "test_macro_f1":
            test_metrics[
                "macro_f1"
            ],


        "evaluation_batch_size":
            batch_size_used,


        "val_wall_clock_seconds":
            val_result[
                "seconds"
            ],

        "test_wall_clock_seconds":
            test_result[
                "seconds"
            ],

        "raw_outputs_path":
            str(
                raw_output_path
            ),

        "test_predictions_path":
            str(
                test_prediction_path
            )
    })


    print(
        "  Val Macro-F1:",
        f"{val_metrics['macro_f1']:.6f}"
    )

    print(
        "  Test Accuracy:",
        f"{test_metrics['accuracy']:.6f}"
    )

    print(
        "  Test Balanced Accuracy:",
        f"{test_metrics['balanced_accuracy']:.6f}"
    )

    print(
        "  Test Macro-F1:",
        f"{test_metrics['macro_f1']:.6f}"
    )

    print(
        "  Batch:",
        batch_size_used
    )


    #  Release GPU memory before the next model

    del model
    del checkpoint

    del val_result
    del test_result

    del val_prediction_df
    del test_prediction_df

    del current_val_loader
    del current_test_loader


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# Create global tables

point_metrics_df = pd.DataFrame(
    all_metric_records
)


reproduction_df = pd.DataFrame(
    reproduction_records
)


#  Add validation ranking

validation_rank_mapping = (

    validation_summary_df
    .set_index(
        "configuration_id"
    )[
        "validation_rank"
    ]
    .to_dict()
)


point_metrics_df[
    "validation_rank"
] = point_metrics_df[
    "configuration_id"
].map(
    validation_rank_mapping
)



point_metrics_df[
    "test_rank_descriptive"
] = (

    point_metrics_df[
        "test_macro_f1"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


point_metrics_df = (

    point_metrics_df
    .sort_values(
        "validation_rank"
    )
    .reset_index(
        drop=True
    )
)


# Save tables

POINT_METRICS_OUTPUT = (

    METRICS_DIR
    / "07_all_models_point_metrics.csv"
)


REPRODUCTION_OUTPUT = (

    METRICS_DIR
    / "08_validation_reproduction_check.csv"
)


point_metrics_df.to_csv(

    POINT_METRICS_OUTPUT,

    index=False
)


reproduction_df.to_csv(

    REPRODUCTION_OUTPUT,

    index=False
)

# Overall verification


if not reproduction_df[
    "within_tolerance"
].all():

    raise RuntimeError(
        "At least one checkpoint did not correctly "
        "reproduce the validation results."
    )


# Display results

print("\n" + "=" * 80)
print("INFERENCE COMPLETED FOR ALL 12 MODELS")
print("=" * 80)


print("\nValidation reproduction:")

display(

    reproduction_df[
        [
            "configuration_id",
            "expected_val_macro_f1",
            "reproduced_val_macro_f1",
            "absolute_difference",
            "within_tolerance"
        ]
    ]
)


print("\nPoint estimates:")

display(

    point_metrics_df[
        [
            "validation_rank",
            "configuration_id",
            "selected_epoch",
            "val_macro_f1",
            "test_accuracy",
            "test_balanced_accuracy",
            "test_macro_f1",
            "test_rank_descriptive"
        ]
    ]
)


print("\nMain output files:")

print(
    POINT_METRICS_OUTPUT
)

print(
    REPRODUCTION_OUTPUT
)

print(
    RAW_OUTPUT_DIR
)

print(
    PREDICTIONS_DIR
)


print(
    "\nLogits and probabilities for all 12 configurations "
    "have been saved."
)

In [ ]:

from pathlib import Path
import shutil

base = Path("/kaggle/working/fruit_R3_recovered_predictions")
predictions = base / "predictions"

files = list(predictions.glob("*_test_predictions.csv"))
print(f"Archivos encontrados: {len(files)}")

assert len(files) == 12, "Se esperaban 12 archivos de predicciones del test."

zip_path = shutil.make_archive(
    str(base / "R3_test_predictions_12_models"),
    "zip",
    root_dir=predictions,
    base_dir="."
)

print("Archivo generado:", zip_path)


# **Cell 6 - Consolidate predictions and similarity groups**

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import zipfile
import io

# LOCATE FILES IN KAGGLE

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

manifest_name = "12_clean_grouped_split_manifest"
zip_name = "R3_test_predictions_12_models.zip"

manifests = []
zip_files = []
prediction_files = []

for root in roots:
    if not root.exists():
        continue

    manifests.extend(
        root.rglob("*12_clean_grouped_split_manifest*.csv")
    )

    zip_files.extend(
        root.rglob(zip_name)
    )

    prediction_files.extend(
        root.rglob("*_test_predictions.csv")
    )

assert manifests, (
    "The split manifest was not found. "
    "Add it to Kaggle Input."
)

manifest_path = manifests[0]

print("Manifest:", manifest_path)

# LOAD TEST MANIFEST

manifest = pd.read_csv(manifest_path)

required = [
    "sha256",
    "split",
    "split_guard_group_id",
    "class_name"
]

assert set(required).issubset(manifest.columns)

manifest = manifest.loc[
    manifest["split"].astype(str).str.lower() == "test",
    required
].copy()

assert len(manifest) == 2841, (
    f"Expected 2841 images, "
    f"but found {len(manifest)}."
)

assert manifest["sha256"].is_unique
assert manifest["split_guard_group_id"].notna().all()

manifest = manifest.set_index("sha256")

print("Test images:", len(manifest))
print(
    "Similarity groups:",
    manifest["split_guard_group_id"].nunique()
)

#  LOAD THE 12 PREDICTION FILES

predictions = {}

if zip_files:

    print("ZIP file found:", zip_files[0])

    with zipfile.ZipFile(zip_files[0]) as z:

        names = [
            n for n in z.namelist()
            if n.endswith("_test_predictions.csv")
        ]

        assert len(names) == 12, (
            f"Found {len(names)} "
            "test files in the ZIP archive."
        )

        for name in sorted(names):

            model = Path(name).name.replace(
                "_test_predictions.csv", ""
            )

            with z.open(name) as f:
                predictions[model] = pd.read_csv(f)

else:

    print("Searching for individual prediction files...")

    for path in sorted(prediction_files):

        model = path.name.replace(
            "_test_predictions.csv", ""
        )

        predictions[model] = pd.read_csv(path)

assert len(predictions) == 12, (
    f"Expected 12 models; "
    f"found {len(predictions)}."
)

print("Models found:", len(predictions))

# CONSOLIDATE PREDICTIONS BY SHA-256

joined = None

for model, pred in sorted(predictions.items()):

    columns = [
        "sha256",
        "target",
        "true_class",
        "correct"
    ]

    assert set(columns).issubset(pred.columns), (
        f"Missing columns in {model}"
    )

    temp = pred[columns].copy()

    assert len(temp) == 2841
    assert temp["sha256"].is_unique

    temp["correct"] = (
        temp["correct"]
        .astype(str)
        .str.lower()
        .map({"true": 1, "false": 0, "1": 1, "0": 0})
    )

    assert temp["correct"].notna().all()

    temp = temp.rename(
        columns={"correct": model}
    )

    if joined is None:

        joined = temp

    else:

        joined = joined.merge(
            temp[["sha256", model]],
            on="sha256",
            how="inner",
            validate="one_to_one"
        )

    print(f"Merged: {model}")

# ADD SIMILARITY GROUP IDENTIFIERS

joined = joined.merge(
    manifest[
        ["split_guard_group_id", "class_name"]
    ],
    left_on="sha256",
    right_index=True,
    how="inner",
    validate="one_to_one"
)

assert len(joined) == 2841

assert joined["sha256"].is_unique

assert (
    joined["true_class"] == joined["class_name"]
).all(), "Inconsistent class labels."

model_columns = sorted(predictions.keys())

assert joined[model_columns].notna().all().all()

joined = joined[
    ["sha256", "target", "true_class"]
    + model_columns
    + ["split_guard_group_id", "class_name"]
]

# VERIFY RESULTS

print("\n" + "=" * 60)
print("CONSOLIDATED FILE VERIFICATION")
print("=" * 60)

print("Images:", len(joined))
print("Models:", len(model_columns))

print(
    "Groups:",
    joined["split_guard_group_id"].nunique()
)

print("\nAccuracy by model:")

for model in model_columns:

    accuracy = joined[model].mean()

    print(f"{model}: {accuracy:.6f}")

group_classes = joined.groupby(
    "split_guard_group_id"
)["class_name"].nunique()

assert group_classes.max() == 1, (
    "Some groups contain different classes."
)

# SAVE RESULTS

output = Path(
    "/kaggle/working/R3_joined_predictions_groups.csv"
)

joined.to_csv(output, index=False)

print("\n" + "=" * 60)
print("FILE GENERATED SUCCESSFULLY")
print("=" * 60)

print("Path:", output)
print("Dimensions:", joined.shape)
print("\nFirst rows:")

display(joined.head())

print("\nREADY FOR R3 STATISTICAL ANALYSIS")


# **Cell 7 - Group-level paired statistical analysis**

In [ ]:

# 12 models | 2841 images | 66 comparisons
# 99999 Monte Carlo permutations
# 5000 class-stratified cluster bootstrap replicates
# Holm correction | Seed 42 | 95% CI

from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.stats import binomtest

# CONFIGURATION

INPUT = Path(
    "/kaggle/working/R3_joined_predictions_groups.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/R3_cluster_statistics"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_PERM = 99_999
N_BOOT = 5_000
ALPHA = 0.05

# HOLM CORRECTION


def holm_adjust(p):
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)

    adjusted_sorted = np.maximum.accumulate(
        (len(p) - np.arange(len(p))) * p[order]
    )

    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0.0, 1.0)

    return adjusted

# GROUP-LEVEL PAIRED PERMUTATION

def permutation_group_p(
    group_diffs, rng, n_perm=N_PERM, batch=2048
):
    """
    Two-sided cluster randomization test.

    Statistic: absolute difference in the total
    number of correct predictions between the two models.

    Each permutation swaps the results
    of both models across entire groups.

    Assumption: exchangeability of model labels
    at the cluster level under H0.
    """

    d = np.asarray(group_diffs, dtype=np.int32)
    d = d[d != 0]

    if d.size == 0:
        return 1.0

    observed = abs(int(d.sum()))
    extreme = 0
    remaining = n_perm

    while remaining > 0:
        size = min(batch, remaining)

        signs = rng.integers(
            0, 2, size=(size, len(d)), dtype=np.int8
        )

        simulated = (
            2 * signs.astype(np.int32) - 1
        ) @ d

        extreme += int(
            np.count_nonzero(
                np.abs(simulated) >= observed
            )
        )

        remaining -= size

    return (extreme + 1) / (n_perm + 1)

# LOAD AND VERIFY PREDICTIONS

df = pd.read_csv(INPUT)

required = {
    "sha256",
    "split_guard_group_id",
    "class_name"
}

assert required.issubset(df.columns), (
    f"Missing columns: {required - set(df.columns)}"
)

model_names = sorted(
    c for c in df.columns
    if "__" in c and c != "true_class"
)

assert len(model_names) == 12, (
    f"Expected 12 models: {model_names}"
)

assert len(df) == 2841
assert df["sha256"].is_unique
assert df["split_guard_group_id"].notna().all()

X = df[model_names].to_numpy(dtype=np.int8)

assert np.isin(X, [0, 1]).all(), (
    "Predictions must be binary: "
    "1=correct, 0=incorrect."
)

# RECONSTRUCT SIMILARITY GROUPS

codes, group_ids = pd.factorize(
    df["split_guard_group_id"], sort=True
)

G = len(group_ids)

class_per_group = (
    df.groupby("split_guard_group_id")["class_name"]
    .nunique()
)

assert class_per_group.max() == 1, (
    "Groups containing different classes were found."
)

group_sizes = np.bincount(
    codes, minlength=G
).astype(np.int32)

group_correct = np.empty(
    (G, len(model_names)), dtype=np.int32
)

for j in range(len(model_names)):
    group_correct[:, j] = np.bincount(
        codes,
        weights=X[:, j],
        minlength=G
    ).astype(np.int32)

group_classes = (
    df.groupby("split_guard_group_id")["class_name"]
    .first()
    .reindex(group_ids)
    .to_numpy()
)

strata = [
    np.flatnonzero(group_classes == c)
    for c in sorted(np.unique(group_classes))
]

assert sum(len(idx) for idx in strata) == G

print("=" * 65)
print("TEST SET STRUCTURE")
print("=" * 65)
print("Images:", len(df))
print("Groups:", G)
print("Groups with multiple images:",
      int((group_sizes > 1).sum()))
print("Images in multi-image groups:",
      int(group_sizes[group_sizes > 1].sum()))
print("Models:", len(model_names))

# PERFORM THE 66 COMPARISONS

rng_perm = np.random.default_rng(SEED)

pairs = list(
    combinations(range(len(model_names)), 2)
)

records = []

for k, (i, j) in enumerate(pairs, start=1):

    a = X[:, i]
    b = X[:, j]

    # McNemar discordances
    n_ab = int(
        np.sum((a == 1) & (b == 0))
    )

    n_ba = int(
        np.sum((a == 0) & (b == 1))
    )

    # Differences aggregated by group
    group_diffs = (
        group_correct[:, i] - group_correct[:, j]
    )

    p_perm = permutation_group_p(
        group_diffs, rng_perm
    )

    if n_ab + n_ba > 0:
        p_mcnemar = float(
            binomtest(
                n_ab, n_ab + n_ba,
                p=0.5,
                alternative="two-sided"
            ).pvalue
        )
    else:
        p_mcnemar = 1.0

    delta_acc = (
        100 * (int(a.sum()) - int(b.sum())) / len(df)
    )

    records.append({
        "model_a": model_names[i],
        "model_b": model_names[j],
        "b": n_ab,
        "c": n_ba,
        "delta_acc_pp": delta_acc,
        "n_discordant_groups": int(
            np.count_nonzero(group_diffs)
        ),
        "p_mcnemar": p_mcnemar,
        "p_cluster_permutation": p_perm
    })

    if k % 10 == 0 or k == len(pairs):
        print(
            f"Completed permutations: "
            f"{k}/{len(pairs)}"
        )

res = pd.DataFrame(records)

# HOLM CORRECTION FOR THE 66 COMPARISONS


res["p_mcnemar_holm"] = holm_adjust(
    res["p_mcnemar"].to_numpy()
)

res["p_cluster_holm"] = holm_adjust(
    res["p_cluster_permutation"].to_numpy()
)

res["mcnemar_significant"] = (
    res["p_mcnemar_holm"] < ALPHA
)

res["cluster_significant"] = (
    res["p_cluster_holm"] < ALPHA
)

# CLASS-STRATIFIED PAIRED BOOTSTRAP

rng_boot = np.random.default_rng(SEED)

boot_deltas = np.empty(
    (N_BOOT, len(pairs)),
    dtype=np.float64
)

i_cols = np.array(
    [i for i, _ in pairs], dtype=int
)

j_cols = np.array(
    [j for _, j in pairs], dtype=int
)

for r in range(N_BOOT):

    # Resampling entire clusters,
    # with replacement within each class
    selected = np.concatenate([
        rng_boot.choice(
            idx, size=len(idx), replace=True
        )
        for idx in strata
    ])

    # The same resample is preserved
    # for all 12 models
    totals = group_correct[selected].sum(axis=0)

    # The number of images may vary
    # because groups have different sizes
    n_images = group_sizes[selected].sum()

    boot_deltas[r] = (
        100.0 *
        (totals[i_cols] - totals[j_cols]) /
        n_images
    )

    if (r + 1) % 1000 == 0:
        print(
            f"Completed bootstrap replicates: {r + 1}/{N_BOOT}"
        )

# 95% percentile confidence intervals
res["ci95_low_pp"] = np.quantile(
    boot_deltas, 0.025, axis=0
)

res["ci95_high_pp"] = np.quantile(
    boot_deltas, 0.975, axis=0
)

# SAVE THE 66 COMPARISONS

full_path = (
    OUTPUT_DIR / "R3_resultados_66_comparaciones.csv"
)

res.to_csv(full_path, index=False)

# EXTRACT SWIN-TINY + CUTMIX COMPARISONS

reference = "swin_tiny__cutmix"

swin = res.loc[
    (res.model_a == reference) |
    (res.model_b == reference)
].copy()

flip = swin.model_b == reference

swin["competitor"] = np.where(
    flip, swin.model_a, swin.model_b
)

old_lo = swin["ci95_low_pp"].copy()
old_hi = swin["ci95_high_pp"].copy()
old_b = swin["b"].copy()

# Orient differences in favor of Swin-Tiny + CutMix
swin.loc[flip, "delta_acc_pp"] *= -1

swin.loc[flip, "ci95_low_pp"] = -old_hi[flip]
swin.loc[flip, "ci95_high_pp"] = -old_lo[flip]

swin.loc[flip, "b"] = swin.loc[flip, "c"]
swin.loc[flip, "c"] = old_b[flip]

cols = [
    "competitor",
    "b",
    "c",
    "delta_acc_pp",
    "ci95_low_pp",
    "ci95_high_pp",
    "p_mcnemar_holm",
    "p_cluster_holm",
    "cluster_significant",
    "n_discordant_groups"
]

swin_output = swin[cols].sort_values("competitor")

swin_path = (
    OUTPUT_DIR / "R3_resultados_swin.csv"
)

swin_output.to_csv(swin_path, index=False)

# FINAL SUMMARY

print("\n" + "=" * 65)
print("FINAL RESULTS - REVIEWER 7, R3")
print("=" * 65)

n_mcnemar = int(
    res["mcnemar_significant"].sum()
)

n_cluster = int(
    res["cluster_significant"].sum()
)

n_changes = int(
    (
        res["mcnemar_significant"] !=
        res["cluster_significant"]
    ).sum()
)

print(f"McNemar + Holm: {n_mcnemar}/66")
print(f"Cluster permutation + Holm: {n_cluster}/66")
print(f"Different significance decisions: {n_changes}")

print("\nSWIN-TINY + CUTMIX COMPARISONS")
print(swin_output.to_string(index=False))

print("\nGENERATED FILES")
print(full_path)
print(swin_path)

print("\nANALYSIS COMPLETED")

# **Cell 8 - Analysis of dual class balancing**

In [ ]:

# Reconstructs the original class weights
# Compares four theoretical balancing schemes
# Does NOT perform training

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# CONFIGURATION

BETA = 0.9999
N_CLASSES = 24

OUTPUT_DIR = Path("/kaggle/working/R4_class_balance")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Search for the original manifest
candidates = []

for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    if root.exists():
        candidates.extend(
            root.rglob("*12_clean_grouped_split_manifest*.csv")
        )

assert candidates, (
    "The manifest was not found. "
    "Add it to Kaggle before running."
)

MANIFEST = candidates[0]
print("Manifest:", MANIFEST)

# RECOVER TRAINING DISTRIBUTION

df = pd.read_csv(MANIFEST)

required = {"split", "class_name", "sha256"}
assert required.issubset(df.columns)

train = df.loc[
    df["split"].astype(str).str.lower() == "train"
].copy()

assert len(train) == 22672, (
    f"Unexpected number of images: {len(train)}"
)

assert train["sha256"].is_unique

counts = train["class_name"].value_counts().sort_index()

assert len(counts) == N_CLASSES
assert (counts > 0).all()

classes = counts.index.to_numpy()
n = counts.to_numpy(dtype=np.float64)

N = int(n.sum())
K = len(n)

#  RECONSTRUCT CLASS-BALANCED LOSS

effective_number = 1.0 - np.power(BETA, n)

cb_weights = (
    (1.0 - BETA) /
    np.maximum(effective_number, 1e-12)
)

# Original normalization: mean = 1
cb_weights = cb_weights / cb_weights.mean()

#  RECONSTRUCT WEIGHTED RANDOM SAMPLER

inverse_frequency = 1.0 / n

p_original = n / N
p_sampler = np.full(K, 1.0 / K)

expected_original = N * p_original
expected_sampler = N * p_sampler

#  RELATIVE INFLUENCE OF FOUR BALANCING SCHEMES


def normalized_mass(p, weights):
    mass = p * weights
    return mass / mass.sum()

mass_none = normalized_mass(
    p_original, np.ones(K)
)

mass_cb_only = normalized_mass(
    p_original, cb_weights
)

mass_sampler_only = normalized_mass(
    p_sampler, np.ones(K)
)

mass_both = normalized_mass(
    p_sampler, cb_weights
)

# Ratio relative to a uniform distribution
uniform = np.full(K, 1.0 / K)

relative_both_uniform = mass_both / uniform

# Ratio between extreme classes
minority_idx = np.argmin(n)
majority_idx = np.argmax(n)

# PER-CLASS RESULTS TABLE

results = pd.DataFrame({
    "class_name": classes,
    "train_count": n.astype(int),
    "original_frequency_pct": 100 * p_original,
    "inverse_frequency_weight": inverse_frequency,
    "class_balanced_weight": cb_weights,
    "expected_samples_sampler": expected_sampler,
    "expected_mass_none_pct": 100 * mass_none,
    "expected_mass_cb_only_pct": 100 * mass_cb_only,
    "expected_mass_sampler_only_pct": 100 * mass_sampler_only,
    "expected_mass_both_pct": 100 * mass_both,
    "both_vs_uniform_ratio": relative_both_uniform
})

results = results.sort_values(
    "train_count"
).reset_index(drop=True)

#  QUANTITATIVE SUMMARY

summary = pd.DataFrame([
    {
        "indicator": "Training images",
        "value": N
    },
    {
        "indicator": "Number of classes",
        "value": K
    },
    {
        "indicator": "Minority class size",
        "value": int(n.min())
    },
    {
        "indicator": "Majority class size",
        "value": int(n.max())
    },
    {
        "indicator": "Original imbalance ratio",
        "value": float(n.max() / n.min())
    },
    {
        "indicator": "Minimum CB weight",
        "value": float(cb_weights.min())
    },
    {
        "indicator": "Maximum CB weight",
        "value": float(cb_weights.max())
    },
    {
        "indicator": "CB max/min weight ratio",
        "value": float(
            cb_weights.max() / cb_weights.min()
        )
    },
    {
        "indicator": "Combined maximum/minimum "
                     "expected class influence ratio",
        "value": float(
            mass_both.max() / mass_both.min()
        )
    }
])

# SAVE RESULTS

results_path = OUTPUT_DIR / "R4_class_balance_by_class.csv"
summary_path = OUTPUT_DIR / "R4_class_balance_summary.csv"

results.to_csv(results_path, index=False)
summary.to_csv(summary_path, index=False)

# VISUALIZATION

plot_data = results.set_index("class_name")

ax = plot_data[
    [
        "expected_mass_none_pct",
        "expected_mass_cb_only_pct",
        "expected_mass_sampler_only_pct",
        "expected_mass_both_pct"
    ]
].plot(
    kind="bar",
    figsize=(17, 6),
    width=0.8
)

ax.set_title(
    "R4 - Expected relative class influence "
    "under four balancing schemes"
)

ax.set_xlabel("Class")
ax.set_ylabel("Normalized expected contribution (%)")

ax.legend([
    "No balancing",
    "Class-Balanced Loss only",
    "WeightedRandomSampler only",
    "Combined"
])

plt.xticks(rotation=75, ha="right")
plt.tight_layout()

figure_path = OUTPUT_DIR / "R4_class_balance_comparison.png"
plt.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()

#  DISPLAY RESULTS

print("\n" + "=" * 65)
print("R4 - CLASS-WEIGHTING ANALYSIS")
print("=" * 65)

print("\nSUMMARY:")
print(summary.to_string(index=False))

print("\nCLASSES SORTED BY FREQUENCY:")
display(results.round(5))

print("\nGENERATED FILES:")
print(results_path)
print(summary_path)
print(figure_path)

print("\nANALYSIS COMPLETED")
